# Telecom AI Knowledge Orchestration Runtime

## Adaptive RAG, MCP, Web Search & LLM Routing

**Notebook 21 — Module 4 Deployment Runtime**

This notebook packages the validated Module 4 research architecture into a CPU-first, portfolio-ready **knowledge orchestration runtime**. It combines Semantic RAG V1, MCP Version A remote knowledge access, hosted Gemma generation, adaptive retrieval, Granite knowledge-scope routing, live web grounding, deterministic local utilities, security controls, and evidence-relative comparative evaluation.

**Canonical final runtime:** **Cell 4D / 6Y v1.3**.

> **Naming note:** `MODULE 4C.6Y` and related `4C6*` identifiers are retained internally as validated legacy/runtime compatibility names. The public notebook topic is now **Telecom AI Knowledge Orchestration Runtime**.

**6Z knowledge-gap escalation was evaluated but intentionally not adopted.** Correctly phrased person/current-information questions already route naturally to the live-external path through Granite, so the additional post-generation escalation layer was not required.


#### Architecture at a Glance

```text
User Question
      ↓
Deterministic Security
      ↓
Granite Knowledge-Scope Router
      ├── Connected Technical Corpus
      │      ↓
      │   Gemma Retrieval Planner
      │      ├── RAG_ONLY
      │      ├── MCP_ONLY
      │      └── HYBRID
      │      ↓
      │   Adaptive Evidence Loop
      │      ↓
      │   Grounded Gemma Answer
      │      ↓
      │   Frozen Granite Comparative Judge
      │
      ├── Live External Scope
      │      ├── Direct date/time → Local Python Runtime
      │      └── Changing public facts → Open-WebSearch CLI
      │
      └── Stable General Knowledge
             ↓
          Gemma Fallback + Scope Caveat
```

The legacy route constant `TELECOM_GROUNDED` is retained for compatibility. Operationally, it means **the connected controlled technical corpus is the appropriate authoritative knowledge source** rather than simply “the question contains telecom words.”


#### Deployment Strategy

- **Knowledge orchestration:** Granite selects the appropriate knowledge scope before downstream execution.
- **Generation:** hosted API through OpenRouter; no local vLLM requirement.
- **Primary generator:** `google/gemma-4-26b-a4b-it`.
- **Router / comparative judge:** `ibm-granite/granite-4.2-8b`.
- **Semantic RAG:** full 1,506,367-vector BGE-M3 / FAISS `IndexFlatIP` corpus.
- **RAG metadata:** disk-backed and resolved on demand rather than reconstructed fully in RAM.
- **MCP:** Version A remote query-time retrieval over dedicated 3GPP and GSMA Telecom Common Corpus sources.
- **Adaptive retrieval:** Gemma selects `RAG_ONLY`, `MCP_ONLY`, or `HYBRID`, formulates the query, checks requirement-bounded sufficiency, and may refine up to three searches.
- **Live external grounding:** `open-websearch@2.1.11` CLI JSON interface, hard-capped at three external operations.
- **Date/time:** deterministic `datetime + zoneinfo` runtime path; web search is skipped.
- **Evaluation:** the frozen Granite evidence-relative judge remains limited to connected-corpus comparisons.


# SECTION 0 — Runtime Environment


#### Cell 0A — Demo Runtime Environment Setup

**Description:** Install only the packages required for the CPU-first deployment runtime; intentionally exclude local vLLM serving.


In [3]:
# ============================================================
# CELL 0A — DEMO RUNTIME ENVIRONMENT SETUP
# ============================================================
#
# Purpose
# -------
#
# Install only the packages required for the lightweight
# Module 4 deployment/demo runtime.
#
# Target architecture:
#
#     Full Semantic RAG V1
#         +
#     MCP Version A
#         +
#     Hosted LLM Generation API
#         +
#     CPU-First Runtime
#
#
# IMPORTANT:
#
# - Do NOT install vLLM.
# - Do NOT load or serve a generation LLM locally.
# - Do NOT install a separate TorchAudio build.
# - Do NOT broadly upgrade Colab's CUDA/PyTorch stack.
# - FAISS retrieval and BGE-M3 query encoding will be
#   validated on CPU.
# - MCP Version A will use remote telecom knowledge access.
#
# After this cell completes:
#
#     RESTART THE COLAB RUNTIME ONCE.
#
# Then continue with Cell 0B.
# ============================================================


# ============================================================
# 0A.1 — INSTALL DEMO RUNTIME DEPENDENCIES
# ============================================================

%pip install -q \
    openai \
    nest_asyncio \
    huggingface_hub \
    kagglehub \
    sentence-transformers==6.0.1 \
    faiss-cpu==1.15.0 \
    duckdb==1.5.5 \
    fastmcp==4.0.2 \
    psutil \
    tqdm


# ============================================================
# 0A.2 — REMOVE TORCHAUDIO
# ============================================================
#
# Transformers may automatically detect TorchAudio if it is
# installed in the Colab environment.
#
# A mismatched TorchAudio build previously caused the
# sentence-transformers import chain to fail.
#
# This deployment runtime is text-only, so TorchAudio is not
# required.
# ============================================================

!pip uninstall -y torchaudio -q


# ============================================================
# 0A.3 — INSTALL JEDI FOR COLAB / IPYTHON COMPATIBILITY
# ============================================================

%pip install -q jedi


# ============================================================
# 0A.4 — FINAL STATUS
# ============================================================

print("=" * 96)
print("MODULE 4 DEMO RUNTIME — ENVIRONMENT INSTALLATION COMPLETE")
print("=" * 96)

print("✓ Hosted LLM API client dependencies installed.")
print("✓ Semantic RAG dependencies installed.")
print("✓ FAISS CPU runtime installed.")
print("✓ BGE-M3 / sentence-transformers runtime installed.")
print("✓ DuckDB installed for MCP Version A remote retrieval.")
print("✓ FastMCP installed.")
print("✓ psutil installed for CPU/RAM monitoring.")
print("✓ TorchAudio removed.")
print("✓ vLLM intentionally NOT installed.")
print("✓ No local generation model runtime installed.")

print()
print("TARGET RUNTIME:")
print("  Generation : Hosted LLM API")
print("  RAG        : Full RAG V1 — CPU")
print("  MCP        : Version A — CPU + Remote Knowledge Access")
print("  GPU        : Not required")

print()
print("IMPORTANT:")
print("Restart the Colab runtime once before running Cell 0B.")

print("=" * 96)


MODULE 4 DEMO RUNTIME — ENVIRONMENT INSTALLATION COMPLETE
✓ Hosted LLM API client dependencies installed.
✓ Semantic RAG dependencies installed.
✓ FAISS CPU runtime installed.
✓ BGE-M3 / sentence-transformers runtime installed.
✓ DuckDB installed for MCP Version A remote retrieval.
✓ FastMCP installed.
✓ psutil installed for CPU/RAM monitoring.
✓ TorchAudio removed.
✓ vLLM intentionally NOT installed.
✓ No local generation model runtime installed.

TARGET RUNTIME:
  Generation : Hosted LLM API
  RAG        : Full RAG V1 — CPU
  MCP        : Version A — CPU + Remote Knowledge Access
  GPU        : Not required

IMPORTANT:
Restart the Colab runtime once before running Cell 0B.


##### Cell 0A — Observation

Environment setup completed successfully for the hosted-generation + RAG + MCP runtime. This cell remains installation-only; a one-time Colab runtime restart may be required after package installation.


#### Cell 0B — Secrets, Imports and Runtime Validation

**Description:** Load secrets from Colab userdata, construct the OpenRouter client, validate package versions, and capture baseline CPU/RAM/disk state.


In [4]:
# ============================================================
# CELL 0B — SECRETS + CORE RUNTIME VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Validate the lightweight Module 4 demo runtime before any
# RAG or MCP data is downloaded.
#
# Validate:
#
#     Google Colab
#     OPENROUTER_API_KEY
#     HF_TOKEN
#     KAGGLE_API_TOKEN
#     OpenRouter / OpenAI-compatible client
#     PyTorch
#     Sentence Transformers
#     FastMCP
#     DuckDB
#     FAISS CPU
#     psutil
#
# Also capture the baseline:
#
#     CPU availability
#     System RAM
#     Process RAM
#     Local disk usage
#
# GPU/CUDA availability is reported for information only.
# The demo runtime does NOT require a GPU.
#
# No datasets or models are downloaded in this cell.
# ============================================================


# ============================================================
# 0B.1 — STANDARD IMPORTS
# ============================================================

import os
import sys
import shutil
import platform
import warnings
import importlib.metadata

from pathlib import Path

from tqdm.auto import tqdm


# ============================================================
# 0B.2 — WARNING FILTERS
# ============================================================

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
)

warnings.filterwarnings(
    "ignore",
    category=ResourceWarning,
)


# ============================================================
# 0B.3 — PROGRESS BAR
# ============================================================

validation_steps = [
    "Read Colab secrets",
    "Import core libraries",
    "Validate Colab runtime",
    "Configure OpenRouter client",
    "Capture CPU / RAM / disk baseline",
    "Validate core demo stack",
]


progress = tqdm(
    total=len(validation_steps),
    desc="Module 4 Demo Runtime Validation",
    unit="step",
)


def advance_progress(label):

    progress.set_postfix_str(
        label
    )

    progress.update(1)


# ============================================================
# 0B.4 — LOAD COLAB SECRETS
# ============================================================

from google.colab import userdata


def read_required_secret(name):

    try:

        value = userdata.get(
            name
        )

    except Exception as exc:

        raise RuntimeError(
            f"Unable to read Colab secret '{name}'."
        ) from exc

    if not value:

        raise RuntimeError(
            f"{name} is missing or empty."
        )

    return value


OPENROUTER_API_KEY = read_required_secret(
    "OPENROUTER_API_KEY"
)

HF_TOKEN = read_required_secret(
    "HF_TOKEN"
)

KAGGLE_API_TOKEN = read_required_secret(
    "KAGGLE_API_TOKEN"
)


os.environ["OPENROUTER_API_KEY"] = (
    OPENROUTER_API_KEY
)

os.environ["HF_TOKEN"] = (
    HF_TOKEN
)

os.environ["HUGGING_FACE_HUB_TOKEN"] = (
    HF_TOKEN
)

os.environ["KAGGLE_API_TOKEN"] = (
    KAGGLE_API_TOKEN
)

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

os.environ["TOKENIZERS_PARALLELISM"] = "false"


print("✓ OPENROUTER_API_KEY loaded securely.")
print("✓ HF_TOKEN loaded securely.")
print("✓ KAGGLE_API_TOKEN loaded securely.")


advance_progress(
    "Secrets loaded"
)


# ============================================================
# 0B.5 — IMPORT CORE LIBRARIES
# ============================================================

import torch
import psutil
import fastmcp
import sentence_transformers
import duckdb
import faiss
import kagglehub

from openai import OpenAI


advance_progress(
    "Core libraries imported"
)


# ============================================================
# 0B.6 — COLAB VALIDATION
# ============================================================

IN_COLAB = (
    "google.colab"
    in
    sys.modules
)


if not IN_COLAB:

    raise RuntimeError(
        "This notebook is expected to run in Google Colab."
    )


advance_progress(
    "Colab runtime validated"
)


# ============================================================
# 0B.7 — OPENROUTER CLIENT CONFIGURATION
# ============================================================
#
# OpenRouter exposes an OpenAI-compatible API.
#
# Instantiating the client does NOT generate tokens and does
# not incur an LLM inference charge.
# ============================================================

OPENROUTER_BASE_URL = (
    "https://openrouter.ai/api/v1"
)

GENERATION_PROVIDER = (
    "OpenRouter"
)

GENERATION_RUNTIME = (
    "Hosted API"
)

MODULE4_DEMO_DEVICE = (
    "cpu"
)


openrouter_client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    timeout=300.0,
)


if openrouter_client is None:

    raise RuntimeError(
        "OpenRouter client initialization failed."
    )


print()
print("✓ OpenRouter client configured.")
print(f"  Provider : {GENERATION_PROVIDER}")
print(f"  Endpoint : {OPENROUTER_BASE_URL}")
print("  Model    : Configurable — not frozen yet")


advance_progress(
    "OpenRouter configured"
)


# ============================================================
# 0B.8 — CPU + OPTIONAL GPU VISIBILITY
# ============================================================
#
# GPU presence is informational only.
#
# All retrieval components in this notebook will explicitly
# target CPU unless changed in a later controlled experiment.
# ============================================================

CPU_LOGICAL_COUNT = (
    psutil.cpu_count(
        logical=True
    )
)

CPU_PHYSICAL_COUNT = (
    psutil.cpu_count(
        logical=False
    )
)


CUDA_AVAILABLE = (
    torch.cuda.is_available()
)


if CUDA_AVAILABLE:

    GPU_COUNT = (
        torch.cuda.device_count()
    )

    GPU_NAME = (
        torch.cuda.get_device_name(
            0
        )
    )

else:

    GPU_COUNT = 0
    GPU_NAME = "None"


# ============================================================
# 0B.9 — BASELINE RAM MEASUREMENT
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


system_memory = (
    psutil.virtual_memory()
)


SYSTEM_RAM_TOTAL_GB = (
    system_memory.total
    /
    BYTES_PER_GIB
)

SYSTEM_RAM_AVAILABLE_GB = (
    system_memory.available
    /
    BYTES_PER_GIB
)


current_process = (
    psutil.Process(
        os.getpid()
    )
)


PROCESS_RSS_GB = (
    current_process
    .memory_info()
    .rss
    /
    BYTES_PER_GIB
)


# ============================================================
# 0B.10 — BASELINE DISK MEASUREMENT
# ============================================================

COLAB_CONTENT_ROOT = (
    Path("/content")
)


disk_usage = (
    shutil.disk_usage(
        COLAB_CONTENT_ROOT
    )
)


DISK_TOTAL_GB = (
    disk_usage.total
    /
    BYTES_PER_GIB
)

DISK_USED_GB = (
    disk_usage.used
    /
    BYTES_PER_GIB
)

DISK_FREE_GB = (
    disk_usage.free
    /
    BYTES_PER_GIB
)


advance_progress(
    "Resource baseline captured"
)


# ============================================================
# 0B.11 — PACKAGE VERSION HELPER
# ============================================================

def package_version(name):

    try:

        return importlib.metadata.version(
            name
        )

    except Exception:

        return "not installed"


# ============================================================
# 0B.12 — CORE DEMO STACK VALIDATION
# ============================================================

print("\nCORE MODULE 4 DEMO IMPORT CHECK")
print("-" * 96)

print(
    f"OpenAI Client            : ✓ "
    f"{package_version('openai')}"
)

print(
    f"FastMCP                  : ✓ "
    f"{package_version('fastmcp')}"
)

print(
    f"Sentence Transformers    : ✓ "
    f"{sentence_transformers.__version__}"
)

print(
    f"DuckDB                   : ✓ "
    f"{duckdb.__version__}"
)

print(
    f"FAISS CPU                : ✓ "
    f"{package_version('faiss-cpu')}"
)

print(
    f"PyTorch                  : ✓ "
    f"{torch.__version__}"
)

print(
    f"psutil                   : ✓ "
    f"{psutil.__version__}"
)

print(
    f"KaggleHub                : ✓ "
    f"{package_version('kagglehub')}"
)

print("-" * 96)

print(
    "✓ Core Module 4 demo software stack imports successfully."
)


advance_progress(
    "Core demo stack validated"
)


progress.close()


# ============================================================
# 0B.13 — RUNTIME RESOURCE SUMMARY
# ============================================================

print("\n" + "=" * 116)

print(
    "MODULE 4 DEMO RUNTIME — CPU + API VALIDATION"
)

print("=" * 116)


print(
    f"Python                     : "
    f"{platform.python_version()}"
)

print(
    f"Platform                   : "
    f"{platform.platform()}"
)

print(
    f"PyTorch                    : "
    f"{torch.__version__}"
)

print()
print("EXECUTION POLICY")
print("-" * 116)

print(
    f"Target Device               : "
    f"{MODULE4_DEMO_DEVICE.upper()}"
)

print(
    f"Generation Provider         : "
    f"{GENERATION_PROVIDER}"
)

print(
    f"Generation Runtime          : "
    f"{GENERATION_RUNTIME}"
)

print(
    f"Generation Model            : "
    f"Configurable"
)

print(
    f"GPU Required                : "
    f"False"
)

print(
    f"CUDA Visible                : "
    f"{CUDA_AVAILABLE}"
)

print(
    f"Detected GPU                : "
    f"{GPU_NAME}"
)

print()
print("CPU")
print("-" * 116)

print(
    f"Physical CPU Cores          : "
    f"{CPU_PHYSICAL_COUNT}"
)

print(
    f"Logical CPU Cores           : "
    f"{CPU_LOGICAL_COUNT}"
)

print()
print("BASELINE MEMORY — BEFORE RAG LOAD")
print("-" * 116)

print(
    f"System RAM Total            : "
    f"{SYSTEM_RAM_TOTAL_GB:.2f} GiB"
)

print(
    f"System RAM Available        : "
    f"{SYSTEM_RAM_AVAILABLE_GB:.2f} GiB"
)

print(
    f"Current Process RSS         : "
    f"{PROCESS_RSS_GB:.2f} GiB"
)

print()
print("BASELINE DISK — BEFORE RAG DOWNLOAD")
print("-" * 116)

print(
    f"Disk Total                  : "
    f"{DISK_TOTAL_GB:.2f} GiB"
)

print(
    f"Disk Used                   : "
    f"{DISK_USED_GB:.2f} GiB"
)

print(
    f"Disk Free                   : "
    f"{DISK_FREE_GB:.2f} GiB"
)

print()
print("AUTHENTICATION")
print("-" * 116)

print(
    f"OpenRouter                  : "
    f"Configured"
)

print(
    f"Hugging Face                : "
    f"Configured"
)

print(
    f"Kaggle                      : "
    f"Configured"
)


# ============================================================
# 0B.14 — FINAL VALIDATION
# ============================================================

MODULE4_CELL0B_PASS = all(
    [
        IN_COLAB,
        bool(OPENROUTER_API_KEY),
        bool(HF_TOKEN),
        bool(KAGGLE_API_TOKEN),
        MODULE4_DEMO_DEVICE == "cpu",
        openrouter_client is not None,
    ]
)


if not MODULE4_CELL0B_PASS:

    raise RuntimeError(
        "Cell 0B demo runtime validation failed."
    )


print()
print("✓ CELL 0B PASSED")
print("✓ Colab runtime healthy.")
print("✓ CPU-first execution policy confirmed.")
print("✓ OpenRouter generation client configured.")
print("✓ Core RAG and MCP dependencies validated.")
print("✓ Baseline RAM and disk measurements captured.")
print("✓ GPU is not required for the demo runtime.")
print("✓ Ready for Cell 0C — RAG artifact discovery and disk validation.")

print("=" * 116)


Module 4 Demo Runtime Validation:   0%|          | 0/6 [00:00<?, ?step/s]

✓ OPENROUTER_API_KEY loaded securely.
✓ HF_TOKEN loaded securely.
✓ KAGGLE_API_TOKEN loaded securely.

✓ OpenRouter client configured.
  Provider : OpenRouter
  Endpoint : https://openrouter.ai/api/v1
  Model    : Configurable — not frozen yet

CORE MODULE 4 DEMO IMPORT CHECK
------------------------------------------------------------------------------------------------
OpenAI Client            : ✓ 2.54.0
FastMCP                  : ✓ 4.0.2
Sentence Transformers    : ✓ 6.0.1
DuckDB                   : ✓ 1.5.5
FAISS CPU                : ✓ 1.15.0
PyTorch                  : ✓ 2.11.0+cpu
psutil                   : ✓ 5.9.5
KaggleHub                : ✓ 1.0.2
------------------------------------------------------------------------------------------------
✓ Core Module 4 demo software stack imports successfully.

MODULE 4 DEMO RUNTIME — CPU + API VALIDATION
Python                     : 3.13.15
Platform                   : Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch                    : 2.11.0

##### Cell 0B — Observation

Runtime validation passed with the required OpenAI, FastMCP, Sentence Transformers, DuckDB, FAISS CPU and KaggleHub stack. The deployment path remains CPU-capable; no local GPU is required for hosted Gemma/Granite inference.


#### Cell 0C — Kaggle RAG Assets + Module 4 Configuration

**Description:** Download and validate the two public Kaggle runtime datasets while keeping the 69 GB Version B knowledge base excluded.


In [5]:
# ============================================================
# CELL 0C — RAG DATA SOURCES + DEMO RUNTIME CONFIGURATION
# ============================================================
#
# Purpose
# -------
#
# Download and validate only the persistent artifacts required
# to operate the FULL Semantic RAG V1 deployment runtime:
#
#     1. Reconciled telecom chunks / metadata
#     2. FAISS semantic vector index
#
# The following original Module 4 artifacts are intentionally
# NOT downloaded:
#
#     BGE-M3 precomputed embeddings
#         -> not required for live query-time retrieval because
#            the indexed vectors already exist inside FAISS.
#
#     Version B BM25 knowledge base
#         -> not required because this demo uses MCP Version A
#            remote knowledge retrieval.
#
# This cell also:
#
#     - measures exact local RAG artifact disk usage;
#     - records disk usage before and after download;
#     - defines the CPU-first RAG/MCP runtime policy;
#     - defines the provider-independent hosted generation policy.
#
# IMPORTANT:
#
# KaggleHub internal download output is suppressed so that
# Colab displays only one controlled progress bar instead of
# generating large numbers of line-by-line progress messages.
#
# No FAISS index, metadata corpus, or BGE-M3 model is loaded
# into RAM in this cell.
# ============================================================


# ============================================================
# 0C.1 — IMPORTS
# ============================================================

import io
import os
import shutil
import contextlib

from pathlib import Path

import kagglehub

from tqdm.notebook import tqdm


# ============================================================
# 0C.2 — VERIFY CELL 0B
# ============================================================

if (
    "MODULE4_CELL0B_PASS"
    not in globals()
    or
    not MODULE4_CELL0B_PASS
):

    raise RuntimeError(
        "Cell 0B must pass before Cell 0C."
    )


# ============================================================
# 0C.3 — CONSTANTS + DISK-SIZE HELPERS
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def gib(byte_count):

    return (
        byte_count
        /
        BYTES_PER_GIB
    )


def directory_size_bytes(root_path):

    root_path = Path(
        root_path
    )

    total_bytes = 0

    for file_path in root_path.rglob("*"):

        try:

            if file_path.is_file():

                total_bytes += (
                    file_path.stat().st_size
                )

        except OSError:

            pass

    return total_bytes


# ============================================================
# 0C.4 — QUIET KAGGLE DATASET DOWNLOAD HELPER
# ============================================================
#
# KaggleHub may emit its own progress output.
#
# In Colab this can sometimes appear as many individual lines
# instead of one updating progress indicator.
#
# Suppress KaggleHub stdout/stderr and let this notebook manage
# the visible progress dashboard itself.
# ============================================================

def quiet_dataset_download(dataset_handle):

    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()

    try:

        with (
            contextlib.redirect_stdout(
                captured_stdout
            ),
            contextlib.redirect_stderr(
                captured_stderr
            ),
        ):

            dataset_path = (
                kagglehub.dataset_download(
                    dataset_handle
                )
            )

        return dataset_path

    except Exception as exc:

        raise RuntimeError(
            f"Dataset download failed: "
            f"{dataset_handle}"
        ) from exc


# ============================================================
# 0C.5 — CAPTURE PRE-DOWNLOAD DISK STATE
# ============================================================

disk_before = (
    shutil.disk_usage(
        "/content"
    )
)


DISK_BEFORE_USED_GIB = (
    gib(
        disk_before.used
    )
)


DISK_BEFORE_FREE_GIB = (
    gib(
        disk_before.free
    )
)


print("=" * 116)
print("MODULE 4 DEMO — FULL RAG ARTIFACT DOWNLOAD")
print("=" * 116)

print(
    f"Disk Used Before Download : "
    f"{DISK_BEFORE_USED_GIB:.2f} GiB"
)

print(
    f"Disk Free Before Download : "
    f"{DISK_BEFORE_FREE_GIB:.2f} GiB"
)

print()


# ============================================================
# 0C.6 — DEFINE REQUIRED RAG DATASETS
# ============================================================

RAG_DATASETS = [

    (
        "Reconciled telecom chunks",
        "cliffordimaguezegie/"
        "telecom-reconciled-chunks",
    ),

    (
        "BGE-M3 FAISS index",
        "cliffordimaguezegie/"
        "telecom-bge-m3-faiss",
    ),

]


# ============================================================
# 0C.7 — CONTROLLED RAG ARTIFACT DOWNLOAD
# ============================================================
#
# One notebook progress bar is used for all downloads.
#
# KaggleHub's internal output remains hidden.
# ============================================================

downloaded_paths = {}


dataset_progress = tqdm(
    total=len(RAG_DATASETS),
    desc="Full RAG Artifact Import",
    unit="dataset",
    leave=True,
)


for label, dataset_handle in RAG_DATASETS:

    dataset_progress.set_postfix_str(
        f"Loading: {label}"
    )

    downloaded_paths[
        label
    ] = quiet_dataset_download(
        dataset_handle
    )

    dataset_progress.update(
        1
    )


dataset_progress.set_postfix_str(
    "Complete"
)

dataset_progress.close()


print()
print("✓ Required RAG artifacts available.")


# ============================================================
# 0C.8 — NORMALIZED RAG PATHS
# ============================================================

MODULE4_RECONCILED_CHUNKS_ROOT = Path(
    downloaded_paths[
        "Reconciled telecom chunks"
    ]
)


MODULE4_BGE_M3_FAISS_ROOT = Path(
    downloaded_paths[
        "BGE-M3 FAISS index"
    ]
)


MODULE4_RAG_DATASET_PATHS = {

    "reconciled_chunks":
        MODULE4_RECONCILED_CHUNKS_ROOT,

    "bge_m3_faiss":
        MODULE4_BGE_M3_FAISS_ROOT,

}


# ============================================================
# 0C.9 — VALIDATE RAG PATHS
# ============================================================

print("\nRAG DATASET PATH VALIDATION")
print("-" * 116)


for name, path in MODULE4_RAG_DATASET_PATHS.items():

    exists = (
        path.exists()
    )

    print(
        f"{name:<28}: "
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if not exists:

        raise FileNotFoundError(
            f"Dataset path missing:\n"
            f"{name} -> {path}"
        )


# ============================================================
# 0C.10 — MEASURE EXACT RAG ARTIFACT SIZES
# ============================================================

print()
print("Measuring RAG artifact disk footprint...")


RAG_DATASET_SIZE_BYTES = {}


size_progress = tqdm(
    total=len(
        MODULE4_RAG_DATASET_PATHS
    ),
    desc="Artifact Size Validation",
    unit="dataset",
    leave=True,
)


for name, path in MODULE4_RAG_DATASET_PATHS.items():

    size_progress.set_postfix_str(
        name
    )

    size_bytes = (
        directory_size_bytes(
            path
        )
    )

    RAG_DATASET_SIZE_BYTES[
        name
    ] = size_bytes

    size_progress.update(
        1
    )


size_progress.set_postfix_str(
    "Complete"
)

size_progress.close()


RECONCILED_CHUNKS_SIZE_GIB = (
    gib(
        RAG_DATASET_SIZE_BYTES[
            "reconciled_chunks"
        ]
    )
)


FAISS_DATASET_SIZE_GIB = (
    gib(
        RAG_DATASET_SIZE_BYTES[
            "bge_m3_faiss"
        ]
    )
)


FULL_RAG_ARTIFACT_SIZE_GIB = (
    RECONCILED_CHUNKS_SIZE_GIB
    +
    FAISS_DATASET_SIZE_GIB
)


# ============================================================
# 0C.11 — RAG ARTIFACT DISK FOOTPRINT
# ============================================================

print("\nFULL RAG ARTIFACT DISK FOOTPRINT")
print("-" * 116)


print(
    f"Reconciled Chunks          : "
    f"{RECONCILED_CHUNKS_SIZE_GIB:.3f} GiB"
)


print(
    f"FAISS Dataset              : "
    f"{FAISS_DATASET_SIZE_GIB:.3f} GiB"
)


print("-" * 116)


print(
    f"Total Required RAG Assets  : "
    f"{FULL_RAG_ARTIFACT_SIZE_GIB:.3f} GiB"
)


# ============================================================
# 0C.12 — CAPTURE POST-DOWNLOAD DISK STATE
# ============================================================

disk_after = (
    shutil.disk_usage(
        "/content"
    )
)


DISK_AFTER_USED_GIB = (
    gib(
        disk_after.used
    )
)


DISK_AFTER_FREE_GIB = (
    gib(
        disk_after.free
    )
)


DISK_DOWNLOAD_DELTA_GIB = (
    DISK_AFTER_USED_GIB
    -
    DISK_BEFORE_USED_GIB
)


print("\nCOLAB DISK STATE AFTER DOWNLOAD")
print("-" * 116)


print(
    f"Disk Used                  : "
    f"{DISK_AFTER_USED_GIB:.2f} GiB"
)


print(
    f"Disk Free                  : "
    f"{DISK_AFTER_FREE_GIB:.2f} GiB"
)


print(
    f"Observed Download Delta    : "
    f"{DISK_DOWNLOAD_DELTA_GIB:.2f} GiB"
)


# ============================================================
# 0C.13 — FULL RAG CONFIGURATION
# ============================================================

MODULE4_RAG_LABEL = (
    "Full Semantic RAG V1"
)


MODULE4_RAG_EMBEDDING_MODEL = (
    "BAAI/bge-m3"
)


MODULE4_RAG_VECTOR_ENGINE = (
    "FAISS"
)


MODULE4_RAG_EXPECTED_VECTORS = (
    1_506_367
)


MODULE4_RAG_EXPECTED_DIMENSION = (
    1024
)


MODULE4_RAG_EXPECTED_METADATA_SHARDS = (
    151
)


MODULE4_RAG_DEVICE = (
    "cpu"
)


MODULE4_RAG_TOP_K = (
    5
)


# ============================================================
# 0C.14 — MCP VERSION A CONFIGURATION
# ============================================================

MODULE4_MCP_VERSION = (
    "Version A"
)


MODULE4_MCP_RUNTIME = (
    "Remote query-time retrieval"
)


MODULE4_MCP_DEVICE = (
    "cpu"
)


MODULE4_MCP_TOOL = (
    "search_telecom_knowledge"
)


MODULE4_MCP_LOCAL_PERSISTENT_KB = (
    False
)


# ============================================================
# 0C.15 — HOSTED GENERATION CONFIGURATION
# ============================================================
#
# Generation model selection remains independent from the
# retrieval architecture.
#
# The exact OpenRouter model will be selected later.
# ============================================================

MODULE4_GENERATION_PROVIDER = (
    "OpenRouter"
)


MODULE4_GENERATION_RUNTIME = (
    "Hosted API"
)


MODULE4_GENERATION_BASE_URL = (
    OPENROUTER_BASE_URL
)


MODULE4_GENERATION_MODEL = (
    "CONFIGURABLE"
)


MODULE4_GENERATION_DEVICE = (
    "REMOTE"
)


MODULE4_GENERATION_SEED = (
    42
)


# ============================================================
# 0C.16 — DEMO ARCHITECTURE CONFIGURATION
# ============================================================

MODULE4_DEMO_RUNTIME = (
    "CPU + Hosted LLM API"
)


MODULE4_DEMO_RETRIEVAL_MODES = (

    "NO_RETRIEVAL",
    "RAG_ONLY",
    "MCP_ONLY",
    "HYBRID",

)


# ============================================================
# 0C.17 — FORMAL SUMMARY
# ============================================================

print("\n" + "=" * 116)

print(
    "MODULE 4 DEMO — FORMAL RUNTIME CONFIGURATION"
)

print("=" * 116)


print("SEMANTIC RAG")
print("-" * 116)


print(
    f"Architecture                : "
    f"{MODULE4_RAG_LABEL}"
)


print(
    f"Embedding Model             : "
    f"{MODULE4_RAG_EMBEDDING_MODEL}"
)


print(
    f"Vector Engine               : "
    f"{MODULE4_RAG_VECTOR_ENGINE}"
)


print(
    f"Expected Vectors            : "
    f"{MODULE4_RAG_EXPECTED_VECTORS:,}"
)


print(
    f"Expected Dimension          : "
    f"{MODULE4_RAG_EXPECTED_DIMENSION}"
)


print(
    f"Expected Metadata Shards    : "
    f"{MODULE4_RAG_EXPECTED_METADATA_SHARDS}"
)


print(
    f"RAG Device                  : "
    f"{MODULE4_RAG_DEVICE.upper()}"
)


print(
    f"RAG Top-K                   : "
    f"{MODULE4_RAG_TOP_K}"
)


print(
    f"Local RAG Artifact Size     : "
    f"{FULL_RAG_ARTIFACT_SIZE_GIB:.3f} GiB"
)


print()
print("MCP KNOWLEDGE ACCESS")
print("-" * 116)


print(
    f"MCP Version                 : "
    f"{MODULE4_MCP_VERSION}"
)


print(
    f"MCP Runtime                 : "
    f"{MODULE4_MCP_RUNTIME}"
)


print(
    f"MCP Device                  : "
    f"{MODULE4_MCP_DEVICE.upper()}"
)


print(
    f"MCP Tool                    : "
    f"{MODULE4_MCP_TOOL}"
)


print(
    f"Persistent Local KB         : "
    f"{MODULE4_MCP_LOCAL_PERSISTENT_KB}"
)


print()
print("LLM GENERATION")
print("-" * 116)


print(
    f"Provider                    : "
    f"{MODULE4_GENERATION_PROVIDER}"
)


print(
    f"Runtime                     : "
    f"{MODULE4_GENERATION_RUNTIME}"
)


print(
    f"Model                       : "
    f"{MODULE4_GENERATION_MODEL}"
)


print(
    f"Generation Device           : "
    f"{MODULE4_GENERATION_DEVICE}"
)


print()
print("DEMO RUNTIME")
print("-" * 116)


print(
    f"Execution                   : "
    f"{MODULE4_DEMO_RUNTIME}"
)


print(
    f"Retrieval Modes             : "
    f"{', '.join(MODULE4_DEMO_RETRIEVAL_MODES)}"
)


print(
    f"Local Version B KB          : "
    f"NOT DOWNLOADED"
)


print(
    f"Standalone Embedding Store  : "
    f"NOT DOWNLOADED"
)


# ============================================================
# 0C.18 — FINAL VALIDATION
# ============================================================

MODULE4_CELL0C_PASS = all(

    path.exists()

    for path
    in MODULE4_RAG_DATASET_PATHS.values()

)


if not MODULE4_CELL0C_PASS:

    raise RuntimeError(
        "Cell 0C RAG dataset/configuration validation failed."
    )


print()
print("✓ CELL 0C PASSED")
print("✓ Full RAG V1 persistent artifacts downloaded.")
print("✓ Exact RAG disk footprint measured.")
print("✓ Precomputed embedding dataset intentionally excluded.")
print("✓ Version B 69 GB knowledge base intentionally excluded.")
print("✓ MCP Version A deployment policy defined.")
print("✓ Hosted OpenRouter generation policy defined.")
print("✓ CPU-first RAG configuration defined.")
print("✓ No RAG assets have been loaded into RAM yet.")
print("✓ Ready for Cell 1A — full FAISS + metadata artifact validation.")

print("=" * 116)


MODULE 4 DEMO — FULL RAG ARTIFACT DOWNLOAD
Disk Used Before Download : 20.61 GiB
Disk Free Before Download : 205.21 GiB



Full RAG Artifact Import:   0%|          | 0/2 [00:00<?, ?dataset/s]

Extracting files...
Extracting files...

✓ Required RAG artifacts available.

RAG DATASET PATH VALIDATION
--------------------------------------------------------------------------------------------------------------------
reconciled_chunks           : ✓ /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1
bge_m3_faiss                : ✓ /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1

Measuring RAG artifact disk footprint...


Artifact Size Validation:   0%|          | 0/2 [00:00<?, ?dataset/s]


FULL RAG ARTIFACT DISK FOOTPRINT
--------------------------------------------------------------------------------------------------------------------
Reconciled Chunks          : 3.791 GiB
FAISS Dataset              : 6.220 GiB
--------------------------------------------------------------------------------------------------------------------
Total Required RAG Assets  : 10.012 GiB

COLAB DISK STATE AFTER DOWNLOAD
--------------------------------------------------------------------------------------------------------------------
Disk Used                  : 30.62 GiB
Disk Free                  : 195.19 GiB
Observed Download Delta    : 10.01 GiB

MODULE 4 DEMO — FORMAL RUNTIME CONFIGURATION
SEMANTIC RAG
--------------------------------------------------------------------------------------------------------------------
Architecture                : Full Semantic RAG V1
Embedding Model             : BAAI/bge-m3
Vector Engine               : FAISS
Expected Vectors            : 1,506,367
E

##### Cell 0C — Observation

The local RAG assets were resolved successfully: approximately **3.791 GiB** of reconciled chunks plus **6.220 GiB** of FAISS assets. MCP remains Version A remote query-time retrieval, avoiding a second large local knowledge store.


# SECTION 1 — Semantic RAG V1 Restoration


#### Cell 1A — Full RAG Artifact Structure Validation

**Description:** Validate FAISS and metadata structure without loading the full metadata corpus into RAM.


In [6]:
# ============================================================
# CELL 1A — FULL RAG ARTIFACT STRUCTURE VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Validate the frozen Semantic RAG V1 artifacts WITHOUT
# reconstructing the complete metadata corpus in memory.
#
# Validate:
#
#     - Frozen FAISS index artifact exists
#     - Reconciled metadata shard structure
#     - Expected metadata shard count
#     - Total metadata row count
#     - Global FAISS-vector ↔ metadata-row ordering
#     - Chunk text availability
#
# Deployment principle:
#
#     DO NOT:
#         - load the full FAISS index yet
#         - concatenate 1.5M metadata rows into Pandas
#         - keep all metadata shards in RAM
#
#     DO:
#         - preserve deterministic shard ordering
#         - count rows using memory-efficient scans
#         - construct a lightweight shard manifest
#         - map global FAISS IDs to shard/local-row positions
#
# The full FAISS index will be loaded separately in Cell 1B,
# where its actual RAM impact can be measured independently.
# ============================================================


# ============================================================
# 1A.1 — IMPORTS
# ============================================================

import gc
import json
import time
import bisect

from pathlib import Path

import pandas as pd
import psutil
import duckdb

from tqdm.notebook import tqdm


# ============================================================
# 1A.2 — VERIFY CELL 0C
# ============================================================

if (
    "MODULE4_CELL0C_PASS"
    not in globals()
    or
    not MODULE4_CELL0C_PASS
):

    raise RuntimeError(
        "Cell 0C must pass before Cell 1A."
    )


required_vars = [

    "MODULE4_RECONCILED_CHUNKS_ROOT",
    "MODULE4_BGE_M3_FAISS_ROOT",
    "MODULE4_RAG_EXPECTED_VECTORS",
    "MODULE4_RAG_EXPECTED_DIMENSION",
    "MODULE4_RAG_EXPECTED_METADATA_SHARDS",

]


missing = [

    name

    for name
    in required_vars

    if name
    not in globals()

]


if missing:

    raise RuntimeError(
        "Required Cell 0C variables are missing:\n"
        f"{missing}"
    )


# ============================================================
# 1A.3 — FROZEN RAG V1 IDENTITY
# ============================================================

RAG_RETRIEVER_VERSION = (
    "V1"
)


RAG_MODEL_NAME = (
    "BAAI/bge-m3"
)


RAG_MAX_SEQ_LENGTH = (
    1280
)


RAG_HISTORICAL_K = (
    7
)


RAG_MODULE4_K = (
    5
)


EXPECTED_RAG_VECTORS = (
    MODULE4_RAG_EXPECTED_VECTORS
)


EXPECTED_RAG_DIMENSION = (
    MODULE4_RAG_EXPECTED_DIMENSION
)


EXPECTED_RAG_METADATA_SHARDS = (
    MODULE4_RAG_EXPECTED_METADATA_SHARDS
)


# ============================================================
# 1A.4 — NORMALIZED PATHS
# ============================================================

RAG_FAISS_ROOT = Path(
    MODULE4_BGE_M3_FAISS_ROOT
)


RAG_CHUNKS_ROOT = Path(
    MODULE4_RECONCILED_CHUNKS_ROOT
)


for path in [

    RAG_FAISS_ROOT,
    RAG_CHUNKS_ROOT,

]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required RAG path does not exist:\n"
            f"{path}"
        )


# ============================================================
# 1A.5 — RESOURCE BASELINE
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def bytes_to_gib(value):

    return (
        value
        /
        BYTES_PER_GIB
    )


current_process = (
    psutil.Process()
)


PROCESS_RAM_BEFORE_GIB = (
    bytes_to_gib(
        current_process
        .memory_info()
        .rss
    )
)


SYSTEM_RAM_BEFORE = (
    psutil.virtual_memory()
)


SYSTEM_RAM_AVAILABLE_BEFORE_GIB = (
    bytes_to_gib(
        SYSTEM_RAM_BEFORE.available
    )
)


# ============================================================
# 1A.6 — FILE DISCOVERY HELPERS
# ============================================================

def find_files(
    root,
    patterns,
):

    matches = []


    for pattern in patterns:

        matches.extend(
            root.rglob(
                pattern
            )
        )


    return sorted(
        set(
            matches
        )
    )


def find_single_file(
    root,
    patterns,
    description,
):

    matches = find_files(
        root,
        patterns,
    )


    if not matches:

        raise FileNotFoundError(
            f"No {description} found under:\n"
            f"{root}"
        )


    if len(matches) > 1:

        raise RuntimeError(
            f"Multiple candidates found for "
            f"{description}.\n\n"
            f"Expected exactly one artifact.\n"
            f"Candidates: "
            f"{[str(path) for path in matches]}"
        )


    return matches[0]


# ============================================================
# 1A.7 — LOCATE FAISS INDEX
# ============================================================
#
# IMPORTANT:
#
# The index is located and inspected on disk here.
#
# It is NOT loaded into RAM in this cell.
# ============================================================

RAG_FAISS_INDEX_PATH = (
    find_single_file(
        RAG_FAISS_ROOT,
        [
            "*.faiss",
            "*.index",
        ],
        "FAISS index",
    )
)


RAG_FAISS_FILE_SIZE_BYTES = (
    RAG_FAISS_INDEX_PATH
    .stat()
    .st_size
)


RAG_FAISS_FILE_SIZE_GIB = (
    bytes_to_gib(
        RAG_FAISS_FILE_SIZE_BYTES
    )
)


print("=" * 116)
print("MODULE 4 DEMO — RAG V1 ARTIFACT STRUCTURE VALIDATION")
print("=" * 116)


print(
    f"FAISS Index               : "
    f"{RAG_FAISS_INDEX_PATH.name}"
)


print(
    f"FAISS File Size           : "
    f"{RAG_FAISS_FILE_SIZE_GIB:.3f} GiB"
)


print(
    "FAISS Load State          : "
    "NOT LOADED"
)


# ============================================================
# 1A.8 — LOCATE METADATA SHARDS
# ============================================================

chunk_candidates = find_files(
    RAG_CHUNKS_ROOT,
    [
        "*.parquet",
        "*.jsonl",
        "*.json",
        "*.csv",
    ],
)


if not chunk_candidates:

    raise FileNotFoundError(
        "No reconciled chunk metadata files found under:\n"
        f"{RAG_CHUNKS_ROOT}"
    )


RAG_DISCOVERED_METADATA_SHARDS = (
    len(
        chunk_candidates
    )
)


print()
print("METADATA DISCOVERY")
print("-" * 116)


print(
    f"Discovered Shards         : "
    f"{RAG_DISCOVERED_METADATA_SHARDS}"
)


print(
    f"Expected Shards           : "
    f"{EXPECTED_RAG_METADATA_SHARDS}"
)


if (
    RAG_DISCOVERED_METADATA_SHARDS
    !=
    EXPECTED_RAG_METADATA_SHARDS
):

    raise RuntimeError(
        "Metadata shard-count mismatch.\n\n"
        f"Expected : "
        f"{EXPECTED_RAG_METADATA_SHARDS}\n"
        f"Observed : "
        f"{RAG_DISCOVERED_METADATA_SHARDS}"
    )


print(
    "✓ Frozen metadata shard count validated."
)


# ============================================================
# 1A.9 — MEMORY-EFFICIENT ROW COUNTERS
# ============================================================
#
# JSONL files are counted as newline-delimited records using
# buffered binary reads.
#
# This avoids materializing each shard as a DataFrame.
#
# Other supported formats use DuckDB to perform disk-backed
# row counting.
# ============================================================

def count_jsonl_rows(
    path,
    buffer_size=16 * 1024 * 1024,
):

    row_count = 0

    final_byte = b""

    file_size = (
        path.stat().st_size
    )


    with path.open(
        "rb"
    ) as file_handle:

        while True:

            chunk = (
                file_handle.read(
                    buffer_size
                )
            )

            if not chunk:

                break

            row_count += (
                chunk.count(
                    b"\n"
                )
            )

            final_byte = (
                chunk[-1:]
            )


    if (
        file_size > 0
        and
        final_byte != b"\n"
    ):

        row_count += 1


    return row_count


def duckdb_escape_path(
    path
):

    return (
        str(path)
        .replace(
            "'",
            "''"
        )
    )


def count_table_rows(
    path,
):

    suffix = (
        path.suffix.lower()
    )


    if suffix == ".jsonl":

        return count_jsonl_rows(
            path
        )


    escaped_path = (
        duckdb_escape_path(
            path
        )
    )


    connection = (
        duckdb.connect()
    )


    try:

        if suffix == ".parquet":

            query = (
                "SELECT COUNT(*) "
                f"FROM read_parquet('{escaped_path}')"
            )


        elif suffix == ".csv":

            query = (
                "SELECT COUNT(*) "
                f"FROM read_csv_auto('{escaped_path}')"
            )


        elif suffix == ".json":

            query = (
                "SELECT COUNT(*) "
                f"FROM read_json_auto('{escaped_path}')"
            )


        else:

            raise ValueError(
                f"Unsupported metadata format: "
                f"{path}"
            )


        result = (
            connection
            .execute(
                query
            )
            .fetchone()
        )


        return int(
            result[0]
        )


    finally:

        connection.close()


# ============================================================
# 1A.10 — BUILD LIGHTWEIGHT SHARD MANIFEST
# ============================================================
#
# Instead of concatenating all metadata rows:
#
#     shard 001
#         global IDs 0 → N
#
#     shard 002
#         global IDs N+1 → ...
#
# The manifest therefore preserves exactly the same global
# deterministic ordering previously used by pd.concat().
# ============================================================

manifest_records = []


cumulative_rows = 0


manifest_start = (
    time.perf_counter()
)


manifest_progress = tqdm(
    total=len(
        chunk_candidates
    ),
    desc="Metadata Manifest",
    unit="shard",
    leave=True,
)


for shard_number, candidate in enumerate(
    chunk_candidates,
    start=1,
):

    manifest_progress.set_postfix_str(
        f"Shard {shard_number:03d}"
    )


    try:

        row_count = (
            count_table_rows(
                candidate
            )
        )


    except Exception as exc:

        manifest_progress.close()

        raise RuntimeError(
            "Failed while validating metadata shard:\n"
            f"{candidate}\n\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc


    global_start = (
        cumulative_rows
    )


    global_end = (
        global_start
        +
        row_count
        -
        1
    )


    manifest_records.append(
        {
            "shard_number":
                shard_number,

            "file":
                candidate.name,

            "path":
                str(candidate),

            "format":
                candidate.suffix.lower(),

            "rows":
                row_count,

            "global_start":
                global_start,

            "global_end":
                global_end,

            "size_gib":
                bytes_to_gib(
                    candidate
                    .stat()
                    .st_size
                ),
        }
    )


    cumulative_rows += (
        row_count
    )


    manifest_progress.update(
        1
    )


manifest_progress.set_postfix_str(
    "Complete"
)

manifest_progress.close()


manifest_time_s = (
    time.perf_counter()
    -
    manifest_start
)


# ============================================================
# 1A.11 — CREATE SMALL MANIFEST DATAFRAME
# ============================================================
#
# Only one row per metadata shard is stored in memory.
#
# Expected size:
#
#     ~151 rows
#
# rather than:
#
#     1,506,367 metadata rows
# ============================================================

RAG_CHUNK_SHARD_MANIFEST = (
    pd.DataFrame(
        manifest_records
    )
)


RAG_CHUNK_SHARD_COUNT = (
    len(
        RAG_CHUNK_SHARD_MANIFEST
    )
)


RAG_METADATA_ROW_COUNT = (
    int(
        cumulative_rows
    )
)


RAG_METADATA_TOTAL_SIZE_GIB = (
    float(
        RAG_CHUNK_SHARD_MANIFEST[
            "size_gib"
        ].sum()
    )
)


print()
print("METADATA MANIFEST SUMMARY")
print("-" * 116)


print(
    f"Metadata Shards           : "
    f"{RAG_CHUNK_SHARD_COUNT}"
)


print(
    f"Metadata Rows             : "
    f"{RAG_METADATA_ROW_COUNT:,}"
)


print(
    f"Expected FAISS Vectors    : "
    f"{EXPECTED_RAG_VECTORS:,}"
)


print(
    f"Metadata Disk Size        : "
    f"{RAG_METADATA_TOTAL_SIZE_GIB:.3f} GiB"
)


print(
    f"Manifest Build Time       : "
    f"{manifest_time_s:.2f} s"
)


# ============================================================
# 1A.12 — VALIDATE GLOBAL ROW COUNT
# ============================================================

RAG_METADATA_ROW_COUNT_PASS = (
    RAG_METADATA_ROW_COUNT
    ==
    EXPECTED_RAG_VECTORS
)


if not RAG_METADATA_ROW_COUNT_PASS:

    raise RuntimeError(
        "Combined metadata row count does not match "
        "the frozen FAISS vector count.\n\n"
        f"Metadata rows : "
        f"{RAG_METADATA_ROW_COUNT:,}\n"
        f"FAISS vectors : "
        f"{EXPECTED_RAG_VECTORS:,}"
    )


print(
    "✓ Metadata row count matches frozen FAISS vector count."
)


# ============================================================
# 1A.13 — VALIDATE GLOBAL RANGE CONTINUITY
# ============================================================

RAG_METADATA_RANGE_PASS = True


previous_end = (
    -1
)


for record in manifest_records:

    expected_start = (
        previous_end
        +
        1
    )


    if (
        record[
            "global_start"
        ]
        !=
        expected_start
    ):

        RAG_METADATA_RANGE_PASS = (
            False
        )

        break


    previous_end = (
        record[
            "global_end"
        ]
    )


if (
    previous_end
    !=
    EXPECTED_RAG_VECTORS
    -
    1
):

    RAG_METADATA_RANGE_PASS = (
        False
    )


if not RAG_METADATA_RANGE_PASS:

    raise RuntimeError(
        "Metadata shard global ranges are not "
        "continuous or do not terminate at the "
        "expected final FAISS vector ID."
    )


print(
    "✓ Global metadata ranges are continuous."
)


# ============================================================
# 1A.14 — GLOBAL VECTOR-ID → SHARD LOOKUP
# ============================================================
#
# This replaces:
#
#     rag_chunks_df.iloc[vector_id]
#
# with:
#
#     vector_id
#         ↓
#     shard
#         ↓
#     local row
#
# without holding the complete corpus in memory.
# ============================================================

RAG_SHARD_START_IDS = (

    RAG_CHUNK_SHARD_MANIFEST[
        "global_start"
    ]
    .astype(
        int
    )
    .tolist()

)


def locate_metadata_row(
    vector_id
):

    vector_id = int(
        vector_id
    )


    if (
        vector_id < 0
        or
        vector_id
        >=
        EXPECTED_RAG_VECTORS
    ):

        raise IndexError(
            f"FAISS vector ID outside valid range: "
            f"{vector_id:,}"
        )


    shard_position = (
        bisect.bisect_right(
            RAG_SHARD_START_IDS,
            vector_id
        )
        -
        1
    )


    shard = (
        RAG_CHUNK_SHARD_MANIFEST
        .iloc[
            shard_position
        ]
    )


    local_row = (
        vector_id
        -
        int(
            shard[
                "global_start"
            ]
        )
    )


    return {

        "vector_id":
            vector_id,

        "shard_number":
            int(
                shard[
                    "shard_number"
                ]
            ),

        "path":
            Path(
                shard[
                    "path"
                ]
            ),

        "format":
            shard[
                "format"
            ],

        "local_row":
            int(
                local_row
            ),

    }


# ============================================================
# 1A.15 — SINGLE-ROW DISK ACCESSOR
# ============================================================
#
# Used only for validation samples.
#
# It reads one requested row from one shard rather than
# materializing the complete metadata corpus.
# ============================================================

def read_jsonl_row(
    path,
    row_number,
):

    with path.open(
        "r",
        encoding="utf-8"
    ) as file_handle:

        for current_row, line in enumerate(
            file_handle
        ):

            if (
                current_row
                ==
                row_number
            ):

                return json.loads(
                    line
                )


    raise IndexError(
        f"Row {row_number:,} not found in "
        f"{path.name}"
    )


def read_disk_row(
    location
):

    path = (
        location[
            "path"
        ]
    )

    local_row = (
        location[
            "local_row"
        ]
    )

    suffix = (
        path.suffix.lower()
    )


    if suffix == ".jsonl":

        return read_jsonl_row(
            path,
            local_row
        )


    escaped_path = (
        duckdb_escape_path(
            path
        )
    )


    connection = (
        duckdb.connect()
    )


    try:

        if suffix == ".parquet":

            query = (
                f"SELECT * "
                f"FROM read_parquet('{escaped_path}') "
                f"LIMIT 1 OFFSET {local_row}"
            )


        elif suffix == ".csv":

            query = (
                f"SELECT * "
                f"FROM read_csv_auto('{escaped_path}') "
                f"LIMIT 1 OFFSET {local_row}"
            )


        elif suffix == ".json":

            query = (
                f"SELECT * "
                f"FROM read_json_auto('{escaped_path}') "
                f"LIMIT 1 OFFSET {local_row}"
            )


        else:

            raise ValueError(
                f"Unsupported metadata format: "
                f"{path}"
            )


        row_df = (
            connection
            .execute(
                query
            )
            .df()
        )


        if row_df.empty:

            raise IndexError(
                f"Row {local_row:,} not found in "
                f"{path.name}"
            )


        return (
            row_df
            .iloc[
                0
            ]
            .to_dict()
        )


    finally:

        connection.close()


# ============================================================
# 1A.16 — IDENTIFY TEXT COLUMN
# ============================================================

TEXT_COLUMN_CANDIDATES = [

    "text",
    "chunk_text",
    "content",
    "page_content",

]


first_location = (
    locate_metadata_row(
        0
    )
)


first_record = (
    read_disk_row(
        first_location
    )
)


RAG_TEXT_COLUMN = next(
    (

        column

        for column
        in TEXT_COLUMN_CANDIDATES

        if column
        in first_record

    ),
    None,
)


if RAG_TEXT_COLUMN is None:

    raise RuntimeError(
        "Could not identify metadata text column.\n"
        f"Available columns: "
        f"{list(first_record.keys())}"
    )


print()
print(
    f"Text Column               : "
    f"{RAG_TEXT_COLUMN}"
)


# ============================================================
# 1A.17 — SAMPLE GLOBAL ALIGNMENT VALIDATION
# ============================================================
#
# Test:
#
#     first vector
#     middle vector
#     final vector
#
# Only three metadata rows are read.
# ============================================================

sample_vector_ids = [

    0,

    EXPECTED_RAG_VECTORS
    //
    2,

    EXPECTED_RAG_VECTORS
    -
    1,

]


sample_results = []


sample_progress = tqdm(
    total=len(
        sample_vector_ids
    ),
    desc="Metadata Sample Validation",
    unit="sample",
    leave=True,
)


RAG_TEXT_SAMPLE_PASS = (
    True
)


for vector_id in sample_vector_ids:

    location = (
        locate_metadata_row(
            vector_id
        )
    )


    record = (
        read_disk_row(
            location
        )
    )


    sample_text = str(
        record.get(
            RAG_TEXT_COLUMN,
            ""
        )
        or
        ""
    ).strip()


    text_valid = (
        len(
            sample_text
        )
        >
        0
    )


    if not text_valid:

        RAG_TEXT_SAMPLE_PASS = (
            False
        )


    sample_results.append(
        {
            "vector_id":
                vector_id,

            "shard_number":
                location[
                    "shard_number"
                ],

            "local_row":
                location[
                    "local_row"
                ],

            "text_chars":
                len(
                    sample_text
                ),

            "valid":
                text_valid,
        }
    )


    sample_progress.update(
        1
    )


sample_progress.set_postfix_str(
    "Complete"
)

sample_progress.close()


if not RAG_TEXT_SAMPLE_PASS:

    raise RuntimeError(
        "One or more sampled metadata rows contain "
        "empty chunk text."
    )


# ============================================================
# 1A.18 — COMPACT SAMPLE SUMMARY
# ============================================================

RAG_METADATA_SAMPLE_RESULTS = (
    pd.DataFrame(
        sample_results
    )
)


print()
print("GLOBAL METADATA SAMPLE VALIDATION")
print("-" * 116)


print(
    RAG_METADATA_SAMPLE_RESULTS
    .to_string(
        index=False
    )
)


print()
print(
    "✓ First, middle and final global metadata "
    "positions validated."
)


# ============================================================
# 1A.19 — RESOURCE STATE AFTER VALIDATION
# ============================================================

gc.collect()


PROCESS_RAM_AFTER_GIB = (
    bytes_to_gib(
        current_process
        .memory_info()
        .rss
    )
)


SYSTEM_RAM_AFTER = (
    psutil.virtual_memory()
)


SYSTEM_RAM_AVAILABLE_AFTER_GIB = (
    bytes_to_gib(
        SYSTEM_RAM_AFTER.available
    )
)


PROCESS_RAM_DELTA_GIB = (
    PROCESS_RAM_AFTER_GIB
    -
    PROCESS_RAM_BEFORE_GIB
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before        : "
    f"{PROCESS_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After         : "
    f"{PROCESS_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"Process RAM Delta         : "
    f"{PROCESS_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available      : "
    f"{SYSTEM_RAM_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print(
    "Full Metadata DataFrame  : "
    "NOT CREATED"
)


print(
    "FAISS Index Loaded        : "
    "False"
)


# ============================================================
# 1A.20 — DEPLOYMENT RUNTIME CONFIG
# ============================================================

MODULE4_RAG_ARTIFACT_CONFIG = {

    "retriever_version":
        RAG_RETRIEVER_VERSION,

    "embedding_model":
        RAG_MODEL_NAME,

    "faiss_index":
        str(
            RAG_FAISS_INDEX_PATH
        ),

    "faiss_file_size_gib":
        RAG_FAISS_FILE_SIZE_GIB,

    "expected_faiss_vectors":
        EXPECTED_RAG_VECTORS,

    "expected_faiss_dimension":
        EXPECTED_RAG_DIMENSION,

    "metadata_root":
        str(
            RAG_CHUNKS_ROOT
        ),

    "metadata_shards":
        RAG_CHUNK_SHARD_COUNT,

    "metadata_rows":
        RAG_METADATA_ROW_COUNT,

    "metadata_disk_size_gib":
        RAG_METADATA_TOTAL_SIZE_GIB,

    "text_column":
        RAG_TEXT_COLUMN,

    "historical_k":
        RAG_HISTORICAL_K,

    "module4_k":
        RAG_MODULE4_K,

    "max_seq_length":
        RAG_MAX_SEQ_LENGTH,

    "metadata_strategy":
        "SHARD_MANIFEST_ON_DEMAND",

    "full_metadata_dataframe":
        False,

    "faiss_loaded":
        False,

}


# ============================================================
# 1A.21 — FINAL VALIDATION
# ============================================================

MODULE4_RAG_ARTIFACT_INSPECTION_PASS = all(
    [

        RAG_FAISS_INDEX_PATH.exists(),

        RAG_CHUNK_SHARD_COUNT
        ==
        EXPECTED_RAG_METADATA_SHARDS,

        RAG_METADATA_ROW_COUNT_PASS,

        RAG_METADATA_RANGE_PASS,

        RAG_TEXT_SAMPLE_PASS,

        RAG_TEXT_COLUMN
        is not None,

    ]
)


MODULE4_CELL1A_PASS = (
    MODULE4_RAG_ARTIFACT_INSPECTION_PASS
)


if not MODULE4_CELL1A_PASS:

    raise RuntimeError(
        "Cell 1A RAG artifact structure validation failed."
    )


# ============================================================
# 1A.22 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — RAG ARTIFACT VALIDATION SUMMARY")
print("=" * 116)


print(
    f"Retriever Version          : "
    f"{RAG_RETRIEVER_VERSION}"
)


print(
    f"Embedding Model            : "
    f"{RAG_MODEL_NAME}"
)


print(
    f"FAISS Artifact             : "
    f"{RAG_FAISS_INDEX_PATH.name}"
)


print(
    f"FAISS File Size            : "
    f"{RAG_FAISS_FILE_SIZE_GIB:.3f} GiB"
)


print(
    f"FAISS Loaded               : "
    f"False"
)


print(
    f"Expected FAISS Vectors     : "
    f"{EXPECTED_RAG_VECTORS:,}"
)


print(
    f"Expected Dimension         : "
    f"{EXPECTED_RAG_DIMENSION}"
)


print(
    f"Metadata Shards            : "
    f"{RAG_CHUNK_SHARD_COUNT}"
)


print(
    f"Metadata Rows              : "
    f"{RAG_METADATA_ROW_COUNT:,}"
)


print(
    f"Metadata Disk Size         : "
    f"{RAG_METADATA_TOTAL_SIZE_GIB:.3f} GiB"
)


print(
    f"Global Metadata Range      : "
    f"0 → "
    f"{RAG_METADATA_ROW_COUNT - 1:,}"
)


print(
    f"Text Column                : "
    f"{RAG_TEXT_COLUMN}"
)


print(
    f"Text Sample PASS           : "
    f"{RAG_TEXT_SAMPLE_PASS}"
)


print(
    f"FAISS ↔ Metadata Mapping   : "
    f"{RAG_METADATA_RANGE_PASS}"
)


print(
    f"Metadata Runtime Strategy  : "
    f"SHARD_MANIFEST_ON_DEMAND"
)


print(
    f"Full Metadata in RAM       : "
    f"False"
)


print(
    f"Process RAM Delta          : "
    f"{PROCESS_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"Validation PASS            : "
    f"{MODULE4_CELL1A_PASS}"
)


print()
print("✓ CELL 1A PASSED")
print("✓ Frozen FAISS artifact located and validated on disk.")
print(f"✓ {RAG_CHUNK_SHARD_COUNT} metadata shards validated.")
print(f"✓ {RAG_METADATA_ROW_COUNT:,} metadata rows accounted for.")
print("✓ Global metadata ordering preserved.")
print("✓ FAISS-vector → shard/local-row mapping created.")
print("✓ First, middle and final metadata samples validated.")
print("✓ Full 1.5M-row Pandas DataFrame was NOT created.")
print("✓ FAISS index was NOT loaded into RAM.")
print("✓ Deployment-safe metadata architecture established.")
print("✓ Ready for Cell 1B — FAISS CPU load + RAM validation.")

print("=" * 116)


MODULE 4 DEMO — RAG V1 ARTIFACT STRUCTURE VALIDATION
FAISS Index               : faiss_index_flat_ip.index
FAISS File Size           : 5.746 GiB
FAISS Load State          : NOT LOADED

METADATA DISCOVERY
--------------------------------------------------------------------------------------------------------------------
Discovered Shards         : 151
Expected Shards           : 151
✓ Frozen metadata shard count validated.


Metadata Manifest:   0%|          | 0/151 [00:00<?, ?shard/s]


METADATA MANIFEST SUMMARY
--------------------------------------------------------------------------------------------------------------------
Metadata Shards           : 151
Metadata Rows             : 1,506,367
Expected FAISS Vectors    : 1,506,367
Metadata Disk Size        : 3.791 GiB
Manifest Build Time       : 19.75 s
✓ Metadata row count matches frozen FAISS vector count.
✓ Global metadata ranges are continuous.

Text Column               : text


Metadata Sample Validation:   0%|          | 0/3 [00:00<?, ?sample/s]


GLOBAL METADATA SAMPLE VALIDATION
--------------------------------------------------------------------------------------------------------------------
 vector_id  shard_number  local_row  text_chars  valid
         0             1          0        2910   True
    753183            76       3183        2981   True
   1506366           151       6366         146   True

✓ First, middle and final global metadata positions validated.

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Process RAM Before        : 0.745 GiB
Process RAM After         : 0.743 GiB
Process RAM Delta         : -0.002 GiB
System RAM Available      : 47.185 GiB
Full Metadata DataFrame  : NOT CREATED
FAISS Index Loaded        : False

MODULE 4 DEMO — RAG ARTIFACT VALIDATION SUMMARY
Retriever Version          : V1
Embedding Model            : BAAI/bge-m3
FAISS Artifact             : faiss_index_flat_ip.index
FAISS File Size            :

##### Cell 1A — Observation

Artifact integrity passed: **151 metadata shards**, **1,506,367 metadata rows**, and **1,506,367 FAISS vectors** remain aligned one-to-one. Metadata is kept disk-backed to control RAM use.


#### Cell 1B — FAISS CPU Load + RAM Validation

**Description:** Load the frozen FAISS `IndexFlatIP` on CPU and measure its real resident-memory footprint.


In [7]:
# ============================================================
# CELL 1B — FAISS CPU LOAD + RAM VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Load the complete frozen Semantic RAG V1 FAISS index on CPU
# and measure its actual runtime memory footprint independently
# from:
#
#     - BGE-M3 query encoder
#     - Metadata corpus
#     - MCP runtime
#     - LLM generation
#
# This provides the first major deployment-capacity test.
#
# Frozen index identity:
#
#     Vectors   : 1,506,367
#     Dimension : 1024
#     Metric    : Inner Product
#
# IMPORTANT:
#
# - The full metadata DataFrame remains unloaded.
# - BGE-M3 remains unloaded.
# - No semantic query is executed yet.
# - The index is loaded normally rather than using a reduced,
#   quantized, or rebuilt representation.
#
# This preserves the original Module 2 RAG V1 FAISS baseline.
# ============================================================


# ============================================================
# 1B.1 — VERIFY CELL 1A
# ============================================================

if (
    "MODULE4_CELL1A_PASS"
    not in globals()
    or
    not MODULE4_CELL1A_PASS
):

    raise RuntimeError(
        "Cell 1A must pass before Cell 1B."
    )


# ============================================================
# 1B.2 — IMPORTS
# ============================================================

import gc
import os
import time

import faiss
import psutil


# ============================================================
# 1B.3 — MEMORY HELPERS
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def bytes_to_gib(
    byte_count
):

    return (
        byte_count
        /
        BYTES_PER_GIB
    )


def current_process_rss_gib():

    process = (
        psutil.Process(
            os.getpid()
        )
    )

    return (
        bytes_to_gib(
            process
            .memory_info()
            .rss
        )
    )


def available_system_ram_gib():

    return (
        bytes_to_gib(
            psutil.virtual_memory()
            .available
        )
    )


# ============================================================
# 1B.4 — CPU THREAD POLICY
# ============================================================
#
# Use the logical CPU count available to the deployment
# runtime.
#
# Current Colab CPU runtime detected 2 logical cores.
# ============================================================

MODULE4_FAISS_CPU_THREADS = max(
    1,
    int(
        psutil.cpu_count(
            logical=True
        )
        or
        1
    ),
)


faiss.omp_set_num_threads(
    MODULE4_FAISS_CPU_THREADS
)


# ============================================================
# 1B.5 — PRE-LOAD RESOURCE STATE
# ============================================================

gc.collect()


FAISS_PROCESS_RAM_BEFORE_GIB = (
    current_process_rss_gib()
)


FAISS_SYSTEM_RAM_BEFORE_GIB = (
    available_system_ram_gib()
)


FAISS_INDEX_DISK_SIZE_GIB = (
    RAG_FAISS_INDEX_PATH
    .stat()
    .st_size
    /
    BYTES_PER_GIB
)


# Keep a minimum amount of RAM available for Python/runtime
# overhead after the FAISS load.
MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB = (
    2.0
)


FAISS_ESTIMATED_POST_LOAD_AVAILABLE_GIB = (
    FAISS_SYSTEM_RAM_BEFORE_GIB
    -
    FAISS_INDEX_DISK_SIZE_GIB
)


print("=" * 116)
print("MODULE 4 DEMO — FAISS CPU LOAD + RAM VALIDATION")
print("=" * 116)


print(
    f"FAISS Index                : "
    f"{RAG_FAISS_INDEX_PATH.name}"
)


print(
    f"Index File Size            : "
    f"{FAISS_INDEX_DISK_SIZE_GIB:.3f} GiB"
)


print(
    f"Process RAM Before          : "
    f"{FAISS_PROCESS_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"System RAM Available       : "
    f"{FAISS_SYSTEM_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Estimated Post-Load RAM    : "
    f"{FAISS_ESTIMATED_POST_LOAD_AVAILABLE_GIB:.3f} GiB"
)


print(
    f"Required Safety Headroom   : "
    f"{MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB:.3f} GiB"
)


print(
    f"FAISS CPU Threads          : "
    f"{MODULE4_FAISS_CPU_THREADS}"
)


# ============================================================
# 1B.6 — PRE-LOAD SAFETY CHECK
# ============================================================

FAISS_PRELOAD_MEMORY_PASS = (

    FAISS_ESTIMATED_POST_LOAD_AVAILABLE_GIB
    >=
    MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB

)


if not FAISS_PRELOAD_MEMORY_PASS:

    raise RuntimeError(
        "Insufficient RAM headroom to safely load the "
        "complete FAISS index.\n\n"
        f"Available RAM        : "
        f"{FAISS_SYSTEM_RAM_BEFORE_GIB:.3f} GiB\n"
        f"FAISS file size      : "
        f"{FAISS_INDEX_DISK_SIZE_GIB:.3f} GiB\n"
        f"Estimated remaining  : "
        f"{FAISS_ESTIMATED_POST_LOAD_AVAILABLE_GIB:.3f} GiB\n"
        f"Required headroom    : "
        f"{MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB:.3f} GiB"
    )


print()
print(
    "✓ Pre-load memory safety check passed."
)


# ============================================================
# 1B.7 — LOAD COMPLETE FAISS INDEX
# ============================================================
#
# This is intentionally a normal FAISS restore.
#
# We are measuring the true RAM requirement of the frozen
# IndexFlatIP architecture rather than changing the index
# representation for deployment.
# ============================================================

print()
print(
    "Loading complete FAISS index on CPU..."
)


faiss_load_start = (
    time.perf_counter()
)


rag_faiss_index = (
    faiss.read_index(
        str(
            RAG_FAISS_INDEX_PATH
        )
    )
)


FAISS_LOAD_TIME_S = (
    time.perf_counter()
    -
    faiss_load_start
)


# ============================================================
# 1B.8 — POST-LOAD RESOURCE STATE
# ============================================================

gc.collect()


FAISS_PROCESS_RAM_AFTER_GIB = (
    current_process_rss_gib()
)


FAISS_SYSTEM_RAM_AFTER_GIB = (
    available_system_ram_gib()
)


FAISS_PROCESS_RAM_DELTA_GIB = (
    FAISS_PROCESS_RAM_AFTER_GIB
    -
    FAISS_PROCESS_RAM_BEFORE_GIB
)


FAISS_SYSTEM_AVAILABLE_DELTA_GIB = (
    FAISS_SYSTEM_RAM_AFTER_GIB
    -
    FAISS_SYSTEM_RAM_BEFORE_GIB
)


# ============================================================
# 1B.9 — INSPECT LOADED INDEX
# ============================================================

RAG_INDEX_TYPE = (
    type(
        rag_faiss_index
    )
    .__name__
)


RAG_INDEX_VECTOR_COUNT = int(
    rag_faiss_index.ntotal
)


RAG_INDEX_DIMENSION = int(
    rag_faiss_index.d
)


RAG_INDEX_METRIC = int(
    rag_faiss_index.metric_type
)


RAG_INDEX_IS_INNER_PRODUCT = (

    RAG_INDEX_METRIC
    ==
    faiss.METRIC_INNER_PRODUCT

)


# ============================================================
# 1B.10 — VALIDATE FROZEN INDEX IDENTITY
# ============================================================

RAG_INDEX_VECTOR_COUNT_PASS = (

    RAG_INDEX_VECTOR_COUNT
    ==
    EXPECTED_RAG_VECTORS

)


RAG_INDEX_DIMENSION_PASS = (

    RAG_INDEX_DIMENSION
    ==
    EXPECTED_RAG_DIMENSION

)


RAG_INDEX_METRIC_PASS = (
    RAG_INDEX_IS_INNER_PRODUCT
)


MODULE4_FAISS_IDENTITY_PASS = all(
    [
        RAG_INDEX_VECTOR_COUNT_PASS,
        RAG_INDEX_DIMENSION_PASS,
        RAG_INDEX_METRIC_PASS,
    ]
)


if not MODULE4_FAISS_IDENTITY_PASS:

    raise RuntimeError(
        "Loaded FAISS index does not match the frozen "
        "Semantic RAG V1 identity.\n\n"
        f"Vectors   : "
        f"{RAG_INDEX_VECTOR_COUNT:,} "
        f"(expected {EXPECTED_RAG_VECTORS:,})\n"
        f"Dimension : "
        f"{RAG_INDEX_DIMENSION} "
        f"(expected {EXPECTED_RAG_DIMENSION})\n"
        f"Metric    : "
        f"{RAG_INDEX_METRIC} "
        f"(expected Inner Product)"
    )


# ============================================================
# 1B.11 — POST-LOAD HEADROOM VALIDATION
# ============================================================
#
# Do not fail simply because BGE-M3 has not yet been tested.
#
# This cell answers only:
#
#     Can the complete frozen FAISS index operate within the
#     current CPU runtime?
#
# BGE-M3 capacity is tested separately in Cell 1C.
# ============================================================

MODULE4_FAISS_POST_LOAD_HEADROOM_PASS = (

    FAISS_SYSTEM_RAM_AFTER_GIB
    >=
    MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB

)


# ============================================================
# 1B.12 — MEMORY-TO-DISK OBSERVATION
# ============================================================
#
# This ratio is diagnostic only.
#
# It indicates how the observed process RSS increase compares
# with the serialized index file size.
# ============================================================

if (
    FAISS_INDEX_DISK_SIZE_GIB
    >
    0
):

    FAISS_RAM_TO_DISK_RATIO = (

        FAISS_PROCESS_RAM_DELTA_GIB
        /
        FAISS_INDEX_DISK_SIZE_GIB

    )

else:

    FAISS_RAM_TO_DISK_RATIO = (
        float("nan")
    )


# ============================================================
# 1B.13 — FORMAL RUNTIME STATE
# ============================================================

MODULE4_RAG_FAISS_LOADED = (
    True
)


MODULE4_RAG_FAISS_DEVICE = (
    "cpu"
)


MODULE4_RAG_FAISS_CONFIG = {

    "index_path":
        str(
            RAG_FAISS_INDEX_PATH
        ),

    "index_type":
        RAG_INDEX_TYPE,

    "vectors":
        RAG_INDEX_VECTOR_COUNT,

    "dimension":
        RAG_INDEX_DIMENSION,

    "metric":
        "INNER_PRODUCT",

    "device":
        MODULE4_RAG_FAISS_DEVICE,

    "cpu_threads":
        MODULE4_FAISS_CPU_THREADS,

    "file_size_gib":
        FAISS_INDEX_DISK_SIZE_GIB,

    "load_time_s":
        FAISS_LOAD_TIME_S,

    "process_ram_before_gib":
        FAISS_PROCESS_RAM_BEFORE_GIB,

    "process_ram_after_gib":
        FAISS_PROCESS_RAM_AFTER_GIB,

    "process_ram_delta_gib":
        FAISS_PROCESS_RAM_DELTA_GIB,

    "system_ram_available_after_gib":
        FAISS_SYSTEM_RAM_AFTER_GIB,

}


# ============================================================
# 1B.14 — FINAL VALIDATION
# ============================================================

MODULE4_FAISS_LOAD_PASS = all(
    [
        MODULE4_RAG_FAISS_LOADED,
        MODULE4_FAISS_IDENTITY_PASS,
        MODULE4_FAISS_POST_LOAD_HEADROOM_PASS,
    ]
)


MODULE4_CELL1B_PASS = (
    MODULE4_FAISS_LOAD_PASS
)


# ============================================================
# 1B.15 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — FAISS CPU LOAD SUMMARY")
print("=" * 116)


print(
    f"Index Type                 : "
    f"{RAG_INDEX_TYPE}"
)


print(
    f"Metric                     : "
    f"INNER_PRODUCT"
)


print(
    f"Vectors                    : "
    f"{RAG_INDEX_VECTOR_COUNT:,}"
)


print(
    f"Dimension                  : "
    f"{RAG_INDEX_DIMENSION}"
)


print(
    f"Device                     : "
    f"{MODULE4_RAG_FAISS_DEVICE.upper()}"
)


print(
    f"CPU Threads                : "
    f"{MODULE4_FAISS_CPU_THREADS}"
)


print(
    f"Index File Size            : "
    f"{FAISS_INDEX_DISK_SIZE_GIB:.3f} GiB"
)


print(
    f"FAISS Load Time            : "
    f"{FAISS_LOAD_TIME_S:.2f} s"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before          : "
    f"{FAISS_PROCESS_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After           : "
    f"{FAISS_PROCESS_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"Process RAM Delta           : "
    f"{FAISS_PROCESS_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"RAM / Disk Ratio           : "
    f"{FAISS_RAM_TO_DISK_RATIO:.3f}"
)


print(
    f"System RAM Available After : "
    f"{FAISS_SYSTEM_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"Required Safety Headroom   : "
    f"{MODULE4_FAISS_MIN_POST_LOAD_HEADROOM_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Vector Count PASS          : "
    f"{RAG_INDEX_VECTOR_COUNT_PASS}"
)


print(
    f"Dimension PASS             : "
    f"{RAG_INDEX_DIMENSION_PASS}"
)


print(
    f"Inner Product Metric PASS  : "
    f"{RAG_INDEX_METRIC_PASS}"
)


print(
    f"Post-Load Headroom PASS    : "
    f"{MODULE4_FAISS_POST_LOAD_HEADROOM_PASS}"
)


print(
    f"FAISS Load PASS            : "
    f"{MODULE4_FAISS_LOAD_PASS}"
)


if not MODULE4_CELL1B_PASS:

    raise RuntimeError(
        "Cell 1B FAISS CPU load validation failed."
    )


print()
print("✓ CELL 1B PASSED")
print("✓ Complete frozen FAISS index loaded on CPU.")
print(f"✓ {RAG_INDEX_VECTOR_COUNT:,} vectors validated.")
print(f"✓ {RAG_INDEX_DIMENSION}-dimensional index validated.")
print("✓ Inner-product retrieval metric validated.")
print("✓ Full metadata corpus remains outside RAM.")
print("✓ Actual FAISS RAM footprint measured.")
print("✓ Runtime retains sufficient post-load RAM headroom.")
print("✓ Ready for Cell 1C — BGE-M3 CPU load + RAM validation.")

print("=" * 116)


MODULE 4 DEMO — FAISS CPU LOAD + RAM VALIDATION
FAISS Index                : faiss_index_flat_ip.index
Index File Size            : 5.746 GiB
Process RAM Before          : 0.743 GiB
System RAM Available       : 47.184 GiB
Estimated Post-Load RAM    : 41.437 GiB
Required Safety Headroom   : 2.000 GiB
FAISS CPU Threads          : 8

✓ Pre-load memory safety check passed.

Loading complete FAISS index on CPU...

MODULE 4 DEMO — FAISS CPU LOAD SUMMARY
Index Type                 : IndexFlatIP
Metric                     : INNER_PRODUCT
Vectors                    : 1,506,367
Dimension                  : 1024
Device                     : CPU
CPU Threads                : 8
Index File Size            : 5.746 GiB
FAISS Load Time            : 8.16 s

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Process RAM Before          : 0.743 GiB
Process RAM After           : 6.490 GiB
Process RAM Delta           : +5.747 GiB

##### Cell 1B — Observation

The **5.746 GiB** FAISS `IndexFlatIP` index loaded successfully. Memory behavior matched expectations for an in-memory exhaustive vector index.


#### Cell 1C — BGE-M3 CPU Load + RAM Validation

**Description:** Load `BAAI/bge-m3` beside the full FAISS index and test one normalized 1024-dimensional query embedding.


In [8]:
# ============================================================
# CELL 1C — BGE-M3 CPU LOAD + RAM VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Load the frozen BGE-M3 query encoder on CPU while the full
# FAISS index remains resident in memory.
#
# This cell measures:
#
#     - RAM before BGE-M3 load
#     - BGE-M3 model load time
#     - BGE-M3 incremental RAM footprint
#     - Combined FAISS + BGE-M3 runtime memory
#     - Remaining system RAM
#     - Query embedding latency
#     - Embedding dimensionality
#
# Deployment question:
#
#     Can the complete semantic retrieval stack:
#
#         Full FAISS Index
#               +
#         BGE-M3 Query Encoder
#
#     coexist within the current CPU runtime?
#
# IMPORTANT:
#
# - FAISS remains fully loaded from Cell 1B.
# - Metadata remains disk-backed / on-demand.
# - BGE-M3 runs strictly on CPU.
# - No GPU migration logic is used.
# - No LLM is loaded locally.
# ============================================================


# ============================================================
# 1C.1 — VERIFY CELL 1B
# ============================================================

if (
    "MODULE4_CELL1B_PASS"
    not in globals()
    or
    not MODULE4_CELL1B_PASS
):

    raise RuntimeError(
        "Cell 1B must pass before Cell 1C."
    )


if (
    "rag_faiss_index"
    not in globals()
):

    raise RuntimeError(
        "FAISS index is no longer resident in memory. "
        "Re-run Cell 1B."
    )


if (
    "MODULE4_RAG_FAISS_LOADED"
    not in globals()
    or
    not MODULE4_RAG_FAISS_LOADED
):

    raise RuntimeError(
        "Cell 1B FAISS runtime state is missing."
    )


# ============================================================
# 1C.2 — IMPORTS
# ============================================================

import gc
import os
import time
import resource

from pathlib import Path

import numpy as np
import psutil
import torch

from sentence_transformers import (
    SentenceTransformer
)

from tqdm.notebook import tqdm


# ============================================================
# 1C.3 — SUPPRESS INTERNAL DOWNLOAD PROGRESS
# ============================================================
#
# Keep notebook output clean.
#
# One controlled progress indicator will be used instead of
# multiple Hugging Face / Transformers download bars.
# ============================================================

try:

    from huggingface_hub.utils import (
        disable_progress_bars
    )

    disable_progress_bars()

except Exception:

    pass


try:

    from transformers.utils import logging as hf_logging

    hf_logging.disable_progress_bar()
    hf_logging.set_verbosity_error()

except Exception:

    pass


# ============================================================
# 1C.4 — MEMORY HELPERS
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def bytes_to_gib(
    byte_count
):

    return (
        byte_count
        /
        BYTES_PER_GIB
    )


def current_process_rss_gib():

    process = (
        psutil.Process(
            os.getpid()
        )
    )

    return (
        bytes_to_gib(
            process
            .memory_info()
            .rss
        )
    )


def available_system_ram_gib():

    return (
        bytes_to_gib(
            psutil.virtual_memory()
            .available
        )
    )


def peak_process_rss_gib():
    """
    Linux reports ru_maxrss in KiB.
    """

    return (
        resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss
        /
        1024**2
    )


def directory_unique_size_bytes(
    root_path
):
    """
    Measure unique underlying files only.

    This avoids double-counting Hugging Face snapshot
    symlinks that point to files in the blobs directory.
    """

    root_path = Path(
        root_path
    )


    if not root_path.exists():

        return 0


    total_bytes = 0
    seen_files = set()


    for file_path in root_path.rglob("*"):

        try:

            if not file_path.is_file():

                continue


            resolved = (
                file_path.resolve()
            )


            if resolved in seen_files:

                continue


            seen_files.add(
                resolved
            )


            total_bytes += (
                resolved.stat().st_size
            )


        except OSError:

            pass


    return total_bytes


# ============================================================
# 1C.5 — CPU EXECUTION POLICY
# ============================================================

MODULE4_BGE_DEVICE = (
    "cpu"
)


MODULE4_BGE_CPU_THREADS = max(
    1,
    int(
        psutil.cpu_count(
            logical=True
        )
        or
        1
    ),
)


torch.set_num_threads(
    MODULE4_BGE_CPU_THREADS
)


# ============================================================
# 1C.6 — BGE-M3 CONFIGURATION
# ============================================================

RAG_MODEL_NAME = (
    "BAAI/bge-m3"
)


RAG_MAX_SEQ_LENGTH = (
    1280
)


EXPECTED_RAG_DIMENSION = (
    1024
)


# Warning threshold rather than an assumption about model size.
#
# We will measure actual remaining memory after loading.
MODULE4_BGE_HEADROOM_WARNING_GIB = (
    1.0
)


# ============================================================
# 1C.7 — PRE-LOAD RESOURCE STATE
# ============================================================

gc.collect()


BGE_PROCESS_RAM_BEFORE_GIB = (
    current_process_rss_gib()
)


BGE_SYSTEM_RAM_BEFORE_GIB = (
    available_system_ram_gib()
)


BGE_PEAK_RAM_BEFORE_GIB = (
    peak_process_rss_gib()
)


print("=" * 116)
print("MODULE 4 DEMO — BGE-M3 CPU LOAD + RAM VALIDATION")
print("=" * 116)


print(
    f"Embedding Model            : "
    f"{RAG_MODEL_NAME}"
)


print(
    f"Device                     : "
    f"{MODULE4_BGE_DEVICE.upper()}"
)


print(
    f"Max Sequence Length        : "
    f"{RAG_MAX_SEQ_LENGTH}"
)


print(
    f"CPU Threads                : "
    f"{MODULE4_BGE_CPU_THREADS}"
)


print()
print("PRE-LOAD MEMORY")
print("-" * 116)


print(
    f"Process RAM                 : "
    f"{BGE_PROCESS_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"System RAM Available       : "
    f"{BGE_SYSTEM_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"FAISS Still Loaded         : "
    f"{MODULE4_RAG_FAISS_LOADED}"
)


print(
    f"FAISS Vectors              : "
    f"{rag_faiss_index.ntotal:,}"
)


print(
    f"FAISS Dimension            : "
    f"{rag_faiss_index.d}"
)


# ============================================================
# 1C.8 — HUGGING FACE CACHE BASELINE
# ============================================================

HF_MODEL_CACHE_ROOT = Path(
    Path.home(),
    ".cache",
    "huggingface",
    "hub",
    "models--BAAI--bge-m3",
)


BGE_CACHE_BEFORE_BYTES = (
    directory_unique_size_bytes(
        HF_MODEL_CACHE_ROOT
    )
)


# ============================================================
# 1C.9 — CONTROLLED LOAD PROGRESS
# ============================================================

load_progress = tqdm(
    total=2,
    desc="BGE-M3 Runtime Validation",
    unit="step",
    leave=True,
)


# ============================================================
# 1C.10 — LOAD BGE-M3 ON CPU
# ============================================================

load_progress.set_postfix_str(
    "Loading BGE-M3 on CPU"
)


encoder_load_start = (
    time.perf_counter()
)


try:

    rag_query_encoder = SentenceTransformer(
        RAG_MODEL_NAME,
        device=MODULE4_BGE_DEVICE,
    )

except Exception as exc:

    load_progress.close()

    raise RuntimeError(
        "BGE-M3 failed to load on CPU.\n\n"
        f"{type(exc).__name__}: {exc}"
    ) from exc


rag_query_encoder.max_seq_length = (
    RAG_MAX_SEQ_LENGTH
)


rag_query_encoder.eval()


BGE_ENCODER_LOAD_TIME_S = (
    time.perf_counter()
    -
    encoder_load_start
)


load_progress.update(
    1
)


# ============================================================
# 1C.11 — POST-LOAD MEMORY STATE
# ============================================================

gc.collect()


BGE_PROCESS_RAM_AFTER_LOAD_GIB = (
    current_process_rss_gib()
)


BGE_SYSTEM_RAM_AFTER_LOAD_GIB = (
    available_system_ram_gib()
)


BGE_PROCESS_RAM_LOAD_DELTA_GIB = (
    BGE_PROCESS_RAM_AFTER_LOAD_GIB
    -
    BGE_PROCESS_RAM_BEFORE_GIB
)


BGE_PEAK_RAM_AFTER_LOAD_GIB = (
    peak_process_rss_gib()
)


# ============================================================
# 1C.12 — MEASURE LOCAL MODEL CACHE FOOTPRINT
# ============================================================

BGE_CACHE_AFTER_BYTES = (
    directory_unique_size_bytes(
        HF_MODEL_CACHE_ROOT
    )
)


BGE_CACHE_TOTAL_GIB = (
    bytes_to_gib(
        BGE_CACHE_AFTER_BYTES
    )
)


BGE_CACHE_DOWNLOAD_DELTA_GIB = (
    bytes_to_gib(
        max(
            0,
            BGE_CACHE_AFTER_BYTES
            -
            BGE_CACHE_BEFORE_BYTES
        )
    )
)


# ============================================================
# 1C.13 — QUERY EMBEDDING FUNCTION
# ============================================================

def embed_rag_query(
    query
):

    if not isinstance(
        query,
        str,
    ):

        query = str(
            query
        )


    query = (
        query.strip()
    )


    if not query:

        raise ValueError(
            "Query cannot be empty."
        )


    start = (
        time.perf_counter()
    )


    embedding = (
        rag_query_encoder.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
    )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    embedding = np.asarray(
        embedding,
        dtype=np.float32,
    )


    expected_shape = (
        1,
        EXPECTED_RAG_DIMENSION,
    )


    if (
        embedding.shape
        !=
        expected_shape
    ):

        raise RuntimeError(
            "Unexpected BGE-M3 embedding shape.\n"
            f"Expected: {expected_shape}\n"
            f"Observed: {embedding.shape}"
        )


    return (
        embedding,
        float(
            elapsed
        ),
    )


# ============================================================
# 1C.14 — EMBEDDING SMOKE TEST
# ============================================================

load_progress.set_postfix_str(
    "Testing query embedding"
)


TEST_QUERY = (
    "What are the primary responsibilities "
    "of the AMF in a 5G Standalone network?"
)


try:

    test_embedding, test_embedding_time_s = (
        embed_rag_query(
            TEST_QUERY
        )
    )

except Exception as exc:

    load_progress.close()

    raise RuntimeError(
        "BGE-M3 query embedding smoke test failed.\n\n"
        f"{type(exc).__name__}: {exc}"
    ) from exc


load_progress.update(
    1
)


load_progress.set_postfix_str(
    "Complete"
)

load_progress.close()


# ============================================================
# 1C.15 — FINAL MEMORY STATE AFTER INFERENCE
# ============================================================

gc.collect()


BGE_PROCESS_RAM_AFTER_TEST_GIB = (
    current_process_rss_gib()
)


BGE_SYSTEM_RAM_AFTER_TEST_GIB = (
    available_system_ram_gib()
)


BGE_TOTAL_PROCESS_DELTA_GIB = (
    BGE_PROCESS_RAM_AFTER_TEST_GIB
    -
    BGE_PROCESS_RAM_BEFORE_GIB
)


BGE_PEAK_RAM_AFTER_TEST_GIB = (
    peak_process_rss_gib()
)


# ============================================================
# 1C.16 — COMPLETE SEMANTIC STACK MEMORY
# ============================================================
#
# Process RSS now includes:
#
#     Python/runtime
#     FAISS
#     BGE-M3
#     query embedding runtime
#
# Metadata remains outside RAM.
# ============================================================

SEMANTIC_STACK_PROCESS_RAM_GIB = (
    BGE_PROCESS_RAM_AFTER_TEST_GIB
)


SEMANTIC_STACK_AVAILABLE_RAM_GIB = (
    BGE_SYSTEM_RAM_AFTER_TEST_GIB
)


if (
    "PROCESS_RSS_GB"
    in globals()
):

    SEMANTIC_STACK_RAM_OVER_BASELINE_GIB = (
        SEMANTIC_STACK_PROCESS_RAM_GIB
        -
        PROCESS_RSS_GB
    )

else:

    SEMANTIC_STACK_RAM_OVER_BASELINE_GIB = (
        float("nan")
    )


# ============================================================
# 1C.17 — ENCODER VALIDATION
# ============================================================

BGE_DEVICE_PASS = (
    str(
        rag_query_encoder.device
    ).startswith(
        "cpu"
    )
)


BGE_DIMENSION_PASS = (
    test_embedding.shape
    ==
    (
        1,
        EXPECTED_RAG_DIMENSION,
    )
)


BGE_FINITE_PASS = (
    np.isfinite(
        test_embedding
    ).all()
)


BGE_NORM = float(
    np.linalg.norm(
        test_embedding[
            0
        ]
    )
)


BGE_NORMALIZATION_PASS = (
    np.isclose(
        BGE_NORM,
        1.0,
        atol=1e-3,
    )
)


MODULE4_RAG_ENCODER_PASS = all(
    [
        BGE_DEVICE_PASS,
        BGE_DIMENSION_PASS,
        BGE_FINITE_PASS,
        BGE_NORMALIZATION_PASS,
    ]
)


# ============================================================
# 1C.18 — DEPLOYMENT HEADROOM CLASSIFICATION
# ============================================================

if (
    SEMANTIC_STACK_AVAILABLE_RAM_GIB
    >=
    2.0
):

    MODULE4_SEMANTIC_RAM_STATUS = (
        "COMFORTABLE"
    )


elif (
    SEMANTIC_STACK_AVAILABLE_RAM_GIB
    >=
    MODULE4_BGE_HEADROOM_WARNING_GIB
):

    MODULE4_SEMANTIC_RAM_STATUS = (
        "TIGHT_BUT_OPERATIONAL"
    )


else:

    MODULE4_SEMANTIC_RAM_STATUS = (
        "CRITICAL_HEADROOM"
    )


MODULE4_SEMANTIC_HEADROOM_PASS = (

    SEMANTIC_STACK_AVAILABLE_RAM_GIB
    >=
    MODULE4_BGE_HEADROOM_WARNING_GIB

)


# ============================================================
# 1C.19 — RUNTIME STATE
# ============================================================

MODULE4_RAG_CURRENT_DEVICE = (
    "cpu"
)


MODULE4_RAG_ENCODER_LOADED = (
    True
)


MODULE4_RAG_ENCODER_CONFIG = {

    "model":
        RAG_MODEL_NAME,

    "device":
        MODULE4_RAG_CURRENT_DEVICE,

    "max_seq_length":
        RAG_MAX_SEQ_LENGTH,

    "dimension":
        EXPECTED_RAG_DIMENSION,

    "cpu_threads":
        MODULE4_BGE_CPU_THREADS,

    "load_time_s":
        BGE_ENCODER_LOAD_TIME_S,

    "embedding_test_time_s":
        test_embedding_time_s,

    "model_cache_gib":
        BGE_CACHE_TOTAL_GIB,

    "ram_before_gib":
        BGE_PROCESS_RAM_BEFORE_GIB,

    "ram_after_load_gib":
        BGE_PROCESS_RAM_AFTER_LOAD_GIB,

    "ram_after_test_gib":
        BGE_PROCESS_RAM_AFTER_TEST_GIB,

    "incremental_ram_gib":
        BGE_TOTAL_PROCESS_DELTA_GIB,

    "system_ram_available_gib":
        SEMANTIC_STACK_AVAILABLE_RAM_GIB,

    "semantic_ram_status":
        MODULE4_SEMANTIC_RAM_STATUS,

}


# ============================================================
# 1C.20 — FINAL VALIDATION
# ============================================================

MODULE4_CELL1C_PASS = (
    MODULE4_RAG_ENCODER_PASS
)


if not MODULE4_CELL1C_PASS:

    raise RuntimeError(
        "Cell 1C BGE-M3 validation failed."
    )


# ============================================================
# 1C.21 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — BGE-M3 CPU LOAD SUMMARY")
print("=" * 116)


print(
    f"Embedding Model             : "
    f"{RAG_MODEL_NAME}"
)


print(
    f"Device                      : "
    f"{rag_query_encoder.device}"
)


print(
    f"Embedding Dimension         : "
    f"{EXPECTED_RAG_DIMENSION}"
)


print(
    f"Max Sequence Length         : "
    f"{RAG_MAX_SEQ_LENGTH}"
)


print(
    f"Encoder Load Time           : "
    f"{BGE_ENCODER_LOAD_TIME_S:.2f} s"
)


print(
    f"Test Embedding Time         : "
    f"{test_embedding_time_s:.4f} s"
)


print(
    f"Embedding Norm              : "
    f"{BGE_NORM:.6f}"
)


print(
    f"Model Cache Footprint       : "
    f"{BGE_CACHE_TOTAL_GIB:.3f} GiB"
)


print(
    f"New Cache Download          : "
    f"{BGE_CACHE_DOWNLOAD_DELTA_GIB:.3f} GiB"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before BGE       : "
    f"{BGE_PROCESS_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After Load       : "
    f"{BGE_PROCESS_RAM_AFTER_LOAD_GIB:.3f} GiB"
)


print(
    f"Process RAM After Test       : "
    f"{BGE_PROCESS_RAM_AFTER_TEST_GIB:.3f} GiB"
)


print(
    f"BGE Incremental RAM          : "
    f"{BGE_TOTAL_PROCESS_DELTA_GIB:+.3f} GiB"
)


print(
    f"Peak Process RAM             : "
    f"{BGE_PEAK_RAM_AFTER_TEST_GIB:.3f} GiB"
)


print(
    f"System RAM Available         : "
    f"{SEMANTIC_STACK_AVAILABLE_RAM_GIB:.3f} GiB"
)


print(
    f"Semantic Runtime Status      : "
    f"{MODULE4_SEMANTIC_RAM_STATUS}"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"CPU Device PASS              : "
    f"{BGE_DEVICE_PASS}"
)


print(
    f"1024-Dimension PASS          : "
    f"{BGE_DIMENSION_PASS}"
)


print(
    f"Finite Embedding PASS        : "
    f"{BGE_FINITE_PASS}"
)


print(
    f"Normalized Embedding PASS    : "
    f"{BGE_NORMALIZATION_PASS}"
)


print(
    f"Encoder PASS                 : "
    f"{MODULE4_RAG_ENCODER_PASS}"
)


print(
    f"≥1 GiB Headroom              : "
    f"{MODULE4_SEMANTIC_HEADROOM_PASS}"
)


print()
print("✓ CELL 1C PASSED")
print("✓ Full FAISS index remains resident on CPU.")
print("✓ BGE-M3 query encoder loaded on CPU.")
print("✓ 1024-dimensional normalized embedding validated.")
print("✓ BGE-M3 incremental RAM footprint measured.")
print("✓ Complete semantic retrieval runtime memory measured.")
print("✓ Full metadata corpus remains outside RAM.")

if MODULE4_SEMANTIC_HEADROOM_PASS:

    print(
        "✓ Runtime retains usable memory headroom."
    )

else:

    print(
        "⚠ Semantic retrieval is operational, but RAM "
        "headroom is below the 1 GiB deployment threshold."
    )


print(
    "✓ Ready for Cell 1D — live semantic RAG retrieval."
)

print("=" * 116)


MODULE 4 DEMO — BGE-M3 CPU LOAD + RAM VALIDATION
Embedding Model            : BAAI/bge-m3
Device                     : CPU
Max Sequence Length        : 1280
CPU Threads                : 8

PRE-LOAD MEMORY
--------------------------------------------------------------------------------------------------------------------
Process RAM                 : 6.491 GiB
System RAM Available       : 41.441 GiB
FAISS Still Loaded         : True
FAISS Vectors              : 1,506,367
FAISS Dimension            : 1024


BGE-M3 Runtime Validation:   0%|          | 0/2 [00:00<?, ?step/s]


MODULE 4 DEMO — BGE-M3 CPU LOAD SUMMARY
Embedding Model             : BAAI/bge-m3
Device                      : cpu
Embedding Dimension         : 1024
Max Sequence Length         : 1280
Encoder Load Time           : 26.38 s
Test Embedding Time         : 1.0869 s
Embedding Norm              : 1.000000
Model Cache Footprint       : 3.714 GiB
New Cache Download          : 3.714 GiB

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Process RAM Before BGE       : 6.491 GiB
Process RAM After Load       : 7.768 GiB
Process RAM After Test       : 8.049 GiB
BGE Incremental RAM          : +1.558 GiB
Peak Process RAM             : 8.193 GiB
System RAM Available         : 40.963 GiB
Semantic Runtime Status      : COMFORTABLE

VALIDATION
--------------------------------------------------------------------------------------------------------------------
CPU Device PASS              : True
1024-Dimension PASS          

##### Cell 1C — Observation

BGE-M3 loaded and operated successfully on CPU. Earlier low-RAM slowdown was environmental rather than architectural; high-RAM execution confirmed the semantic stack is viable without a GPU.


#### Cell 1D — Live Semantic RAG Retrieval Validation

**Description:** Execute query encoding → FAISS Top-5 search → on-demand metadata resolution using the full frozen corpus.


In [9]:
# ============================================================
# CELL 1D — LIVE SEMANTIC RAG RETRIEVAL VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Validate the complete deployment-safe Semantic RAG V1
# retrieval path:
#
#     User Query
#         ↓
#     BGE-M3 Query Encoder — CPU
#         ↓
#     1024-D Normalized Query Vector
#         ↓
#     Full FAISS Index — CPU
#         ↓
#     Top-K Vector IDs + Similarity Scores
#         ↓
#     Shard Manifest
#         ↓
#     On-Demand Metadata Resolution
#         ↓
#     Retrieved Telecom Evidence
#
# This cell measures:
#
#     - Query embedding latency
#     - FAISS search latency
#     - Metadata resolution latency
#     - Total retrieval latency
#     - Memory before / after retrieval
#
# IMPORTANT:
#
# - Full FAISS remains resident in RAM.
# - BGE-M3 remains resident in RAM.
# - The 1.5M-row metadata corpus remains disk-backed.
# - Only metadata corresponding to retrieved vector IDs is read.
# - No LLM generation occurs in this cell.
# ============================================================


# ============================================================
# 1D.1 — VERIFY CELL 1C
# ============================================================

if (
    "MODULE4_CELL1C_PASS"
    not in globals()
    or
    not MODULE4_CELL1C_PASS
):

    raise RuntimeError(
        "Cell 1C must pass before Cell 1D."
    )


required_runtime_objects = [

    "rag_faiss_index",
    "rag_query_encoder",
    "embed_rag_query",
    "locate_metadata_row",
    "RAG_CHUNK_SHARD_MANIFEST",
    "RAG_TEXT_COLUMN",

]


missing_runtime_objects = [

    name

    for name
    in required_runtime_objects

    if name not in globals()

]


if missing_runtime_objects:

    raise RuntimeError(
        "Required Semantic RAG runtime objects are missing:\n"
        f"{missing_runtime_objects}"
    )


# ============================================================
# 1D.2 — IMPORTS
# ============================================================

import gc
import json
import os
import time

from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil


# ============================================================
# 1D.3 — RETRIEVAL CONFIGURATION
# ============================================================

MODULE4_RAG_TOP_K = (
    5
)


MODULE4_RAG_SNIPPET_CHARS = (
    220
)


# ============================================================
# 1D.4 — MEMORY HELPERS
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def bytes_to_gib(
    byte_count
):

    return (
        byte_count
        /
        BYTES_PER_GIB
    )


def current_process_rss_gib():

    process = (
        psutil.Process(
            os.getpid()
        )
    )

    return (
        bytes_to_gib(
            process
            .memory_info()
            .rss
        )
    )


def available_system_ram_gib():

    return (
        bytes_to_gib(
            psutil.virtual_memory()
            .available
        )
    )


# ============================================================
# 1D.5 — JSONL MULTI-ROW READER
# ============================================================
#
# A FAISS top-k search may return several vectors from the
# same metadata shard.
#
# Instead of opening and scanning that shard once per result,
# this function reads all required local rows from the shard
# in ONE sequential pass.
#
# Only matching JSON lines are parsed.
# ============================================================

def read_jsonl_rows(
    path,
    requested_rows,
):

    requested_rows = sorted(
        set(
            int(row)
            for row
            in requested_rows
        )
    )


    if not requested_rows:

        return {}


    requested_set = set(
        requested_rows
    )


    max_requested_row = max(
        requested_rows
    )


    resolved = {}


    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file_handle:

        for current_row, line in enumerate(
            file_handle
        ):

            if current_row in requested_set:

                resolved[
                    current_row
                ] = json.loads(
                    line
                )


            if current_row >= max_requested_row:

                break


    missing_rows = (

        requested_set
        -
        set(
            resolved.keys()
        )

    )


    if missing_rows:

        raise IndexError(
            "One or more requested metadata rows "
            "could not be resolved.\n"
            f"File: {path}\n"
            f"Missing local rows: "
            f"{sorted(missing_rows)}"
        )


    return resolved


# ============================================================
# 1D.6 — MULTI-RESULT METADATA RESOLVER
# ============================================================
#
# Global FAISS IDs are first mapped to:
#
#     shard
#     +
#     local row
#
# Results from the same shard are then grouped so that each
# JSONL shard is scanned only once.
#
# Non-JSONL formats retain the single-row disk-backed fallback
# established in Cell 1A.
# ============================================================

def resolve_rag_metadata(
    vector_ids,
):

    vector_ids = [
        int(vector_id)
        for vector_id
        in vector_ids
    ]


    locations = [

        locate_metadata_row(
            vector_id
        )

        for vector_id
        in vector_ids

    ]


    grouped_locations = defaultdict(
        list
    )


    for location in locations:

        grouped_locations[
            str(
                location[
                    "path"
                ]
            )
        ].append(
            location
        )


    records_by_vector_id = {}


    for path_string, group in grouped_locations.items():

        path = Path(
            path_string
        )


        suffix = (
            path.suffix.lower()
        )


        if suffix == ".jsonl":

            local_rows = [

                item[
                    "local_row"
                ]

                for item
                in group

            ]


            resolved_rows = (
                read_jsonl_rows(
                    path,
                    local_rows,
                )
            )


            for item in group:

                vector_id = (
                    item[
                        "vector_id"
                    ]
                )


                local_row = (
                    item[
                        "local_row"
                    ]
                )


                records_by_vector_id[
                    vector_id
                ] = (
                    resolved_rows[
                        local_row
                    ]
                )


        else:

            # Deployment-safe fallback for supported
            # Parquet / CSV / JSON formats.
            #
            # Top-K is small, so this remains bounded and does
            # not materialize the complete metadata corpus.

            for item in group:

                vector_id = (
                    item[
                        "vector_id"
                    ]
                )


                records_by_vector_id[
                    vector_id
                ] = (
                    read_disk_row(
                        item
                    )
                )


    return [

        records_by_vector_id[
            vector_id
        ]

        for vector_id
        in vector_ids

    ]


# ============================================================
# 1D.7 — METADATA FIELD HELPER
# ============================================================
#
# The reconciled corpus contains heterogeneous source metadata.
#
# Use the first available field from common naming variants
# without changing the frozen metadata itself.
# ============================================================

def first_metadata_value(
    record,
    candidates,
    default="",
):

    for field in candidates:

        value = (
            record.get(
                field
            )
        )


        if value is None:

            continue


        value = str(
            value
        ).strip()


        if value:

            return value


    return default


# ============================================================
# 1D.8 — NORMALIZE RETRIEVAL RESULT
# ============================================================

def normalize_rag_result(
    rank,
    vector_id,
    score,
    record,
):

    text = str(
        record.get(
            RAG_TEXT_COLUMN,
            ""
        )
        or
        ""
    ).strip()


    source = (
        first_metadata_value(
            record,
            [
                "source",
                "source_name",
                "organization",
                "publisher",
                "corpus",
                "dataset",
            ],
            default="Unknown",
        )
    )


    title = (
        first_metadata_value(
            record,
            [
                "title",
                "document_title",
                "doc_title",
                "file_name",
                "filename",
                "document",
            ],
            default="",
        )
    )


    return {

        "rank":
            int(
                rank
            ),

        "vector_id":
            int(
                vector_id
            ),

        "score":
            float(
                score
            ),

        "source":
            source,

        "title":
            title,

        "text":
            text,

        "text_chars":
            len(
                text
            ),

        "metadata":
            record,

    }


# ============================================================
# 1D.9 — LIVE SEMANTIC RETRIEVAL FUNCTION
# ============================================================
#
# This becomes the reusable deployment RAG retrieval function
# for later RAG_ONLY and HYBRID execution modes.
# ============================================================

def retrieve_rag(
    query,
    top_k=MODULE4_RAG_TOP_K,
):

    if not isinstance(
        query,
        str,
    ):

        query = str(
            query
        )


    query = (
        query.strip()
    )


    if not query:

        raise ValueError(
            "Query cannot be empty."
        )


    top_k = int(
        top_k
    )


    if top_k <= 0:

        raise ValueError(
            "top_k must be greater than zero."
        )


    total_start = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # STEP 1 — QUERY ENCODING
    # --------------------------------------------------------

    query_embedding, embedding_time_s = (
        embed_rag_query(
            query
        )
    )


    # --------------------------------------------------------
    # STEP 2 — FAISS SEARCH
    # --------------------------------------------------------

    faiss_start = (
        time.perf_counter()
    )


    scores, vector_ids = (
        rag_faiss_index.search(
            query_embedding,
            top_k,
        )
    )


    faiss_time_s = (
        time.perf_counter()
        -
        faiss_start
    )


    scores = (
        scores[
            0
        ]
    )


    vector_ids = (
        vector_ids[
            0
        ]
    )


    valid_pairs = [

        (
            float(score),
            int(vector_id),
        )

        for score, vector_id
        in zip(
            scores,
            vector_ids,
        )

        if int(vector_id) >= 0

    ]


    if not valid_pairs:

        raise RuntimeError(
            "FAISS returned no valid retrieval results."
        )


    valid_scores = [

        score

        for score, _
        in valid_pairs

    ]


    valid_vector_ids = [

        vector_id

        for _, vector_id
        in valid_pairs

    ]


    # --------------------------------------------------------
    # STEP 3 — ON-DEMAND METADATA RESOLUTION
    # --------------------------------------------------------

    metadata_start = (
        time.perf_counter()
    )


    metadata_records = (
        resolve_rag_metadata(
            valid_vector_ids
        )
    )


    metadata_time_s = (
        time.perf_counter()
        -
        metadata_start
    )


    # --------------------------------------------------------
    # STEP 4 — NORMALIZE RESULTS
    # --------------------------------------------------------

    results = [

        normalize_rag_result(
            rank=rank,
            vector_id=vector_id,
            score=score,
            record=record,
        )

        for rank, (
            score,
            vector_id,
            record,
        )
        in enumerate(
            zip(
                valid_scores,
                valid_vector_ids,
                metadata_records,
            ),
            start=1,
        )

    ]


    total_time_s = (
        time.perf_counter()
        -
        total_start
    )


    return {

        "query":
            query,

        "top_k":
            top_k,

        "retrieved":
            len(
                results
            ),

        "results":
            results,

        "timing": {

            "embedding_s":
                float(
                    embedding_time_s
                ),

            "faiss_search_s":
                float(
                    faiss_time_s
                ),

            "metadata_resolution_s":
                float(
                    metadata_time_s
                ),

            "total_s":
                float(
                    total_time_s
                ),

        },

    }


# ============================================================
# 1D.10 — PRE-RETRIEVAL MEMORY
# ============================================================

gc.collect()


RAG_LIVE_RAM_BEFORE_GIB = (
    current_process_rss_gib()
)


RAG_LIVE_AVAILABLE_BEFORE_GIB = (
    available_system_ram_gib()
)


# ============================================================
# 1D.11 — LIVE RETRIEVAL TEST
# ============================================================

TEST_QUERY = (
    "What are the primary responsibilities "
    "of the AMF in a 5G Standalone network?"
)


print("=" * 116)
print("MODULE 4 DEMO — LIVE SEMANTIC RAG RETRIEVAL")
print("=" * 116)


print(
    f"Query                      : "
    f"{TEST_QUERY}"
)


print(
    f"Top-K                      : "
    f"{MODULE4_RAG_TOP_K}"
)


print(
    f"Encoder Device             : "
    f"{rag_query_encoder.device}"
)


print(
    f"FAISS Vectors              : "
    f"{rag_faiss_index.ntotal:,}"
)


print(
    "Metadata Strategy          : "
    "SHARD_MANIFEST_ON_DEMAND"
)


print()


rag_live_result = (
    retrieve_rag(
        TEST_QUERY,
        top_k=MODULE4_RAG_TOP_K,
    )
)


# ============================================================
# 1D.12 — POST-RETRIEVAL MEMORY
# ============================================================

gc.collect()


RAG_LIVE_RAM_AFTER_GIB = (
    current_process_rss_gib()
)


RAG_LIVE_AVAILABLE_AFTER_GIB = (
    available_system_ram_gib()
)


RAG_LIVE_RAM_DELTA_GIB = (
    RAG_LIVE_RAM_AFTER_GIB
    -
    RAG_LIVE_RAM_BEFORE_GIB
)


# ============================================================
# 1D.13 — VALIDATE RETRIEVAL RESULTS
# ============================================================

retrieved_results = (
    rag_live_result[
        "results"
    ]
)


RAG_RESULT_COUNT_PASS = (
    len(
        retrieved_results
    )
    ==
    MODULE4_RAG_TOP_K
)


RAG_VECTOR_ID_PASS = all(

    0
    <=
    result[
        "vector_id"
    ]
    <
    EXPECTED_RAG_VECTORS

    for result
    in retrieved_results

)


RAG_SCORE_PASS = all(

    np.isfinite(
        result[
            "score"
        ]
    )

    for result
    in retrieved_results

)


RAG_TEXT_PASS = all(

    bool(
        result[
            "text"
        ].strip()
    )

    for result
    in retrieved_results

)


RAG_RANK_PASS = (

    [
        result[
            "rank"
        ]

        for result
        in retrieved_results

    ]

    ==
    list(
        range(
            1,
            len(
                retrieved_results
            )
            +
            1,
        )
    )

)


RAG_SCORE_ORDER_PASS = all(

    retrieved_results[
        index
    ][
        "score"
    ]
    >=
    retrieved_results[
        index
        +
        1
    ][
        "score"
    ]

    for index
    in range(
        len(
            retrieved_results
        )
        -
        1
    )

)


MODULE4_RAG_LIVE_RETRIEVAL_PASS = all(
    [
        RAG_RESULT_COUNT_PASS,
        RAG_VECTOR_ID_PASS,
        RAG_SCORE_PASS,
        RAG_TEXT_PASS,
        RAG_RANK_PASS,
        RAG_SCORE_ORDER_PASS,
    ]
)


# ============================================================
# 1D.14 — COMPACT RETRIEVAL RESULT TABLE
# ============================================================
#
# Display only short snippets.
#
# Full retrieved text remains available inside:
#
#     rag_live_result["results"]
#
# This prevents long corpus samples from cluttering the
# notebook output.
# ============================================================

display_rows = []


for result in retrieved_results:

    clean_text = " ".join(
        result[
            "text"
        ].split()
    )


    snippet = (
        clean_text[
            :MODULE4_RAG_SNIPPET_CHARS
        ]
    )


    if (
        len(
            clean_text
        )
        >
        MODULE4_RAG_SNIPPET_CHARS
    ):

        snippet += "..."


    display_rows.append(
        {

            "rank":
                result[
                    "rank"
                ],

            "score":
                round(
                    result[
                        "score"
                    ],
                    6,
                ),

            "vector_id":
                result[
                    "vector_id"
                ],

            "source":
                result[
                    "source"
                ],

            "title":
                result[
                    "title"
                ],

            "text_chars":
                result[
                    "text_chars"
                ],

            "snippet":
                snippet,

        }
    )


RAG_LIVE_RESULTS_DF = (
    pd.DataFrame(
        display_rows
    )
)


print("TOP-K RETRIEVAL RESULTS")
print("-" * 116)


print(
    RAG_LIVE_RESULTS_DF
    .to_string(
        index=False
    )
)


# ============================================================
# 1D.15 — LATENCY SUMMARY
# ============================================================

timing = (
    rag_live_result[
        "timing"
    ]
)


print()
print("LATENCY")
print("-" * 116)


print(
    f"Query Encoding             : "
    f"{timing['embedding_s']:.4f} s"
)


print(
    f"FAISS Search               : "
    f"{timing['faiss_search_s']:.4f} s"
)


print(
    f"Metadata Resolution        : "
    f"{timing['metadata_resolution_s']:.4f} s"
)


print(
    f"Total Retrieval            : "
    f"{timing['total_s']:.4f} s"
)


# ============================================================
# 1D.16 — MEMORY SUMMARY
# ============================================================

print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before         : "
    f"{RAG_LIVE_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After          : "
    f"{RAG_LIVE_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"Retrieval RAM Delta        : "
    f"{RAG_LIVE_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available       : "
    f"{RAG_LIVE_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print(
    "Full Metadata in RAM      : "
    "False"
)


# ============================================================
# 1D.17 — FORMAL RUNTIME CONFIG
# ============================================================

MODULE4_RAG_LIVE_CONFIG = {

    "retriever_version":
        RAG_RETRIEVER_VERSION,

    "embedding_model":
        RAG_MODEL_NAME,

    "embedding_device":
        "cpu",

    "faiss_device":
        "cpu",

    "faiss_vectors":
        int(
            rag_faiss_index.ntotal
        ),

    "dimension":
        int(
            rag_faiss_index.d
        ),

    "top_k":
        MODULE4_RAG_TOP_K,

    "metadata_strategy":
        "SHARD_MANIFEST_ON_DEMAND",

    "full_metadata_dataframe":
        False,

    "embedding_latency_s":
        timing[
            "embedding_s"
        ],

    "faiss_latency_s":
        timing[
            "faiss_search_s"
        ],

    "metadata_latency_s":
        timing[
            "metadata_resolution_s"
        ],

    "total_retrieval_latency_s":
        timing[
            "total_s"
        ],

    "process_ram_gib":
        RAG_LIVE_RAM_AFTER_GIB,

    "available_ram_gib":
        RAG_LIVE_AVAILABLE_AFTER_GIB,

}


# ============================================================
# 1D.18 — FINAL VALIDATION
# ============================================================

MODULE4_CELL1D_PASS = (
    MODULE4_RAG_LIVE_RETRIEVAL_PASS
)


if not MODULE4_CELL1D_PASS:

    raise RuntimeError(
        "Cell 1D live Semantic RAG retrieval validation failed."
    )


# ============================================================
# 1D.19 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — LIVE SEMANTIC RAG SUMMARY")
print("=" * 116)


print(
    f"Query                      : "
    f"{TEST_QUERY}"
)


print(
    f"Retrieved Evidence         : "
    f"{len(retrieved_results)}"
)


print(
    f"Top-K                      : "
    f"{MODULE4_RAG_TOP_K}"
)


print(
    f"Embedding Dimension        : "
    f"{EXPECTED_RAG_DIMENSION}"
)


print(
    f"FAISS Corpus               : "
    f"{rag_faiss_index.ntotal:,} vectors"
)


print(
    f"Metadata Corpus            : "
    f"{RAG_METADATA_ROW_COUNT:,} rows"
)


print(
    "Metadata Runtime          : "
    "Disk-backed / on-demand"
)


print(
    f"Embedding Latency          : "
    f"{timing['embedding_s']:.4f} s"
)


print(
    f"FAISS Search Latency       : "
    f"{timing['faiss_search_s']:.4f} s"
)


print(
    f"Metadata Resolution        : "
    f"{timing['metadata_resolution_s']:.4f} s"
)


print(
    f"End-to-End Retrieval       : "
    f"{timing['total_s']:.4f} s"
)


print(
    f"Retrieval RAM Delta        : "
    f"{RAG_LIVE_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"Available RAM After        : "
    f"{RAG_LIVE_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Top-K Count PASS           : "
    f"{RAG_RESULT_COUNT_PASS}"
)


print(
    f"Vector IDs PASS            : "
    f"{RAG_VECTOR_ID_PASS}"
)


print(
    f"Similarity Scores PASS     : "
    f"{RAG_SCORE_PASS}"
)


print(
    f"Score Ordering PASS        : "
    f"{RAG_SCORE_ORDER_PASS}"
)


print(
    f"Retrieved Text PASS        : "
    f"{RAG_TEXT_PASS}"
)


print(
    f"Live Retrieval PASS        : "
    f"{MODULE4_RAG_LIVE_RETRIEVAL_PASS}"
)


print()
print("✓ CELL 1D PASSED")
print("✓ BGE-M3 query encoding operational on CPU.")
print("✓ Full 1,506,367-vector FAISS search operational.")
print("✓ Top-k semantic evidence retrieved successfully.")
print("✓ Metadata resolved directly from disk.")
print("✓ Full metadata DataFrame remains outside RAM.")
print("✓ Retrieval latency measured by pipeline stage.")
print("✓ Retrieval memory impact measured.")
print("✓ Semantic RAG V1 deployment runtime validated.")
print("✓ Ready for the MCP Version A retrieval stage.")

print("=" * 116)


MODULE 4 DEMO — LIVE SEMANTIC RAG RETRIEVAL
Query                      : What are the primary responsibilities of the AMF in a 5G Standalone network?
Top-K                      : 5
Encoder Device             : cpu
FAISS Vectors              : 1,506,367
Metadata Strategy          : SHARD_MANIFEST_ON_DEMAND

TOP-K RETRIEVAL RESULTS
--------------------------------------------------------------------------------------------------------------------
 rank    score  vector_id    source     title  text_chars                                                                                                                                                                                                                         snippet
    1 0.706518    1033100 standards    rel_15        2840 the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), t

##### Cell 1D — Observation

Live top-5 Semantic RAG retrieval passed. In the final 5G validation run, the selected `RAG_ONLY` search returned **5 candidates / 5 presented evidence items** with approximately **0.65 s** selected retrieval wall time.


# SECTION 2 — MCP Version A Restoration


#### Cell 2A — Remote Knowledge Source Configuration

**Description:** Restore MCP Version A source discovery for GSMA TCC and dedicated 3GPP while keeping DuckDB in memory and avoiding a persistent local KB.


In [10]:
# ============================================================
# CELL 2A — CONFIGURE MCP VERSION A REMOTE KNOWLEDGE SOURCES
# ============================================================
#
# Purpose
# -------
#
# Restore the frozen Module 3 Version A knowledge-source
# configuration for the deployment runtime.
#
# Version A knowledge sources:
#
#     GSMA Telco Common Corpus
#         → remote Parquet
#         → DuckDB HTTPFS at query time
#
#     GSMA 3GPP Corpus
#         → remote marked/**/raw.md
#         → direct HTTP access at query time
#
# IMPORTANT:
#
# - No telecom corpus is downloaded.
# - No local BM25 index is created.
# - No persistent DuckDB knowledge base is restored.
# - The ~69 GB Version B artifact remains excluded.
#
# This cell performs source discovery and configuration only.
# Actual remote retrieval is validated in Cell 2B.
# ============================================================


# ============================================================
# 2A.1 — VERIFY RAG RUNTIME
# ============================================================

if (
    "MODULE4_CELL1D_PASS"
    not in globals()
    or
    not MODULE4_CELL1D_PASS
):

    raise RuntimeError(
        "Cell 1D must pass before Cell 2A."
    )


# ============================================================
# 2A.2 — IMPORTS
# ============================================================

import gc
import os
import time

import psutil

from huggingface_hub import HfApi
from tqdm.notebook import tqdm


# ============================================================
# 2A.3 — VERSION A ARCHITECTURE
# ============================================================

ARCHITECTURE_VERSION = (
    "Version A"
)


RETRIEVAL_ARCHITECTURE = (
    "Remote raw-corpus retrieval"
)


# ============================================================
# 2A.4 — REMOTE TELECOM CORPORA
# ============================================================
#
# Preserve the original Version A source scope.
# ============================================================

REMOTE_CORPORA = {

    "tcc": {

        "repo_id":
            "GSMA/Telco-Common-Corpus",

        "repo_type":
            "dataset",

        "source_family":
            "TCC",

        "file_pattern":
            "data/*.parquet",

        "format":
            "parquet",
    },

    "3gpp": {

        "repo_id":
            "GSMA/3GPP",

        "repo_type":
            "dataset",

        "source_family":
            "3GPP",

        "file_pattern":
            "marked/**/raw.md",

        "format":
            "markdown",
    },

}


# ============================================================
# 2A.5 — FROZEN VERSION A RETRIEVAL CONTROLS
# ============================================================

MAX_REMOTE_SHARDS = (
    5
)


PARALLEL_SHARD_CONCURRENCY = (
    5
)


TOP_K_RESULTS = (
    5
)


RETRIEVAL_SCORING = (
    "lexical + phrase + proximity"
)


MAX_MCP_SEARCHES = (
    3
)


MAX_RETRIEVED_SOURCES = (
    5
)


MCP_EXCERPT_CHARS = (
    2500
)


# ============================================================
# 2A.6 — DUCKDB RUNTIME POLICY
# ============================================================
#
# The original Version A used os.cpu_count().
#
# On this CPU runtime this resolves naturally to the available
# logical processor count rather than retaining the previous
# L4-host CPU configuration.
# ============================================================

DUCKDB_THREADS = (
    os.cpu_count()
    or
    1
)


DUCKDB_DATABASE = (
    ":memory:"
)


# ============================================================
# 2A.7 — HISTORICAL SOURCE-COUNT REFERENCE
# ============================================================
#
# These are reference values observed in the frozen Version A
# experiment.
#
# Because Version A accesses live remote repositories, source
# counts are reported rather than forced to remain identical.
# ============================================================

VERSION_A_REFERENCE_TCC_FILES = (
    100
)


VERSION_A_REFERENCE_3GPP_FILES = (
    15_052
)


# ============================================================
# 2A.8 — MEMORY BASELINE
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def process_ram_gib():

    return (
        psutil.Process(
            os.getpid()
        )
        .memory_info()
        .rss
        /
        BYTES_PER_GIB
    )


def available_ram_gib():

    return (
        psutil.virtual_memory()
        .available
        /
        BYTES_PER_GIB
    )


gc.collect()


MCP_CONFIG_RAM_BEFORE_GIB = (
    process_ram_gib()
)


MCP_CONFIG_AVAILABLE_BEFORE_GIB = (
    available_ram_gib()
)


# ============================================================
# 2A.9 — HUGGING FACE API
# ============================================================

hf_api = HfApi(
    token=HF_TOKEN
)


# ============================================================
# 2A.10 — REMOTE REGISTRY CONTAINERS
# ============================================================

REMOTE_CORPUS_FILES = {

    "tcc": [],
    "3gpp": [],

}


REMOTE_CORPUS_URLS = {

    "tcc": [],
    "3gpp": [],

}


# ============================================================
# 2A.11 — DIRECT HUGGING FACE URL HELPER
# ============================================================

def build_hf_resolve_url(
    repo_id,
    file_path,
):

    return (
        "https://huggingface.co/datasets/"
        f"{repo_id}/resolve/main/"
        f"{file_path}"
    )


# ============================================================
# 2A.12 — FILE-SCOPE HELPERS
# ============================================================

def is_tcc_runtime_file(
    file_path,
):

    file_path = str(
        file_path
    )


    return (

        file_path.startswith(
            "data/"
        )

        and

        file_path.endswith(
            ".parquet"
        )

        and

        file_path.count("/")
        ==
        1
    )


def is_3gpp_runtime_file(
    file_path,
):

    file_path = str(
        file_path
    )


    return (

        file_path.startswith(
            "marked/"
        )

        and

        file_path.endswith(
            "/raw.md"
        )
    )


# ============================================================
# 2A.13 — CONTROLLED REMOTE SOURCE DISCOVERY
# ============================================================
#
# Keep notebook output compact.
#
# One progress bar represents the two remote repositories.
# No individual source filenames are printed.
# ============================================================

source_progress = tqdm(
    total=2,
    desc="Version A Source Discovery",
    unit="source",
    leave=True,
)


source_discovery_start = (
    time.perf_counter()
)


# ------------------------------------------------------------
# TCC
# ------------------------------------------------------------

source_progress.set_postfix_str(
    "GSMA Telco Common Corpus"
)


tcc_repo_files = (
    hf_api.list_repo_files(
        repo_id=
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_id"
            ],

        repo_type=
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_type"
            ],
    )
)


tcc_files = sorted([

    file_path

    for file_path
    in tcc_repo_files

    if is_tcc_runtime_file(
        file_path
    )

])


if not tcc_files:

    source_progress.close()

    raise RuntimeError(
        "No Version A TCC runtime Parquet files "
        "were discovered."
    )


tcc_urls = [

    build_hf_resolve_url(
        REMOTE_CORPORA[
            "tcc"
        ][
            "repo_id"
        ],
        file_path,
    )

    for file_path
    in tcc_files

]


REMOTE_CORPUS_FILES[
    "tcc"
] = tcc_files


REMOTE_CORPUS_URLS[
    "tcc"
] = tcc_urls


source_progress.update(
    1
)


# ------------------------------------------------------------
# DEDICATED 3GPP
# ------------------------------------------------------------

source_progress.set_postfix_str(
    "GSMA 3GPP"
)


gpp_repo_files = (
    hf_api.list_repo_files(
        repo_id=
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_id"
            ],

        repo_type=
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_type"
            ],
    )
)


gpp_files = sorted([

    file_path

    for file_path
    in gpp_repo_files

    if is_3gpp_runtime_file(
        file_path
    )

])


if not gpp_files:

    source_progress.close()

    raise RuntimeError(
        "No dedicated Version A 3GPP raw.md files "
        "were discovered."
    )


gpp_urls = [

    build_hf_resolve_url(
        REMOTE_CORPORA[
            "3gpp"
        ][
            "repo_id"
        ],
        file_path,
    )

    for file_path
    in gpp_files

]


REMOTE_CORPUS_FILES[
    "3gpp"
] = gpp_files


REMOTE_CORPUS_URLS[
    "3gpp"
] = gpp_urls


source_progress.update(
    1
)


source_progress.set_postfix_str(
    "Complete"
)

source_progress.close()


SOURCE_DISCOVERY_TIME_S = (
    time.perf_counter()
    -
    source_discovery_start
)


# ============================================================
# 2A.14 — GLOBAL REMOTE SOURCE REGISTRY
# ============================================================

REMOTE_SOURCE_REGISTRY = {

    "tcc": {

        "repo_id":
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_id"
            ],

        "source_family":
            "TCC",

        "format":
            "parquet",

        "file_count":
            len(
                tcc_files
            ),

        "files":
            tcc_files,

        "urls":
            tcc_urls,
    },


    "3gpp": {

        "repo_id":
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_id"
            ],

        "source_family":
            "3GPP",

        "format":
            "markdown",

        "file_count":
            len(
                gpp_files
            ),

        "files":
            gpp_files,

        "urls":
            gpp_urls,
    },

}


# ============================================================
# 2A.15 — VALIDATE SOURCE REGISTRY
# ============================================================

MCP_TCC_SOURCE_PASS = (

    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "file_count"
    ]
    >
    0
)


MCP_3GPP_SOURCE_PASS = (

    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "file_count"
    ]
    >
    0
)


MCP_REMOTE_REGISTRY_PASS = all(
    [
        len(
            REMOTE_SOURCE_REGISTRY
        )
        ==
        2,

        MCP_TCC_SOURCE_PASS,

        MCP_3GPP_SOURCE_PASS,
    ]
)


if not MCP_REMOTE_REGISTRY_PASS:

    raise RuntimeError(
        "MCP Version A remote source registry "
        "validation failed."
    )


# ============================================================
# 2A.16 — REFERENCE-COUNT COMPARISON
# ============================================================

TCC_REFERENCE_MATCH = (

    len(
        tcc_files
    )
    ==
    VERSION_A_REFERENCE_TCC_FILES
)


GPP_REFERENCE_MATCH = (

    len(
        gpp_files
    )
    ==
    VERSION_A_REFERENCE_3GPP_FILES
)


# ============================================================
# 2A.17 — MEMORY AFTER CONFIGURATION
# ============================================================

gc.collect()


MCP_CONFIG_RAM_AFTER_GIB = (
    process_ram_gib()
)


MCP_CONFIG_AVAILABLE_AFTER_GIB = (
    available_ram_gib()
)


MCP_CONFIG_RAM_DELTA_GIB = (

    MCP_CONFIG_RAM_AFTER_GIB
    -
    MCP_CONFIG_RAM_BEFORE_GIB
)


# ============================================================
# 2A.18 — FORMAL CONFIGURATION
# ============================================================

MODULE4_MCP_VERSION_A_CONFIG = {

    "architecture":
        ARCHITECTURE_VERSION,

    "retrieval":
        RETRIEVAL_ARCHITECTURE,

    "tcc_repo":
        REMOTE_CORPORA[
            "tcc"
        ][
            "repo_id"
        ],

    "tcc_files":
        len(
            tcc_files
        ),

    "gpp_repo":
        REMOTE_CORPORA[
            "3gpp"
        ][
            "repo_id"
        ],

    "gpp_files":
        len(
            gpp_files
        ),

    "duckdb_database":
        DUCKDB_DATABASE,

    "duckdb_threads":
        DUCKDB_THREADS,

    "max_remote_shards":
        MAX_REMOTE_SHARDS,

    "parallel_shard_concurrency":
        PARALLEL_SHARD_CONCURRENCY,

    "top_k":
        TOP_K_RESULTS,

    "retrieval_scoring":
        RETRIEVAL_SCORING,

    "persistent_local_kb":
        False,

}


# ============================================================
# 2A.19 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — MCP VERSION A REMOTE SOURCE CONFIGURATION")
print("=" * 116)


print(
    f"Architecture                : "
    f"{ARCHITECTURE_VERSION}"
)


print(
    f"Retrieval Architecture      : "
    f"{RETRIEVAL_ARCHITECTURE}"
)


print()
print("REMOTE KNOWLEDGE SOURCES")
print("-" * 116)


print(
    f"TCC Repository              : "
    f"{REMOTE_CORPORA['tcc']['repo_id']}"
)


print(
    f"TCC Runtime Files           : "
    f"{len(tcc_files):,}"
)


print(
    f"TCC Historical Reference    : "
    f"{VERSION_A_REFERENCE_TCC_FILES:,}"
)


print(
    f"TCC Reference Match         : "
    f"{TCC_REFERENCE_MATCH}"
)


print()


print(
    f"3GPP Repository             : "
    f"{REMOTE_CORPORA['3gpp']['repo_id']}"
)


print(
    f"3GPP Runtime Files          : "
    f"{len(gpp_files):,}"
)


print(
    f"3GPP Historical Reference   : "
    f"{VERSION_A_REFERENCE_3GPP_FILES:,}"
)


print(
    f"3GPP Reference Match        : "
    f"{GPP_REFERENCE_MATCH}"
)


print()
print("RETRIEVAL POLICY")
print("-" * 116)


print(
    f"DuckDB Database             : "
    f"{DUCKDB_DATABASE}"
)


print(
    f"DuckDB Threads              : "
    f"{DUCKDB_THREADS}"
)


print(
    f"Scoring                     : "
    f"{RETRIEVAL_SCORING}"
)


print(
    f"Max Remote TCC Shards       : "
    f"{MAX_REMOTE_SHARDS}"
)


print(
    f"Parallel Shard Calls        : "
    f"{PARALLEL_SHARD_CONCURRENCY}"
)


print(
    f"Top-K                       : "
    f"{TOP_K_RESULTS}"
)


print(
    f"Persistent Local KB         : "
    f"False"
)


print()
print("RUNTIME IMPACT")
print("-" * 116)


print(
    f"Source Discovery Time       : "
    f"{SOURCE_DISCOVERY_TIME_S:.2f} s"
)


print(
    f"Process RAM Before           : "
    f"{MCP_CONFIG_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{MCP_CONFIG_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{MCP_CONFIG_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{MCP_CONFIG_AVAILABLE_AFTER_GIB:.3f} GiB"
)


# ============================================================
# 2A.20 — FINAL VALIDATION
# ============================================================

MODULE4_CELL2A_PASS = (
    MCP_REMOTE_REGISTRY_PASS
)


if not MODULE4_CELL2A_PASS:

    raise RuntimeError(
        "Cell 2A MCP Version A configuration failed."
    )


print()
print("✓ CELL 2A PASSED")
print("✓ MCP Version A architecture restored.")
print("✓ TCC remote source registry created.")
print("✓ Dedicated 3GPP remote source registry created.")
print("✓ No telecom source corpus downloaded.")
print("✓ No persistent knowledge index created.")
print("✓ Version B 69 GB knowledge base remains excluded.")
print("✓ Full Semantic RAG remains resident and unchanged.")
print("✓ Ready for Cell 2B — remote TCC + 3GPP access validation.")

print("=" * 116)


Version A Source Discovery:   0%|          | 0/2 [00:00<?, ?source/s]


MODULE 4 DEMO — MCP VERSION A REMOTE SOURCE CONFIGURATION
Architecture                : Version A
Retrieval Architecture      : Remote raw-corpus retrieval

REMOTE KNOWLEDGE SOURCES
--------------------------------------------------------------------------------------------------------------------
TCC Repository              : GSMA/Telco-Common-Corpus
TCC Runtime Files           : 100
TCC Historical Reference    : 100
TCC Reference Match         : True

3GPP Repository             : GSMA/3GPP
3GPP Runtime Files          : 15,052
3GPP Historical Reference   : 15,052
3GPP Reference Match        : True

RETRIEVAL POLICY
--------------------------------------------------------------------------------------------------------------------
DuckDB Database             : :memory:
DuckDB Threads              : 8
Scoring                     : lexical + phrase + proximity
Max Remote TCC Shards       : 5
Parallel Shard Calls        : 5
Top-K                       : 5
Persistent Local KB         : F

##### Cell 2A — Observation

Remote MCP source configuration passed with **100 TCC Parquet shards** and **15,052 dedicated 3GPP `raw.md` files**. This preserves broad telecom knowledge access with negligible local storage growth.


#### Cell 2B — Remote Source Access Validation

**Description:** Verify live TCC access through DuckDB HTTPFS and direct HTTP access to dedicated 3GPP documents.


In [11]:
# ============================================================
# CELL 2B — MCP VERSION A REMOTE ACCESS VALIDATION
# ============================================================
#
# Purpose
# -------
#
# Validate the two remote knowledge-access paths used by
# MCP Version A:
#
#     TCC
#       → Remote Parquet
#       → DuckDB HTTPFS
#
#     3GPP
#       → Remote marked/**/raw.md
#       → Direct HTTP
#
# This is ACCESS VALIDATION only.
#
# No routing, ranking, MCP tool execution or LLM generation
# occurs in this cell.
# ============================================================


# ============================================================
# 2B.1 — VERIFY CELL 2A
# ============================================================

if (
    "MODULE4_CELL2A_PASS"
    not in globals()
    or
    not MODULE4_CELL2A_PASS
):

    raise RuntimeError(
        "Cell 2A must pass before Cell 2B."
    )


# ============================================================
# 2B.2 — IMPORTS
# ============================================================

import gc
import os
import time
import urllib.request

import duckdb
import psutil


# ============================================================
# 2B.3 — VALIDATION CONFIGURATION
# ============================================================

TCC_VALIDATION_URL = (
    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "urls"
    ][0]
)


GPP_VALIDATION_URL = (
    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "urls"
    ][0]
)


GPP_VALIDATION_BYTES = (
    8192
)


REMOTE_ACCESS_TIMEOUT_S = (
    30
)


# ============================================================
# 2B.4 — MEMORY HELPERS
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


def current_process_ram_gib():

    return (
        psutil.Process(
            os.getpid()
        )
        .memory_info()
        .rss
        /
        BYTES_PER_GIB
    )


def available_system_ram_gib():

    return (
        psutil.virtual_memory()
        .available
        /
        BYTES_PER_GIB
    )


gc.collect()


MCP_ACCESS_RAM_BEFORE_GIB = (
    current_process_ram_gib()
)


MCP_ACCESS_AVAILABLE_BEFORE_GIB = (
    available_system_ram_gib()
)


# ============================================================
# 2B.5 — START VALIDATION
# ============================================================

print("=" * 116)
print("MODULE 4 DEMO — MCP VERSION A REMOTE ACCESS VALIDATION")
print("=" * 116)

print(
    f"TCC Repository              : "
    f"{REMOTE_CORPORA['tcc']['repo_id']}"
)

print(
    f"3GPP Repository             : "
    f"{REMOTE_CORPORA['3gpp']['repo_id']}"
)

print(
    f"DuckDB Threads              : "
    f"{DUCKDB_THREADS}"
)

print()


# ============================================================
# 2B.6 — TCC REMOTE PARQUET ACCESS
# ============================================================

print("[1/2] Validating TCC remote Parquet via DuckDB HTTPFS...")


tcc_start = (
    time.perf_counter()
)


tcc_conn = duckdb.connect(
    DUCKDB_DATABASE
)


try:

    # --------------------------------------------------------
    # RUNTIME CONFIG
    # --------------------------------------------------------

    tcc_conn.execute(
        f"SET threads = {int(DUCKDB_THREADS)}"
    )


    # --------------------------------------------------------
    # HTTPFS
    # --------------------------------------------------------
    #
    # LOAD first because the extension may already exist.
    # Install only if required.
    # --------------------------------------------------------

    try:

        tcc_conn.execute(
            "LOAD httpfs"
        )

    except Exception:

        tcc_conn.execute(
            "INSTALL httpfs"
        )

        tcc_conn.execute(
            "LOAD httpfs"
        )


    # --------------------------------------------------------
    # REMOTE SCHEMA VALIDATION
    # --------------------------------------------------------

    tcc_schema = (
        tcc_conn.execute(
            """
            DESCRIBE
            SELECT *
            FROM read_parquet(?)
            """,
            [
                TCC_VALIDATION_URL
            ],
        )
        .df()
    )


    if tcc_schema.empty:

        raise RuntimeError(
            "Remote TCC Parquet schema could not be read."
        )


    tcc_columns = (
        tcc_schema[
            "column_name"
        ]
        .astype(str)
        .tolist()
    )


    if (
        "text"
        not in tcc_columns
    ):

        raise RuntimeError(
            "Expected 'text' column is missing "
            "from the TCC Parquet schema."
        )


    # --------------------------------------------------------
    # SMALL BOUNDED SAMPLE
    # --------------------------------------------------------
    #
    # LIMIT 3 ensures this remains an access test rather than
    # a real corpus retrieval.
    # --------------------------------------------------------

    tcc_sample = (
        tcc_conn.execute(
            """
            SELECT
                collection,
                identifier,
                title,
                LEFT(
                    CAST(text AS VARCHAR),
                    300
                ) AS text_preview

            FROM read_parquet(?)

            WHERE text IS NOT NULL

            LIMIT 3
            """,
            [
                TCC_VALIDATION_URL
            ],
        )
        .df()
    )


    if tcc_sample.empty:

        raise RuntimeError(
            "TCC remote source was reachable "
            "but returned no validation rows."
        )


finally:

    tcc_conn.close()


TCC_ACCESS_TIME_S = (
    time.perf_counter()
    -
    tcc_start
)


TCC_REMOTE_ACCESS_PASS = (
    not tcc_sample.empty
    and
    "text" in tcc_columns
)


print(
    f"      Columns detected      : "
    f"{len(tcc_columns)}"
)

print(
    f"      Sample rows           : "
    f"{len(tcc_sample)}"
)

print(
    f"      Access latency        : "
    f"{TCC_ACCESS_TIME_S:.3f} s"
)

print(
    f"      Status                : "
    f"{'PASS' if TCC_REMOTE_ACCESS_PASS else 'FAIL'}"
)


# ============================================================
# 2B.7 — DEDICATED 3GPP REMOTE ACCESS
# ============================================================

print()
print("[2/2] Validating dedicated 3GPP raw.md via direct HTTP...")


gpp_start = (
    time.perf_counter()
)


gpp_request = urllib.request.Request(

    GPP_VALIDATION_URL,

    headers={

        "Range":
            f"bytes=0-{GPP_VALIDATION_BYTES - 1}",

        "User-Agent":
            "Telecom-AI-MCP-Version-A-Demo",

    },

)


try:

    with urllib.request.urlopen(
        gpp_request,
        timeout=REMOTE_ACCESS_TIMEOUT_S,
    ) as response:

        gpp_raw = (
            response.read(
                GPP_VALIDATION_BYTES
            )
        )


        gpp_http_status = (
            response.status
        )


        gpp_content_type = (
            response.headers.get(
                "Content-Type",
                "",
            )
        )


except Exception as exc:

    raise RuntimeError(
        "Unable to access the dedicated "
        "3GPP remote raw.md document."
    ) from exc


GPP_ACCESS_TIME_S = (
    time.perf_counter()
    -
    gpp_start
)


gpp_text = (
    gpp_raw
    .decode(
        "utf-8",
        errors="replace",
    )
    .strip()
)


GPP_REMOTE_ACCESS_PASS = (

    len(
        gpp_text
    )
    >
    0

    and

    gpp_http_status
    in (
        200,
        206,
    )
)


print(
    f"      HTTP status           : "
    f"{gpp_http_status}"
)

print(
    f"      Content type          : "
    f"{gpp_content_type}"
)

print(
    f"      Bytes read            : "
    f"{len(gpp_raw):,}"
)

print(
    f"      Text characters       : "
    f"{len(gpp_text):,}"
)

print(
    f"      Access latency        : "
    f"{GPP_ACCESS_TIME_S:.3f} s"
)

print(
    f"      Status                : "
    f"{'PASS' if GPP_REMOTE_ACCESS_PASS else 'FAIL'}"
)


# ============================================================
# 2B.8 — SMALL VALIDATION PREVIEW
# ============================================================

print()
print("TCC VALIDATION SAMPLE")
print("-" * 116)


tcc_preview = (
    tcc_sample.copy()
)


tcc_preview[
    "text_preview"
] = (
    tcc_preview[
        "text_preview"
    ]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)


display(
    tcc_preview
)


print()
print("3GPP VALIDATION PREVIEW")
print("-" * 116)


gpp_preview = " ".join(
    gpp_text[
        :600
    ].split()
)


print(
    gpp_preview
)


# ============================================================
# 2B.9 — MEMORY AFTER ACCESS
# ============================================================

gc.collect()


MCP_ACCESS_RAM_AFTER_GIB = (
    current_process_ram_gib()
)


MCP_ACCESS_AVAILABLE_AFTER_GIB = (
    available_system_ram_gib()
)


MCP_ACCESS_RAM_DELTA_GIB = (

    MCP_ACCESS_RAM_AFTER_GIB
    -
    MCP_ACCESS_RAM_BEFORE_GIB
)


# ============================================================
# 2B.10 — FINAL VALIDATION
# ============================================================

MODULE4_MCP_REMOTE_ACCESS_PASS = all(
    [
        TCC_REMOTE_ACCESS_PASS,
        GPP_REMOTE_ACCESS_PASS,
    ]
)


MODULE4_CELL2B_PASS = (
    MODULE4_MCP_REMOTE_ACCESS_PASS
)


if not MODULE4_CELL2B_PASS:

    raise RuntimeError(
        "Cell 2B MCP Version A remote "
        "access validation failed."
    )


# ============================================================
# 2B.11 — FORMAL RUNTIME RECORD
# ============================================================

MODULE4_MCP_REMOTE_ACCESS_CONFIG = {

    "architecture":
        "Version A",

    "tcc_access":
        "DuckDB HTTPFS",

    "gpp_access":
        "Direct HTTP raw.md",

    "tcc_validation_file":
        REMOTE_SOURCE_REGISTRY[
            "tcc"
        ][
            "files"
        ][0],

    "gpp_validation_file":
        REMOTE_SOURCE_REGISTRY[
            "3gpp"
        ][
            "files"
        ][0],

    "tcc_latency_s":
        TCC_ACCESS_TIME_S,

    "gpp_latency_s":
        GPP_ACCESS_TIME_S,

    "persistent_local_kb":
        False,

}


# ============================================================
# 2B.12 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — MCP VERSION A REMOTE ACCESS SUMMARY")
print("=" * 116)


print(
    f"TCC Access Method           : "
    f"DuckDB HTTPFS"
)

print(
    f"TCC Access Latency          : "
    f"{TCC_ACCESS_TIME_S:.3f} s"
)

print(
    f"TCC Access PASS             : "
    f"{TCC_REMOTE_ACCESS_PASS}"
)


print()


print(
    f"3GPP Access Method          : "
    f"Direct HTTP raw.md"
)

print(
    f"3GPP Access Latency         : "
    f"{GPP_ACCESS_TIME_S:.3f} s"
)

print(
    f"3GPP HTTP Status            : "
    f"{gpp_http_status}"
)

print(
    f"3GPP Access PASS            : "
    f"{GPP_REMOTE_ACCESS_PASS}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{MCP_ACCESS_RAM_BEFORE_GIB:.3f} GiB"
)

print(
    f"Process RAM After            : "
    f"{MCP_ACCESS_RAM_AFTER_GIB:.3f} GiB"
)

print(
    f"RAM Delta                   : "
    f"{MCP_ACCESS_RAM_DELTA_GIB:+.3f} GiB"
)

print(
    f"System RAM Available        : "
    f"{MCP_ACCESS_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"TCC Remote Access PASS      : "
    f"{TCC_REMOTE_ACCESS_PASS}"
)

print(
    f"3GPP Remote Access PASS     : "
    f"{GPP_REMOTE_ACCESS_PASS}"
)

print(
    f"Overall Access PASS         : "
    f"{MODULE4_MCP_REMOTE_ACCESS_PASS}"
)


print()
print("✓ CELL 2B PASSED")
print("✓ TCC remote Parquet access validated through DuckDB HTTPFS.")
print("✓ Dedicated 3GPP raw.md access validated through direct HTTP.")
print("✓ Only bounded validation data was read.")
print("✓ No source corpus downloaded.")
print("✓ No persistent knowledge index created.")
print("✓ Full Semantic RAG remains resident and unchanged.")
print("✓ Ready for Cell 2C — Version A routing + retrieval engine restoration.")

print("=" * 116)


MODULE 4 DEMO — MCP VERSION A REMOTE ACCESS VALIDATION
TCC Repository              : GSMA/Telco-Common-Corpus
3GPP Repository             : GSMA/3GPP
DuckDB Threads              : 8

[1/2] Validating TCC remote Parquet via DuckDB HTTPFS...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      Columns detected      : 13
      Sample rows           : 3
      Access latency        : 5.243 s
      Status                : PASS

[2/2] Validating dedicated 3GPP raw.md via direct HTTP...
      HTTP status           : 206
      Content type          : text/plain; charset=utf-8
      Bytes read            : 8,192
      Text characters       : 8,180
      Access latency        : 0.447 s
      Status                : PASS

TCC VALIDATION SAMPLE
--------------------------------------------------------------------------------------------------------------------


,collection,identifier,title,text_preview
0,IETF-RFCs,RFC8202,June 2017,### 9.2 Informative References [Err4519] RFC E...
1,3GPP-TSG,C1-072238,C1-072238,"**Source:** Huawei **Title:** Draft CR, MRFC a..."
2,USPTO,US-12347254-B2,Using combination of GPS and BLE beaconing to ...,# Using combination of GPS and BLE beaconing t...



3GPP VALIDATION PREVIEW
--------------------------------------------------------------------------------------------------------------------
# **3rd Generation Partnership Project; Technical Specification Group Services and System Aspects; Technical Specifications and Technical Reports for a UTRAN-based 3GPP system (Release 10)** ![3GPP logo](935eed7aa61f7777f62cfc032e11bee9_img.jpg) The 3GPP logo is displayed within a rectangular border. It features the letters '3GPP' in a stylized, bold font. The '3' is on the left, followed by 'G', 'P', and 'P'. Below the 'G' is a small red signal icon consisting of three curved lines. A small 'TM' trademark symbol is located in the top right corner of the logo's border. 3GPP logo The presen

MODULE 4 DEMO — MCP VERSION A REMOTE ACCESS SUMMARY
TCC Access Method           : DuckDB HTTPFS
TCC Access Latency          : 5.243 s
TCC Access PASS             : True

3GPP Access Method          : Direct HTTP raw.md
3GPP Access Latency         : 0.447 s
3GP

##### Cell 2B — Observation

Both remote source access paths validated successfully. Performance remains source/network dependent, with dedicated 3GPP access generally faster than broader TCC scans.


#### Cell 2C — Three-Route Retrieval Engine Restoration

**Description:** Restore Version A routing and retrieval for dedicated 3GPP, TCC/non-3GPP, and internal hybrid 3GPP + TCC queries.


In [12]:
# ============================================================
# CELL 2C — RESTORE MCP VERSION A ROUTING + RETRIEVAL ENGINES
# ============================================================
#
# Architecture
# ------------
#
#                     User Query
#                         │
#                         ▼
#               Telecom Query Processor
#                         │
#                         ▼
#               Deterministic Router
#                  /       |       \
#                 /        |        \
#              3GPP       TCC      HYBRID
#                │         │        │   │
#                │         │        │   │
#       Spec Inference     │        │   │
#                │         │        │   │
#         Remote raw.md    │        │   │
#                │         │        │   │
#          Window Rank     │        │   │
#                          │        │   │
#                   Shard Selection │
#                          │        │
#                    DuckDB HTTPFS  │
#                          │        │
#                    Remote Parquet │
#                          │        │
#                          └────┬───┘
#                               │
#                               ▼
#                    Ranked Top-K Evidence
#
# No persistent local knowledge index is created.
# ============================================================


# ============================================================
# 2C.1 — VERIFY PREVIOUS STATE
# ============================================================

if (
    "MODULE4_CELL2B_PASS" not in globals()
    or
    not MODULE4_CELL2B_PASS
):
    raise RuntimeError(
        "Cell 2B must pass before Cell 2C."
    )


# ============================================================
# 2C.2 — IMPORTS
# ============================================================

import asyncio
import gc
import os
import re
import time
import urllib.request

from collections import defaultdict

import duckdb
import pandas as pd
import psutil


# ============================================================
# 2C.3 — VERSION A RETRIEVAL PARAMETERS
# ============================================================
#
# Retained from the frozen Module 3 Version A implementation.
# ============================================================

PER_SHARD_LIMIT = 10

TCC_SEARCH_TIMEOUT_SECONDS = 120

EARLY_TEXT_CHARS = 3000

PROXIMITY_WINDOW = 200


GPP_FETCH_TIMEOUT_SECONDS = 60

GPP_WINDOW_CHARS = 5000

GPP_WINDOW_OVERLAP = 1000

GPP_WINDOWS_PER_SPEC = 2

MAX_GPP_SPEC_CANDIDATES = 3


# ============================================================
# 2C.4 — QUERY PROCESSING
# ============================================================

LOW_INFORMATION_TERMS = {
    "5g",
    "nr",
    "network",
    "procedure",
    "function",
    "role",
    "system",
    "handling",
    "management",
}


QUERY_STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to",
    "for", "in", "on", "with", "by", "from",
    "as", "at", "what", "which", "how", "why",
    "when", "where", "is", "are", "was", "were",
    "be", "been", "being", "do", "does", "did",
    "explain", "describe", "identify", "including",
    "according", "primary",
}


def normalize_query(query):

    if not query or not str(query).strip():

        raise ValueError(
            "Query must not be empty."
        )

    return re.sub(
        r"\s+",
        " ",
        str(query).strip().lower(),
    )


def tokenize_query(
    query,
    remove_stopwords=True,
):

    tokens = re.findall(
        r"[a-z0-9]+(?:[.-][a-z0-9]+)*",
        normalize_query(query),
    )

    if not remove_stopwords:
        return tokens

    return [
        token
        for token in tokens
        if token not in QUERY_STOPWORDS
    ]


def build_query_phrases(query):

    terms = tokenize_query(
        query
    )

    bigrams = [
        " ".join(
            terms[i:i + 2]
        )
        for i in range(
            len(terms) - 1
        )
    ]

    trigrams = [
        " ".join(
            terms[i:i + 3]
        )
        for i in range(
            len(terms) - 2
        )
    ]

    return (
        bigrams,
        trigrams,
    )


def build_proximity_pairs(query):

    terms = tokenize_query(
        query
    )

    return [
        (
            terms[i],
            terms[i + 1],
        )
        for i in range(
            len(terms) - 1
        )
        if terms[i] != terms[i + 1]
    ]


def term_weight(term):

    if term in LOW_INFORMATION_TERMS:
        return 1.0

    if len(term) <= 2:
        return 0.5

    return 2.0


def escape_sql_literal(value):

    return str(
        value
    ).replace(
        "'",
        "''",
    )


def build_query_features(query):

    normalized = normalize_query(
        query
    )

    terms = tokenize_query(
        query
    )

    (
        bigrams,
        trigrams,
    ) = build_query_phrases(
        query
    )

    proximity_pairs = (
        build_proximity_pairs(
            query
        )
    )

    return {
        "normalized_query":
            normalized,

        "terms":
            terms,

        "bigrams":
            bigrams,

        "trigrams":
            trigrams,

        "proximity_pairs":
            proximity_pairs,
    }


def contains_signal(
    normalized_query,
    signal,
):

    pattern = (
        r"(?<![a-z0-9])"
        +
        re.escape(
            signal.lower()
        )
        +
        r"(?![a-z0-9])"
    )

    return (
        re.search(
            pattern,
            normalized_query,
        )
        is not None
    )


# ============================================================
# 2C.5 — TELECOM SOURCE SIGNALS
# ============================================================

GPP_SOURCE_SIGNALS = {
    "3gpp",
    "3gpp ts",
    "3gpp tr",
    "5g standalone",
    "5g sa",
    "5gs",
    "5g core",
    "ng-ran",
    "amf",
    "smf",
    "upf",
    "ausf",
    "udm",
    "nssf",
    "pcf",
    "pdu session",
    "registration",
    "mobility management",
    "s-nssai",
    "network slicing",
    "network slice",
    "5qi",
    "qos flow",
    "ngap",
    "xnap",
    "xn interface",
    "rrc",
    "radio link failure",
    "rlf",
    "n1 interface",
    "n2 interface",
    "n3 interface",
    "n4 interface",
    "pfcp",
}


TCC_IETF_SIGNALS = {
    "ietf",
    "rfc",
    "quic",
    "http",
    "http3",
    "http/3",
    "tls",
    "tcp",
    "udp",
    "dns",
}


TCC_RESEARCH_SIGNALS = {
    "research",
    "paper",
    "study",
    "ieee",
    "openalex",
}


TCC_PATENT_SIGNALS = {
    "patent",
    "invention",
    "uspto",
    "epo",
}


TCC_KNOWLEDGE_SIGNALS = {
    "wikipedia",
    "wikidata",
    "general telecom",
    "general telecommunications",
}


# ============================================================
# 2C.6 — 3GPP DOMAIN → SPECIFICATION RULES
# ============================================================

GPP_SPEC_RULES = [

    {
        "name":
            "5GS architecture",

        "signals": {
            "amf",
            "smf",
            "upf",
            "nssf",
            "s-nssai",
            "network slice",
            "network slicing",
            "5qi",
            "qos flow",
            "pdu session",
            "5g core",
            "5g standalone",
            "5g sa",
        },

        "specs": [
            "23.501",
        ],
    },

    {
        "name":
            "5GS procedures",

        "signals": {
            "registration",
            "mobility management",
            "pdu session",
            "handover",
            "inter-gnb handover",
            "service request",
            "session release",
            "pdu session release",
        },

        "specs": [
            "23.502",
        ],
    },

    {
        "name":
            "5GS policy and QoS",

        "signals": {
            "policy control",
            "pcf",
            "qos policy",
            "5qi",
        },

        "specs": [
            "23.503",
        ],
    },

    {
        "name":
            "5G security",

        "signals": {
            "authentication",
            "ausf",
            "udm",
            "security",
            "5g aka",
            "aka",
        },

        "specs": [
            "33.501",
        ],
    },

    {
        "name":
            "5GS NAS",

        "signals": {
            "nas",
            "5gmm",
            "5gsm",
        },

        "specs": [
            "24.501",
        ],
    },

    {
        "name":
            "PFCP and N4",

        "signals": {
            "pfcp",
            "n4",
            "n4 interface",
        },

        "specs": [
            "29.244",
        ],
    },

    {
        "name":
            "NR RRC",

        "signals": {
            "rrc",
            "radio link failure",
            "rlf",
            "rrc re-establishment",
        },

        "specs": [
            "38.331",
        ],
    },

    {
        "name":
            "NR architecture",

        "signals": {
            "nr architecture",
            "gnb",
            "ng-ran",
            "handover",
            "inter-gnb handover",
        },

        "specs": [
            "38.300",
        ],
    },

    {
        "name":
            "NGAP",

        "signals": {
            "ngap",
            "n2",
            "n2 interface",
        },

        "specs": [
            "38.413",
        ],
    },

    {
        "name":
            "XnAP",

        "signals": {
            "xnap",
            "xn",
            "xn interface",
            "inter-gnb handover",
        },

        "specs": [
            "38.423",
        ],
    },

]


# ============================================================
# 2C.7 — 3GPP SPECIFICATION INFERENCE
# ============================================================

def extract_explicit_3gpp_specs(query):

    normalized = normalize_query(
        query
    )

    matches = re.findall(
        r"\b"
        r"(?:3gpp\s*)?"
        r"(?:ts\s*|tr\s*)?"
        r"(\d{2}\.\d{3})"
        r"\b",
        normalized,
    )

    return list(
        dict.fromkeys(
            matches
        )
    )


def infer_3gpp_spec_candidates(
    query,
    max_specs=MAX_GPP_SPEC_CANDIDATES,
):

    normalized = normalize_query(
        query
    )

    candidates = []
    reasons = []


    # Explicit specification references first.

    for spec in extract_explicit_3gpp_specs(
        query
    ):

        if spec not in candidates:

            candidates.append(
                spec
            )

            reasons.append(
                f"explicit:{spec}"
            )


    # Telecom-domain inference.

    for rule in GPP_SPEC_RULES:

        matched_signals = [
            signal
            for signal in rule[
                "signals"
            ]
            if contains_signal(
                normalized,
                signal,
            )
        ]

        if not matched_signals:
            continue

        for spec in rule[
            "specs"
        ]:

            if spec not in candidates:

                candidates.append(
                    spec
                )

        reasons.append(
            f"{rule['name']}: "
            +
            ", ".join(
                matched_signals
            )
        )

        if len(
            candidates
        ) >= max_specs:

            break


    return {

        "specs":
            candidates[
                :max_specs
            ],

        "reasons":
            reasons,
    }


# ============================================================
# 2C.8 — TCC COLLECTION ROUTING
# ============================================================

def select_tcc_collections(query):

    normalized = normalize_query(
        query
    )

    collections = []
    reasons = []


    ietf_hits = [
        signal
        for signal in TCC_IETF_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]

    if ietf_hits:

        collections.extend(
            [
                "IETF-RFCs",
                "IETF-Drafts",
            ]
        )

        reasons.append(
            "IETF: "
            +
            ", ".join(
                ietf_hits
            )
        )


    research_hits = [
        signal
        for signal in TCC_RESEARCH_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]

    if research_hits:

        collections.extend(
            [
                "IEEE-Access",
                "OpenAlex",
            ]
        )

        reasons.append(
            "Research: "
            +
            ", ".join(
                research_hits
            )
        )


    patent_hits = [
        signal
        for signal in TCC_PATENT_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]

    if patent_hits:

        collections.extend(
            [
                "USPTO",
                "EPO",
            ]
        )

        reasons.append(
            "Patents: "
            +
            ", ".join(
                patent_hits
            )
        )


    knowledge_hits = [
        signal
        for signal in TCC_KNOWLEDGE_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]

    if knowledge_hits:

        collections.extend(
            [
                "Wikipedia-Telecom",
                "Wikidata-Telecom",
            ]
        )

        reasons.append(
            "General knowledge: "
            +
            ", ".join(
                knowledge_hits
            )
        )


    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]

    if gpp_hits:

        collections.append(
            "3GPP-TSG"
        )

        reasons.append(
            "TCC 3GPP contribution material"
        )


    if not collections:

        collections = [
            "Wikipedia-Telecom",
        ]

        reasons.append(
            "generic TCC fallback"
        )


    collections = list(
        dict.fromkeys(
            collections
        )
    )


    return {

        "collections":
            collections,

        "reasons":
            reasons,
    }


# ============================================================
# 2C.9 — DETERMINISTIC SOURCE ROUTER
# ============================================================

def route_telecom_query(query):

    normalized = normalize_query(
        query
    )


    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal,
        )
    ]


    tcc_signals = (
        TCC_IETF_SIGNALS
        |
        TCC_RESEARCH_SIGNALS
        |
        TCC_PATENT_SIGNALS
        |
        TCC_KNOWLEDGE_SIGNALS
    )


    tcc_hits = [
        signal
        for signal in tcc_signals
        if contains_signal(
            normalized,
            signal,
        )
    ]


    explicit_specs = (
        extract_explicit_3gpp_specs(
            query
        )
    )


    if explicit_specs:

        route = "3gpp"

    elif gpp_hits and tcc_hits:

        route = "hybrid"

    elif gpp_hits:

        route = "3gpp"

    elif tcc_hits:

        route = "tcc"

    else:

        route = "tcc"


    gpp_selection = (
        infer_3gpp_spec_candidates(
            query
        )
    )


    tcc_selection = (
        select_tcc_collections(
            query
        )
    )


    # --------------------------------------------------------
    # Prevent duplicate standards retrieval in HYBRID.
    # --------------------------------------------------------

    if route == "hybrid":

        tcc_selection[
            "collections"
        ] = [
            collection
            for collection
            in tcc_selection[
                "collections"
            ]
            if collection != "3GPP-TSG"
        ]


        tcc_selection[
            "reasons"
        ] = [
            reason
            for reason
            in tcc_selection[
                "reasons"
            ]
            if reason
            !=
            "TCC 3GPP contribution material"
        ]


        if not tcc_selection[
            "collections"
        ]:

            tcc_selection[
                "collections"
            ] = [
                "Wikipedia-Telecom"
            ]

            tcc_selection[
                "reasons"
            ].append(
                "hybrid TCC fallback"
            )


    return {

        "route":
            route,

        "gpp_signal_hits":
            gpp_hits,

        "tcc_signal_hits":
            tcc_hits,

        "gpp_specs":
            gpp_selection[
                "specs"
            ],

        "gpp_reasons":
            gpp_selection[
                "reasons"
            ],

        "tcc_collections":
            tcc_selection[
                "collections"
            ],

        "tcc_reasons":
            tcc_selection[
                "reasons"
            ],
    }


# ============================================================
# 2C.10 — TCC REMOTE SHARD REGISTRY
# ============================================================

TCC_REMOTE_SHARDS = [

    {
        "shard_idx":
            shard_idx,

        "file_path":
            file_path,

        "url":
            url,
    }

    for shard_idx, (
        file_path,
        url,
    )
    in enumerate(
        zip(
            REMOTE_SOURCE_REGISTRY[
                "tcc"
            ][
                "files"
            ],
            REMOTE_SOURCE_REGISTRY[
                "tcc"
            ][
                "urls"
            ],
        )
    )
]


if not TCC_REMOTE_SHARDS:

    raise RuntimeError(
        "No TCC remote shards are available."
    )


# Lightweight in-memory query→shard statistics only.
# No source documents are stored.

SHARD_TERM_STATS = defaultdict(
    lambda:
        defaultdict(float)
)


# ============================================================
# 2C.11 — TCC SHARD SELECTION
# ============================================================

def select_spread_shards(
    parquet_files,
    max_shards=MAX_REMOTE_SHARDS,
):

    total_files = len(
        parquet_files
    )

    if total_files <= max_shards:

        return list(
            parquet_files
        )


    if max_shards <= 1:

        return [
            parquet_files[0]
        ]


    selected = []
    selected_indices = set()


    step = (
        (total_files - 1)
        /
        (max_shards - 1)
    )


    for i in range(
        max_shards
    ):

        idx = round(
            i * step
        )

        if idx not in selected_indices:

            selected.append(
                parquet_files[
                    idx
                ]
            )

            selected_indices.add(
                idx
            )


    if len(
        selected
    ) < max_shards:

        for idx, item in enumerate(
            parquet_files
        ):

            if idx not in selected_indices:

                selected.append(
                    item
                )

                selected_indices.add(
                    idx
                )

            if len(
                selected
            ) >= max_shards:

                break


    return selected[
        :max_shards
    ]


def get_query_shard_scores(
    query,
    parquet_files,
):

    terms = tokenize_query(
        query
    )

    shard_scores = []


    for shard in parquet_files:

        shard_idx = shard[
            "shard_idx"
        ]

        score = 0.0


        for term in terms:

            score += (
                SHARD_TERM_STATS[
                    term
                ].get(
                    shard_idx,
                    0.0,
                )
            )


        shard_scores.append(
            {
                "shard_idx":
                    shard_idx,

                "score":
                    score,

                "parquet_file":
                    shard,
            }
        )


    return shard_scores


def select_adaptive_shards(
    query,
    parquet_files,
    max_shards=MAX_REMOTE_SHARDS,
):

    if not parquet_files:

        return {
            "mode":
                "EMPTY",

            "shards":
                [],
        }


    max_shards = min(
        max_shards,
        len(
            parquet_files
        ),
    )


    scores = (
        get_query_shard_scores(
            query,
            parquet_files,
        )
    )


    useful = [
        item
        for item in scores
        if item[
            "score"
        ] > 0
    ]


    if not useful:

        return {

            "mode":
                "SPREAD",

            "shards":
                select_spread_shards(
                    parquet_files,
                    max_shards,
                ),
        }


    ranked = sorted(
        scores,
        key=lambda item:
            item[
                "score"
            ],
        reverse=True,
    )


    return {

        "mode":
            "ADAPTIVE",

        "shards":
            [
                item[
                    "parquet_file"
                ]
                for item in ranked[
                    :max_shards
                ]
            ],
    }


def update_shard_term_stats(
    query,
    shard_idx,
    relevance_score,
):

    if relevance_score <= 0:
        return


    for term in tokenize_query(
        query
    ):

        SHARD_TERM_STATS[
            term
        ][
            shard_idx
        ] += float(
            relevance_score
        )


# ============================================================
# 2C.12 — DUCKDB HTTPFS
# ============================================================

def load_httpfs(con):

    try:

        con.execute(
            "LOAD httpfs"
        )

    except Exception:

        con.execute(
            "INSTALL httpfs"
        )

        con.execute(
            "LOAD httpfs"
        )


# Bootstrap once.

_bootstrap_conn = duckdb.connect(
    database=":memory:"
)

load_httpfs(
    _bootstrap_conn
)

_bootstrap_conn.close()


# ============================================================
# 2C.13 — TCC SQL HELPERS
# ============================================================

def build_collection_filter_sql(
    collections
):

    if not collections:

        raise ValueError(
            "At least one TCC collection must be selected."
        )


    values = [

        "'"
        +
        escape_sql_literal(
            collection
        )
        +
        "'"

        for collection
        in collections
    ]


    return (
        "("
        +
        ", ".join(
            values
        )
        +
        ")"
    )


def build_tcc_search_sql(
    query,
    parquet_url,
    collections,
    limit=PER_SHARD_LIMIT,
):

    features = (
        build_query_features(
            query
        )
    )


    normalized_query = (
        escape_sql_literal(
            features[
                "normalized_query"
            ]
        )
    )


    terms = features[
        "terms"
    ]

    bigrams = features[
        "bigrams"
    ]

    trigrams = features[
        "trigrams"
    ]

    proximity_pairs = features[
        "proximity_pairs"
    ]


    if not terms:

        raise ValueError(
            "Query contains no searchable terms."
        )


    collection_sql = (
        build_collection_filter_sql(
            collections
        )
    )


    score_parts = []
    matched_term_parts = []
    matched_phrase_parts = []
    proximity_parts = []


    # --------------------------------------------------------
    # Exact query
    # --------------------------------------------------------

    score_parts.append(
        f"""
        CASE
            WHEN title_l LIKE '%{normalized_query}%'
                THEN 40
            WHEN early_text_l LIKE '%{normalized_query}%'
                THEN 30
            WHEN text_l LIKE '%{normalized_query}%'
                THEN 20
            ELSE 0
        END
        """
    )


    # --------------------------------------------------------
    # Individual terms
    # --------------------------------------------------------

    for term in terms:

        safe_term = (
            escape_sql_literal(
                term
            )
        )

        weight = (
            term_weight(
                term
            )
        )


        score_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_term}%'
                    THEN {8 * weight}
                WHEN early_text_l LIKE '%{safe_term}%'
                    THEN {5 * weight}
                WHEN text_l LIKE '%{safe_term}%'
                    THEN {2 * weight}
                ELSE 0
            END
            """
        )


        matched_term_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_term}%'
                  OR text_l LIKE '%{safe_term}%'
                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # Bigrams
    # --------------------------------------------------------

    for phrase in bigrams:

        safe_phrase = (
            escape_sql_literal(
                phrase
            )
        )


        score_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_phrase}%'
                    THEN 14
                WHEN early_text_l LIKE '%{safe_phrase}%'
                    THEN 10
                WHEN text_l LIKE '%{safe_phrase}%'
                    THEN 6
                ELSE 0
            END
            """
        )


        matched_phrase_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_phrase}%'
                  OR text_l LIKE '%{safe_phrase}%'
                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # Trigrams
    # --------------------------------------------------------

    for phrase in trigrams:

        safe_phrase = (
            escape_sql_literal(
                phrase
            )
        )


        score_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_phrase}%'
                    THEN 20
                WHEN early_text_l LIKE '%{safe_phrase}%'
                    THEN 14
                WHEN text_l LIKE '%{safe_phrase}%'
                    THEN 8
                ELSE 0
            END
            """
        )


        matched_phrase_parts.append(
            f"""
            CASE
                WHEN title_l LIKE '%{safe_phrase}%'
                  OR text_l LIKE '%{safe_phrase}%'
                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # Proximity
    # --------------------------------------------------------

    for (
        term_a,
        term_b,
    ) in proximity_pairs:

        safe_a = escape_sql_literal(
            re.escape(
                term_a
            )
        )

        safe_b = escape_sql_literal(
            re.escape(
                term_b
            )
        )


        proximity_parts.append(
            f"""
            CASE
                WHEN regexp_matches(
                    text_l,
                    '{safe_a}.{{0,{PROXIMITY_WINDOW}}}{safe_b}'
                )
                OR regexp_matches(
                    text_l,
                    '{safe_b}.{{0,{PROXIMITY_WINDOW}}}{safe_a}'
                )
                THEN 10
                ELSE 0
            END
            """
        )


    score_expr = (
        " + ".join(
            score_parts
        )
        if score_parts
        else "0"
    )


    matched_terms_expr = (
        " + ".join(
            matched_term_parts
        )
        if matched_term_parts
        else "0"
    )


    matched_phrases_expr = (
        " + ".join(
            matched_phrase_parts
        )
        if matched_phrase_parts
        else "0"
    )


    proximity_expr = (
        " + ".join(
            proximity_parts
        )
        if proximity_parts
        else "0"
    )


    safe_url = (
        escape_sql_literal(
            parquet_url
        )
    )


    return f"""
        WITH scoped AS (

            SELECT
                identifier,
                collection,
                date,
                title,
                creator,
                text,

                lower(
                    coalesce(
                        title,
                        ''
                    )
                ) AS title_l,

                lower(
                    coalesce(
                        text,
                        ''
                    )
                ) AS text_l,

                lower(
                    substr(
                        coalesce(
                            text,
                            ''
                        ),
                        1,
                        {EARLY_TEXT_CHARS}
                    )
                ) AS early_text_l

            FROM read_parquet(
                '{safe_url}'
            )

            WHERE collection IN
                {collection_sql}
        )

        SELECT
            identifier,
            collection,
            date,
            title,
            creator,
            text,

            'TCC'
                AS source_family,

            ({matched_terms_expr})
                AS matched_terms,

            ({matched_phrases_expr})
                AS matched_phrases,

            ({proximity_expr})
                AS proximity_score,

            (
                ({score_expr})
                +
                ({proximity_expr})
            )
                AS relevance_score

        FROM scoped

        WHERE
            (
                title_l
                    LIKE '%{normalized_query}%'

                OR

                text_l
                    LIKE '%{normalized_query}%'

                OR

                ({matched_terms_expr}) > 0
            )

        ORDER BY
            relevance_score DESC,
            date DESC NULLS LAST

        LIMIT {int(limit)}
    """


# ============================================================
# 2C.14 — SEARCH ONE REMOTE TCC SHARD
# ============================================================

def search_single_tcc_shard(
    query,
    shard,
    collections,
    limit=PER_SHARD_LIMIT,
):

    start_time = (
        time.perf_counter()
    )


    con = duckdb.connect(
        database=":memory:"
    )


    try:

        con.execute(
            f"SET threads = {int(DUCKDB_THREADS)}"
        )

        load_httpfs(
            con
        )


        sql = (
            build_tcc_search_sql(
                query=query,
                parquet_url=
                    shard[
                        "url"
                    ],
                collections=
                    collections,
                limit=
                    limit,
            )
        )


        result_df = (
            con.execute(
                sql
            )
            .fetchdf()
        )


        records = (
            result_df.to_dict(
                orient="records"
            )
        )


        for item in records:

            item[
                "shard_idx"
            ] = (
                shard[
                    "shard_idx"
                ]
            )

            item[
                "source_path"
            ] = (
                shard[
                    "file_path"
                ]
            )

            item[
                "source_shard"
            ] = (
                shard[
                    "url"
                ]
            )


        return {

            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "results":
                records,

            "error":
                None,
        }


    except Exception as exc:

        return {

            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "results":
                [],

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        }


    finally:

        con.close()


async def search_tcc_shard_with_timeout(
    query,
    shard,
    collections,
    limit=PER_SHARD_LIMIT,
):

    try:

        return await asyncio.wait_for(

            asyncio.to_thread(
                search_single_tcc_shard,
                query,
                shard,
                collections,
                limit,
            ),

            timeout=
                TCC_SEARCH_TIMEOUT_SECONDS,
        )


    except asyncio.TimeoutError:

        return {

            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                float(
                    TCC_SEARCH_TIMEOUT_SECONDS
                ),

            "results":
                [],

            "error":
                (
                    "TimeoutError: TCC shard search "
                    f"exceeded "
                    f"{TCC_SEARCH_TIMEOUT_SECONDS}s"
                ),
        }


async def search_selected_tcc_shards(
    query,
    selected_shards,
    collections,
    limit=PER_SHARD_LIMIT,
):

    semaphore = asyncio.Semaphore(
        PARALLEL_SHARD_CONCURRENCY
    )


    async def run_one(shard):

        async with semaphore:

            return await (
                search_tcc_shard_with_timeout(
                    query=query,
                    shard=shard,
                    collections=collections,
                    limit=limit,
                )
            )


    tasks = [
        run_one(
            shard
        )
        for shard
        in selected_shards
    ]


    return await asyncio.gather(
        *tasks
    )


def merge_tcc_results(
    shard_results
):

    combined = []


    for shard_result in shard_results:

        for item in shard_result[
            "results"
        ]:

            combined.append(
                dict(
                    item
                )
            )


    combined.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0,
                )
                or 0
            ),
        reverse=True,
    )


    unique = []
    seen = set()


    for item in combined:

        key = (
            str(
                item.get(
                    "collection",
                    "",
                )
            ),
            str(
                item.get(
                    "identifier",
                    "",
                )
            ),
            str(
                item.get(
                    "title",
                    "",
                )
            ),
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        unique.append(
            item
        )


    return unique


def learn_from_tcc_results(
    query,
    ranked_results,
):

    for item in ranked_results:

        shard_idx = (
            item.get(
                "shard_idx"
            )
        )

        relevance_score = float(
            item.get(
                "relevance_score",
                0,
            )
            or 0
        )


        if (
            shard_idx is None
            or
            relevance_score <= 0
        ):
            continue


        update_shard_term_stats(
            query=query,
            shard_idx=shard_idx,
            relevance_score=
                relevance_score,
        )


async def retrieve_tcc_remote(
    query,
    collections=None,
    top_k=TOP_K_RESULTS,
    max_shards=MAX_REMOTE_SHARDS,
):

    start_time = (
        time.perf_counter()
    )


    if collections is None:

        selection = (
            select_tcc_collections(
                query
            )
        )

        collections = (
            selection[
                "collections"
            ]
        )


    collections = list(
        dict.fromkeys(
            collections
        )
    )


    shard_selection = (
        select_adaptive_shards(
            query=query,
            parquet_files=
                TCC_REMOTE_SHARDS,
            max_shards=
                max_shards,
        )
    )


    selected_shards = (
        shard_selection[
            "shards"
        ]
    )


    shard_results = (
        await search_selected_tcc_shards(
            query=query,
            selected_shards=
                selected_shards,
            collections=
                collections,
            limit=
                PER_SHARD_LIMIT,
        )
    )


    ranked_results = (
        merge_tcc_results(
            shard_results
        )
    )


    final_results = (
        ranked_results[
            :top_k
        ]
    )


    learn_from_tcc_results(
        query=query,
        ranked_results=
            final_results,
    )


    errors = [
        item
        for item in shard_results
        if item.get(
            "error"
        )
        is not None
    ]


    return {

        "engine":
            "tcc_remote",

        "query":
            query,

        "collections":
            collections,

        "shard_selection":
            shard_selection[
                "mode"
            ],

        "selected_shards":
            selected_shards,

        "shard_results":
            shard_results,

        "shard_errors":
            len(
                errors
            ),

        "results":
            final_results,

        "result_count":
            len(
                final_results
            ),

        "retrieval_time_s":
            (
                time.perf_counter()
                -
                start_time
            ),
    }


# ============================================================
# 2C.15 — BUILD DEDICATED 3GPP SPECIFICATION REGISTRY
# ============================================================

def format_3gpp_spec_number(
    identifier
):

    value = str(
        identifier
    )


    match = re.match(
        r"^(\d{5})(-\d+)?$",
        value,
    )


    if not match:
        return None


    base = match.group(
        1
    )

    suffix = (
        match.group(
            2
        )
        or
        ""
    )


    return (
        f"{base[:2]}."
        f"{base[2:]}"
        f"{suffix}"
    )


def parse_3gpp_path(
    file_path,
    url,
):

    release_match = re.search(
        r"/Rel-(\d+)/",
        f"/{file_path}",
    )


    spec_match = re.search(
        r"/(\d{5}(?:-\d+)?)/raw\.md$",
        f"/{file_path}",
    )


    if not spec_match:
        return None


    spec_number = (
        format_3gpp_spec_number(
            spec_match.group(
                1
            )
        )
    )


    if not spec_number:
        return None


    release = (
        int(
            release_match.group(
                1
            )
        )
        if release_match
        else None
    )


    return {

        "spec_number":
            spec_number,

        "release":
            release,

        "file_path":
            file_path,

        "url":
            url,
    }


GPP_SPEC_PATH_INDEX = defaultdict(
    list
)


for (
    file_path,
    url,
) in zip(

    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "files"
    ],

    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "urls"
    ],

):

    metadata = (
        parse_3gpp_path(
            file_path,
            url,
        )
    )


    if metadata:

        GPP_SPEC_PATH_INDEX[
            metadata[
                "spec_number"
            ]
        ].append(
            metadata
        )


# Latest available release first.

for spec_number in GPP_SPEC_PATH_INDEX:

    GPP_SPEC_PATH_INDEX[
        spec_number
    ].sort(

        key=lambda item:
            (
                item[
                    "release"
                ]
                if item[
                    "release"
                ] is not None
                else -1
            ),

        reverse=True,
    )


def get_latest_3gpp_spec(
    spec_number
):

    available = (
        GPP_SPEC_PATH_INDEX.get(
            spec_number,
            [],
        )
    )


    if not available:
        return None


    return available[
        0
    ]


# ============================================================
# 2C.16 — FETCH + RANK 3GPP SPECIFICATIONS
# ============================================================

def fetch_3gpp_document(
    candidate
):

    start_time = (
        time.perf_counter()
    )


    request = urllib.request.Request(

        candidate[
            "url"
        ],

        headers={
            "User-Agent":
                "Telecom-AI-MCP-Version-A-Demo"
        },
    )


    try:

        with urllib.request.urlopen(
            request,
            timeout=
                GPP_FETCH_TIMEOUT_SECONDS,
        ) as response:

            raw = (
                response.read()
            )

            status = getattr(
                response,
                "status",
                None,
            )


        text = raw.decode(
            "utf-8",
            errors="replace",
        )


        return {

            **candidate,

            "http_status":
                status,

            "bytes":
                len(
                    raw
                ),

            "text":
                text,

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "error":
                None,
        }


    except Exception as exc:

        return {

            **candidate,

            "http_status":
                None,

            "bytes":
                0,

            "text":
                "",

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
        }


def extract_3gpp_title(
    text,
    spec_number,
):

    for line in str(
        text
    ).splitlines()[
        :120
    ]:

        clean = re.sub(
            r"^#+\s*",
            "",
            line,
        ).strip()


        clean = re.sub(
            r"[*_~]+",
            "",
            clean,
        ).strip()


        if (
            len(
                clean
            ) >= 20

            and

            not clean.startswith(
                "!["
            )

            and

            "3gpp logo"
            not in clean.lower()
        ):

            return clean[
                :500
            ]


    return (
        f"3GPP Specification "
        f"{spec_number}"
    )


def score_text_block(
    query,
    title,
    text,
):

    features = (
        build_query_features(
            query
        )
    )


    normalized_query = (
        features[
            "normalized_query"
        ]
    )

    terms = (
        features[
            "terms"
        ]
    )

    bigrams = (
        features[
            "bigrams"
        ]
    )

    trigrams = (
        features[
            "trigrams"
        ]
    )

    proximity_pairs = (
        features[
            "proximity_pairs"
        ]
    )


    title_l = str(
        title
        or
        ""
    ).lower()


    text_l = str(
        text
        or
        ""
    ).lower()


    early_text_l = (
        text_l[
            :EARLY_TEXT_CHARS
        ]
    )


    score = 0.0
    matched_terms = 0
    matched_phrases = 0
    proximity_score = 0.0


    # Exact query.

    if normalized_query:

        if normalized_query in title_l:

            score += 40

        elif normalized_query in early_text_l:

            score += 30

        elif normalized_query in text_l:

            score += 20


    # Individual terms.

    for term in terms:

        weight = (
            term_weight(
                term
            )
        )


        if (
            term in title_l
            or
            term in text_l
        ):

            matched_terms += 1


        if term in title_l:

            score += (
                8
                *
                weight
            )

        elif term in early_text_l:

            score += (
                5
                *
                weight
            )

        elif term in text_l:

            score += (
                2
                *
                weight
            )


    # Bigrams.

    for phrase in bigrams:

        if (
            phrase in title_l
            or
            phrase in text_l
        ):

            matched_phrases += 1


        if phrase in title_l:

            score += 14

        elif phrase in early_text_l:

            score += 10

        elif phrase in text_l:

            score += 6


    # Trigrams.

    for phrase in trigrams:

        if (
            phrase in title_l
            or
            phrase in text_l
        ):

            matched_phrases += 1


        if phrase in title_l:

            score += 20

        elif phrase in early_text_l:

            score += 14

        elif phrase in text_l:

            score += 8


    # Proximity.

    for (
        term_a,
        term_b,
    ) in proximity_pairs:

        pattern_ab = re.compile(
            re.escape(
                term_a
            )
            +
            rf".{{0,{PROXIMITY_WINDOW}}}"
            +
            re.escape(
                term_b
            ),
            flags=re.DOTALL,
        )


        pattern_ba = re.compile(
            re.escape(
                term_b
            )
            +
            rf".{{0,{PROXIMITY_WINDOW}}}"
            +
            re.escape(
                term_a
            ),
            flags=re.DOTALL,
        )


        if (
            pattern_ab.search(
                text_l
            )
            or
            pattern_ba.search(
                text_l
            )
        ):

            proximity_score += 10


    score += (
        proximity_score
    )


    return {

        "matched_terms":
            matched_terms,

        "matched_phrases":
            matched_phrases,

        "proximity_score":
            proximity_score,

        "relevance_score":
            score,
    }


def rank_3gpp_document(
    query,
    document,
    windows_per_spec=
        GPP_WINDOWS_PER_SPEC,
):

    text = document.get(
        "text",
        "",
    )


    if not text:
        return []


    spec_number = (
        document[
            "spec_number"
        ]
    )


    title = (
        extract_3gpp_title(
            text,
            spec_number,
        )
    )


    step = max(
        1,
        (
            GPP_WINDOW_CHARS
            -
            GPP_WINDOW_OVERLAP
        ),
    )


    ranked = []


    for start in range(
        0,
        len(
            text
        ),
        step,
    ):

        window = (
            text[
                start:
                start
                +
                GPP_WINDOW_CHARS
            ]
        )


        if not window.strip():
            continue


        scoring = (
            score_text_block(
                query=query,
                title=title,
                text=window,
            )
        )


        if scoring[
            "relevance_score"
        ] <= 0:

            continue


        ranked.append(
            {

                "identifier":
                    spec_number,

                "collection":
                    "3GPP-Specifications",

                "date":
                    None,

                "title":
                    title,

                "creator":
                    "3GPP",

                "text":
                    window,

                "source_family":
                    "3GPP",

                "source_path":
                    document[
                        "file_path"
                    ],

                "source_shard":
                    document[
                        "url"
                    ],

                "release":
                    document[
                        "release"
                    ],

                "window_start":
                    start,

                **scoring,
            }
        )


    ranked.sort(
        key=lambda item:
            float(
                item[
                    "relevance_score"
                ]
            ),
        reverse=True,
    )


    return ranked[
        :windows_per_spec
    ]


async def retrieve_3gpp_remote(
    query,
    specs=None,
    top_k=TOP_K_RESULTS,
):

    start_time = (
        time.perf_counter()
    )


    if specs is None:

        inference = (
            infer_3gpp_spec_candidates(
                query
            )
        )

        specs = (
            inference[
                "specs"
            ]
        )


    specs = list(
        dict.fromkeys(
            specs
        )
    )


    selected_specs = []
    missing_specs = []


    for spec in specs:

        candidate = (
            get_latest_3gpp_spec(
                spec
            )
        )


        if candidate is None:

            missing_specs.append(
                spec
            )

            continue


        selected_specs.append(
            candidate
        )


    fetched_documents = []


    if selected_specs:

        fetched_documents = (
            await asyncio.gather(
                *[
                    asyncio.to_thread(
                        fetch_3gpp_document,
                        candidate,
                    )
                    for candidate
                    in selected_specs
                ]
            )
        )


    document_errors = [
        item
        for item in fetched_documents
        if item.get(
            "error"
        )
        is not None
    ]


    ranked_results = []


    for document in fetched_documents:

        if document.get(
            "error"
        ) is not None:

            continue


        ranked_results.extend(
            rank_3gpp_document(
                query=query,
                document=document,
            )
        )


    ranked_results.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0,
                )
                or 0
            ),
        reverse=True,
    )


    final_results = (
        ranked_results[
            :top_k
        ]
    )


    return {

        "engine":
            "3gpp_remote",

        "query":
            query,

        "requested_specs":
            specs,

        "selected_specs":
            selected_specs,

        "missing_specs":
            missing_specs,

        "fetched_documents":
            fetched_documents,

        "document_errors":
            len(
                document_errors
            ),

        "results":
            final_results,

        "result_count":
            len(
                final_results
            ),

        "retrieval_time_s":
            (
                time.perf_counter()
                -
                start_time
            ),
    }


# ============================================================
# 2C.17 — CROSS-SOURCE MERGE
# ============================================================

def merge_cross_source_results(
    result_groups,
    top_k=TOP_K_RESULTS,
):

    combined = []


    for group in result_groups:

        combined.extend(
            group
        )


    combined.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0,
                )
                or 0
            ),
        reverse=True,
    )


    unique = []
    seen = set()


    for item in combined:

        key = (
            str(
                item.get(
                    "source_family",
                    "",
                )
            ),
            str(
                item.get(
                    "source_path",
                    "",
                )
            ),
            str(
                item.get(
                    "identifier",
                    "",
                )
            ),
            int(
                item.get(
                    "window_start",
                    0,
                )
                or 0
            ),
        )


        if key in seen:
            continue


        seen.add(
            key
        )


        unique.append(
            item
        )


        if len(
            unique
        ) >= top_k:

            break


    return unique


# ============================================================
# 2C.18 — UNIFIED VERSION A RETRIEVER
# ============================================================

async def retrieve_version_a(
    query,
    top_k=TOP_K_RESULTS,
):

    overall_start = (
        time.perf_counter()
    )


    routing = (
        route_telecom_query(
            query
        )
    )


    route = routing[
        "route"
    ]


    # --------------------------------------------------------
    # 3GPP
    # --------------------------------------------------------

    if route == "3gpp":

        gpp_result = (
            await retrieve_3gpp_remote(
                query=query,
                specs=
                    routing[
                        "gpp_specs"
                    ],
                top_k=
                    top_k,
            )
        )


        final_results = (
            gpp_result[
                "results"
            ]
        )


        return {

            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "3GPP"
                ],

            "tcc":
                None,

            "3gpp":
                gpp_result,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                ),
        }


    # --------------------------------------------------------
    # TCC
    # --------------------------------------------------------

    if route == "tcc":

        tcc_result = (
            await retrieve_tcc_remote(
                query=query,
                collections=
                    routing[
                        "tcc_collections"
                    ],
                top_k=
                    top_k,
                max_shards=
                    MAX_REMOTE_SHARDS,
            )
        )


        final_results = (
            tcc_result[
                "results"
            ]
        )


        return {

            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "TCC"
                ],

            "tcc":
                tcc_result,

            "3gpp":
                None,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                ),
        }


    # --------------------------------------------------------
    # HYBRID
    # --------------------------------------------------------

    if route == "hybrid":

        (
            gpp_result,
            tcc_result,
        ) = await asyncio.gather(

            retrieve_3gpp_remote(
                query=query,
                specs=
                    routing[
                        "gpp_specs"
                    ],
                top_k=
                    top_k,
            ),

            retrieve_tcc_remote(
                query=query,
                collections=
                    routing[
                        "tcc_collections"
                    ],
                top_k=
                    top_k,
                max_shards=
                    MAX_REMOTE_SHARDS,
            ),
        )


        final_results = (
            merge_cross_source_results(
                [
                    gpp_result[
                        "results"
                    ],
                    tcc_result[
                        "results"
                    ],
                ],
                top_k=
                    top_k,
            )
        )


        return {

            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "3GPP",
                    "TCC",
                ],

            "tcc":
                tcc_result,

            "3gpp":
                gpp_result,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                ),
        }


    raise RuntimeError(
        f"Unsupported Version A route: {route}"
    )


# ============================================================
# 2C.19 — STATIC ROUTER VALIDATION
# ============================================================

ROUTER_VALIDATION_CASES = [

    {
        "query":
            (
                "Explain AMF registration and mobility "
                "management in a 5G Standalone network"
            ),

        "expected":
            "3gpp",
    },

    {
        "query":
            (
                "Explain QUIC transport and relevant "
                "IETF RFC mechanisms"
            ),

        "expected":
            "tcc",
    },

    {
        "query":
            (
                "Explain AMF traffic transported "
                "using QUIC"
            ),

        "expected":
            "hybrid",
    },

]


ROUTER_VALIDATION_RESULTS = []


for case in ROUTER_VALIDATION_CASES:

    routing = (
        route_telecom_query(
            case[
                "query"
            ]
        )
    )


    ROUTER_VALIDATION_RESULTS.append(
        {
            "query":
                case[
                    "query"
                ],

            "expected":
                case[
                    "expected"
                ],

            "observed":
                routing[
                    "route"
                ],

            "pass":
                (
                    routing[
                        "route"
                    ]
                    ==
                    case[
                        "expected"
                    ]
                ),

            "3gpp_specs":
                routing[
                    "gpp_specs"
                ],

            "tcc_collections":
                routing[
                    "tcc_collections"
                ],
        }
    )


MODULE4_VERSION_A_ROUTER_PASS = all(
    item[
        "pass"
    ]
    for item
    in ROUTER_VALIDATION_RESULTS
)


if not MODULE4_VERSION_A_ROUTER_PASS:

    raise RuntimeError(
        "Version A source-router validation failed."
    )


# ============================================================
# 2C.20 — LIVE 3GPP RETRIEVAL SMOKE TEST
# ============================================================
#
# Use the same AMF subject used for RAG validation.
#
# This intentionally validates the 3GPP-only branch first.
# TCC + Hybrid execution will be tested through the MCP
# three-route validation after the tool is exposed.
# ============================================================

MCP_VERSION_A_TEST_QUERY = (
    "What are the primary responsibilities "
    "of the AMF in a 5G Standalone network?"
)


gc.collect()


MCP_RETRIEVAL_RAM_BEFORE_GIB = (
    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


MCP_RETRIEVAL_AVAILABLE_BEFORE_GIB = (
    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


print("=" * 116)
print("MODULE 4 DEMO — MCP VERSION A LIVE RETRIEVAL")
print("=" * 116)


print(
    f"Query                      : "
    f"{MCP_VERSION_A_TEST_QUERY}"
)


mcp_routing_preview = (
    route_telecom_query(
        MCP_VERSION_A_TEST_QUERY
    )
)


print(
    f"Route                      : "
    f"{mcp_routing_preview['route'].upper()}"
)


print(
    f"3GPP Specifications        : "
    f"{mcp_routing_preview['gpp_specs']}"
)


print(
    f"TCC Collections            : "
    f"{mcp_routing_preview['tcc_collections']}"
)


print()
print("Executing live Version A retrieval...")
print()


MODULE4_VERSION_A_SMOKE_RESULT = (
    await retrieve_version_a(
        MCP_VERSION_A_TEST_QUERY,
        top_k=TOP_K_RESULTS,
    )
)


gc.collect()


MCP_RETRIEVAL_RAM_AFTER_GIB = (
    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


MCP_RETRIEVAL_AVAILABLE_AFTER_GIB = (
    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


MCP_RETRIEVAL_RAM_DELTA_GIB = (
    MCP_RETRIEVAL_RAM_AFTER_GIB
    -
    MCP_RETRIEVAL_RAM_BEFORE_GIB
)


# ============================================================
# 2C.21 — COMPACT RESULT DISPLAY
# ============================================================

mcp_display_rows = []


for rank, item in enumerate(
    MODULE4_VERSION_A_SMOKE_RESULT[
        "results"
    ],
    start=1,
):

    text = " ".join(
        str(
            item.get(
                "text",
                "",
            )
        ).split()
    )


    mcp_display_rows.append(
        {

            "rank":
                rank,

            "score":
                float(
                    item.get(
                        "relevance_score",
                        0,
                    )
                    or 0
                ),

            "source":
                item.get(
                    "source_family",
                    "",
                ),

            "identifier":
                item.get(
                    "identifier",
                    "",
                ),

            "release":
                item.get(
                    "release",
                    "",
                ),

            "title":
                str(
                    item.get(
                        "title",
                        "",
                    )
                )[
                    :80
                ],

            "snippet":
                text[
                    :220
                ]
                +
                (
                    "..."
                    if len(
                        text
                    ) > 220
                    else ""
                ),
        }
    )


MCP_VERSION_A_RESULTS_DF = (
    pd.DataFrame(
        mcp_display_rows
    )
)


print("TOP-K MCP VERSION A EVIDENCE")
print("-" * 116)


display(
    MCP_VERSION_A_RESULTS_DF
)


# ============================================================
# 2C.22 — VALIDATION
# ============================================================

MCP_VERSION_A_RESULT_PASS = (

    MODULE4_VERSION_A_SMOKE_RESULT[
        "result_count"
    ]
    >
    0
)


MCP_VERSION_A_ROUTE_PASS = (

    MODULE4_VERSION_A_SMOKE_RESULT[
        "route"
    ]
    ==
    "3gpp"
)


MCP_VERSION_A_SOURCE_PASS = (

    MODULE4_VERSION_A_SMOKE_RESULT[
        "sources_searched"
    ]
    ==
    [
        "3GPP"
    ]
)


MCP_VERSION_A_TEXT_PASS = all(

    bool(
        str(
            item.get(
                "text",
                "",
            )
        ).strip()
    )

    for item
    in MODULE4_VERSION_A_SMOKE_RESULT[
        "results"
    ]

)


MODULE4_VERSION_A_RETRIEVAL_PASS = all(
    [
        MODULE4_VERSION_A_ROUTER_PASS,
        MCP_VERSION_A_RESULT_PASS,
        MCP_VERSION_A_ROUTE_PASS,
        MCP_VERSION_A_SOURCE_PASS,
        MCP_VERSION_A_TEXT_PASS,
    ]
)


MODULE4_CELL2C_PASS = (
    MODULE4_VERSION_A_RETRIEVAL_PASS
)


if not MODULE4_CELL2C_PASS:

    raise RuntimeError(
        "Cell 2C Version A retrieval validation failed."
    )


# ============================================================
# 2C.23 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — MCP VERSION A RETRIEVAL SUMMARY")
print("=" * 116)


print(
    f"Architecture                : "
    f"{ARCHITECTURE_VERSION}"
)


print(
    f"Route                       : "
    f"{MODULE4_VERSION_A_SMOKE_RESULT['route'].upper()}"
)


print(
    f"Sources Searched            : "
    f"{MODULE4_VERSION_A_SMOKE_RESULT['sources_searched']}"
)


print(
    f"Evidence Returned           : "
    f"{MODULE4_VERSION_A_SMOKE_RESULT['result_count']}"
)


print(
    f"Total Retrieval Latency     : "
    f"{MODULE4_VERSION_A_SMOKE_RESULT['retrieval_time_s']:.3f} s"
)


gpp_runtime = (
    MODULE4_VERSION_A_SMOKE_RESULT[
        "3gpp"
    ]
)


if gpp_runtime:

    print(
        f"Requested Specs             : "
        f"{gpp_runtime['requested_specs']}"
    )

    print(
        f"Remote Documents Fetched    : "
        f"{len(gpp_runtime['fetched_documents'])}"
    )

    print(
        f"3GPP Document Errors        : "
        f"{gpp_runtime['document_errors']}"
    )


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{MCP_RETRIEVAL_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{MCP_RETRIEVAL_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{MCP_RETRIEVAL_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{MCP_RETRIEVAL_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Router Validation PASS      : "
    f"{MODULE4_VERSION_A_ROUTER_PASS}"
)


print(
    f"Live Result PASS            : "
    f"{MCP_VERSION_A_RESULT_PASS}"
)


print(
    f"Route PASS                  : "
    f"{MCP_VERSION_A_ROUTE_PASS}"
)


print(
    f"Source PASS                 : "
    f"{MCP_VERSION_A_SOURCE_PASS}"
)


print(
    f"Evidence Text PASS          : "
    f"{MCP_VERSION_A_TEXT_PASS}"
)


print(
    f"Version A Retrieval PASS    : "
    f"{MODULE4_VERSION_A_RETRIEVAL_PASS}"
)


print()
print("✓ CELL 2C PASSED")
print("✓ Telecom-aware Version A query processor restored.")
print("✓ Deterministic 3GPP / TCC / Hybrid router restored.")
print("✓ Adaptive TCC remote-shard selection restored.")
print("✓ TCC DuckDB HTTPFS retrieval engine restored.")
print("✓ Dedicated 3GPP specification inference restored.")
print("✓ 3GPP remote document ranking restored.")
print("✓ Hybrid parallel retrieval capability restored.")
print("✓ Live 3GPP evidence retrieval validated.")
print("✓ No persistent local MCP knowledge base created.")
print("✓ Ready for Cell 2D — FastMCP knowledge tool + three-route validation.")

print("=" * 116)

# ============================================================
# 2C.24 — TCC + HYBRID LIVE RETRIEVAL VALIDATION
# ============================================================
#
# The 3GPP-only route has already passed above.
#
# Now validate the remaining Version A retrieval paths
# DIRECTLY through retrieve_version_a():
#
#     1. TCC-only
#     2. HYBRID 3GPP + TCC
#
# FastMCP is deliberately NOT involved yet.
#
# This isolates retrieval-engine correctness from the
# MCP client/server boundary that will be tested in Cell 2D.
# ============================================================


VERSION_A_ADDITIONAL_ROUTE_CASES = [

    {
        "id":
            "TCC_ONLY",

        "query":
            (
                "Explain QUIC connection migration and the "
                "relevant IETF mechanisms."
            ),

        "expected_route":
            "tcc",

        "expected_sources":
            {
                "TCC"
            },

        "required_evidence_sources":
            {
                "TCC"
            },
    },


    {
        "id":
            "HYBRID",

        "query":
            (
                "Explain how QUIC transport could relate to "
                "traffic carried through a 5G Core network."
            ),

        "expected_route":
            "hybrid",

        "expected_sources":
            {
                "3GPP",
                "TCC",
            },

        "required_evidence_sources":
            {
                "3GPP",
                "TCC",
            },
    },

]


# ============================================================
# 2C.25 — MEMORY BASELINE
# ============================================================

gc.collect()


VERSION_A_MULTI_ROUTE_RAM_BEFORE_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


VERSION_A_MULTI_ROUTE_AVAILABLE_BEFORE_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


# ============================================================
# 2C.26 — STORE EXISTING 3GPP RESULT
# ============================================================

VERSION_A_ROUTE_RESULTS = {

    "3GPP_ONLY":
        MODULE4_VERSION_A_SMOKE_RESULT

}


VERSION_A_ROUTE_VALIDATION = []


# Existing 3GPP validation record.

gpp_evidence_sources = {

    item.get(
        "source_family",
        "UNKNOWN"
    )

    for item
    in MODULE4_VERSION_A_SMOKE_RESULT[
        "results"
    ]
}


VERSION_A_ROUTE_VALIDATION.append({

    "test":
        "3GPP_ONLY",

    "expected_route":
        "3gpp",

    "actual_route":
        MODULE4_VERSION_A_SMOKE_RESULT[
            "route"
        ],

    "sources_searched":
        ", ".join(
            MODULE4_VERSION_A_SMOKE_RESULT[
                "sources_searched"
            ]
        ),

    "evidence_sources":
        ", ".join(
            sorted(
                gpp_evidence_sources
            )
        ),

    "result_count":
        MODULE4_VERSION_A_SMOKE_RESULT[
            "result_count"
        ],

    "retrieval_time_s":
        MODULE4_VERSION_A_SMOKE_RESULT[
            "retrieval_time_s"
        ],

    "passed":
        all([
            MODULE4_VERSION_A_SMOKE_RESULT[
                "route"
            ]
            ==
            "3gpp",

            set(
                MODULE4_VERSION_A_SMOKE_RESULT[
                    "sources_searched"
                ]
            )
            ==
            {
                "3GPP"
            },

            "3GPP"
            in
            gpp_evidence_sources,

            MODULE4_VERSION_A_SMOKE_RESULT[
                "result_count"
            ]
            >
            0,
        ]),
})


# ============================================================
# 2C.27 — EXECUTE TCC + HYBRID
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — VERSION A THREE-ROUTE RETRIEVAL VALIDATION")
print("=" * 116)

print(
    "Execution Layer             : "
    "Direct Version A Retrieval Engine"
)

print(
    "MCP Boundary                : "
    "Not yet involved"
)

print(
    "Already Validated           : "
    "3GPP_ONLY"
)

print(
    "Additional Routes           : "
    "TCC_ONLY / HYBRID"
)

print()


route_progress = tqdm(
    total=
        len(
            VERSION_A_ADDITIONAL_ROUTE_CASES
        ),

    desc=
        "Version A Route Validation",

    unit=
        "route",

    leave=True,
)


for case in VERSION_A_ADDITIONAL_ROUTE_CASES:

    route_progress.set_postfix_str(
        case[
            "id"
        ]
    )


    result = (
        await retrieve_version_a(

            query=
                case[
                    "query"
                ],

            top_k=
                TOP_K_RESULTS,
        )
    )


    VERSION_A_ROUTE_RESULTS[
        case[
            "id"
        ]
    ] = result


    evidence_sources = {

        item.get(
            "source_family",
            "UNKNOWN"
        )

        for item
        in result[
            "results"
        ]
    }


    actual_sources = set(
        result[
            "sources_searched"
        ]
    )


    route_correct = (

        result[
            "route"
        ]
        ==
        case[
            "expected_route"
        ]
    )


    sources_correct = (

        actual_sources
        ==
        case[
            "expected_sources"
        ]
    )


    evidence_sources_correct = (

        case[
            "required_evidence_sources"
        ]
        .issubset(
            evidence_sources
        )
    )


    results_returned = (

        result[
            "result_count"
        ]
        >
        0
    )


    case_passed = all(
        [
            route_correct,
            sources_correct,
            evidence_sources_correct,
            results_returned,
        ]
    )


    VERSION_A_ROUTE_VALIDATION.append({

        "test":
            case[
                "id"
            ],

        "expected_route":
            case[
                "expected_route"
            ],

        "actual_route":
            result[
                "route"
            ],

        "sources_searched":
            ", ".join(
                sorted(
                    actual_sources
                )
            ),

        "evidence_sources":
            ", ".join(
                sorted(
                    evidence_sources
                )
            ),

        "result_count":
            result[
                "result_count"
            ],

        "retrieval_time_s":
            result[
                "retrieval_time_s"
            ],

        "passed":
            case_passed,
    })


    route_progress.update(
        1
    )


route_progress.set_postfix_str(
    "Complete"
)

route_progress.close()


# ============================================================
# 2C.28 — MEMORY AFTER VALIDATION
# ============================================================

gc.collect()


VERSION_A_MULTI_ROUTE_RAM_AFTER_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


VERSION_A_MULTI_ROUTE_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


VERSION_A_MULTI_ROUTE_RAM_DELTA_GIB = (

    VERSION_A_MULTI_ROUTE_RAM_AFTER_GIB
    -
    VERSION_A_MULTI_ROUTE_RAM_BEFORE_GIB
)


# ============================================================
# 2C.29 — VALIDATION SUMMARY TABLE
# ============================================================

VERSION_A_ROUTE_VALIDATION_DF = (

    pd.DataFrame(
        VERSION_A_ROUTE_VALIDATION
    )
)


print()
print("THREE-ROUTE RETRIEVAL RESULTS")
print("-" * 116)


display(

    VERSION_A_ROUTE_VALIDATION_DF[
        [
            "test",
            "expected_route",
            "actual_route",
            "sources_searched",
            "evidence_sources",
            "result_count",
            "retrieval_time_s",
            "passed",
        ]
    ]
    .round(
        {
            "retrieval_time_s":
                3
        }
    )
)


# ============================================================
# 2C.30 — ROUTE TRACE
# ============================================================

print()
print("ROUTE TRACE")
print("-" * 116)


for record in VERSION_A_ROUTE_VALIDATION:

    result = (
        VERSION_A_ROUTE_RESULTS[
            record[
                "test"
            ]
        ]
    )


    print(
        f"{record['test']:<12} | "
        f"Route={record['actual_route'].upper():<6} | "
        f"Sources={record['sources_searched']:<10} | "
        f"Evidence={record['result_count']} | "
        f"Latency={record['retrieval_time_s']:.3f}s"
    )


    routing = (
        result.get(
            "routing",
            {}
        )
    )


    if result[
        "route"
    ] in (
        "3gpp",
        "hybrid",
    ):

        print(
            f"{'':14}"
            f"3GPP Specs       : "
            f"{routing.get('gpp_specs', [])}"
        )


    if result[
        "route"
    ] in (
        "tcc",
        "hybrid",
    ):

        tcc_result = (
            result.get(
                "tcc"
            )
        )


        print(
            f"{'':14}"
            f"TCC Collections  : "
            f"{routing.get('tcc_collections', [])}"
        )


        if tcc_result:

            print(
                f"{'':14}"
                f"TCC Shards       : "
                f"{len(tcc_result.get('selected_shards', []))}"
                f" | "
                f"Mode="
                f"{tcc_result.get('shard_selection', '')}"
                f" | "
                f"Errors="
                f"{tcc_result.get('shard_errors', 0)}"
            )


# ============================================================
# 2C.31 — FINAL THREE-ROUTE VALIDATION
# ============================================================

MODULE4_VERSION_A_3GPP_PASS = bool(

    VERSION_A_ROUTE_VALIDATION_DF.loc[
        VERSION_A_ROUTE_VALIDATION_DF[
            "test"
        ]
        ==
        "3GPP_ONLY",

        "passed",
    ].iloc[0]
)


MODULE4_VERSION_A_TCC_PASS = bool(

    VERSION_A_ROUTE_VALIDATION_DF.loc[
        VERSION_A_ROUTE_VALIDATION_DF[
            "test"
        ]
        ==
        "TCC_ONLY",

        "passed",
    ].iloc[0]
)


MODULE4_VERSION_A_HYBRID_PASS = bool(

    VERSION_A_ROUTE_VALIDATION_DF.loc[
        VERSION_A_ROUTE_VALIDATION_DF[
            "test"
        ]
        ==
        "HYBRID",

        "passed",
    ].iloc[0]
)


MODULE4_VERSION_A_ALL_ROUTES_PASS = all(
    [
        MODULE4_VERSION_A_3GPP_PASS,
        MODULE4_VERSION_A_TCC_PASS,
        MODULE4_VERSION_A_HYBRID_PASS,
    ]
)


# Upgrade Cell 2C PASS:
# Cell 2C now means the full Version A retrieval architecture
# has passed across all three supported routing paths.

MODULE4_CELL2C_PASS = (
    MODULE4_VERSION_A_ALL_ROUTES_PASS
)


if not MODULE4_CELL2C_PASS:

    raise RuntimeError(
        "Cell 2C three-route Version A "
        "retrieval validation failed."
    )


# ============================================================
# 2C.32 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — VERSION A RETRIEVAL ENGINE FINAL SUMMARY")
print("=" * 116)


print(
    f"3GPP-only Route             : "
    f"{'PASS' if MODULE4_VERSION_A_3GPP_PASS else 'FAIL'}"
)


print(
    f"TCC-only Route              : "
    f"{'PASS' if MODULE4_VERSION_A_TCC_PASS else 'FAIL'}"
)


print(
    f"Hybrid Route                : "
    f"{'PASS' if MODULE4_VERSION_A_HYBRID_PASS else 'FAIL'}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{VERSION_A_MULTI_ROUTE_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{VERSION_A_MULTI_ROUTE_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{VERSION_A_MULTI_ROUTE_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{VERSION_A_MULTI_ROUTE_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("✓ CELL 2C PASSED")
print("✓ 3GPP-only direct retrieval validated.")
print("✓ TCC-only direct retrieval validated.")
print("✓ Hybrid 3GPP + TCC direct retrieval validated.")
print("✓ Query determines the Version A knowledge route.")
print("✓ Retrieval engine validated independently of FastMCP.")
print("✓ No persistent MCP knowledge index created.")
print("✓ Ready for Cell 2D — FastMCP knowledge-service validation.")

print("=" * 116)


MODULE 4 DEMO — MCP VERSION A LIVE RETRIEVAL
Query                      : What are the primary responsibilities of the AMF in a 5G Standalone network?
Route                      : 3GPP
3GPP Specifications        : ['23.501']
TCC Collections            : ['3GPP-TSG']

Executing live Version A retrieval...

TOP-K MCP VERSION A EVIDENCE
--------------------------------------------------------------------------------------------------------------------


,rank,score,source,identifier,release,title,snippet
0,1,60.0,3GPP,23.501,20,3GPP TS 23.501 V20.0.0 (2025-12) ---,r IP address preservation during EPC to 5GC mo...
1,2,54.0,3GPP,23.501,20,3GPP TS 23.501 V20.0.0 (2025-12) ---,is(are) configured to broadcast in the system ...



MODULE 4 DEMO — MCP VERSION A RETRIEVAL SUMMARY
Architecture                : Version A
Route                       : 3GPP
Sources Searched            : ['3GPP']
Evidence Returned           : 2
Total Retrieval Latency     : 0.584 s
Requested Specs             : ['23.501']
Remote Documents Fetched    : 1
3GPP Document Errors        : 0

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Process RAM Before           : 8.061 GiB
Process RAM After            : 8.075 GiB
RAM Delta                   : +0.014 GiB
System RAM Available        : 40.955 GiB

VALIDATION
--------------------------------------------------------------------------------------------------------------------
Router Validation PASS      : True
Live Result PASS            : True
Route PASS                  : True
Source PASS                 : True
Evidence Text PASS          : True
Version A Retrieval PASS    : True

✓ CELL 2C PASSED
✓ Telecom

Version A Route Validation:   0%|          | 0/2 [00:00<?, ?route/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


THREE-ROUTE RETRIEVAL RESULTS
--------------------------------------------------------------------------------------------------------------------


,test,expected_route,actual_route,sources_searched,evidence_sources,result_count,retrieval_time_s,passed
0,3GPP_ONLY,3gpp,3gpp,3GPP,3GPP,2,0.584,True
1,TCC_ONLY,tcc,tcc,TCC,TCC,5,7.017,True
2,HYBRID,hybrid,hybrid,"3GPP, TCC","3GPP, TCC",5,7.185,True



ROUTE TRACE
--------------------------------------------------------------------------------------------------------------------
3GPP_ONLY    | Route=3GPP   | Sources=3GPP       | Evidence=2 | Latency=0.584s
              3GPP Specs       : ['23.501']
TCC_ONLY     | Route=TCC    | Sources=TCC        | Evidence=5 | Latency=7.017s
              TCC Collections  : ['IETF-RFCs', 'IETF-Drafts']
              TCC Shards       : 5 | Mode=SPREAD | Errors=0
HYBRID       | Route=HYBRID | Sources=3GPP, TCC  | Evidence=5 | Latency=7.185s
              3GPP Specs       : ['23.501']
              TCC Collections  : ['IETF-RFCs', 'IETF-Drafts']
              TCC Shards       : 5 | Mode=ADAPTIVE | Errors=0

MODULE 4 DEMO — VERSION A RETRIEVAL ENGINE FINAL SUMMARY
3GPP-only Route             : PASS
TCC-only Route              : PASS
Hybrid Route                : PASS

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Proc

##### Cell 2C — Observation

All three retrieval routes—dedicated 3GPP, TCC, and internal hybrid—validated without shard errors. The route-dependent latency profile remained consistent with the intended Version A design.


#### Cell 2D — FastMCP Knowledge Tool + Route Validation

**Description:** Expose the restored Version A retrieval engine through the real FastMCP client/tool boundary.


In [13]:
# ============================================================
# CELL 2D — FASTMCP KNOWLEDGE TOOL + THREE-ROUTE VALIDATION
# ============================================================
#
# External contract
# -----------------
#
#     search_telecom_knowledge(
#         query,
#         top_k=5
#     )
#
#
# Internal architecture
# ---------------------
#
# FastMCP Client
#      │
#      ▼
# search_telecom_knowledge()
#      │
#      ▼
# Version A Router
#   /    |    \
#  /     |     \
# 3GPP  TCC   HYBRID
#  │      │      │
#  └──────┴──────┘
#         │
#         ▼
# Bounded ranked evidence
#
#
# IMPORTANT
# ---------
#
# The caller does NOT choose the underlying knowledge source.
#
# The Version A retrieval service decides whether the query
# requires:
#
#     - dedicated 3GPP
#     - TCC
#     - Hybrid 3GPP + TCC
#
# Full remote documents are not returned through MCP.
# ============================================================


# ============================================================
# 2D.1 — VERIFY CELL 2C
# ============================================================

if (
    "MODULE4_CELL2C_PASS"
    not in globals()
    or
    not MODULE4_CELL2C_PASS
):

    raise RuntimeError(
        "Cell 2C must pass before Cell 2D."
    )


# ============================================================
# 2D.2 — IMPORTS
# ============================================================

import gc
import os
import re
import time

import pandas as pd
import psutil

from fastmcp import FastMCP, Client
from tqdm.notebook import tqdm


# ============================================================
# 2D.3 — MCP SERVER
# ============================================================

mcp = FastMCP(
    "Telecom Knowledge Service — Version A"
)


# ============================================================
# 2D.4 — JSON-SAFE RESULT HELPERS
# ============================================================

def clean_mcp_text(
    value,
    max_chars=MCP_EXCERPT_CHARS,
):

    if value is None:
        return ""


    text = str(
        value
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


    if len(
        text
    ) > max_chars:

        text = (
            text[
                :max_chars
            ].rstrip()
            +
            " ..."
        )


    return text


def safe_float(
    value,
    default=0.0,
):

    try:

        if value is None:

            return float(
                default
            )


        return float(
            value
        )


    except Exception:

        return float(
            default
        )


def safe_int(
    value,
    default=0,
):

    try:

        if value is None:

            return int(
                default
            )


        return int(
            value
        )


    except Exception:

        return int(
            default
        )


# ============================================================
# 2D.5 — FORMAT ONE MCP EVIDENCE ITEM
# ============================================================

def format_mcp_evidence(
    item,
    rank,
):

    source_family = str(
        item.get(
            "source_family",
            "",
        )
        or
        ""
    )


    identifier = str(
        item.get(
            "identifier",
            "",
        )
        or
        ""
    )


    collection = str(
        item.get(
            "collection",
            "",
        )
        or
        ""
    )


    title = clean_mcp_text(
        item.get(
            "title",
            "",
        ),
        max_chars=500,
    )


    source_path = str(
        item.get(
            "source_path",
            "",
        )
        or
        ""
    )


    release = item.get(
        "release"
    )


    if release is not None:

        release = str(
            release
        )


    evidence_text = clean_mcp_text(
        item.get(
            "text",
            "",
        ),
        max_chars=
            MCP_EXCERPT_CHARS,
    )


    return {

        "rank":
            safe_int(
                rank
            ),

        "source_family":
            source_family,

        "collection":
            collection,

        "identifier":
            identifier,

        "title":
            title,

        "release":
            release,

        "source_path":
            source_path,

        # Retain provenance for later demo inspection.
        "source_shard":
            str(
                item.get(
                    "source_shard",
                    "",
                )
                or
                ""
            ),

        "section_heading":
            clean_mcp_text(
                item.get(
                    "section_heading",
                    "",
                ),
                max_chars=500,
            ),

        # Retrieval diagnostics.
        "relevance_score":
            safe_float(
                item.get(
                    "relevance_score",
                    0,
                )
            ),

        "matched_terms":
            safe_int(
                item.get(
                    "matched_terms",
                    0,
                )
            ),

        "matched_phrases":
            safe_int(
                item.get(
                    "matched_phrases",
                    0,
                )
            ),

        "proximity_score":
            safe_float(
                item.get(
                    "proximity_score",
                    0,
                )
            ),

        "primary_coverage":
            safe_float(
                item.get(
                    "primary_coverage",
                    0,
                )
            ),

        # Bounded content exposed to downstream generation.
        "evidence":
            evidence_text,

        "evidence_chars":
            len(
                evidence_text
            ),
    }


# ============================================================
# 2D.6 — BUILD LIGHTWEIGHT MCP TRACE
# ============================================================

def build_mcp_trace(
    retrieval_result
):

    routing = (
        retrieval_result.get(
            "routing",
            {},
        )
    )


    trace = {

        "architecture":
            ARCHITECTURE_VERSION,

        "retrieval_architecture":
            RETRIEVAL_ARCHITECTURE,

        "route":
            retrieval_result.get(
                "route",
                "",
            ),

        "sources_searched":
            list(
                retrieval_result.get(
                    "sources_searched",
                    [],
                )
            ),

        "retrieval_time_s":
            safe_float(
                retrieval_result.get(
                    "retrieval_time_s",
                    0,
                )
            ),

        "gpp_specs":
            list(
                routing.get(
                    "gpp_specs",
                    [],
                )
            ),

        "tcc_collections":
            list(
                routing.get(
                    "tcc_collections",
                    [],
                )
            ),
    }


    # --------------------------------------------------------
    # 3GPP trace
    # --------------------------------------------------------

    gpp_result = (
        retrieval_result.get(
            "3gpp"
        )
    )


    if gpp_result is not None:

        trace[
            "3gpp"
        ] = {

            "requested_specs":
                list(
                    gpp_result.get(
                        "requested_specs",
                        [],
                    )
                ),

            "selected_specs":
                [

                    {
                        "spec_number":
                            str(
                                item.get(
                                    "spec_number",
                                    "",
                                )
                            ),

                        "release":
                            item.get(
                                "release"
                            ),
                    }

                    for item
                    in gpp_result.get(
                        "selected_specs",
                        [],
                    )
                ],

            "missing_specs":
                list(
                    gpp_result.get(
                        "missing_specs",
                        [],
                    )
                ),

            "document_errors":
                safe_int(
                    gpp_result.get(
                        "document_errors",
                        0,
                    )
                ),

            "retrieval_time_s":
                safe_float(
                    gpp_result.get(
                        "retrieval_time_s",
                        0,
                    )
                ),
        }


    else:

        trace[
            "3gpp"
        ] = None


    # --------------------------------------------------------
    # TCC trace
    # --------------------------------------------------------

    tcc_result = (
        retrieval_result.get(
            "tcc"
        )
    )


    if tcc_result is not None:

        trace[
            "tcc"
        ] = {

            "collections":
                list(
                    tcc_result.get(
                        "collections",
                        [],
                    )
                ),

            "shard_selection":
                str(
                    tcc_result.get(
                        "shard_selection",
                        "",
                    )
                ),

            "shards_searched":
                len(
                    tcc_result.get(
                        "selected_shards",
                        [],
                    )
                ),

            "shard_errors":
                safe_int(
                    tcc_result.get(
                        "shard_errors",
                        0,
                    )
                ),

            "retrieval_time_s":
                safe_float(
                    tcc_result.get(
                        "retrieval_time_s",
                        0,
                    )
                ),
        }


    else:

        trace[
            "tcc"
        ] = None


    return trace


# ============================================================
# 2D.7 — UNIFIED MCP KNOWLEDGE TOOL
# ============================================================

@mcp.tool()
async def search_telecom_knowledge(
    query: str,
    top_k: int = TOP_K_RESULTS,
) -> dict:

    """
    Search authoritative telecommunications documentation.

    The knowledge service dynamically selects the appropriate
    Version A retrieval route:

        - dedicated 3GPP
        - GSMA Telco Common Corpus
        - Hybrid 3GPP + TCC

    The caller does not select the underlying source.

    Returns ranked bounded evidence together with lightweight
    retrieval provenance and timing metadata.
    """


    # --------------------------------------------------------
    # Input validation
    # --------------------------------------------------------

    if (
        not query
        or
        not query.strip()
    ):

        raise ValueError(
            "query must not be empty."
        )


    requested_top_k = safe_int(
        top_k,
        default=
            TOP_K_RESULTS,
    )


    effective_top_k = max(
        1,
        min(
            requested_top_k,
            TOP_K_RESULTS,
            MAX_RETRIEVED_SOURCES,
        ),
    )


    tool_start = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # Dynamic Version A retrieval
    # --------------------------------------------------------

    retrieval_result = (
        await retrieve_version_a(
            query=
                query,

            top_k=
                effective_top_k,
        )
    )


    # --------------------------------------------------------
    # Format bounded MCP evidence
    # --------------------------------------------------------

    raw_results = (
        retrieval_result.get(
            "results",
            [],
        )
    )


    evidence = [

        format_mcp_evidence(
            item=
                item,

            rank=
                rank,
        )

        for rank, item
        in enumerate(
            raw_results[
                :effective_top_k
            ],
            start=1,
        )
    ]


    # --------------------------------------------------------
    # Evidence-source distribution
    # --------------------------------------------------------

    source_distribution = {}


    for item in evidence:

        source_family = (
            item[
                "source_family"
            ]
            or
            "UNKNOWN"
        )


        source_distribution[
            source_family
        ] = (

            source_distribution.get(
                source_family,
                0,
            )
            +
            1
        )


    # --------------------------------------------------------
    # Lightweight trace
    # --------------------------------------------------------

    trace = (
        build_mcp_trace(
            retrieval_result
        )
    )


    tool_elapsed_s = (
        time.perf_counter()
        -
        tool_start
    )


    trace[
        "tool_elapsed_s"
    ] = safe_float(
        tool_elapsed_s
    )


    # --------------------------------------------------------
    # Final MCP payload
    # --------------------------------------------------------

    return {

        "query":
            query,

        "route":
            retrieval_result.get(
                "route",
                "",
            ),

        "sources_searched":
            list(
                retrieval_result.get(
                    "sources_searched",
                    [],
                )
            ),

        "source_distribution":
            source_distribution,

        "result_count":
            len(
                evidence
            ),

        "top_k":
            effective_top_k,

        "evidence":
            evidence,

        "trace":
            trace,
    }


# ============================================================
# 2D.8 — STATIC MCP SERVICE VALIDATION
# ============================================================

if mcp is None:

    raise RuntimeError(
        "FastMCP service was not created."
    )


if not callable(
    search_telecom_knowledge
):

    raise RuntimeError(
        "search_telecom_knowledge() "
        "was not registered correctly."
    )


# ============================================================
# 2D.9 — THREE-ROUTE VALIDATION CASES
# ============================================================

MCP_VALIDATION_CASES = [

    # --------------------------------------------------------
    # 3GPP-only
    # --------------------------------------------------------

    {
        "id":
            "3GPP_ONLY",

        "query":
            (
                "Explain the primary responsibilities of the AMF "
                "in a 5G Standalone network, including registration "
                "and mobility management."
            ),

        "expected_route":
            "3gpp",

        "expected_sources":
            {
                "3GPP"
            },

        "require_evidence_sources":
            {
                "3GPP"
            },
    },


    # --------------------------------------------------------
    # TCC-only
    # --------------------------------------------------------

    {
        "id":
            "TCC_ONLY",

        "query":
            (
                "Explain QUIC connection migration and the "
                "relevant IETF mechanisms."
            ),

        "expected_route":
            "tcc",

        "expected_sources":
            {
                "TCC"
            },

        "require_evidence_sources":
            {
                "TCC"
            },
    },


    # --------------------------------------------------------
    # Hybrid
    # --------------------------------------------------------

    {
        "id":
            "HYBRID",

        "query":
            (
                "Explain how QUIC transport could relate to "
                "traffic carried through a 5G Core network."
            ),

        "expected_route":
            "hybrid",

        "expected_sources":
            {
                "3GPP",
                "TCC",
            },

        "require_evidence_sources":
            {
                "3GPP",
                "TCC",
            },
    },

]


MCP_VALIDATION_TOP_K = (
    TOP_K_RESULTS
)


# ============================================================
# 2D.10 — MEMORY BASELINE
# ============================================================

BYTES_PER_GIB = (
    1024 ** 3
)


gc.collect()


MCP_BOUNDARY_RAM_BEFORE_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    BYTES_PER_GIB
)


MCP_BOUNDARY_AVAILABLE_BEFORE_GIB = (

    psutil.virtual_memory()
    .available
    /
    BYTES_PER_GIB
)


# ============================================================
# 2D.11 — CREATE REAL FASTMCP CLIENT
# ============================================================

mcp_client = Client(
    mcp
)


mcp_validation_records = []

mcp_validation_payloads = {}


print("=" * 116)

print(
    "MODULE 4 DEMO — VERSION A THREE-ROUTE FASTMCP VALIDATION"
)

print("=" * 116)


print(
    "Invocation Path             : "
    "FastMCP Client → MCP Tool → Router → Remote Retrieval"
)


print(
    f"MCP Service                 : "
    f"Telecom Knowledge Service — Version A"
)


print(
    f"Exposed Tool                : "
    f"search_telecom_knowledge"
)


print(
    f"Validation Cases            : "
    f"{len(MCP_VALIDATION_CASES)}"
)


print(
    f"Maximum Top-K               : "
    f"{MCP_VALIDATION_TOP_K}"
)


print(
    f"Evidence Excerpt Limit      : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)


print()


# ============================================================
# 2D.12 — EXECUTE ALL THREE ROUTES THROUGH MCP
# ============================================================

validation_progress = tqdm(
    total=
        len(
            MCP_VALIDATION_CASES
        ),

    desc=
        "FastMCP Route Validation",

    unit=
        "route",

    leave=True,
)


async with mcp_client:

    for case in MCP_VALIDATION_CASES:

        validation_progress.set_postfix_str(
            case[
                "id"
            ]
        )


        # ----------------------------------------------------
        # MCP client → server → tool → retrieval → response
        # ----------------------------------------------------

        roundtrip_start = (
            time.perf_counter()
        )


        call_result = (
            await mcp_client.call_tool(

                "search_telecom_knowledge",

                {
                    "query":
                        case[
                            "query"
                        ],

                    "top_k":
                        MCP_VALIDATION_TOP_K,
                },
            )
        )


        roundtrip_s = (
            time.perf_counter()
            -
            roundtrip_start
        )


        # ----------------------------------------------------
        # Structured MCP response
        # ----------------------------------------------------

        payload = getattr(
            call_result,
            "data",
            None,
        )


        if payload is None:

            payload = getattr(
                call_result,
                "structured_content",
                None,
            )


        if not isinstance(
            payload,
            dict,
        ):

            validation_progress.close()

            raise RuntimeError(
                f"{case['id']} did not return "
                "a structured MCP dictionary."
            )


        mcp_validation_payloads[
            case[
                "id"
            ]
        ] = payload


        # ----------------------------------------------------
        # Response components
        # ----------------------------------------------------

        evidence = (
            payload.get(
                "evidence",
                [],
            )
        )


        trace = (
            payload.get(
                "trace",
                {},
            )
        )


        route = (
            payload.get(
                "route",
                "",
            )
        )


        sources_searched = set(
            payload.get(
                "sources_searched",
                [],
            )
        )


        evidence_sources = {

            item.get(
                "source_family",
                "UNKNOWN",
            )

            for item
            in evidence
        }


        retrieval_time_s = safe_float(
            trace.get(
                "retrieval_time_s",
                0,
            )
        )


        tool_elapsed_s = safe_float(
            trace.get(
                "tool_elapsed_s",
                0,
            )
        )


        boundary_overhead_s = max(
            0.0,
            (
                roundtrip_s
                -
                tool_elapsed_s
            ),
        )


        # ----------------------------------------------------
        # Validation checks
        # ----------------------------------------------------

        route_correct = (
            route
            ==
            case[
                "expected_route"
            ]
        )


        sources_correct = (
            sources_searched
            ==
            case[
                "expected_sources"
            ]
        )


        results_returned = (
            len(
                evidence
            )
            >
            0
        )


        top_k_respected = (
            len(
                evidence
            )
            <=
            TOP_K_RESULTS
        )


        source_limit_respected = (
            len(
                evidence
            )
            <=
            MAX_RETRIEVED_SOURCES
        )


        excerpt_limit_respected = all(

            len(
                str(
                    item.get(
                        "evidence",
                        "",
                    )
                )
            )
            <=
            (
                MCP_EXCERPT_CHARS
                +
                4
            )

            for item
            in evidence
        )


        expected_evidence_present = (

            case[
                "require_evidence_sources"
            ]
            .issubset(
                evidence_sources
            )
        )


        latency_recorded = (

            retrieval_time_s
            >
            0

            and

            tool_elapsed_s
            >
            0

            and

            roundtrip_s
            >
            0
        )


        case_passed = all(
            [
                route_correct,
                sources_correct,
                results_returned,
                top_k_respected,
                source_limit_respected,
                excerpt_limit_respected,
                expected_evidence_present,
                latency_recorded,
            ]
        )


        # ----------------------------------------------------
        # Record runtime results
        # ----------------------------------------------------

        mcp_validation_records.append(
            {

                "test":
                    case[
                        "id"
                    ],

                "expected_route":
                    case[
                        "expected_route"
                    ],

                "actual_route":
                    route,

                "sources_searched":
                    ", ".join(
                        sorted(
                            sources_searched
                        )
                    ),

                "evidence_sources":
                    ", ".join(
                        sorted(
                            evidence_sources
                        )
                    ),

                "result_count":
                    len(
                        evidence
                    ),

                "retrieval_time_s":
                    retrieval_time_s,

                "tool_elapsed_s":
                    tool_elapsed_s,

                "mcp_roundtrip_s":
                    roundtrip_s,

                "mcp_overhead_s":
                    boundary_overhead_s,

                "route_correct":
                    route_correct,

                "sources_correct":
                    sources_correct,

                "evidence_sources_correct":
                    expected_evidence_present,

                "top_k_respected":
                    top_k_respected,

                "excerpt_limit_respected":
                    excerpt_limit_respected,

                "passed":
                    case_passed,
            }
        )


        validation_progress.update(
            1
        )


validation_progress.set_postfix_str(
    "Complete"
)

validation_progress.close()


# ============================================================
# 2D.13 — VALIDATION DATAFRAME
# ============================================================

MCP_VALIDATION_DF = (
    pd.DataFrame(
        mcp_validation_records
    )
)


# ============================================================
# 2D.14 — MEMORY AFTER THREE-ROUTE TEST
# ============================================================

gc.collect()


MCP_BOUNDARY_RAM_AFTER_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    BYTES_PER_GIB
)


MCP_BOUNDARY_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory()
    .available
    /
    BYTES_PER_GIB
)


MCP_BOUNDARY_RAM_DELTA_GIB = (

    MCP_BOUNDARY_RAM_AFTER_GIB
    -
    MCP_BOUNDARY_RAM_BEFORE_GIB
)


# ============================================================
# 2D.15 — GLOBAL VALIDATION
# ============================================================

ALL_MCP_CASES_PASSED = all(

    record[
        "passed"
    ]

    for record
    in mcp_validation_records
)


MODULE4_MCP_TOOL_PASS = (

    ALL_MCP_CASES_PASSED
)


MODULE4_CELL2D_PASS = (

    MODULE4_MCP_TOOL_PASS
)


if not MODULE4_CELL2D_PASS:

    raise RuntimeError(
        "Version A three-route FastMCP validation failed."
    )


# ============================================================
# 2D.16 — COMPACT RESULT TABLE
# ============================================================

print()
print("THREE-ROUTE MCP VALIDATION RESULTS")
print("-" * 116)


display(

    MCP_VALIDATION_DF[
        [
            "test",
            "actual_route",
            "sources_searched",
            "evidence_sources",
            "result_count",
            "retrieval_time_s",
            "tool_elapsed_s",
            "mcp_roundtrip_s",
            "mcp_overhead_s",
            "passed",
        ]
    ]
    .round(
        {
            "retrieval_time_s":
                3,

            "tool_elapsed_s":
                3,

            "mcp_roundtrip_s":
                3,

            "mcp_overhead_s":
                3,
        }
    )
)


# ============================================================
# 2D.17 — ROUTE-SPECIFIC TRACE SUMMARY
# ============================================================

print()
print("ROUTE TRACE")
print("-" * 116)


for record in mcp_validation_records:

    payload = (
        mcp_validation_payloads[
            record[
                "test"
            ]
        ]
    )


    trace = payload.get(
        "trace",
        {},
    )


    print(
        f"{record['test']:<12} | "
        f"Route={record['actual_route'].upper():<6} | "
        f"Evidence={record['result_count']} | "
        f"Retrieval={record['retrieval_time_s']:.3f}s | "
        f"MCP={record['mcp_roundtrip_s']:.3f}s"
    )


    if trace.get(
        "3gpp"
    ):

        print(
            f"{'':14}"
            f"3GPP Specs       : "
            f"{trace.get('gpp_specs', [])}"
        )


    if trace.get(
        "tcc"
    ):

        tcc_trace = (
            trace[
                "tcc"
            ]
        )


        print(
            f"{'':14}"
            f"TCC Collections  : "
            f"{tcc_trace.get('collections', [])}"
        )


        print(
            f"{'':14}"
            f"TCC Shards       : "
            f"{tcc_trace.get('shards_searched', 0)}"
            f" | "
            f"Mode={tcc_trace.get('shard_selection', '')}"
            f" | "
            f"Errors={tcc_trace.get('shard_errors', 0)}"
        )


# ============================================================
# 2D.18 — FINAL SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — FASTMCP KNOWLEDGE SERVICE SUMMARY")
print("=" * 116)


print(
    f"MCP Service                 : "
    f"Telecom Knowledge Service — Version A"
)


print(
    f"Exposed Tool                : "
    f"search_telecom_knowledge(query, top_k=5)"
)


print(
    f"Internal Routes             : "
    f"3GPP / TCC / HYBRID"
)


print(
    f"Validation Cases            : "
    f"{len(MCP_VALIDATION_CASES)}"
)


print(
    f"Cases Passed                : "
    f"{sum(record['passed'] for record in mcp_validation_records)}"
    f"/"
    f"{len(mcp_validation_records)}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{MCP_BOUNDARY_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{MCP_BOUNDARY_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{MCP_BOUNDARY_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{MCP_BOUNDARY_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


for record in mcp_validation_records:

    print(
        f"{record['test']:<28}: "
        f"{'PASS' if record['passed'] else 'FAIL'}"
    )


print(
    f"{'Overall MCP Validation':<28}: "
    f"{'PASS' if MODULE4_CELL2D_PASS else 'FAIL'}"
)


print()
print("✓ CELL 2D PASSED")
print("✓ Real FastMCP client/server boundary validated.")
print("✓ 3GPP-only route validated through MCP.")
print("✓ TCC-only route validated through MCP.")
print("✓ Hybrid 3GPP + TCC route validated through MCP.")
print("✓ Source selection remains internal to one MCP tool.")
print("✓ Top-K evidence limit enforced.")
print("✓ 2,500-character evidence excerpt limit enforced.")
print("✓ Retrieval and MCP round-trip timing captured.")
print("✓ No persistent local MCP knowledge base created.")
print("✓ Ready for Section 2 — Hosted LLM generation setup.")

print("=" * 116)


MODULE 4 DEMO — VERSION A THREE-ROUTE FASTMCP VALIDATION
Invocation Path             : FastMCP Client → MCP Tool → Router → Remote Retrieval
MCP Service                 : Telecom Knowledge Service — Version A
Exposed Tool                : search_telecom_knowledge
Validation Cases            : 3
Maximum Top-K               : 5
Evidence Excerpt Limit      : 2,500 chars/source



FastMCP Route Validation:   0%|          | 0/3 [00:00<?, ?route/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


THREE-ROUTE MCP VALIDATION RESULTS
--------------------------------------------------------------------------------------------------------------------


,test,actual_route,sources_searched,evidence_sources,result_count,retrieval_time_s,tool_elapsed_s,mcp_roundtrip_s,mcp_overhead_s,passed
0,3GPP_ONLY,3gpp,3GPP,3GPP,4,1.233,1.234,2.427,1.193,True
1,TCC_ONLY,tcc,TCC,TCC,5,6.729,6.730,6.733,0.004,True
2,HYBRID,hybrid,"3GPP, TCC","3GPP, TCC",5,7.425,7.426,7.429,0.003,True



ROUTE TRACE
--------------------------------------------------------------------------------------------------------------------
3GPP_ONLY    | Route=3GPP   | Evidence=4 | Retrieval=1.233s | MCP=2.427s
              3GPP Specs       : ['23.501', '23.502']
TCC_ONLY     | Route=TCC    | Evidence=5 | Retrieval=6.729s | MCP=6.733s
              TCC Collections  : ['IETF-RFCs', 'IETF-Drafts']
              TCC Shards       : 5 | Mode=ADAPTIVE | Errors=0
HYBRID       | Route=HYBRID | Evidence=5 | Retrieval=7.425s | MCP=7.429s
              3GPP Specs       : ['23.501']
              TCC Collections  : ['IETF-RFCs', 'IETF-Drafts']
              TCC Shards       : 5 | Mode=ADAPTIVE | Errors=0

MODULE 4 DEMO — FASTMCP KNOWLEDGE SERVICE SUMMARY
MCP Service                 : Telecom Knowledge Service — Version A
Exposed Tool                : search_telecom_knowledge(query, top_k=5)
Internal Routes             : 3GPP / TCC / HYBRID
Validation Cases            : 3
Cases Passed                : 3/3

##### Cell 2D — Observation

FastMCP route validation passed across the three knowledge-access scenarios. Most latency originates in the underlying remote retrieval rather than FastMCP orchestration overhead.


# SECTION 3 — Hosted Generation + Hybrid Evidence Runtime


#### Cell 3A — OpenRouter + Gemma 4 Generation Validation

**Description:** Validate hosted generation independently of RAG and MCP using the final Gemma model configuration.


In [14]:
# ============================================================
# CELL 3A — OPENROUTER + GEMMA 4 HOSTED GENERATION VALIDATION
# ============================================================
#
# Architecture validated in this cell:
#
#       Test Prompt
#           │
#           ▼
#   OpenAI-Compatible Client
#           │
#           ▼
#       OpenRouter
#           │
#           ▼
# Gemma 4 26B A4B IT
#           │
#           ▼
#      Text Response
#
#
# IMPORTANT
# ---------
#
# RAG is NOT invoked.
# MCP is NOT invoked.
#
# This cell isolates the hosted generation layer so that
# provider/model latency can be measured independently.
# ============================================================


# ============================================================
# 3A.1 — VERIFY PREVIOUS ARCHITECTURE
# ============================================================

if (
    "MODULE4_CELL2D_PASS" not in globals()
    or
    not MODULE4_CELL2D_PASS
):
    raise RuntimeError(
        "Cell 2D must pass before Cell 3A."
    )


# ============================================================
# 3A.2 — IMPORTS
# ============================================================

import gc
import os
import time

import psutil

from openai import OpenAI


# ============================================================
# 3A.3 — OPENROUTER CONFIGURATION
# ============================================================

OPENROUTER_BASE_URL = (
    "https://openrouter.ai/api/v1"
)

LLM_PROVIDER = (
    "OpenRouter"
)

LLM_MODEL = (
    "google/gemma-4-26b-a4b-it"
)

LLM_DISPLAY_NAME = (
    "Gemma 4 26B A4B IT"
)

LLM_MAX_TOKENS = 1800

# Preserve previous experimental convention:
# provider/API default temperature.
LLM_TEMPERATURE = None


# ============================================================
# 3A.4 — API KEY VALIDATION
# ============================================================

if (
    "OPENROUTER_API_KEY" not in globals()
    or
    not OPENROUTER_API_KEY
):

    OPENROUTER_API_KEY = (
        os.environ.get(
            "OPENROUTER_API_KEY"
        )
    )


if not OPENROUTER_API_KEY:

    raise RuntimeError(
        "OPENROUTER_API_KEY is not available. "
        "Cell 0B must load the OpenRouter secret."
    )


# ============================================================
# 3A.5 — OPENROUTER CLIENT
# ============================================================

openrouter_client = OpenAI(

    base_url=
        OPENROUTER_BASE_URL,

    api_key=
        OPENROUTER_API_KEY,
)


# Keep a compatibility alias for later cells.
openrouter = (
    openrouter_client
)


# ============================================================
# 3A.6 — MODEL / CONTROL VALIDATION
# ============================================================

if LLM_PROVIDER != "OpenRouter":

    raise RuntimeError(
        "Hosted generation provider must be OpenRouter."
    )


if LLM_MODEL != "google/gemma-4-26b-a4b-it":

    raise RuntimeError(
        "Unexpected hosted generation model."
    )


if LLM_MAX_TOKENS != 1800:

    raise RuntimeError(
        "LLM_MAX_TOKENS must remain 1800."
    )


# ============================================================
# 3A.7 — STANDALONE GENERATION HELPER
# ============================================================

def generate_hosted_llm(
    messages,
    max_tokens=LLM_MAX_TOKENS,
):
    """
    Execute one hosted OpenRouter Chat Completions request.

    Temperature remains intentionally unset when
    LLM_TEMPERATURE is None.
    """

    if not isinstance(
        messages,
        list,
    ):

        raise TypeError(
            "messages must be a list."
        )


    request_kwargs = {

        "model":
            LLM_MODEL,

        "messages":
            messages,

        "max_tokens":
            int(
                max_tokens
            ),
    }


    if LLM_TEMPERATURE is not None:

        request_kwargs[
            "temperature"
        ] = float(
            LLM_TEMPERATURE
        )


    start_time = (
        time.perf_counter()
    )


    response = (
        openrouter_client
        .chat
        .completions
        .create(
            **request_kwargs
        )
    )


    elapsed_s = (
        time.perf_counter()
        -
        start_time
    )


    if (
        not response.choices
        or
        response.choices[0].message is None
    ):

        raise RuntimeError(
            "OpenRouter returned no completion message."
        )


    text = (
        response
        .choices[0]
        .message
        .content
    )


    text = str(
        text
        or
        ""
    ).strip()


    if not text:

        raise RuntimeError(
            "OpenRouter returned an empty completion."
        )


    usage = getattr(
        response,
        "usage",
        None,
    )


    return {

        "text":
            text,

        "elapsed_s":
            elapsed_s,

        "response_id":
            getattr(
                response,
                "id",
                None,
            ),

        "response_model":
            getattr(
                response,
                "model",
                None,
            ),

        "prompt_tokens":
            (
                getattr(
                    usage,
                    "prompt_tokens",
                    None,
                )
                if usage
                else None
            ),

        "completion_tokens":
            (
                getattr(
                    usage,
                    "completion_tokens",
                    None,
                )
                if usage
                else None
            ),

        "total_tokens":
            (
                getattr(
                    usage,
                    "total_tokens",
                    None,
                )
                if usage
                else None
            ),

        "finish_reason":
            getattr(
                response.choices[0],
                "finish_reason",
                None,
            ),
    }


# ============================================================
# 3A.8 — MEMORY BASELINE
# ============================================================

gc.collect()


HOSTED_LLM_RAM_BEFORE_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


HOSTED_LLM_AVAILABLE_BEFORE_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


# ============================================================
# 3A.9 — MINIMAL GENERATION SMOKE TEST
# ============================================================
#
# This deliberately does NOT test telecom grounding.
#
# It only validates that the hosted LLM endpoint works.
# Grounding will be tested later using retrieved evidence.
# ============================================================

HOSTED_LLM_TEST_MESSAGES = [

    {
        "role":
            "system",

        "content":
            (
                "You are a telecommunications engineering assistant. "
                "Answer the user's test request in one concise sentence."
            ),
    },

    {
        "role":
            "user",

        "content":
            (
                "Confirm that the hosted telecom language-model "
                "generation path is operational."
            ),
    },

]


print("=" * 116)
print("MODULE 4 DEMO — HOSTED LLM GENERATION VALIDATION")
print("=" * 116)


print(
    f"Provider                    : "
    f"{LLM_PROVIDER}"
)


print(
    f"Model                       : "
    f"{LLM_MODEL}"
)


print(
    f"Display Name                : "
    f"{LLM_DISPLAY_NAME}"
)


print(
    f"Max Output Tokens           : "
    f"{LLM_MAX_TOKENS}"
)


print(
    "Temperature                 : "
    "API DEFAULT"
)


print()
print("Executing standalone hosted generation...")
print()


HOSTED_LLM_SMOKE_RESULT = (
    generate_hosted_llm(

        messages=
            HOSTED_LLM_TEST_MESSAGES,

        # Keep this connectivity test very small.
        max_tokens=
            80,
    )
)


# ============================================================
# 3A.10 — MEMORY AFTER GENERATION
# ============================================================

gc.collect()


HOSTED_LLM_RAM_AFTER_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


HOSTED_LLM_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


HOSTED_LLM_RAM_DELTA_GIB = (

    HOSTED_LLM_RAM_AFTER_GIB
    -
    HOSTED_LLM_RAM_BEFORE_GIB
)


# ============================================================
# 3A.11 — VALIDATION
# ============================================================

HOSTED_LLM_TEXT_PASS = (

    bool(
        HOSTED_LLM_SMOKE_RESULT[
            "text"
        ]
    )
)


HOSTED_LLM_LATENCY_PASS = (

    HOSTED_LLM_SMOKE_RESULT[
        "elapsed_s"
    ]
    >
    0
)


HOSTED_LLM_FINISH_PASS = (

    HOSTED_LLM_SMOKE_RESULT[
        "finish_reason"
    ]
    is not None
)


MODULE4_HOSTED_LLM_PASS = all(
    [
        HOSTED_LLM_TEXT_PASS,
        HOSTED_LLM_LATENCY_PASS,
        HOSTED_LLM_FINISH_PASS,
    ]
)


MODULE4_CELL3A_PASS = (
    MODULE4_HOSTED_LLM_PASS
)


if not MODULE4_CELL3A_PASS:

    raise RuntimeError(
        "Hosted LLM validation failed."
    )


# ============================================================
# 3A.12 — RESULT
# ============================================================

print("HOSTED MODEL RESPONSE")
print("-" * 116)


print(
    HOSTED_LLM_SMOKE_RESULT[
        "text"
    ]
)


print()
print("=" * 116)
print("MODULE 4 DEMO — HOSTED GENERATION SUMMARY")
print("=" * 116)


print(
    f"Provider                    : "
    f"{LLM_PROVIDER}"
)


print(
    f"Requested Model             : "
    f"{LLM_MODEL}"
)


print(
    f"Response Model              : "
    f"{HOSTED_LLM_SMOKE_RESULT['response_model']}"
)


print(
    f"Generation Latency          : "
    f"{HOSTED_LLM_SMOKE_RESULT['elapsed_s']:.3f} s"
)


print(
    f"Prompt Tokens               : "
    f"{HOSTED_LLM_SMOKE_RESULT['prompt_tokens']}"
)


print(
    f"Completion Tokens           : "
    f"{HOSTED_LLM_SMOKE_RESULT['completion_tokens']}"
)


print(
    f"Total Tokens                : "
    f"{HOSTED_LLM_SMOKE_RESULT['total_tokens']}"
)


print(
    f"Finish Reason               : "
    f"{HOSTED_LLM_SMOKE_RESULT['finish_reason']}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{HOSTED_LLM_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{HOSTED_LLM_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{HOSTED_LLM_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{HOSTED_LLM_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Response Text PASS          : "
    f"{HOSTED_LLM_TEXT_PASS}"
)


print(
    f"Latency Capture PASS        : "
    f"{HOSTED_LLM_LATENCY_PASS}"
)


print(
    f"Finish Status PASS          : "
    f"{HOSTED_LLM_FINISH_PASS}"
)


print(
    f"Hosted Generation PASS      : "
    f"{MODULE4_HOSTED_LLM_PASS}"
)


print()
print("✓ CELL 3A PASSED")
print("✓ OpenRouter client validated.")
print("✓ Gemma 4 26B A4B IT hosted generation validated.")
print("✓ Generation latency captured independently.")
print("✓ Token usage captured where exposed by provider.")
print("✓ No Semantic RAG retrieval executed.")
print("✓ No MCP retrieval executed.")
print("✓ Local RAG + MCP memory state remains intact.")
print("✓ Ready for Cell 3B — Hybrid evidence contract + context builder.")

print("=" * 116)


MODULE 4 DEMO — HOSTED LLM GENERATION VALIDATION
Provider                    : OpenRouter
Model                       : google/gemma-4-26b-a4b-it
Display Name                : Gemma 4 26B A4B IT
Max Output Tokens           : 1800
Temperature                 : API DEFAULT

Executing standalone hosted generation...

HOSTED MODEL RESPONSE
--------------------------------------------------------------------------------------------------------------------
The hosted telecom language-model generation path is confirmed as fully operational and performing within specified parameters.

MODULE 4 DEMO — HOSTED GENERATION SUMMARY
Provider                    : OpenRouter
Requested Model             : google/gemma-4-26b-a4b-it
Response Model              : google/gemma-4-26b-a4b-it
Generation Latency          : 2.159 s
Prompt Tokens               : 51
Completion Tokens           : 20
Total Tokens                : 71
Finish Reason               : stop

MEMORY IMPACT
----------------------------------

##### Cell 3A — Observation

Hosted `google/gemma-4-26b-a4b-it` generation passed with valid response/token accounting and negligible local RAM growth, confirming the deployment notebook does not require local model hosting.


#### Cell 3B — Hybrid Evidence Contract + Context Builder

**Description:** Normalize RAG and MCP results into one evidence schema and enforce the final context/fusion limits.


In [15]:
# ============================================================
# CELL 3B — HYBRID EVIDENCE CONTRACT + CONTEXT BUILDER
# ============================================================
#
# Purpose
# -------
#
# Normalize:
#
#       Semantic RAG V1
#              +
#       MCP Version A
#
# into ONE generator-facing evidence contract.
#
#
# IMPORTANT
# ---------
#
# This cell does NOT:
#
#     - execute RAG retrieval
#     - execute MCP retrieval
#     - execute Hybrid fusion
#     - call the hosted LLM
#
#
# It only defines and validates:
#
#     1. common evidence schema
#     2. provenance preservation
#     3. per-item evidence limit
#     4. total context budget
#     5. generator context formatting
#
# ============================================================


# ============================================================
# 3B.1 — VERIFY PREVIOUS CELLS
# ============================================================

if (
    "MODULE4_CELL2D_PASS" not in globals()
    or
    not MODULE4_CELL2D_PASS
):
    raise RuntimeError(
        "Cell 2D must pass before Cell 3B."
    )


if (
    "MODULE4_CELL3A_PASS" not in globals()
    or
    not MODULE4_CELL3A_PASS
):
    raise RuntimeError(
        "Cell 3A must pass before Cell 3B."
    )


# ============================================================
# 3B.2 — IMPORTS
# ============================================================

import copy
import hashlib
import re


# ============================================================
# 3B.3 — FROZEN MODULE 4 EVIDENCE CONTROLS
# ============================================================

MODULE4_RAG_RETRIEVAL_K = 5

MODULE4_MCP_RETRIEVAL_K = 5

COMMON_MAX_EVIDENCE_ITEMS = 5

NORMALIZED_EVIDENCE_MAX_CHARS = 2_500

COMMON_CONTEXT_MAX_CHARS = 12_500


# Controls required by the later Hybrid fusion stage.
HYBRID_FUSION_METHOD = (
    "RECIPROCAL_RANK_FUSION"
)

HYBRID_RRF_K = 60

HYBRID_NEAR_DUPLICATE_THRESHOLD = 0.88

HYBRID_MIN_ITEMS_PER_RETRIEVER = 2


# ============================================================
# 3B.4 — CONTROL VALIDATION
# ============================================================

if MODULE4_RAG_RETRIEVAL_K != 5:

    raise RuntimeError(
        "RAG retrieval Top-K must remain 5."
    )


if MODULE4_MCP_RETRIEVAL_K != 5:

    raise RuntimeError(
        "MCP retrieval Top-K must remain 5."
    )


if COMMON_MAX_EVIDENCE_ITEMS != 5:

    raise RuntimeError(
        "Maximum presented evidence must remain 5."
    )


if NORMALIZED_EVIDENCE_MAX_CHARS != 2500:

    raise RuntimeError(
        "Evidence item limit must remain 2,500 characters."
    )


if COMMON_CONTEXT_MAX_CHARS != 12500:

    raise RuntimeError(
        "Context evidence budget must remain 12,500 characters."
    )


# ============================================================
# 3B.5 — TEXT NORMALIZATION
# ============================================================

def normalize_evidence_text(
    text
):
    """
    Normalize whitespace while preserving technical content.
    """

    if text is None:
        return ""


    return re.sub(
        r"\s+",
        " ",
        str(
            text
        ),
    ).strip()


def canonical_evidence_text(
    text
):
    """
    Canonical representation used later for duplicate detection.
    """

    text = normalize_evidence_text(
        text
    ).lower()


    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )


    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def bound_evidence_text(
    text,
    max_chars=NORMALIZED_EVIDENCE_MAX_CHARS,
):
    """
    Bound one evidence item before it enters the common contract.
    """

    text = normalize_evidence_text(
        text
    )


    if len(
        text
    ) <= max_chars:

        return text


    return (
        text[
            :max_chars
        ]
        .rstrip()
    )


# ============================================================
# 3B.6 — STABLE EVIDENCE IDENTIFIER
# ============================================================

def build_evidence_id(
    retrieval_system,
    document_id,
    chunk_id,
    text,
):
    """
    Produce a stable evidence identity while keeping RAG and MCP
    provenance distinct.
    """

    payload = (
        f"{retrieval_system}|"
        f"{document_id}|"
        f"{chunk_id}|"
        f"{canonical_evidence_text(text)}"
    )


    digest = (
        hashlib.sha256(
            payload.encode(
                "utf-8"
            )
        )
        .hexdigest()[
            :20
        ]
    )


    return (
        f"{retrieval_system}:"
        f"{digest}"
    )


# ============================================================
# 3B.7 — RAG SOURCE-FAMILY INFERENCE
# ============================================================

def infer_rag_source_family(
    item
):
    """
    Infer a human-readable source family from RAG metadata.

    This affects provenance/display only.
    It does NOT alter RAG ranking.
    """

    metadata = (
        item.get(
            "metadata",
            {}
        )
        or
        {}
    )


    fields = [

        item.get(
            "source",
            ""
        ),

        item.get(
            "title",
            ""
        ),

        item.get(
            "file_path",
            ""
        ),

        item.get(
            "document_id",
            ""
        ),

        metadata.get(
            "source",
            ""
        ),

        metadata.get(
            "title",
            ""
        ),

        metadata.get(
            "file_path",
            ""
        ),

        metadata.get(
            "document_id",
            ""
        ),
    ]


    blob = " ".join(
        str(
            value
        ).lower()

        for value
        in fields

        if value is not None
    )


    source_rules = [

        (
            [
                "3gpp",
                "23.",
                "24.",
                "28.",
                "29.",
                "38.",
            ],
            "3GPP",
        ),

        (
            [
                "etsi"
            ],
            "ETSI",
        ),

        (
            [
                "itu-t",
                "itu "
            ],
            "ITU-T",
        ),

        (
            [
                "gsma"
            ],
            "GSMA",
        ),

        (
            [
                "o-ran",
                "o ran",
                "oran"
            ],
            "O-RAN",
        ),

        (
            [
                "tm forum",
                "tmforum"
            ],
            "TM Forum",
        ),

        (
            [
                "camara"
            ],
            "CAMARA",
        ),

        (
            [
                "kubernetes"
            ],
            "Kubernetes",
        ),

        (
            [
                "ietf",
                "rfc"
            ],
            "IETF",
        ),
    ]


    for signals, family in source_rules:

        if any(
            signal in blob
            for signal
            in signals
        ):

            return family


    return str(
        item.get(
            "source",
            ""
        )
        or
        metadata.get(
            "source",
            ""
        )
        or
        "RAG_CORPUS"
    ).strip()


# ============================================================
# 3B.8 — NORMALIZE RAG RESULTS
# ============================================================

def normalize_rag_results(
    rag_result
):
    """
    Convert native Semantic RAG V1 results into the common
    Module 4 evidence representation.
    """

    normalized = []


    raw_results = (
        rag_result.get(
            "results",
            []
        )
        or
        []
    )


    for fallback_rank, item in enumerate(
        raw_results,
        start=1,
    ):

        metadata = (
            item.get(
                "metadata",
                {}
            )
            or
            {}
        )


        text = bound_evidence_text(
            item.get(
                "text",
                ""
            )
        )


        document_id = str(

            item.get(
                "document_id",
                ""
            )

            or

            metadata.get(
                "document_id",
                ""
            )

            or

            item.get(
                "file_path",
                ""
            )

            or

            metadata.get(
                "file_path",
                ""
            )

            or

            item.get(
                "chunk_id",
                ""
            )

            or

            item.get(
                "vector_id",
                ""
            )
        )


        chunk_id = str(

            item.get(
                "chunk_id",
                ""
            )

            or

            metadata.get(
                "chunk_id",
                ""
            )

            or

            ""
        )


        rank = int(
            item.get(
                "rank",
                fallback_rank
            )
            or
            fallback_rank
        )


        score = float(
            item.get(
                "score",
                0.0
            )
            or
            0.0
        )


        family = (
            infer_rag_source_family(
                item
            )
        )


        title = str(

            item.get(
                "title",
                ""
            )

            or

            metadata.get(
                "title",
                ""
            )

            or

            ""
        )


        source = (

            item.get(
                "source"
            )

            or

            metadata.get(
                "source"
            )
        )


        file_path = (

            item.get(
                "file_path"
            )

            or

            metadata.get(
                "file_path"
            )
        )


        normalized.append({

            "evidence_id":
                build_evidence_id(
                    retrieval_system=
                        "RAG",

                    document_id=
                        document_id,

                    chunk_id=
                        chunk_id,

                    text=
                        text,
                ),

            "retrieval_system":
                "RAG",

            "retrieval_systems":
                [
                    "RAG"
                ],

            "source_family":
                family,

            "source_families":
                [
                    family
                ],

            "document_id":
                document_id,

            "title":
                title,

            "chunk_id":
                chunk_id,

            # Native RAG similarity score.
            "score":
                score,

            "native_ranks": {
                "RAG":
                    rank,
            },

            "native_scores": {
                "RAG":
                    score,
            },

            # Assigned only during Hybrid fusion.
            "fusion_score":
                None,

            "text":
                text,

            "text_chars":
                len(
                    text
                ),

            "provenance": [

                {
                    "retrieval_system":
                        "RAG",

                    "vector_id":
                        item.get(
                            "vector_id"
                        ),

                    "chunk_id":
                        chunk_id,

                    "document_id":
                        document_id,

                    "source":
                        source,

                    "file_path":
                        file_path,

                    "page":
                        (
                            item.get(
                                "page"
                            )
                            or
                            metadata.get(
                                "page"
                            )
                        ),

                    "native_rank":
                        rank,

                    "native_score":
                        score,
                }
            ],
        })


    return normalized


# ============================================================
# 3B.9 — NORMALIZE MCP RESULTS
# ============================================================

def normalize_mcp_results(
    mcp_payload
):
    """
    Convert Version A MCP evidence into the same Module 4
    evidence representation used by Semantic RAG.
    """

    normalized = []


    raw_evidence = (
        mcp_payload.get(
            "evidence",
            []
        )
        or
        []
    )


    for fallback_rank, item in enumerate(
        raw_evidence,
        start=1,
    ):

        text = bound_evidence_text(
            item.get(
                "evidence",
                ""
            )
        )


        source_path = str(
            item.get(
                "source_path",
                ""
            )
            or
            ""
        )


        identifier = str(
            item.get(
                "identifier",
                ""
            )
            or
            ""
        )


        title = str(
            item.get(
                "title",
                ""
            )
            or
            ""
        )


        document_id = (
            identifier
            or
            source_path
            or
            title
        )


        rank = int(
            item.get(
                "rank",
                fallback_rank
            )
            or
            fallback_rank
        )


        score = float(

            item.get(
                "relevance_score",

                item.get(
                    "bm25_score",
                    0.0
                )
            )

            or
            0.0
        )


        family = str(
            item.get(
                "source_family",
                ""
            )
            or
            "MCP"
        )


        normalized.append({

            "evidence_id":
                build_evidence_id(
                    retrieval_system=
                        "MCP",

                    document_id=
                        document_id,

                    chunk_id=
                        "",

                    text=
                        text,
                ),

            "retrieval_system":
                "MCP",

            "retrieval_systems":
                [
                    "MCP"
                ],

            "source_family":
                family,

            "source_families":
                [
                    family
                ],

            "document_id":
                document_id,

            "title":
                title,

            "chunk_id":
                "",

            # Native MCP lexical relevance score.
            "score":
                score,

            "native_ranks": {
                "MCP":
                    rank,
            },

            "native_scores": {
                "MCP":
                    score,
            },

            # Assigned only during Hybrid fusion.
            "fusion_score":
                None,

            "text":
                text,

            "text_chars":
                len(
                    text
                ),

            "provenance": [

                {
                    "retrieval_system":
                        "MCP",

                    "source_family":
                        family,

                    "collection":
                        item.get(
                            "collection"
                        ),

                    "identifier":
                        identifier,

                    "release":
                        item.get(
                            "release"
                        ),

                    "section_heading":
                        item.get(
                            "section_heading"
                        ),

                    "source_path":
                        source_path,

                    "source_shard":
                        item.get(
                            "source_shard"
                        ),

                    "native_rank":
                        rank,

                    "native_score":
                        score,
                }
            ],
        })


    return normalized


# ============================================================
# 3B.10 — COMMON GENERATOR CONTEXT BUDGET
# ============================================================

def apply_context_budget(
    evidence,
    max_items=
        COMMON_MAX_EVIDENCE_ITEMS,
    max_chars=
        COMMON_CONTEXT_MAX_CHARS,
):
    """
    Enforce the same generator-facing evidence budget
    regardless of retrieval architecture.
    """

    selected = []

    used_chars = 0


    for source_item in evidence:

        if len(
            selected
        ) >= max_items:

            break


        remaining = (
            max_chars
            -
            used_chars
        )


        if remaining <= 0:

            break


        item = copy.deepcopy(
            source_item
        )


        text = str(
            item.get(
                "text",
                ""
            )
        )


        if len(
            text
        ) > remaining:

            # Avoid adding an unusably tiny final fragment.
            if remaining < 300:

                break


            text = (
                text[
                    :remaining
                ]
                .rstrip()
            )


        item[
            "text"
        ] = text


        item[
            "text_chars"
        ] = len(
            text
        )


        selected.append(
            item
        )


        used_chars += len(
            text
        )


    for rank, item in enumerate(
        selected,
        start=1,
    ):

        item[
            "presentation_rank"
        ] = rank


    return (
        selected,
        used_chars,
    )


# ============================================================
# 3B.11 — SINGLE-RETRIEVER SELECTION
# ============================================================

def select_single_mode_evidence(
    candidates
):
    """
    Preserve native ranking for RAG_ONLY or MCP_ONLY while
    applying the common generator-facing evidence budget.
    """

    ordered = sorted(

        candidates,

        key=lambda item:
            min(
                item.get(
                    "native_ranks",
                    {
                        "":
                            999999
                    },
                ).values(),

                default=
                    999999,
            ),
    )


    return apply_context_budget(
        ordered
    )


# ============================================================
# 3B.12 — GENERATOR CONTEXT BUILDER
# ============================================================

def build_evidence_context(
    evidence
):
    """
    Convert normalized evidence into the text context that
    will later be supplied to the hosted LLM.
    """

    blocks = []


    for fallback_rank, item in enumerate(
        evidence,
        start=1,
    ):

        presentation_rank = (
            item.get(
                "presentation_rank",
                fallback_rank
            )
        )


        systems = "/".join(
            item.get(
                "retrieval_systems",
                []
            )
        )


        header = (

            f"[E{presentation_rank}] "

            f"Retrieval={systems} | "

            f"Source="
            f"{item.get('source_family', '')} | "

            f"Title="
            f"{item.get('title', '')} | "

            f"Document="
            f"{item.get('document_id', '')}"
        )


        blocks.append(

            header
            +
            "\n"
            +
            item.get(
                "text",
                ""
            )
        )


    return "\n\n".join(
        blocks
    )


# ============================================================
# 3B.13 — CONTRACT VALIDATION SAMPLE
# ============================================================
#
# No retrieval is executed.
#
# The RAG sample validates the expected adapter contract.
#
# For MCP, reuse a REAL payload from Cell 2D where available,
# avoiding another remote query.
# ============================================================

CONTRACT_TEST_RAG_RESULT = {

    "results": [

        {
            "rank":
                1,

            "score":
                0.91,

            "vector_id":
                123,

            "text":
                (
                    "The AMF provides access and mobility "
                    "management functions for the 5G Core."
                ),

            "metadata": {

                "document_id":
                    "23.501",

                "chunk_id":
                    "contract-test-rag-1",

                "title":
                    "3GPP TS 23.501",

                "source":
                    "3GPP",
            },
        }
    ]
}


CONTRACT_TEST_MCP_PAYLOAD = None


if (
    "mcp_validation_payloads"
    in globals()
    and
    isinstance(
        mcp_validation_payloads,
        dict
    )
    and
    "3GPP_ONLY"
    in mcp_validation_payloads
):

    CONTRACT_TEST_MCP_PAYLOAD = (
        mcp_validation_payloads[
            "3GPP_ONLY"
        ]
    )


else:

    # Fallback only if the previous payload is unavailable.
    CONTRACT_TEST_MCP_PAYLOAD = {

        "evidence": [

            {
                "rank":
                    1,

                "source_family":
                    "3GPP",

                "collection":
                    "",

                "identifier":
                    "23.501",

                "title":
                    "3GPP TS 23.501",

                "release":
                    "20",

                "source_path":
                    "",

                "relevance_score":
                    60.0,

                "evidence":
                    (
                        "The AMF provides access and mobility "
                        "management functions for the 5G Core."
                    ),
            }
        ]
    }


CONTRACT_TEST_RAG_NORMALIZED = (
    normalize_rag_results(
        CONTRACT_TEST_RAG_RESULT
    )
)


CONTRACT_TEST_MCP_NORMALIZED = (
    normalize_mcp_results(
        CONTRACT_TEST_MCP_PAYLOAD
    )
)


CONTRACT_TEST_COMBINED = (

    CONTRACT_TEST_RAG_NORMALIZED
    +
    CONTRACT_TEST_MCP_NORMALIZED
)


(
    CONTRACT_TEST_SELECTED,
    CONTRACT_TEST_CONTEXT_CHARS,
) = apply_context_budget(
    CONTRACT_TEST_COMBINED
)


CONTRACT_TEST_CONTEXT = (
    build_evidence_context(
        CONTRACT_TEST_SELECTED
    )
)


# ============================================================
# 3B.14 — SCHEMA VALIDATION
# ============================================================

REQUIRED_EVIDENCE_FIELDS = {

    "evidence_id",
    "retrieval_system",
    "retrieval_systems",
    "source_family",
    "source_families",
    "document_id",
    "title",
    "chunk_id",
    "score",
    "native_ranks",
    "native_scores",
    "fusion_score",
    "text",
    "text_chars",
    "provenance",
}


all_contract_items = (
    CONTRACT_TEST_RAG_NORMALIZED
    +
    CONTRACT_TEST_MCP_NORMALIZED
)


SCHEMA_PASS = all(

    REQUIRED_EVIDENCE_FIELDS
    .issubset(
        item.keys()
    )

    for item
    in all_contract_items
)


TEXT_BOUND_PASS = all(

    item[
        "text_chars"
    ]
    <=
    NORMALIZED_EVIDENCE_MAX_CHARS

    for item
    in all_contract_items
)


PROVENANCE_PASS = all(

    isinstance(
        item[
            "provenance"
        ],
        list
    )

    and

    len(
        item[
            "provenance"
        ]
    )
    > 0

    for item
    in all_contract_items
)


NATIVE_SCORE_SEPARATION_PASS = all(

    (
        item[
            "retrieval_system"
        ]
        in
        item[
            "native_scores"
        ]
    )

    and

    (
        item[
            "retrieval_system"
        ]
        in
        item[
            "native_ranks"
        ]
    )

    for item
    in all_contract_items
)


CONTEXT_ITEM_LIMIT_PASS = (

    len(
        CONTRACT_TEST_SELECTED
    )
    <=
    COMMON_MAX_EVIDENCE_ITEMS
)


CONTEXT_CHAR_LIMIT_PASS = (

    CONTRACT_TEST_CONTEXT_CHARS
    <=
    COMMON_CONTEXT_MAX_CHARS
)


CONTEXT_LABEL_PASS = (

    "[E1]"
    in
    CONTRACT_TEST_CONTEXT
)


MODULE4_EVIDENCE_CONTRACT_PASS = all(
    [
        SCHEMA_PASS,
        TEXT_BOUND_PASS,
        PROVENANCE_PASS,
        NATIVE_SCORE_SEPARATION_PASS,
        CONTEXT_ITEM_LIMIT_PASS,
        CONTEXT_CHAR_LIMIT_PASS,
        CONTEXT_LABEL_PASS,
    ]
)


MODULE4_CELL3B_PASS = (
    MODULE4_EVIDENCE_CONTRACT_PASS
)


if not MODULE4_CELL3B_PASS:

    raise RuntimeError(
        "Hybrid evidence contract validation failed."
    )


# ============================================================
# 3B.15 — SUMMARY
# ============================================================

print("=" * 116)
print("MODULE 4 DEMO — HYBRID EVIDENCE CONTRACT")
print("=" * 116)


print(
    f"RAG Retrieval Top-K         : "
    f"{MODULE4_RAG_RETRIEVAL_K}"
)


print(
    f"MCP Retrieval Top-K         : "
    f"{MODULE4_MCP_RETRIEVAL_K}"
)


print(
    f"Max Presented Evidence      : "
    f"{COMMON_MAX_EVIDENCE_ITEMS}"
)


print(
    f"Max Chars / Evidence        : "
    f"{NORMALIZED_EVIDENCE_MAX_CHARS:,}"
)


print(
    f"Total Evidence Budget       : "
    f"{COMMON_CONTEXT_MAX_CHARS:,} chars"
)


print()
print("HYBRID FUSION CONTROLS")
print("-" * 116)


print(
    f"Fusion Method               : "
    f"{HYBRID_FUSION_METHOD}"
)


print(
    f"RRF k                       : "
    f"{HYBRID_RRF_K}"
)


print(
    f"Near-Duplicate Threshold    : "
    f"{HYBRID_NEAR_DUPLICATE_THRESHOLD}"
)


print(
    f"Minimum / Retriever         : "
    f"{HYBRID_MIN_ITEMS_PER_RETRIEVER}"
)


print()
print("CONTRACT TEST")
print("-" * 116)


print(
    f"Normalized RAG Items        : "
    f"{len(CONTRACT_TEST_RAG_NORMALIZED)}"
)


print(
    f"Normalized MCP Items        : "
    f"{len(CONTRACT_TEST_MCP_NORMALIZED)}"
)


print(
    f"Context Test Items          : "
    f"{len(CONTRACT_TEST_SELECTED)}"
)


print(
    f"Context Evidence Chars      : "
    f"{CONTRACT_TEST_CONTEXT_CHARS:,}"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Common Schema PASS          : "
    f"{SCHEMA_PASS}"
)


print(
    f"Per-Item Bound PASS         : "
    f"{TEXT_BOUND_PASS}"
)


print(
    f"Provenance PASS             : "
    f"{PROVENANCE_PASS}"
)


print(
    f"Native Scores Separate PASS : "
    f"{NATIVE_SCORE_SEPARATION_PASS}"
)


print(
    f"Evidence Count PASS         : "
    f"{CONTEXT_ITEM_LIMIT_PASS}"
)


print(
    f"Context Budget PASS         : "
    f"{CONTEXT_CHAR_LIMIT_PASS}"
)


print(
    f"Evidence Labels PASS        : "
    f"{CONTEXT_LABEL_PASS}"
)


print(
    f"Evidence Contract PASS      : "
    f"{MODULE4_EVIDENCE_CONTRACT_PASS}"
)


print()
print("✓ CELL 3B PASSED")
print("✓ RAG and MCP share one normalized evidence contract.")
print("✓ Native retrieval scores remain separate.")
print("✓ Source provenance is retained.")
print("✓ Evidence excerpts are bounded to 2,500 characters.")
print("✓ Generator-facing evidence is limited to five items.")
print("✓ Total evidence budget is limited to 12,500 characters.")
print("✓ Generator context builder validated.")
print("✓ No new retrieval or hosted LLM request executed.")
print("✓ Ready for Cell 3C — Hybrid evidence fusion engine.")

print("=" * 116)


MODULE 4 DEMO — HYBRID EVIDENCE CONTRACT
RAG Retrieval Top-K         : 5
MCP Retrieval Top-K         : 5
Max Presented Evidence      : 5
Max Chars / Evidence        : 2,500
Total Evidence Budget       : 12,500 chars

HYBRID FUSION CONTROLS
--------------------------------------------------------------------------------------------------------------------
Fusion Method               : RECIPROCAL_RANK_FUSION
RRF k                       : 60
Near-Duplicate Threshold    : 0.88
Minimum / Retriever         : 2

CONTRACT TEST
--------------------------------------------------------------------------------------------------------------------
Normalized RAG Items        : 1
Normalized MCP Items        : 4
Context Test Items          : 5
Context Evidence Chars      : 10,073

VALIDATION
--------------------------------------------------------------------------------------------------------------------
Common Schema PASS          : True
Per-Item Bound PASS         : True
Provenance PASS           

##### Cell 3B — Observation

The hybrid evidence contract remains fixed at **Top-5 evidence**, **2,500 characters per item**, **12,500 total context characters**, RRF `k=60`, and near-duplicate threshold `0.88`.


#### Cell 3C — Live Hybrid Evidence Fusion

**Description:** Run fresh RAG and MCP retrieval concurrently, fuse results, deduplicate, and preserve representation from both retrieval mechanisms.


In [16]:
# ============================================================
# CELL 3C — LIVE HYBRID EVIDENCE FUSION ENGINE
# ============================================================
#
# SYSTEM-LEVEL HYBRID
#
#                    SAME QUESTION
#                         │
#             ┌───────────┴───────────┐
#             │                       │
#             ▼                       ▼
#     Semantic RAG V1          MCP Version A
#       BGE-M3 + FAISS         FastMCP boundary
#             │                       │
#             │                 internal router
#             │                3GPP/TCC/HYBRID
#             │                       │
#             └───────────┬───────────┘
#                         ▼
#                Common Evidence Contract
#                         │
#                         ▼
#              Reciprocal Rank Fusion
#                         │
#                         ▼
#              Exact Duplicate Merge
#                         │
#                         ▼
#              Near-Duplicate Merge
#                    Jaccard >= 0.88
#                         │
#                         ▼
#            Minimum RAG/MCP Representation
#                         │
#                         ▼
#              Final Top-5 / 12,500 chars
#
#
# No LLM generation occurs in this cell.
# ============================================================


# ============================================================
# 3C.1 — VERIFY CELL 3B
# ============================================================

if (
    "MODULE4_CELL3B_PASS" not in globals()
    or
    not MODULE4_CELL3B_PASS
):
    raise RuntimeError(
        "Cell 3B must pass before Cell 3C."
    )


# ============================================================
# 3C.2 — IMPORTS
# ============================================================

import asyncio
import copy
import gc
import inspect
import os
import time

import pandas as pd
import psutil

from fastmcp import Client


# ============================================================
# 3C.3 — FROZEN HYBRID SELECTION POLICY
# ============================================================

HYBRID_SELECTION_POLICY = (
    "RRF_WITH_POST_DEDUP_MINIMUM_RETRIEVER_REPRESENTATION"
)


# ============================================================
# 3C.4 — NEAR-DUPLICATE SIMILARITY
# ============================================================

def evidence_token_set(
    text
):

    return set(
        re.findall(
            r"[a-z0-9]+",
            canonical_evidence_text(
                text
            ),
        )
    )


def text_jaccard(
    text_a,
    text_b,
):

    tokens_a = evidence_token_set(
        text_a
    )

    tokens_b = evidence_token_set(
        text_b
    )


    if (
        not tokens_a
        or
        not tokens_b
    ):

        return 0.0


    return (
        len(
            tokens_a
            &
            tokens_b
        )
        /
        len(
            tokens_a
            |
            tokens_b
        )
    )


# ============================================================
# 3C.5 — RECIPROCAL RANK FUSION
# ============================================================

def add_rrf_score(
    item
):

    item = copy.deepcopy(
        item
    )


    components = {}


    for system, rank in (
        item.get(
            "native_ranks",
            {},
        ).items()
    ):

        rank = int(
            rank
        )


        if rank > 0:

            components[
                system
            ] = (
                1.0
                /
                (
                    HYBRID_RRF_K
                    +
                    rank
                )
            )


    item[
        "rrf_components"
    ] = components


    item[
        "fusion_score"
    ] = float(
        sum(
            components.values()
        )
    )


    return item


# ============================================================
# 3C.6 — MERGE DUPLICATE EVIDENCE
# ============================================================

def merge_duplicate_evidence(
    primary,
    secondary,
):

    merged = copy.deepcopy(
        primary
    )


    systems = list(
        dict.fromkeys(
            primary.get(
                "retrieval_systems",
                [],
            )
            +
            secondary.get(
                "retrieval_systems",
                [],
            )
        )
    )


    families = list(
        dict.fromkeys(
            primary.get(
                "source_families",
                [],
            )
            +
            secondary.get(
                "source_families",
                [],
            )
        )
    )


    merged[
        "retrieval_systems"
    ] = systems


    merged[
        "retrieval_system"
    ] = (
        "BOTH"
        if (
            "RAG" in systems
            and
            "MCP" in systems
        )
        else
        systems[0]
    )


    merged[
        "source_families"
    ] = families


    merged[
        "source_family"
    ] = (
        families[0]
        if len(
            families
        ) == 1
        else
        "MULTI"
    )


    # Preserve both provenance chains.
    merged[
        "provenance"
    ] = (
        primary.get(
            "provenance",
            [],
        )
        +
        secondary.get(
            "provenance",
            [],
        )
    )


    # Keep the best native rank for each retriever.
    native_ranks = dict(
        primary.get(
            "native_ranks",
            {},
        )
    )


    for system, rank in (
        secondary.get(
            "native_ranks",
            {},
        ).items()
    ):

        if (
            system
            not in native_ranks
            or
            rank
            <
            native_ranks[
                system
            ]
        ):

            native_ranks[
                system
            ] = rank


    merged[
        "native_ranks"
    ] = native_ranks


    # Native scores remain retriever-specific.
    native_scores = dict(
        primary.get(
            "native_scores",
            {},
        )
    )


    native_scores.update(
        secondary.get(
            "native_scores",
            {},
        )
    )


    merged[
        "native_scores"
    ] = native_scores


    # Merge RRF components.
    rrf_components = dict(
        primary.get(
            "rrf_components",
            {},
        )
    )


    for system, score in (
        secondary.get(
            "rrf_components",
            {},
        ).items()
    ):

        rrf_components[
            system
        ] = max(
            float(
                score
            ),
            float(
                rrf_components.get(
                    system,
                    0.0,
                )
            ),
        )


    merged[
        "rrf_components"
    ] = rrf_components


    merged[
        "fusion_score"
    ] = float(
        sum(
            rrf_components.values()
        )
    )


    return merged


# ============================================================
# 3C.7 — EXACT + NEAR DUPLICATE CONSOLIDATION
# ============================================================

def deduplicate_hybrid_evidence(
    evidence
):

    stats = {

        "input_items":
            len(
                evidence
            ),

        "exact_duplicate_merges":
            0,

        "near_duplicate_merges":
            0,
    }


    # --------------------------------------------------------
    # Exact duplicates
    # --------------------------------------------------------

    exact_map = {}


    for item in evidence:

        fingerprint = (
            hashlib.sha256(
                canonical_evidence_text(
                    item[
                        "text"
                    ]
                )
                .encode(
                    "utf-8"
                )
            )
            .hexdigest()
        )


        if fingerprint in exact_map:

            exact_map[
                fingerprint
            ] = merge_duplicate_evidence(

                exact_map[
                    fingerprint
                ],

                item,
            )


            stats[
                "exact_duplicate_merges"
            ] += 1


        else:

            exact_map[
                fingerprint
            ] = copy.deepcopy(
                item
            )


    exact_unique = list(
        exact_map.values()
    )


    exact_unique.sort(

        key=lambda item:
            float(
                item.get(
                    "fusion_score",
                    0.0,
                )
            ),

        reverse=True,
    )


    # --------------------------------------------------------
    # Near duplicates
    # --------------------------------------------------------

    near_unique = []


    for candidate in exact_unique:

        duplicate_index = None


        for index, existing in enumerate(
            near_unique
        ):

            similarity = text_jaccard(

                candidate[
                    "text"
                ],

                existing[
                    "text"
                ],
            )


            if (
                similarity
                >=
                HYBRID_NEAR_DUPLICATE_THRESHOLD
            ):

                duplicate_index = index

                break


        if duplicate_index is None:

            near_unique.append(
                copy.deepcopy(
                    candidate
                )
            )


        else:

            near_unique[
                duplicate_index
            ] = merge_duplicate_evidence(

                near_unique[
                    duplicate_index
                ],

                candidate,
            )


            stats[
                "near_duplicate_merges"
            ] += 1


    near_unique.sort(

        key=lambda item:
            float(
                item.get(
                    "fusion_score",
                    0.0,
                )
            ),

        reverse=True,
    )


    stats[
        "output_items"
    ] = len(
        near_unique
    )


    return (
        near_unique,
        stats,
    )


# ============================================================
# 3C.8 — HYBRID RETRIEVER REPRESENTATION
# ============================================================

def select_hybrid_evidence(
    fused_evidence
):

    ranked = sorted(

        fused_evidence,

        key=lambda item:
            float(
                item.get(
                    "fusion_score",
                    0.0,
                )
            ),

        reverse=True,
    )


    available_rag = sum(

        "RAG"
        in item.get(
            "retrieval_systems",
            [],
        )

        for item
        in ranked
    )


    available_mcp = sum(

        "MCP"
        in item.get(
            "retrieval_systems",
            [],
        )

        for item
        in ranked
    )


    required_rag = min(
        HYBRID_MIN_ITEMS_PER_RETRIEVER,
        available_rag,
    )


    required_mcp = min(
        HYBRID_MIN_ITEMS_PER_RETRIEVER,
        available_mcp,
    )


    selected_ids = []


    def get_selected_items():

        selected_set = set(
            selected_ids
        )


        return [

            item

            for item
            in ranked

            if item[
                "evidence_id"
            ]
            in selected_set
        ]


    def representation_count(
        system
    ):

        return sum(

            system
            in item.get(
                "retrieval_systems",
                [],
            )

            for item
            in get_selected_items()
        )


    # --------------------------------------------------------
    # Ensure sufficient distinct RAG evidence
    # --------------------------------------------------------

    for item in ranked:

        if (
            representation_count(
                "RAG"
            )
            >=
            required_rag
        ):

            break


        if (
            "RAG"
            in item.get(
                "retrieval_systems",
                [],
            )
            and
            item[
                "evidence_id"
            ]
            not in selected_ids
        ):

            selected_ids.append(
                item[
                    "evidence_id"
                ]
            )


    # --------------------------------------------------------
    # Ensure sufficient distinct MCP evidence
    # --------------------------------------------------------

    for item in ranked:

        if (
            representation_count(
                "MCP"
            )
            >=
            required_mcp
        ):

            break


        if (
            "MCP"
            in item.get(
                "retrieval_systems",
                [],
            )
            and
            item[
                "evidence_id"
            ]
            not in selected_ids
        ):

            selected_ids.append(
                item[
                    "evidence_id"
                ]
            )


    # --------------------------------------------------------
    # Fill remaining positions by global RRF
    # --------------------------------------------------------

    for item in ranked:

        if (
            len(
                selected_ids
            )
            >=
            COMMON_MAX_EVIDENCE_ITEMS
        ):

            break


        if (
            item[
                "evidence_id"
            ]
            not in selected_ids
        ):

            selected_ids.append(
                item[
                    "evidence_id"
                ]
            )


    # Membership is fixed first.
    # Then presentation order follows global RRF.
    selected_set = set(
        selected_ids
    )


    selected = [

        item

        for item
        in ranked

        if item[
            "evidence_id"
        ]
        in selected_set
    ]


    (
        selected,
        context_chars,
    ) = apply_context_budget(
        selected
    )


    return {

        "evidence":
            selected,

        "context_chars":
            context_chars,

        "available_rag_after_dedup":
            int(
                available_rag
            ),

        "available_mcp_after_dedup":
            int(
                available_mcp
            ),

        "required_rag":
            int(
                required_rag
            ),

        "required_mcp":
            int(
                required_mcp
            ),
    }


# ============================================================
# 3C.9 — COMPLETE HYBRID FUSION
# ============================================================

def fuse_hybrid_candidates(
    rag_candidates,
    mcp_candidates,
):

    start = (
        time.perf_counter()
    )


    if (
        len(
            rag_candidates
        )
        >
        MODULE4_RAG_RETRIEVAL_K
    ):

        raise RuntimeError(
            "RAG candidate count exceeds Module 4 Top-K."
        )


    if (
        len(
            mcp_candidates
        )
        >
        MODULE4_MCP_RETRIEVAL_K
    ):

        raise RuntimeError(
            "MCP candidate count exceeds Module 4 Top-K."
        )


    # --------------------------------------------------------
    # Reciprocal Rank Fusion
    # --------------------------------------------------------

    scored = [

        add_rrf_score(
            item
        )

        for item in (

            list(
                rag_candidates
            )

            +

            list(
                mcp_candidates
            )
        )
    ]


    # --------------------------------------------------------
    # Exact + near duplicate consolidation
    # --------------------------------------------------------

    (
        deduplicated,
        dedup_stats,
    ) = deduplicate_hybrid_evidence(
        scored
    )


    # --------------------------------------------------------
    # Retriever representation + final Top-5
    # --------------------------------------------------------

    hybrid_selection = (
        select_hybrid_evidence(
            deduplicated
        )
    )


    selected = (
        hybrid_selection[
            "evidence"
        ]
    )


    context_chars = (
        hybrid_selection[
            "context_chars"
        ]
    )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    rag_supported = sum(

        "RAG"
        in item.get(
            "retrieval_systems",
            [],
        )

        for item
        in selected
    )


    mcp_supported = sum(

        "MCP"
        in item.get(
            "retrieval_systems",
            [],
        )

        for item
        in selected
    )


    both_supported = sum(

        (
            "RAG"
            in item.get(
                "retrieval_systems",
                [],
            )

            and

            "MCP"
            in item.get(
                "retrieval_systems",
                [],
            )
        )

        for item
        in selected
    )


    return {

        "evidence":
            selected,

        "context":
            build_evidence_context(
                selected
            ),

        "context_chars":
            context_chars,

        "timing": {

            "fusion_only_s":
                elapsed,
        },

        "diagnostics": {

            **dedup_stats,

            "rag_input_candidates":
                len(
                    rag_candidates
                ),

            "mcp_input_candidates":
                len(
                    mcp_candidates
                ),

            "available_rag_after_dedup":
                hybrid_selection[
                    "available_rag_after_dedup"
                ],

            "available_mcp_after_dedup":
                hybrid_selection[
                    "available_mcp_after_dedup"
                ],

            "required_rag":
                hybrid_selection[
                    "required_rag"
                ],

            "required_mcp":
                hybrid_selection[
                    "required_mcp"
                ],

            "selected_items":
                len(
                    selected
                ),

            "rag_supported_items":
                int(
                    rag_supported
                ),

            "mcp_supported_items":
                int(
                    mcp_supported
                ),

            "both_supported_items":
                int(
                    both_supported
                ),

            "selection_policy":
                HYBRID_SELECTION_POLICY,
        },
    }


# ============================================================
# 3C.10 — RAG RETRIEVAL ADAPTER
# ============================================================

def execute_rag_v1(
    query
):
    """
    Call the already validated Cell 1D retrieve_rag() function
    without assuming whether its Top-K parameter is named
    'top_k' or 'k'.
    """

    if (
        "retrieve_rag"
        not in globals()
        or
        not callable(
            retrieve_rag
        )
    ):

        raise RuntimeError(
            "Cell 1D retrieve_rag() is unavailable."
        )


    parameters = (
        inspect.signature(
            retrieve_rag
        ).parameters
    )


    if "top_k" in parameters:

        return retrieve_rag(
            query,
            top_k=
                MODULE4_RAG_RETRIEVAL_K,
        )


    if "k" in parameters:

        return retrieve_rag(
            query,
            k=
                MODULE4_RAG_RETRIEVAL_K,
        )


    return retrieve_rag(
        query
    )


def retrieve_rag_candidates(
    query
):

    start = (
        time.perf_counter()
    )


    raw = execute_rag_v1(
        query
    )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    if not isinstance(
        raw,
        dict
    ):

        raise RuntimeError(
            "RAG retriever did not return a dictionary."
        )


    normalized = (
        normalize_rag_results(
            raw
        )
    )


    return {

        "query":
            query,

        "retrieval_system":
            "RAG",

        "retrieval_top_k":
            MODULE4_RAG_RETRIEVAL_K,

        "candidate_count":
            len(
                normalized
            ),

        "evidence":
            normalized,

        "timing": {

            "retrieval_time_s":
                float(
                    elapsed
                ),
        },

        "raw":
            raw,
    }


# ============================================================
# 3C.11 — MCP RETRIEVAL ADAPTER
# ============================================================

async def retrieve_mcp_candidates(
    query
):

    start = (
        time.perf_counter()
    )


    client = Client(
        mcp
    )


    async with client:

        result = (
            await client.call_tool(

                "search_telecom_knowledge",

                {
                    "query":
                        query,

                    "top_k":
                        MODULE4_MCP_RETRIEVAL_K,
                },
            )
        )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    payload = getattr(
        result,
        "data",
        None,
    )


    if payload is None:

        payload = getattr(
            result,
            "structured_content",
            None,
        )


    if not isinstance(
        payload,
        dict
    ):

        raise RuntimeError(
            "MCP returned no structured payload."
        )


    normalized = (
        normalize_mcp_results(
            payload
        )
    )


    return {

        "query":
            query,

        "retrieval_system":
            "MCP",

        "retrieval_top_k":
            MODULE4_MCP_RETRIEVAL_K,

        "candidate_count":
            len(
                normalized
            ),

        "evidence":
            normalized,

        "timing": {

            "mcp_roundtrip_s":
                float(
                    elapsed
                ),

            "retrieval_time_s":
                float(
                    payload.get(
                        "trace",
                        {},
                    ).get(
                        "retrieval_time_s",
                        0.0,
                    )
                    or
                    0.0
                ),
        },

        "trace":
            payload.get(
                "trace",
                {},
            ),

        "payload":
            payload,
    }


# ============================================================
# 3C.12 — SYSTEM-LEVEL RAG + MCP RETRIEVAL
# ============================================================

async def retrieve_module4_hybrid(
    query
):
    """
    Execute RAG and MCP concurrently, then construct:

        RAG_ONLY
        MCP_ONLY
        HYBRID

    from the SAME retrieval run.
    """

    pair_start = (
        time.perf_counter()
    )


    # RAG is synchronous and CPU-bound.
    # Dispatch it to a worker so the MCP coroutine can execute
    # concurrently.

    rag_task = asyncio.to_thread(
        retrieve_rag_candidates,
        query,
    )


    mcp_task = retrieve_mcp_candidates(
        query
    )


    (
        rag,
        mcp_result,
    ) = await asyncio.gather(
        rag_task,
        mcp_task,
    )


    parallel_retrieval_wall_s = (
        time.perf_counter()
        -
        pair_start
    )


    # --------------------------------------------------------
    # RAG-only view
    # --------------------------------------------------------

    (
        rag_selected,
        rag_context_chars,
    ) = select_single_mode_evidence(
        rag[
            "evidence"
        ]
    )


    rag_mode = {

        "mode":
            "RAG_ONLY",

        "candidate_count":
            rag[
                "candidate_count"
            ],

        "evidence":
            rag_selected,

        "context":
            build_evidence_context(
                rag_selected
            ),

        "context_chars":
            rag_context_chars,

        "timing":
            rag[
                "timing"
            ],
    }


    # --------------------------------------------------------
    # MCP-only view
    # --------------------------------------------------------

    (
        mcp_selected,
        mcp_context_chars,
    ) = select_single_mode_evidence(
        mcp_result[
            "evidence"
        ]
    )


    mcp_mode = {

        "mode":
            "MCP_ONLY",

        "candidate_count":
            mcp_result[
                "candidate_count"
            ],

        "evidence":
            mcp_selected,

        "context":
            build_evidence_context(
                mcp_selected
            ),

        "context_chars":
            mcp_context_chars,

        "timing":
            mcp_result[
                "timing"
            ],

        "trace":
            mcp_result[
                "trace"
            ],
    }


    # --------------------------------------------------------
    # Hybrid fused view
    # --------------------------------------------------------

    hybrid = fuse_hybrid_candidates(

        rag_candidates=
            rag[
                "evidence"
            ],

        mcp_candidates=
            mcp_result[
                "evidence"
            ],
    )


    hybrid_total_wall_s = (

        parallel_retrieval_wall_s

        +

        hybrid[
            "timing"
        ][
            "fusion_only_s"
        ]
    )


    hybrid_mode = {

        "mode":
            "HYBRID",

        "candidate_count": {

            "RAG":
                rag[
                    "candidate_count"
                ],

            "MCP":
                mcp_result[
                    "candidate_count"
                ],

            "total_before_dedup":
                (
                    rag[
                        "candidate_count"
                    ]
                    +
                    mcp_result[
                        "candidate_count"
                    ]
                ),
        },

        "evidence":
            hybrid[
                "evidence"
            ],

        "context":
            hybrid[
                "context"
            ],

        "context_chars":
            hybrid[
                "context_chars"
            ],

        "timing": {

            "rag_retrieval_s":
                rag[
                    "timing"
                ][
                    "retrieval_time_s"
                ],

            "mcp_retrieval_s":
                mcp_result[
                    "timing"
                ][
                    "retrieval_time_s"
                ],

            "mcp_roundtrip_s":
                mcp_result[
                    "timing"
                ][
                    "mcp_roundtrip_s"
                ],

            "parallel_retrieval_wall_s":
                float(
                    parallel_retrieval_wall_s
                ),

            "fusion_only_s":
                hybrid[
                    "timing"
                ][
                    "fusion_only_s"
                ],

            "hybrid_total_wall_s":
                float(
                    hybrid_total_wall_s
                ),
        },

        "diagnostics":
            hybrid[
                "diagnostics"
            ],

        "mcp_trace":
            mcp_result[
                "trace"
            ],
    }


    return {

        "query":
            query,

        "RAG_ONLY":
            rag_mode,

        "MCP_ONLY":
            mcp_mode,

        "HYBRID":
            hybrid_mode,
    }


# ============================================================
# 3C.13 — LIVE SYSTEM-LEVEL HYBRID QUERY
# ============================================================
#
# Use a 3GPP-native question for this first integration test.
#
# MCP Version A should internally select 3GPP.
#
# System-level architecture is nevertheless HYBRID because
# BOTH Semantic RAG and MCP are queried.
# ============================================================

MODULE4_HYBRID_TEST_QUERY = (

    "What are the primary responsibilities of the AMF "
    "in a 5G Standalone network?"
)


# ============================================================
# 3C.14 — MEMORY BASELINE
# ============================================================

gc.collect()


HYBRID_RAM_BEFORE_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


print("=" * 116)
print("MODULE 4 DEMO — LIVE SYSTEM-LEVEL HYBRID RETRIEVAL")
print("=" * 116)

print(
    f"Query                       : "
    f"{MODULE4_HYBRID_TEST_QUERY}"
)

print(
    "System Retrieval           : "
    "Semantic RAG V1 + MCP Version A"
)

print(
    "Execution                  : "
    "Concurrent"
)

print(
    "Hybrid Fusion              : "
    "RRF + Exact Dedup + Near Dedup + Representation Control"
)

print()

print(
    "Executing live RAG + MCP retrieval..."
)


MODULE4_HYBRID_LIVE_RESULT = (
    await retrieve_module4_hybrid(
        MODULE4_HYBRID_TEST_QUERY
    )
)


# ============================================================
# 3C.15 — MEMORY AFTER
# ============================================================

gc.collect()


HYBRID_RAM_AFTER_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


HYBRID_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


HYBRID_RAM_DELTA_GIB = (

    HYBRID_RAM_AFTER_GIB
    -
    HYBRID_RAM_BEFORE_GIB
)


# ============================================================
# 3C.16 — EXTRACT RESULTS
# ============================================================

rag_view = (
    MODULE4_HYBRID_LIVE_RESULT[
        "RAG_ONLY"
    ]
)


mcp_view = (
    MODULE4_HYBRID_LIVE_RESULT[
        "MCP_ONLY"
    ]
)


hybrid_view = (
    MODULE4_HYBRID_LIVE_RESULT[
        "HYBRID"
    ]
)


hybrid_diag = (
    hybrid_view[
        "diagnostics"
    ]
)


hybrid_timing = (
    hybrid_view[
        "timing"
    ]
)


# ============================================================
# 3C.17 — FINAL EVIDENCE TABLE
# ============================================================

HYBRID_EVIDENCE_DF = pd.DataFrame(

    [

        {
            "rank":
                item[
                    "presentation_rank"
                ],

            "retrieval":
                "/".join(
                    item[
                        "retrieval_systems"
                    ]
                ),

            "source":
                item[
                    "source_family"
                ],

            "document":
                item[
                    "document_id"
                ],

            "RAG_rank":
                item.get(
                    "native_ranks",
                    {},
                ).get(
                    "RAG"
                ),

            "MCP_rank":
                item.get(
                    "native_ranks",
                    {},
                ).get(
                    "MCP"
                ),

            "fusion_score":
                item.get(
                    "fusion_score"
                ),

            "chars":
                item[
                    "text_chars"
                ],

            "title":
                item[
                    "title"
                ][:80],
        }

        for item
        in hybrid_view[
            "evidence"
        ]
    ]
)


print()
print("FINAL HYBRID EVIDENCE")
print("-" * 116)


display(
    HYBRID_EVIDENCE_DF.round(
        {
            "fusion_score":
                6
        }
    )
)


# ============================================================
# 3C.18 — VALIDATION
# ============================================================

RAG_RESULTS_PASS = (

    rag_view[
        "candidate_count"
    ]
    >
    0
)


MCP_RESULTS_PASS = (

    mcp_view[
        "candidate_count"
    ]
    >
    0
)


HYBRID_RESULT_PASS = (

    len(
        hybrid_view[
            "evidence"
        ]
    )
    >
    0
)


HYBRID_TOPK_PASS = (

    len(
        hybrid_view[
            "evidence"
        ]
    )
    <=
    COMMON_MAX_EVIDENCE_ITEMS
)


HYBRID_CONTEXT_PASS = (

    hybrid_view[
        "context_chars"
    ]
    <=
    COMMON_CONTEXT_MAX_CHARS
)


RAG_REPRESENTATION_PASS = (

    hybrid_diag[
        "rag_supported_items"
    ]
    >=
    hybrid_diag[
        "required_rag"
    ]
)


MCP_REPRESENTATION_PASS = (

    hybrid_diag[
        "mcp_supported_items"
    ]
    >=
    hybrid_diag[
        "required_mcp"
    ]
)


MCP_ROUTE_PASS = (

    hybrid_view[
        "mcp_trace"
    ].get(
        "route"
    )
    ==
    "3gpp"
)


FUSION_TIMING_PASS = (

    hybrid_timing[
        "fusion_only_s"
    ]
    >=
    0
)


MODULE4_HYBRID_FUSION_PASS = all(
    [
        RAG_RESULTS_PASS,
        MCP_RESULTS_PASS,
        HYBRID_RESULT_PASS,
        HYBRID_TOPK_PASS,
        HYBRID_CONTEXT_PASS,
        RAG_REPRESENTATION_PASS,
        MCP_REPRESENTATION_PASS,
        MCP_ROUTE_PASS,
        FUSION_TIMING_PASS,
    ]
)


MODULE4_CELL3C_PASS = (
    MODULE4_HYBRID_FUSION_PASS
)


if not MODULE4_CELL3C_PASS:

    raise RuntimeError(
        "Live Hybrid evidence fusion validation failed."
    )


# ============================================================
# 3C.19 — SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — LIVE HYBRID RETRIEVAL SUMMARY")
print("=" * 116)


print(
    f"RAG Candidates              : "
    f"{rag_view['candidate_count']}"
)


print(
    f"MCP Candidates              : "
    f"{mcp_view['candidate_count']}"
)


print(
    f"Raw Candidate Pool          : "
    f"{hybrid_view['candidate_count']['total_before_dedup']}"
)


print(
    f"Exact Duplicate Merges      : "
    f"{hybrid_diag['exact_duplicate_merges']}"
)


print(
    f"Near Duplicate Merges       : "
    f"{hybrid_diag['near_duplicate_merges']}"
)


print(
    f"Distinct After Dedup        : "
    f"{hybrid_diag['output_items']}"
)


print(
    f"Final Presented Evidence    : "
    f"{hybrid_diag['selected_items']}"
)


print(
    f"RAG-Supported Final Items   : "
    f"{hybrid_diag['rag_supported_items']}"
)


print(
    f"MCP-Supported Final Items   : "
    f"{hybrid_diag['mcp_supported_items']}"
)


print(
    f"Supported by Both           : "
    f"{hybrid_diag['both_supported_items']}"
)


print(
    f"Context Evidence Chars      : "
    f"{hybrid_view['context_chars']:,}"
)


print(
    f"MCP Internal Route          : "
    f"{hybrid_view['mcp_trace'].get('route', '').upper()}"
)


print()
print("TIMING")
print("-" * 116)


print(
    f"RAG Retrieval               : "
    f"{hybrid_timing['rag_retrieval_s']:.3f} s"
)


print(
    f"MCP Retrieval               : "
    f"{hybrid_timing['mcp_retrieval_s']:.3f} s"
)


print(
    f"MCP Round Trip              : "
    f"{hybrid_timing['mcp_roundtrip_s']:.3f} s"
)


print(
    f"Parallel Retrieval Wall     : "
    f"{hybrid_timing['parallel_retrieval_wall_s']:.3f} s"
)


print(
    f"Fusion Only                 : "
    f"{hybrid_timing['fusion_only_s']:.6f} s"
)


print(
    f"Hybrid Total Wall           : "
    f"{hybrid_timing['hybrid_total_wall_s']:.3f} s"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{HYBRID_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{HYBRID_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{HYBRID_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{HYBRID_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"RAG Evidence PASS           : "
    f"{RAG_RESULTS_PASS}"
)


print(
    f"MCP Evidence PASS           : "
    f"{MCP_RESULTS_PASS}"
)


print(
    f"Hybrid Fusion PASS          : "
    f"{HYBRID_RESULT_PASS}"
)


print(
    f"Top-5 Limit PASS            : "
    f"{HYBRID_TOPK_PASS}"
)


print(
    f"Context Budget PASS         : "
    f"{HYBRID_CONTEXT_PASS}"
)


print(
    f"RAG Representation PASS     : "
    f"{RAG_REPRESENTATION_PASS}"
)


print(
    f"MCP Representation PASS     : "
    f"{MCP_REPRESENTATION_PASS}"
)


print(
    f"MCP Internal Routing PASS   : "
    f"{MCP_ROUTE_PASS}"
)


print(
    f"Overall Hybrid PASS         : "
    f"{MODULE4_HYBRID_FUSION_PASS}"
)


print()
print("✓ CELL 3C PASSED")
print("✓ Same live query executed through RAG and MCP.")
print("✓ RAG and MCP retrieval executed concurrently.")
print("✓ Native scores remained retriever-specific.")
print("✓ Reciprocal Rank Fusion applied.")
print("✓ Exact and near-duplicate consolidation applied.")
print("✓ Minimum retriever representation enforced.")
print("✓ Final evidence bounded to Top-5 / 12,500 characters.")
print("✓ System-level Hybrid retrieval validated.")
print("✓ No hosted LLM generation executed.")
print("✓ Ready for Cell 3D — Hybrid grounded generation.")

print("=" * 116)

# ============================================================
# 3C.17A — FINAL HYBRID EVIDENCE CONTENT PREVIEW
# ============================================================
#
# Human-readable inspection of the evidence that will later
# be supplied to the hosted LLM.
#
# No new retrieval is executed.
# ============================================================

HYBRID_EVIDENCE_PREVIEW_CHARS = 500


print()
print("FINAL HYBRID EVIDENCE — CONTENT PREVIEW")
print("-" * 116)


for item in hybrid_view["evidence"]:

    rank = item["presentation_rank"]

    retrieval = "/".join(
        item.get(
            "retrieval_systems",
            []
        )
    )

    source = item.get(
        "source_family",
        ""
    )

    document = item.get(
        "document_id",
        ""
    )

    title = item.get(
        "title",
        ""
    )

    text = item.get(
        "text",
        ""
    )


    preview = (
        text[
            :HYBRID_EVIDENCE_PREVIEW_CHARS
        ]
        .strip()
    )


    if (
        len(text)
        >
        HYBRID_EVIDENCE_PREVIEW_CHARS
    ):

        preview += " ..."


    print()
    print(
        f"[E{rank}] "
        f"Retrieval={retrieval} | "
        f"Source={source}"
    )

    print(
        f"Document : {document}"
    )

    print(
        f"Title    : {title}"
    )

    print(
        f"Preview  : {preview}"
    )

    print("-" * 116)


MODULE 4 DEMO — LIVE SYSTEM-LEVEL HYBRID RETRIEVAL
Query                       : What are the primary responsibilities of the AMF in a 5G Standalone network?
System Retrieval           : Semantic RAG V1 + MCP Version A
Execution                  : Concurrent
Hybrid Fusion              : RRF + Exact Dedup + Near Dedup + Representation Control

Executing live RAG + MCP retrieval...

FINAL HYBRID EVIDENCE
--------------------------------------------------------------------------------------------------------------------


,rank,retrieval,source,document,RAG_rank,MCP_rank,fusion_score,chars,title
0,1,RAG,3GPP,standards/3gpp_rel18/original/rel_15.docx,1.0,NaN,0.016393,2500,rel_15
1,2,MCP,3GPP,23.501,NaN,1.0,0.016393,2499,3GPP TS 23.501 V20.0.0 (2025-12) ---
2,3,RAG,ETSI,standards/etsi/marked/TR/tr/tr_121915v150000p/...,2.0,NaN,0.016129,2500,raw
3,4,MCP,3GPP,23.501,NaN,2.0,0.016129,2500,3GPP TS 23.501 V20.0.0 (2025-12) ---
4,5,RAG,3GPP,standards/3gpp_rel18/original/29518-i40.docx,3.0,NaN,0.015873,2500,29518-i40



MODULE 4 DEMO — LIVE HYBRID RETRIEVAL SUMMARY
RAG Candidates              : 5
MCP Candidates              : 2
Raw Candidate Pool          : 7
Exact Duplicate Merges      : 0
Near Duplicate Merges       : 0
Distinct After Dedup        : 7
Final Presented Evidence    : 5
RAG-Supported Final Items   : 3
MCP-Supported Final Items   : 2
Supported by Both           : 0
Context Evidence Chars      : 12,499
MCP Internal Route          : 3GPP

TIMING
--------------------------------------------------------------------------------------------------------------------
RAG Retrieval               : 0.568 s
MCP Retrieval               : 0.783 s
MCP Round Trip              : 0.794 s
Parallel Retrieval Wall     : 0.795 s
Fusion Only                 : 0.015365 s
Hybrid Total Wall           : 0.811 s

MEMORY IMPACT
--------------------------------------------------------------------------------------------------------------------
Process RAM Before           : 9.084 GiB
Process RAM After            : 9

##### Cell 3C — Observation

Live hybrid fusion validated successfully and produced the expected five-item evidence package. Parallel retrieval and lightweight fusion preserve the original Module 4 evidence contract.


#### Cell 3D — Live Hybrid Grounded Generation

**Description:** Send the fused evidence context to hosted Gemma and validate grouped/individual `[E#]` citations.


In [17]:
# ============================================================
# CELL 3D — LIVE HYBRID GROUNDED GENERATION
# ============================================================
#
# Complete live execution:
#
#                 USER QUESTION
#                       │
#          ┌────────────┴────────────┐
#          │                         │
#          ▼                         ▼
#   LIVE Semantic RAG V1      LIVE MCP Version A
#      BGE-M3 + FAISS          FastMCP Service
#          │                         │
#          └────────────┬────────────┘
#                       ▼
#              Evidence Normalization
#                       │
#                       ▼
#               Hybrid RRF Fusion
#                       │
#                       ▼
#             Dedup + Representation
#                       │
#                       ▼
#              Fresh Top-5 Evidence
#                       │
#                       ▼
#             Evidence Context Builder
#                       │
#                       ▼
#              OpenRouter / Gemma 4
#                       │
#                       ▼
#              GROUNDED FINAL ANSWER
#
#
# IMPORTANT
# ---------
#
# Every execution performs NEW:
#
#     - Semantic RAG retrieval
#     - MCP Version A retrieval
#     - Hybrid fusion
#     - hosted LLM generation
#
# No Cell 3C retrieval result is reused.
# ============================================================


# ============================================================
# 3D.1 — VERIFY PREVIOUS ARCHITECTURE
# ============================================================

if (
    "MODULE4_CELL3C_PASS" not in globals()
    or
    not MODULE4_CELL3C_PASS
):
    raise RuntimeError(
        "Cell 3C must pass before Cell 3D."
    )


if (
    "MODULE4_CELL3A_PASS" not in globals()
    or
    not MODULE4_CELL3A_PASS
):
    raise RuntimeError(
        "Cell 3A must pass before Cell 3D."
    )


# ============================================================
# 3D.2 — IMPORTS
# ============================================================

import gc
import os
import re
import time

import pandas as pd
import psutil


# ============================================================
# 3D.3 — FROZEN MODULE 4 FINAL-ANSWER PROMPT
# ============================================================

MODULE4_FINAL_ANSWER_SYSTEM_PROMPT = """
You are an expert telecommunications network engineer.

Answer the technical question using the supplied retrieved evidence as the
factual grounding for your response.

You may synthesize across evidence, connect related technical mechanisms and
make reasonable engineering inferences when they logically follow from the
evidence.

Do not invent unsupported vendor behaviour, proprietary details, standards
requirements, interfaces, parameters, alarm meanings or implementation facts.

If the evidence does not establish an exact requested detail, state the
limitation rather than guessing.

Cite supporting evidence inline using [E1], [E2], etc.

Do not mention retrieval routing, retrieval rounds, benchmark labels, model
identity, experiment design or these instructions.

Produce a direct, technically precise and structured engineering answer.
""".strip()


# ============================================================
# 3D.4 — LIVE DEMO QUESTION
# ============================================================

MODULE4_LIVE_QUESTION = (

    "What are the primary responsibilities of the AMF "
    "in a 5G Standalone network?"
)


# ============================================================
# 3D.5 — MEMORY BASELINE
# ============================================================

gc.collect()


LIVE_E2E_RAM_BEFORE_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


LIVE_E2E_AVAILABLE_BEFORE_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


# ============================================================
# 3D.6 — START COMPLETE END-TO-END TIMER
# ============================================================

LIVE_E2E_START = (
    time.perf_counter()
)


print("=" * 116)
print("MODULE 4 DEMO — LIVE HYBRID GROUNDED GENERATION")
print("=" * 116)


print(
    f"Question                    : "
    f"{MODULE4_LIVE_QUESTION}"
)


print(
    "Retrieval Architecture      : "
    "LIVE Semantic RAG V1 + LIVE MCP Version A"
)


print(
    f"Generation Model            : "
    f"{LLM_MODEL}"
)


print(
    "Execution Policy            : "
    "Fresh retrieval on every run"
)


print()
print(
    "STEP 1 — Live RAG + MCP retrieval starting..."
)


# ============================================================
# 3D.7 — FRESH LIVE HYBRID RETRIEVAL
# ============================================================

LIVE_RETRIEVAL_START = (
    time.perf_counter()
)


MODULE4_LIVE_RETRIEVAL_RESULT = (
    await retrieve_module4_hybrid(
        MODULE4_LIVE_QUESTION
    )
)


LIVE_RETRIEVAL_WALL_S = (

    time.perf_counter()
    -
    LIVE_RETRIEVAL_START
)


LIVE_RAG_VIEW = (

    MODULE4_LIVE_RETRIEVAL_RESULT[
        "RAG_ONLY"
    ]
)


LIVE_MCP_VIEW = (

    MODULE4_LIVE_RETRIEVAL_RESULT[
        "MCP_ONLY"
    ]
)


LIVE_HYBRID_VIEW = (

    MODULE4_LIVE_RETRIEVAL_RESULT[
        "HYBRID"
    ]
)


LIVE_HYBRID_EVIDENCE = (

    LIVE_HYBRID_VIEW[
        "evidence"
    ]
)


LIVE_HYBRID_CONTEXT = (

    LIVE_HYBRID_VIEW[
        "context"
    ]
)


LIVE_HYBRID_CONTEXT_CHARS = (

    LIVE_HYBRID_VIEW[
        "context_chars"
    ]
)


LIVE_HYBRID_EVIDENCE_COUNT = (

    len(
        LIVE_HYBRID_EVIDENCE
    )
)


print(
    f"STEP 1 — Retrieval complete → "
    f"{LIVE_RETRIEVAL_WALL_S:.3f} s"
)


print(
    f"         RAG candidates      → "
    f"{LIVE_RAG_VIEW['candidate_count']}"
)


print(
    f"         MCP candidates      → "
    f"{LIVE_MCP_VIEW['candidate_count']}"
)


print(
    f"         Final Hybrid items  → "
    f"{LIVE_HYBRID_EVIDENCE_COUNT}"
)


print(
    f"         Context             → "
    f"{LIVE_HYBRID_CONTEXT_CHARS:,} chars"
)


print(
    f"         MCP internal route  → "
    f"{LIVE_HYBRID_VIEW['mcp_trace'].get('route', '').upper()}"
)


# ============================================================
# 3D.8 — LIVE EVIDENCE VALIDATION
# ============================================================

if LIVE_HYBRID_EVIDENCE_COUNT == 0:

    raise RuntimeError(
        "Live Hybrid retrieval returned no evidence."
    )


if (
    LIVE_HYBRID_EVIDENCE_COUNT
    >
    COMMON_MAX_EVIDENCE_ITEMS
):

    raise RuntimeError(
        "Live Hybrid evidence exceeds Top-5."
    )


if (
    LIVE_HYBRID_CONTEXT_CHARS
    >
    COMMON_CONTEXT_MAX_CHARS
):

    raise RuntimeError(
        "Live Hybrid context exceeds 12,500 characters."
    )


# ============================================================
# 3D.9 — LIVE EVIDENCE CONTENT PREVIEW
# ============================================================

LIVE_EVIDENCE_PREVIEW_CHARS = 400


print()
print("LIVE HYBRID EVIDENCE — CONTENT PREVIEW")
print("-" * 116)


for item in LIVE_HYBRID_EVIDENCE:

    rank = item[
        "presentation_rank"
    ]


    retrieval = "/".join(
        item.get(
            "retrieval_systems",
            []
        )
    )


    preview = (
        item.get(
            "text",
            ""
        )[
            :LIVE_EVIDENCE_PREVIEW_CHARS
        ]
        .strip()
    )


    if (
        len(
            item.get(
                "text",
                ""
            )
        )
        >
        LIVE_EVIDENCE_PREVIEW_CHARS
    ):

        preview += " ..."


    print()

    print(
        f"[E{rank}] "
        f"Retrieval={retrieval} | "
        f"Source={item.get('source_family', '')}"
    )


    print(
        f"Document : "
        f"{item.get('document_id', '')}"
    )


    print(
        f"Title    : "
        f"{item.get('title', '')}"
    )


    print(
        f"Preview  : "
        f"{preview}"
    )


    print(
        "-" * 116
    )


# ============================================================
# 3D.10 — GENERATOR USER PROMPT
# ============================================================

LIVE_GENERATION_USER_PROMPT = (

    f"QUESTION\n"
    f"--------\n"
    f"{MODULE4_LIVE_QUESTION}\n\n"

    f"TECHNICAL EVIDENCE\n"
    f"------------------\n"
    f"{LIVE_HYBRID_CONTEXT}"
)


LIVE_GENERATION_MESSAGES = [

    {
        "role":
            "system",

        "content":
            MODULE4_FINAL_ANSWER_SYSTEM_PROMPT,
    },

    {
        "role":
            "user",

        "content":
            LIVE_GENERATION_USER_PROMPT,
    },
]


# ============================================================
# 3D.11 — LIVE HOSTED GENERATION
# ============================================================

print()
print(
    "STEP 2 — Grounded hosted generation starting..."
)


LIVE_GENERATION_START = (
    time.perf_counter()
)


MODULE4_LIVE_GENERATION_RESULT = (

    generate_hosted_llm(

        messages=
            LIVE_GENERATION_MESSAGES,

        max_tokens=
            LLM_MAX_TOKENS,
    )
)


LIVE_GENERATION_WALL_S = (

    time.perf_counter()
    -
    LIVE_GENERATION_START
)


MODULE4_LIVE_ANSWER = (

    MODULE4_LIVE_GENERATION_RESULT[
        "text"
    ]
)


print(
    f"STEP 2 — Generation complete → "
    f"{LIVE_GENERATION_WALL_S:.3f} s"
)


# ============================================================
# 3D.12 — END COMPLETE TIMER
# ============================================================

LIVE_E2E_TOTAL_S = (

    time.perf_counter()
    -
    LIVE_E2E_START
)


# ============================================================
# 3D.13 — MEMORY AFTER
# ============================================================

gc.collect()


LIVE_E2E_RAM_AFTER_GIB = (

    psutil.Process(
        os.getpid()
    )
    .memory_info()
    .rss
    /
    (1024 ** 3)
)


LIVE_E2E_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory()
    .available
    /
    (1024 ** 3)
)


LIVE_E2E_RAM_DELTA_GIB = (

    LIVE_E2E_RAM_AFTER_GIB
    -
    LIVE_E2E_RAM_BEFORE_GIB
)


# ============================================================
# 3D.14 — ANSWER CITATION VALIDATION
# ============================================================
#
# Supports:
#
#     [E1]
#     [E1, E3]
#     [E2, E4, E5]
#
# Every E-number appearing inside a bracketed evidence
# citation group is extracted.
# ============================================================

def extract_evidence_citations(
    answer
):

    answer = str(
        answer
        or
        ""
    )


    citations = set()


    citation_blocks = re.findall(

        r"\[[^\]]*E\d+[^\]]*\]",

        answer,
    )


    for block in citation_blocks:

        for value in re.findall(

            r"E(\d+)",

            block,
        ):

            citations.add(
                int(
                    value
                )
            )


    return sorted(
        citations
    )


LIVE_ANSWER_CITATIONS = (
    extract_evidence_citations(
        MODULE4_LIVE_ANSWER
    )
)


LIVE_ANSWER_NONEMPTY_PASS = (

    len(
        MODULE4_LIVE_ANSWER.strip()
    )
    >=
    100
)


LIVE_HAS_EVIDENCE_CITATION_PASS = (

    len(
        LIVE_ANSWER_CITATIONS
    )
    >
    0
)


LIVE_CITATION_RANGE_PASS = all(

    1
    <=
    citation
    <=
    LIVE_HYBRID_EVIDENCE_COUNT

    for citation
    in LIVE_ANSWER_CITATIONS
)


LIVE_FINISH_REASON_PASS = (

    MODULE4_LIVE_GENERATION_RESULT[
        "finish_reason"
    ]
    ==
    "stop"
)


LIVE_CITATION_COVERAGE_PCT = (

    (
        len(
            LIVE_ANSWER_CITATIONS
        )
        /
        LIVE_HYBRID_EVIDENCE_COUNT
    )
    *
    100.0

    if LIVE_HYBRID_EVIDENCE_COUNT
    else
    0.0
)


# ============================================================
# 3D.15 — LIVE RETRIEVAL VALIDATION
# ============================================================

LIVE_RAG_PASS = (

    LIVE_RAG_VIEW[
        "candidate_count"
    ]
    >
    0
)


LIVE_MCP_PASS = (

    LIVE_MCP_VIEW[
        "candidate_count"
    ]
    >
    0
)


LIVE_HYBRID_PASS = (

    LIVE_HYBRID_EVIDENCE_COUNT
    >
    0
)


LIVE_CONTEXT_PASS = (

    LIVE_HYBRID_CONTEXT_CHARS
    <=
    COMMON_CONTEXT_MAX_CHARS
)


# ============================================================
# 3D.16 — OVERALL VALIDATION
# ============================================================

MODULE4_LIVE_E2E_PASS = all(
    [
        LIVE_RAG_PASS,
        LIVE_MCP_PASS,
        LIVE_HYBRID_PASS,
        LIVE_CONTEXT_PASS,
        LIVE_ANSWER_NONEMPTY_PASS,
        LIVE_HAS_EVIDENCE_CITATION_PASS,
        LIVE_CITATION_RANGE_PASS,
        LIVE_FINISH_REASON_PASS,
    ]
)


MODULE4_CELL3D_PASS = (
    MODULE4_LIVE_E2E_PASS
)


if not MODULE4_CELL3D_PASS:

    raise RuntimeError(
        "Live Hybrid grounded generation failed."
    )


# ============================================================
# 3D.17 — SAVE COMPLETE LIVE RESULT
# ============================================================

MODULE4_LIVE_E2E_RESULT = {

    "question":
        MODULE4_LIVE_QUESTION,

    "mode":
        "HYBRID",

    "model":
        LLM_MODEL,

    "retrieval":
        MODULE4_LIVE_RETRIEVAL_RESULT,

    "evidence":
        LIVE_HYBRID_EVIDENCE,

    "context":
        LIVE_HYBRID_CONTEXT,

    "context_chars":
        LIVE_HYBRID_CONTEXT_CHARS,

    "answer":
        MODULE4_LIVE_ANSWER,

    "citations":
        LIVE_ANSWER_CITATIONS,

    "citation_coverage_pct":
        LIVE_CITATION_COVERAGE_PCT,

    "generation":
        MODULE4_LIVE_GENERATION_RESULT,

    "timing": {

        "retrieval_wall_s":
            LIVE_RETRIEVAL_WALL_S,

        "generation_wall_s":
            LIVE_GENERATION_WALL_S,

        "end_to_end_s":
            LIVE_E2E_TOTAL_S,
    },
}


# ============================================================
# 3D.18 — FINAL GROUNDED ANSWER
# ============================================================

print()
print("=" * 116)
print("LIVE GROUNDED ANSWER")
print("=" * 116)

print()

print(
    MODULE4_LIVE_ANSWER
)


# ============================================================
# 3D.19 — LIVE END-TO-END SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — LIVE END-TO-END SUMMARY")
print("=" * 116)


print(
    f"Model                       : "
    f"{LLM_MODEL}"
)


print(
    "Retrieval Architecture      : "
    "Semantic RAG V1 + MCP Version A"
)


print(
    f"RAG Candidates              : "
    f"{LIVE_RAG_VIEW['candidate_count']}"
)


print(
    f"MCP Candidates              : "
    f"{LIVE_MCP_VIEW['candidate_count']}"
)


print(
    f"Final Evidence              : "
    f"{LIVE_HYBRID_EVIDENCE_COUNT}"
)


print(
    f"Context Chars               : "
    f"{LIVE_HYBRID_CONTEXT_CHARS:,}"
)


print(
    f"MCP Internal Route          : "
    f"{LIVE_HYBRID_VIEW['mcp_trace'].get('route', '').upper()}"
)


print(
    f"Evidence Citations Used     : "
    f"{LIVE_ANSWER_CITATIONS}"
)


print(
    f"Citation Coverage           : "
    f"{LIVE_CITATION_COVERAGE_PCT:.1f}%"
)


print()
print("TIMING")
print("-" * 116)


print(
    f"RAG Retrieval               : "
    f"{LIVE_HYBRID_VIEW['timing']['rag_retrieval_s']:.3f} s"
)


print(
    f"MCP Retrieval               : "
    f"{LIVE_HYBRID_VIEW['timing']['mcp_retrieval_s']:.3f} s"
)


print(
    f"Parallel Retrieval Wall     : "
    f"{LIVE_HYBRID_VIEW['timing']['parallel_retrieval_wall_s']:.3f} s"
)


print(
    f"Fusion Only                 : "
    f"{LIVE_HYBRID_VIEW['timing']['fusion_only_s']:.6f} s"
)


print(
    f"Generation                  : "
    f"{LIVE_GENERATION_WALL_S:.3f} s"
)


print(
    f"TOTAL END-TO-END            : "
    f"{LIVE_E2E_TOTAL_S:.3f} s"
)


print()
print("TOKEN USAGE")
print("-" * 116)


print(
    f"Prompt Tokens               : "
    f"{MODULE4_LIVE_GENERATION_RESULT['prompt_tokens']}"
)


print(
    f"Completion Tokens           : "
    f"{MODULE4_LIVE_GENERATION_RESULT['completion_tokens']}"
)


print(
    f"Total Tokens                : "
    f"{MODULE4_LIVE_GENERATION_RESULT['total_tokens']}"
)


print(
    f"Finish Reason               : "
    f"{MODULE4_LIVE_GENERATION_RESULT['finish_reason']}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{LIVE_E2E_RAM_BEFORE_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{LIVE_E2E_RAM_AFTER_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{LIVE_E2E_RAM_DELTA_GIB:+.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{LIVE_E2E_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)


print(
    f"Live RAG PASS               : "
    f"{LIVE_RAG_PASS}"
)


print(
    f"Live MCP PASS               : "
    f"{LIVE_MCP_PASS}"
)


print(
    f"Live Hybrid PASS            : "
    f"{LIVE_HYBRID_PASS}"
)


print(
    f"Context Budget PASS         : "
    f"{LIVE_CONTEXT_PASS}"
)


print(
    f"Answer Non-Empty PASS       : "
    f"{LIVE_ANSWER_NONEMPTY_PASS}"
)


print(
    f"Evidence Citation PASS      : "
    f"{LIVE_HAS_EVIDENCE_CITATION_PASS}"
)


print(
    f"Citation Range PASS         : "
    f"{LIVE_CITATION_RANGE_PASS}"
)


print(
    f"Finish Reason PASS          : "
    f"{LIVE_FINISH_REASON_PASS}"
)


print(
    f"Live End-to-End PASS        : "
    f"{MODULE4_LIVE_E2E_PASS}"
)


print()
print("✓ CELL 3D PASSED")
print("✓ Fresh Semantic RAG retrieval executed.")
print("✓ Fresh MCP Version A retrieval executed.")
print("✓ RAG and MCP executed concurrently.")
print("✓ Fresh Hybrid evidence fusion executed.")
print("✓ Fresh evidence content inspected.")
print("✓ Hosted Gemma 4 generation executed.")
print("✓ Individual and grouped [E#] citations validated.")
print("✓ Full retrieval + generation timing captured.")
print("✓ Complete live Module 4 deployment path validated.")

print("=" * 116)


MODULE 4 DEMO — LIVE HYBRID GROUNDED GENERATION
Question                    : What are the primary responsibilities of the AMF in a 5G Standalone network?
Retrieval Architecture      : LIVE Semantic RAG V1 + LIVE MCP Version A
Generation Model            : google/gemma-4-26b-a4b-it
Execution Policy            : Fresh retrieval on every run

STEP 1 — Live RAG + MCP retrieval starting...
STEP 1 — Retrieval complete → 0.832 s
         RAG candidates      → 5
         MCP candidates      → 2
         Final Hybrid items  → 5
         Context             → 12,499 chars
         MCP internal route  → 3GPP

LIVE HYBRID EVIDENCE — CONTENT PREVIEW
--------------------------------------------------------------------------------------------------------------------

[E1] Retrieval=RAG | Source=3GPP
Document : standards/3gpp_rel18/original/rel_15.docx
Title    : rel_15
Preview  : the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not speci

##### Cell 3D — Observation

End-to-end RAG + MCP fusion → grounded Gemma generation passed. Retrieval and generation timings remain separated because hosted-provider latency varies independently of local retrieval latency.


# SECTION 4 — Flexible Knowledge Routing + Evaluation


#### Cell 4A — Granite Judge Setup + Validation

**Description:** Validate Granite 4.2 8B as the independent structured comparative judge for connected-corpus evidence.


In [18]:
# ============================================================
# CELL 4A — GRANITE JUDGE SETUP + VALIDATION
# ============================================================
#
# Purpose:
#
#   Validate the independent LLM judge before integrating it
#   into the Standalone-vs-Grounded comparison.
#
# No RAG.
# No MCP.
# No Gemma generation.
# ============================================================


# ============================================================
# 4A.1 — IMPORTS
# ============================================================

import gc
import json
import os
import time

import psutil
from openai import OpenAI


# ============================================================
# 4A.2 — JUDGE CONFIGURATION
# ============================================================

JUDGE_PROVIDER = "OpenRouter"

JUDGE_MODEL = (
    "ibm-granite/granite-4.2-8b"
)

JUDGE_DISPLAY_NAME = (
    "Granite 4.2 8B"
)

JUDGE_REASONING_EFFORT = "low"

JUDGE_MAX_TOKENS = 300


# ============================================================
# 4A.3 — OPENROUTER CLIENT
# ============================================================
#
# Reuse the existing OpenRouter API key configured in Cell 0B.
# ============================================================

if (
    "OPENROUTER_API_KEY" not in globals()
    or
    not OPENROUTER_API_KEY
):
    raise RuntimeError(
        "OPENROUTER_API_KEY is unavailable."
    )


judge_client = OpenAI(

    base_url=
        "https://openrouter.ai/api/v1",

    api_key=
        OPENROUTER_API_KEY,
)


# ============================================================
# 4A.4 — REUSABLE GRANITE JUDGE FUNCTION
# ============================================================
#
# This function will later be reused for the real comparative
# hallucination-risk evaluation.
#
# response_format requests valid JSON.
#
# Low reasoning effort is selected to balance judge quality
# against dashboard latency.
# ============================================================

def call_granite_judge(
    messages,
    max_tokens=JUDGE_MAX_TOKENS,
):

    start = (
        time.perf_counter()
    )


    response = (
        judge_client.chat.completions.create(

            model=
                JUDGE_MODEL,

            messages=
                messages,

            max_tokens=
                max_tokens,

            temperature=
                1.0,

            top_p=
                0.95,

            response_format={
                "type":
                    "json_object"
            },

            extra_body={

                "reasoning": {

                    "effort":
                        JUDGE_REASONING_EFFORT,

                    "exclude":
                        True,
                }
            },
        )
    )


    elapsed = (
        time.perf_counter()
        -
        start
    )


    message = (
        response.choices[0].message
    )


    content = str(
        message.content
        or
        ""
    ).strip()


    usage = getattr(
        response,
        "usage",
        None,
    )


    prompt_tokens = int(
        getattr(
            usage,
            "prompt_tokens",
            0,
        )
        or
        0
    )


    completion_tokens = int(
        getattr(
            usage,
            "completion_tokens",
            0,
        )
        or
        0
    )


    total_tokens = int(
        getattr(
            usage,
            "total_tokens",
            0,
        )
        or
        0
    )


    return {

        "text":
            content,

        "elapsed_s":
            float(
                elapsed
            ),

        "prompt_tokens":
            prompt_tokens,

        "completion_tokens":
            completion_tokens,

        "total_tokens":
            total_tokens,

        "finish_reason":
            response.choices[
                0
            ].finish_reason,

        "requested_model":
            JUDGE_MODEL,

        "response_model":
            getattr(
                response,
                "model",
                "",
            ),
    }


# ============================================================
# 4A.5 — VALIDATION PROMPT
# ============================================================
#
# This is intentionally small.
#
# We are testing:
#
#   - model availability
#   - low-effort reasoning request
#   - JSON output
#   - response completeness
#
# We are NOT yet judging a telecom answer.
# ============================================================

JUDGE_VALIDATION_SYSTEM_PROMPT = """
You are an independent technical evaluation model.

Return only a valid JSON object matching the requested fields.
Do not include markdown or explanatory text outside the JSON.
""".strip()


JUDGE_VALIDATION_USER_PROMPT = """
Confirm that you can operate as an independent evaluator comparing two
telecommunications answers against supplied technical evidence.

Return exactly these fields:

{
  "judge_status": "operational",
  "comparison_ready": true,
  "supports_claim_evaluation": true,
  "message": "A short confirmation."
}
""".strip()


JUDGE_VALIDATION_MESSAGES = [

    {
        "role":
            "system",

        "content":
            JUDGE_VALIDATION_SYSTEM_PROMPT,
    },

    {
        "role":
            "user",

        "content":
            JUDGE_VALIDATION_USER_PROMPT,
    },
]


# ============================================================
# 4A.6 — MEMORY BASELINE
# ============================================================

gc.collect()


process = psutil.Process(
    os.getpid()
)


JUDGE_RAM_BEFORE_GIB = (

    process.memory_info().rss
    /
    (1024 ** 3)
)


# ============================================================
# 4A.7 — LIVE JUDGE VALIDATION
# ============================================================

print("=" * 116)
print("MODULE 4 DEMO — INDEPENDENT JUDGE VALIDATION")
print("=" * 116)

print(
    f"Provider                    : "
    f"{JUDGE_PROVIDER}"
)

print(
    f"Judge Model                 : "
    f"{JUDGE_MODEL}"
)

print(
    f"Display Name                : "
    f"{JUDGE_DISPLAY_NAME}"
)

print(
    f"Reasoning Effort            : "
    f"{JUDGE_REASONING_EFFORT.upper()}"
)

print(
    f"Maximum Output Tokens       : "
    f"{JUDGE_MAX_TOKENS}"
)

print(
    "Structured Output           : JSON"
)

print()
print(
    "Executing independent judge validation..."
)


JUDGE_VALIDATION_RESULT = (
    call_granite_judge(
        messages=
            JUDGE_VALIDATION_MESSAGES
    )
)


# ============================================================
# 4A.8 — PARSE STRUCTURED OUTPUT
# ============================================================

JUDGE_JSON_PARSE_PASS = False
JUDGE_VALIDATION_PAYLOAD = {}


try:

    JUDGE_VALIDATION_PAYLOAD = (
        json.loads(
            JUDGE_VALIDATION_RESULT[
                "text"
            ]
        )
    )

    JUDGE_JSON_PARSE_PASS = (
        isinstance(
            JUDGE_VALIDATION_PAYLOAD,
            dict,
        )
    )


except Exception:

    JUDGE_JSON_PARSE_PASS = False


# ============================================================
# 4A.9 — MEMORY AFTER
# ============================================================

gc.collect()


JUDGE_RAM_AFTER_GIB = (

    process.memory_info().rss
    /
    (1024 ** 3)
)


JUDGE_AVAILABLE_AFTER_GIB = (

    psutil.virtual_memory().available
    /
    (1024 ** 3)
)


JUDGE_RAM_DELTA_GIB = (

    JUDGE_RAM_AFTER_GIB
    -
    JUDGE_RAM_BEFORE_GIB
)


# ============================================================
# 4A.10 — VALIDATION CHECKS
# ============================================================

JUDGE_RESPONSE_NONEMPTY_PASS = (

    len(
        JUDGE_VALIDATION_RESULT[
            "text"
        ]
    )
    >
    0
)


JUDGE_STATUS_PASS = (

    JUDGE_VALIDATION_PAYLOAD.get(
        "judge_status"
    )
    ==
    "operational"
)


JUDGE_COMPARISON_READY_PASS = (

    JUDGE_VALIDATION_PAYLOAD.get(
        "comparison_ready"
    )
    is True
)


JUDGE_CLAIM_EVALUATION_PASS = (

    JUDGE_VALIDATION_PAYLOAD.get(
        "supports_claim_evaluation"
    )
    is True
)


JUDGE_FINISH_REASON_PASS = (

    JUDGE_VALIDATION_RESULT[
        "finish_reason"
    ]
    ==
    "stop"
)


MODULE4_JUDGE_VALIDATION_PASS = all(
    [
        JUDGE_RESPONSE_NONEMPTY_PASS,
        JUDGE_JSON_PARSE_PASS,
        JUDGE_STATUS_PASS,
        JUDGE_COMPARISON_READY_PASS,
        JUDGE_CLAIM_EVALUATION_PASS,
        JUDGE_FINISH_REASON_PASS,
    ]
)


MODULE4_CELL4A_PASS = (
    MODULE4_JUDGE_VALIDATION_PASS
)


if not MODULE4_CELL4A_PASS:

    raise RuntimeError(
        "Granite judge validation failed."
    )


# ============================================================
# 4A.11 — DISPLAY RESPONSE
# ============================================================

print()
print("JUDGE RESPONSE")
print("-" * 116)

print(
    json.dumps(
        JUDGE_VALIDATION_PAYLOAD,
        indent=2,
    )
)


# ============================================================
# 4A.12 — SUMMARY
# ============================================================

print()
print("=" * 116)
print("MODULE 4 DEMO — JUDGE VALIDATION SUMMARY")
print("=" * 116)

print(
    f"Requested Model             : "
    f"{JUDGE_VALIDATION_RESULT['requested_model']}"
)

print(
    f"Response Model              : "
    f"{JUDGE_VALIDATION_RESULT['response_model']}"
)

print(
    f"Judge Latency               : "
    f"{JUDGE_VALIDATION_RESULT['elapsed_s']:.3f} s"
)

print(
    f"Prompt Tokens               : "
    f"{JUDGE_VALIDATION_RESULT['prompt_tokens']}"
)

print(
    f"Completion Tokens           : "
    f"{JUDGE_VALIDATION_RESULT['completion_tokens']}"
)

print(
    f"Total Tokens                : "
    f"{JUDGE_VALIDATION_RESULT['total_tokens']}"
)

print(
    f"Finish Reason               : "
    f"{JUDGE_VALIDATION_RESULT['finish_reason']}"
)


print()
print("MEMORY IMPACT")
print("-" * 116)

print(
    f"Process RAM Before           : "
    f"{JUDGE_RAM_BEFORE_GIB:.3f} GiB"
)

print(
    f"Process RAM After            : "
    f"{JUDGE_RAM_AFTER_GIB:.3f} GiB"
)

print(
    f"RAM Delta                   : "
    f"{JUDGE_RAM_DELTA_GIB:+.3f} GiB"
)

print(
    f"System RAM Available        : "
    f"{JUDGE_AVAILABLE_AFTER_GIB:.3f} GiB"
)


print()
print("VALIDATION")
print("-" * 116)

print(
    f"JSON Parse PASS             : "
    f"{JUDGE_JSON_PARSE_PASS}"
)

print(
    f"Judge Status PASS           : "
    f"{JUDGE_STATUS_PASS}"
)

print(
    f"Comparison Ready PASS       : "
    f"{JUDGE_COMPARISON_READY_PASS}"
)

print(
    f"Claim Evaluation PASS       : "
    f"{JUDGE_CLAIM_EVALUATION_PASS}"
)

print(
    f"Finish Reason PASS          : "
    f"{JUDGE_FINISH_REASON_PASS}"
)

print(
    f"Judge Validation PASS       : "
    f"{MODULE4_JUDGE_VALIDATION_PASS}"
)


print()
print("✓ CELL 4A PASSED")
print("✓ Granite 4.2 8B judge endpoint validated.")
print("✓ Low-effort reasoning configured.")
print("✓ Structured JSON output validated.")
print("✓ Judge latency and token usage captured.")
print("✓ No RAG retrieval executed.")
print("✓ No MCP retrieval executed.")
print("✓ No Gemma generation executed.")
print("✓ Ready for Standalone-vs-Grounded comparison.")

print("=" * 116)


MODULE 4 DEMO — INDEPENDENT JUDGE VALIDATION
Provider                    : OpenRouter
Judge Model                 : ibm-granite/granite-4.2-8b
Display Name                : Granite 4.2 8B
Reasoning Effort            : LOW
Maximum Output Tokens       : 300
Structured Output           : JSON

Executing independent judge validation...

JUDGE RESPONSE
--------------------------------------------------------------------------------------------------------------------
{
  "judge_status": "operational",
  "comparison_ready": true,
  "supports_claim_evaluation": true,
  "message": "Confirmation received. Ready to evaluate telecommunications answers against technical evidence."
}

MODULE 4 DEMO — JUDGE VALIDATION SUMMARY
Requested Model             : ibm-granite/granite-4.2-8b
Response Model              : ibm-granite/granite-4.2-8b
Judge Latency               : 0.708 s
Prompt Tokens               : 130
Completion Tokens           : 58
Total Tokens                : 188
Finish Reason            

##### Cell 4A — Observation

Granite returned valid structured output and remains the router/comparative-evaluation model. The comparative judge is deliberately restricted to connected technical-corpus evidence rather than reused for live-web or general-knowledge answers.


#### Cell 4B — Editable User Question

**Description:** Define the only user-editable question input for the final runtime. No expected route is supplied.


In [19]:
# ============================================================
# CELL 4C.6A — EDITABLE USER QUESTION
# ============================================================
#
# PURPOSE
# -------
#
# This is the ONLY cell that needs to change when testing
# different user questions.
#
# The question may be:
#
#   - Telecom
#   - Non-telecom
#   - Ambiguous
#   - Technical
#   - General knowledge
#   - Adversarial / prompt-injection attempt
#
# Cell 4C and the final 6Y runtime determine the correct execution path.
#
# No expected domain route is supplied here.
#
# ============================================================


MODULE4_4C6_RAW_PROMPT = """
What is 5G?
""".strip()


if not MODULE4_4C6_RAW_PROMPT:
    raise ValueError(
        "MODULE4_4C6_RAW_PROMPT must not be empty."
    )


print("=" * 116)
print("MODULE 4C.6A — EDITABLE USER QUESTION")
print("=" * 116)

print(
    f"Question : "
    f"{MODULE4_4C6_RAW_PROMPT}"
)

print()
print(
    "✓ Question loaded."
)

print(
    "✓ No expected domain route has been hard-coded."
)

print(
    "✓ Granite will determine the initial knowledge scope; 6Y will execute the appropriate grounded, live, runtime, or fallback path."
)

print("=" * 116)


MODULE 4C.6A — EDITABLE USER QUESTION
Question : What is 5G?

✓ Question loaded.
✓ No expected domain route has been hard-coded.
✓ Granite will determine the initial knowledge scope; 6Y will execute the appropriate grounded, live, runtime, or fallback path.


##### Cell 4B — Observation

`What is 5G?` remains the default reproducibility question for the public notebook because it exercises the connected-corpus path and the comparative evaluation layer. Change only this cell when testing another user question.


#### Cell 4C — Granite Knowledge-Scope Router

**Description:** Define the low-latency corpus-aware base router in non-thinking mode. The final 6Y runtime extends this decision into connected-corpus, live-external, and stable-general execution.


In [20]:
# ============================================================
# CELL 4C — CORPUS-AWARE GRANITE KNOWLEDGE-SCOPE ROUTER
#                NON-THINKING ROUTER MODE
# ============================================================
#
# PURPOSE
# -------
#
# Replace the earlier:
#
#     "Is this explicitly a telecom question?"
#
# semantic decision with:
#
#     "Is this question meaningfully covered by the connected
#      grounded knowledge corpus?"
#
#
# IMPORTANT
# ---------
#
# This cell defines the base connected-corpus routing layer.
# The final 6Y runtime extends this into the three-way knowledge-scope strategy.
#
# It does NOT change:
#
# - Cell 4C.6A question handling
# - deterministic security
# - Gemma 4 baseline generation
# - RAG retrieval
# - MCP retrieval
# - Hybrid fusion
# - evidence scanning
# - grounded generation
# - fallback generation
# - risk applicability
# - Cell 4C.6S orchestration
#
#
# ROUTER EXECUTION CONTROL
# ------------------------
#
# Granite 4.2 8B is used in NON-THINKING mode for this routing task.
#
# Reason:
#
# - this is a compact semantic classification task
# - extended reasoning is unnecessary
# - reasoning tokens can consume a short completion budget before
#   final JSON content is emitted
# - disabling reasoning should preserve the low-latency router role
#
#
# DOWNSTREAM ROUTE VALUES REMAIN UNCHANGED
# ----------------------------------------
#
# TELECOM_GROUNDED
# GENERAL_KNOWLEDGE_FALLBACK
#
# TELECOM_GROUNDED now operationally means:
#
#     Connected grounded corpus coverage is applicable.
#
# ============================================================


import json
import re
import time


# ============================================================
# 4C.6R.0 — BASE KNOWLEDGE-SCOPE ROUTE CONSTANTS
# ============================================================
#
# These constants were originally defined in an earlier experimental
# routing cell. They are now defined locally so Cell 4C is self-contained
# and reproducible in a fresh sequential notebook run.
# ============================================================

TELECOM_GROUNDED_MODE = "TELECOM_GROUNDED"

GENERAL_KNOWLEDGE_FALLBACK_MODE = (
    "GENERAL_KNOWLEDGE_FALLBACK"
)

VALID_RESPONSE_MODES = {
    TELECOM_GROUNDED_MODE,
    GENERAL_KNOWLEDGE_FALLBACK_MODE,
}


# ============================================================
# 4C.6R.1 — VERIFY EXISTING NOTEBOOK RUNTIME
# ============================================================

required_4c6r = [

    "openrouter",

    "JUDGE_MODEL",

    "TELECOM_GROUNDED_MODE",

    "GENERAL_KNOWLEDGE_FALLBACK_MODE",

    "VALID_RESPONSE_MODES",
]


for name in required_4c6r:

    if name not in globals():

        raise RuntimeError(
            f"Cell 4C.6R missing required runtime object: {name}"
        )


if openrouter is None:

    raise RuntimeError(
        "Existing Notebook 21 `openrouter` client is unavailable."
    )


# ============================================================
# 4C.6R.2 — COMPACT CONNECTED-CORPUS COVERAGE PROFILE
# ============================================================
#
# This intentionally describes knowledge families rather than
# enumerating every document, standard, RFC or corpus record.
# ============================================================

MODULE4_CORPUS_COVERAGE_PROFILE = """
The connected grounded knowledge corpus covers these major technical families:

1. Mobile and telecommunications engineering:
   2G, 3G, 4G/LTE, 5G, 5G-Advanced, RAN, NR, NG-RAN, mobile core,
   5G Core network functions, interfaces, procedures, protocols,
   mobility, registration, session management, QoS, slicing,
   security, OSS, network operations, automation, orchestration,
   deployment and performance.

2. Telecommunications standards and industry ecosystems:
   dedicated 3GPP specifications and 3GPP working-group material,
   ETSI, ITU-T, GSMA, O-RAN, TM Forum and CAMARA.

3. Internet and packet-networking technologies represented through TCC:
   IETF RFCs, Internet-Drafts, proceedings and mailing-list material,
   including Internet protocols and mechanisms such as TCP/IP, UDP,
   DNS, HTTP/2, HTTP/3, TLS, QUIC, NETCONF, RESTCONF and related
   networking topics.

4. Cloud-native and platform technologies represented in the corpus:
   Kubernetes, containers, container orchestration, microservices,
   cloud-native network functions (CNFs), virtualization,
   cloud infrastructure, cloud-native networking, eBPF,
   observability and telemetry.

5. Research and technical knowledge represented through TCC:
   IEEE Access, OpenAlex research literature, telecom/networking
   patents from USPTO and EPO, telecom-focused Wikidata and
   telecom-focused Wikipedia.

The coverage description is semantic, not an exhaustive keyword list.

A question does not need to mention telecom or 5G if its subject itself
is materially represented in one of these connected knowledge families.
""".strip()


# ============================================================
# 4C.6R.3 — CORPUS-AWARE ROUTER SYSTEM PROMPT
# ============================================================

MODULE4_CORPUS_ROUTER_SYSTEM_PROMPT = f"""
You are the semantic knowledge-scope router for a grounded technical AI system.

Your task is to determine whether the CONNECTED GROUNDED KNOWLEDGE CORPUS
is likely to contain useful factual evidence for answering the user's question.

Do NOT simply decide whether the question explicitly mentions telecommunications.

CONNECTED GROUNDED KNOWLEDGE COVERAGE
-------------------------------------
{MODULE4_CORPUS_COVERAGE_PROFILE}

AVAILABLE RESPONSE MODES
------------------------

{TELECOM_GROUNDED_MODE}

Choose this when the subject of the question is directly and meaningfully
covered by the connected grounded corpus.

This includes:
- core telecommunications questions;
- standards and protocols represented in the corpus;
- IETF and Internet-networking topics represented in the corpus;
- Kubernetes and cloud-native/platform topics represented in the corpus;
- relevant research, patents and technical knowledge represented above.

A question does NOT need to mention telecom, mobile networks or 5G to use
this route if the subject itself is represented in the connected corpus.


{GENERAL_KNOWLEDGE_FALLBACK_MODE}

Choose this when the subject is outside the connected grounded corpus.

Examples include unrelated biography, general history, cooking, geology,
entertainment, ordinary general-knowledge topics and unrelated engineering.


DECISION RULES
--------------

1. Judge semantic corpus coverage, not simple keyword presence.

2. Do not require explicit telecom terminology when the technology itself
   is represented in the connected corpus.

3. Do not classify an unrelated topic as covered merely because words such
   as "network", "system", "technology", "platform", "function" or
   "communication" appear in the question.

4. Consider the complete meaning and intent of the question.

5. Do not retrieve evidence.

6. Do not answer the user's question.

7. Return exactly one response mode.

Return JSON only, with no Markdown and no explanatory text outside the JSON.

Use exactly this structure:

{{
  "response_mode": "{TELECOM_GROUNDED_MODE}",
  "reason": "Short semantic corpus-coverage reason."
}}

or:

{{
  "response_mode": "{GENERAL_KNOWLEDGE_FALLBACK_MODE}",
  "reason": "Short semantic corpus-coverage reason."
}}
""".strip()


# ============================================================
# 4C.6R.4 — ROUTER JSON EXTRACTION
# ============================================================

def _extract_4c6r_json(
    text,
):

    if not isinstance(
        text,
        str,
    ):

        raise TypeError(
            "Granite router response must be text."
        )


    cleaned = (
        text
        .strip()
    )


    cleaned = re.sub(

        r"^```(?:json)?\s*",

        "",

        cleaned,

        flags=
            re.IGNORECASE,
    )


    cleaned = re.sub(

        r"\s*```$",

        "",

        cleaned,
    )


    # --------------------------------------------------------
    # DIRECT JSON
    # --------------------------------------------------------

    try:

        return json.loads(
            cleaned
        )

    except json.JSONDecodeError:

        pass


    # --------------------------------------------------------
    # JSON OBJECT EMBEDDED IN TEXT
    # --------------------------------------------------------

    match = re.search(

        r"\{.*\}",

        cleaned,

        flags=
            re.DOTALL,
    )


    if match is not None:

        try:

            return json.loads(
                match.group(0)
            )

        except json.JSONDecodeError:

            pass


    raise RuntimeError(
        "Granite corpus-aware router did not return valid JSON."
    )


# ============================================================
# 4C.6R.5 — CORPUS-AWARE GRANITE ROUTER
# ============================================================
#
# IMPORTANT:
#
# Granite reasoning is explicitly disabled for this routing call.
#
# This keeps the router focused on fast semantic classification
# and prevents a short max_tokens budget being consumed entirely
# by reasoning before final JSON is emitted.
# ============================================================

def classify_4c_response_mode(
    question,
):

    if (
        not isinstance(
            question,
            str,
        )
        or
        not question.strip()
    ):

        raise ValueError(
            "Router question must be a non-empty string."
        )


    start = (
        time.perf_counter()
    )


    response = (
        openrouter.chat.completions.create(

            model=
                JUDGE_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        MODULE4_CORPUS_ROUTER_SYSTEM_PROMPT,
                },
                {
                    "role":
                        "user",

                    "content":
                        question.strip(),
                },
            ],

            temperature=
                0,

            max_tokens=
                120,

            # ------------------------------------------------
            # OpenRouter-specific control.
            #
            # Granite 4.2 supports non-thinking mode.
            #
            # Using extra_body keeps the existing OpenAI-
            # compatible client unchanged.
            # ------------------------------------------------

            extra_body={
                "reasoning": {
                    "effort":
                        "none",
                }
            },
        )
    )


    elapsed_s = (
        time.perf_counter()
        -
        start
    )


    choice = (
        response
        .choices[0]
    )


    message = (
        choice
        .message
    )


    raw = str(
        message.content
        or
        ""
    ).strip()


    # --------------------------------------------------------
    # DIAGNOSTIC ONLY
    #
    # This does not expose or use reasoning for classification.
    # It only lets us identify a provider/model behaviour issue
    # if final content is unexpectedly empty again.
    # --------------------------------------------------------

    reasoning_present = bool(

        getattr(
            message,
            "reasoning",
            None,
        )

        or

        getattr(
            message,
            "reasoning_content",
            None,
        )
    )


    if not raw:

        finish_reason = str(
            getattr(
                choice,
                "finish_reason",
                "",
            )
            or
            ""
        )


        completion_tokens = (

            getattr(
                getattr(
                    response,
                    "usage",
                    None,
                ),
                "completion_tokens",
                None,
            )
        )


        raise RuntimeError(

            "Granite corpus-aware router returned empty final content. "
            f"finish_reason={finish_reason!r}, "
            f"reasoning_present={reasoning_present}, "
            f"completion_tokens={completion_tokens!r}."
        )


    parsed = (
        _extract_4c6r_json(
            raw
        )
    )


    response_mode = str(

        parsed.get(
            "response_mode",
            "",
        )
    ).strip()


    reason = str(

        parsed.get(
            "reason",
            "",
        )
    ).strip()


    if (
        response_mode
        not in
        VALID_RESPONSE_MODES
    ):

        raise RuntimeError(

            "Granite corpus-aware router returned unsupported "
            f"response mode: {response_mode!r}"
        )


    if not reason:

        raise RuntimeError(
            "Granite corpus-aware router returned no routing reason."
        )


    return {

        "response_mode":
            response_mode,

        "reason":
            reason,

        "elapsed_s":
            elapsed_s,
    }


# ============================================================
# 4C.6R.6 — STATIC VALIDATION
# ============================================================
#
# No model/API request is executed here.
# ============================================================

if not MODULE4_CORPUS_COVERAGE_PROFILE:

    raise RuntimeError(
        "Corpus coverage profile is empty."
    )


if not MODULE4_CORPUS_ROUTER_SYSTEM_PROMPT:

    raise RuntimeError(
        "Corpus-aware router prompt is empty."
    )


MODULE4_4C6R_PASS = True


# ============================================================
# 4C.6R.7 — STATUS
# ============================================================

print("=" * 116)
print("MODULE 4C.6R — CORPUS-AWARE GRANITE ROUTER")
print("=" * 116)


print(
    f"Router Model                : "
    f"{JUDGE_MODEL}"
)


print(
    "Router Mode                 : NON-THINKING"
)


print(
    "Router API Calls / Question : 1"
)


print(
    "Pre-Routing Retrieval       : NONE"
)


print(
    "Additional Embedding Call   : NONE"
)


print(
    "Additional Classifier       : NONE"
)


print(
    "Decision Basis              : CONNECTED GROUNDED CORPUS COVERAGE"
)


print(
    f"Grounded Route Value         : "
    f"{TELECOM_GROUNDED_MODE}"
)


print(
    f"Fallback Route Value         : "
    f"{GENERAL_KNOWLEDGE_FALLBACK_MODE}"
)


print()


print(
    "✓ Existing Notebook 21 `openrouter` client retained."
)


print(
    "✓ Granite corpus-aware semantic routing retained."
)


print(
    "✓ Granite reasoning disabled for the routing task."
)


print(
    "✓ Existing downstream route values retained."
)


print(
    "✓ No retrieval or additional model call introduced."
)


print(
    "✓ classify_4c_response_mode() replaced in memory."
)


print(
    "✓ Ready for isolated router validation before Cell 4C.6S."
)


print("=" * 116)


MODULE 4C.6R — CORPUS-AWARE GRANITE ROUTER
Router Model                : ibm-granite/granite-4.2-8b
Router Mode                 : NON-THINKING
Router API Calls / Question : 1
Pre-Routing Retrieval       : NONE
Additional Embedding Call   : NONE
Additional Classifier       : NONE
Decision Basis              : CONNECTED GROUNDED CORPUS COVERAGE
Grounded Route Value         : TELECOM_GROUNDED
Fallback Route Value         : GENERAL_KNOWLEDGE_FALLBACK

✓ Existing Notebook 21 `openrouter` client retained.
✓ Granite corpus-aware semantic routing retained.
✓ Granite reasoning disabled for the routing task.
✓ Existing downstream route values retained.
✓ No retrieval or additional model call introduced.
✓ classify_4c_response_mode() replaced in memory.
✓ Ready for isolated router validation before Cell 4C.6S.


##### Cell 4C — Observation

The Granite router is **knowledge-scope based**, not keyword based. It decides whether the connected controlled corpus, live external information, or stable general knowledge is the appropriate authoritative source; legacy `TELECOM_GROUNDED` naming is retained only for compatibility.


#### Cell 4D — Final Flexible Knowledge Runtime — 6Y v1.3

**Description:** Execute the final validated runtime: deterministic security, baseline comparison, three-way knowledge routing, local date/time utility, Open-WebSearch live grounding, adaptive RAG/MCP/Hybrid retrieval, grounded generation, and the frozen comparative judge where applicable.


In [24]:
# ============================================================
# CELL 4D — FINAL FLEXIBLE KNOWLEDGE RUNTIME (6Y)
# ============================================================
#
# PURPOSE
# -------
#
# Execute ANY question supplied in Cell 4C.6A through the
# governed Telecom AI runtime.
#
#
# ARCHITECTURE
# ------------
#
# Raw User Question
#        │
#        ▼
# Deterministic Security Guardrail
#        │
#        ├── detect prompt injection
#        ├── detect evidence manipulation
#        └── preserve legitimate intent
#                 │
#                 ▼
#          Sanitized Question
#                 │
#                 ├──────────────► Gemma 4 Only
#                 │                comparison baseline
#                 │                no RAG / MCP
#                 │
#                 ▼
#          Granite Knowledge-Scope Router
#                 │
#          ┌──────┴───────────────┐
#          │                      │
#   CORPUS-COVERED          OUTSIDE CORPUS
#          │                      │
#          ▼                      ▼
#   Grounded LLM             Skip Retrieval
#          │                      │
#          ▼                      ▼
#   decides/searches          General LLM
#          │                      │
#          ▼                      ▼
#  Governed Hybrid Tool       Scope Caveat
#  (RAG + MCP internal)
#          │
#          ▼
#  Evidence Security Scan
#          │
#          ▼
#   tool_result to LLM
#          │
#          ├─ sufficient → final grounded answer
#          └─ insufficient → refined tool search (max 3)
#
#
# IMPORTANT
# ---------
#
# Granite decides the knowledge-scope route.
#
# On the grounded route, the LLM decides WHEN retrieval is needed,
# WHAT search query to issue, WHETHER the returned evidence is
# sufficient, and WHETHER a refined second/third search is needed.
#
# Python does NOT formulate retrieval queries or pre-fetch evidence.
# Python only validates tool calls, executes the existing Hybrid
# RAG + MCP service, applies deterministic evidence security checks,
# and enforces the maximum-search / evidence limits.
#
#
# STABLE HOSTED-LLM EXECUTION
# ---------------------------
#
# Gemma 4 Only + Granite Router may run concurrently because
# they are different models.
#
# The routed Gemma generation starts only AFTER the baseline
# Gemma generation has completed.
#
# Therefore two Gemma generations are never executed
# concurrently through the hosted provider.
#
#
# EVALUATION BOUNDARY
# -------------------
#
# Telecom evidence-relative hallucination risk is marked
# APPLICABLE for TELECOM_GROUNDED responses.
#
# The frozen 4B single-call Granite comparative judge is
# integrated directly into this routing-runtime cell.
#
# Grounded responses are judged immediately before the cell ends.
# General-Knowledge Fallback responses skip the evidence judge.
#
# ============================================================


import asyncio
import gc
import json
import os
import re
import time

import psutil



# ============================================================
# 4D.0 — SELF-CONTAINED RUNTIME COMPATIBILITY HELPERS
# ============================================================
#
# The original research notebook inherited these helpers from
# earlier experimental 4C cells. The cleaned Git notebook must
# define them explicitly so a fresh sequential run has no hidden
# notebook-state dependency.
#
# Existing implementations are preserved if already present.
# ============================================================

if "INJECTION_PATTERNS_4C3" not in globals():
    INJECTION_PATTERNS_4C3 = (
        r"\bignore (?:all |any |the )?(?:previous|prior|earlier) instructions?\b",
        r"\bdisregard (?:all |any |the )?(?:previous|prior|earlier) instructions?\b",
        r"\boverride (?:the )?(?:system|developer|previous|prior) instructions?\b",
        r"\breveal (?:the )?(?:system|developer) prompt\b",
        r"\bshow (?:me )?(?:the )?(?:system|developer) prompt\b",
        r"\bprint (?:the )?(?:system|developer) prompt\b",
        r"\byou are now\b",
        r"\bact as\b.*\binstead\b",
        r"\bforget (?:all |the )?(?:previous|prior|earlier) instructions?\b",
    )

if "EVIDENCE_MANIPULATION_PATTERNS_4C3" not in globals():
    EVIDENCE_MANIPULATION_PATTERNS_4C3 = (
        r"\bignore (?:the )?(?:retrieved )?evidence\b",
        r"\bignore (?:the )?(?:provided )?context\b",
        r"\bdo not cite\b",
        r"\bfabricate (?:a |the )?citation\b",
        r"\binvent (?:a |the )?citation\b",
        r"\bpretend (?:the )?evidence\b",
        r"\bclaim (?:that )?(?:the )?evidence says\b",
        r"\bchange (?:the )?evidence\b",
        r"\boverride (?:the )?evidence\b",
    )


if "get_injection_matches_4c3" not in globals():

    def get_injection_matches_4c3(text):
        value = str(text or "")
        matches = []

        for pattern in (
            tuple(INJECTION_PATTERNS_4C3)
            +
            tuple(EVIDENCE_MANIPULATION_PATTERNS_4C3)
        ):
            if re.search(
                pattern,
                value,
                flags=re.IGNORECASE | re.DOTALL,
            ):
                matches.append(pattern)

        return matches


if "scan_for_injection_4c3" not in globals():

    def scan_for_injection_4c3(text):
        value = str(text or "")

        return any(
            re.search(
                pattern,
                value,
                flags=re.IGNORECASE | re.DOTALL,
            )
            is not None

            for pattern in INJECTION_PATTERNS_4C3
        )


if "scan_for_evidence_manipulation_4c3" not in globals():

    def scan_for_evidence_manipulation_4c3(text):
        value = str(text or "")

        return any(
            re.search(
                pattern,
                value,
                flags=re.IGNORECASE | re.DOTALL,
            )
            is not None

            for pattern in EVIDENCE_MANIPULATION_PATTERNS_4C3
        )


if "preprocess_user_prompt_4c3" not in globals():

    def preprocess_user_prompt_4c3(prompt):
        start = time.perf_counter()

        raw_prompt = str(prompt or "").strip()

        injection_matches = [
            pattern
            for pattern in INJECTION_PATTERNS_4C3
            if re.search(
                pattern,
                raw_prompt,
                flags=re.IGNORECASE | re.DOTALL,
            )
        ]

        evidence_matches = [
            pattern
            for pattern in EVIDENCE_MANIPULATION_PATTERNS_4C3
            if re.search(
                pattern,
                raw_prompt,
                flags=re.IGNORECASE | re.DOTALL,
            )
        ]

        sanitized_prompt = raw_prompt

        for pattern in (
            tuple(injection_matches)
            +
            tuple(evidence_matches)
        ):
            sanitized_prompt = re.sub(
                pattern,
                " ",
                sanitized_prompt,
                flags=re.IGNORECASE | re.DOTALL,
            )

        sanitized_prompt = re.sub(
            r"\s+",
            " ",
            sanitized_prompt,
        ).strip()

        return {
            "raw_prompt": raw_prompt,
            "sanitized_prompt": sanitized_prompt,
            "injection_detected": bool(injection_matches),
            "evidence_manipulation_detected": bool(evidence_matches),
            "matched_patterns": (
                injection_matches
                +
                evidence_matches
            ),
            "elapsed_s": float(
                time.perf_counter()
                -
                start
            ),
        }


# ------------------------------------------------------------
# Hosted Gemma compatibility wrappers
# ------------------------------------------------------------

if "run_4c_gemma_only" not in globals():

    def run_4c_gemma_only(question):
        messages = [
            {
                "role": "system",
                "content": (
                    "Answer the user's question directly and accurately "
                    "using only your pretrained general knowledge. "
                    "Do not claim access to retrieved evidence, live web "
                    "information, RAG, MCP, or external tools."
                ),
            },
            {
                "role": "user",
                "content": str(question),
            },
        ]

        result = generate_hosted_llm(
            messages=messages,
            max_tokens=LLM_MAX_TOKENS,
        )

        return {
            "answer": str(result["text"]).strip(),
            "elapsed_s": float(result["elapsed_s"]),
            "prompt_tokens": result.get("prompt_tokens"),
            "completion_tokens": result.get("completion_tokens"),
            "total_tokens": result.get("total_tokens"),
        }


if "run_4c_general_fallback" not in globals():

    def run_4c_general_fallback(question):
        messages = [
            {
                "role": "system",
                "content": (
                    "Answer the user's question using stable general "
                    "knowledge only. Do not claim live or retrieved "
                    "knowledge. Begin the answer exactly with:\n\n"
                    "⚠ Outside Telecom Knowledge Domain: This response "
                    "uses the model's general knowledge and is not grounded "
                    "in the connected Telecom RAG/MCP sources.\n\n"
                    "Then provide the useful answer."
                ),
            },
            {
                "role": "user",
                "content": str(question),
            },
        ]

        result = generate_hosted_llm(
            messages=messages,
            max_tokens=LLM_MAX_TOKENS,
        )

        answer = str(result["text"]).strip()

        required_prefix = (
            "⚠ Outside Telecom Knowledge Domain: This response uses "
            "the model's general knowledge and is not grounded in the "
            "connected Telecom RAG/MCP sources."
        )

        if required_prefix not in answer:
            answer = (
                required_prefix
                +
                "\n\n"
                +
                answer
            )

        return {
            "answer": answer,
            "elapsed_s": float(result["elapsed_s"]),
            "prompt_tokens": result.get("prompt_tokens"),
            "completion_tokens": result.get("completion_tokens"),
            "total_tokens": result.get("total_tokens"),
        }


if "run_4c_telecom_grounded" not in globals():

    def run_4c_telecom_grounded(
        question,
        context,
    ):
        if (
            "MODULE4_FINAL_ANSWER_SYSTEM_PROMPT"
            not in globals()
        ):
            raise RuntimeError(
                "MODULE4_FINAL_ANSWER_SYSTEM_PROMPT is unavailable. "
                "Run Cell 3D before Cell 4D."
            )

        user_prompt = (
            "QUESTION\n"
            "--------\n"
            f"{str(question).strip()}\n\n"
            "TECHNICAL EVIDENCE\n"
            "------------------\n"
            f"{str(context).strip()}"
        )

        messages = [
            {
                "role": "system",
                "content": MODULE4_FINAL_ANSWER_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

        result = generate_hosted_llm(
            messages=messages,
            max_tokens=LLM_MAX_TOKENS,
        )

        return {
            "answer": str(result["text"]).strip(),
            "elapsed_s": float(result["elapsed_s"]),
            "prompt_tokens": result.get("prompt_tokens"),
            "completion_tokens": result.get("completion_tokens"),
            "total_tokens": result.get("total_tokens"),
        }




# ============================================================
# 4C.6Y.J — FROZEN 4B COMPARATIVE JUDGE HELPERS
# ============================================================
#
# Copied from the validated Cell 4B single-call judge path.
# Generation and retrieval helpers are intentionally excluded;
# 6T reuses the answers/evidence already produced above.
# ============================================================

# 4B.3 — PUBLIC DISPLAY LABELS
# ============================================================

GEMMA_ONLY_LABEL = (
    "Gemma 4 Only"
)


GEMMA_RAG_MCP_LABEL = (
    "Gemma 4 + RAG + MCP"
)


RISK_DISPLAY_LABEL = (
    "Estimated Hallucination Risk"
)


RISK_DEFINITION = (
    "Estimated Hallucination Risk reflects how strongly the "
    "answer is supported by the retrieved technical evidence. "
    "It does not prove that unsupported claims are false."
)



# 4B.6 — ESTIMATED HALLUCINATION RISK WEIGHTS
# ============================================================

HALLUCINATION_RISK_WEIGHTS = {

    "supported":
        0.00,

    "partially_supported":
        0.35,

    "unsupported":
        0.75,

    "contradicted":
        1.00,
}


VALID_JUDGE_STATUSES = set(
    HALLUCINATION_RISK_WEIGHTS.keys()
)


# ============================================================
# 4B.7 — CLAIM TEXT NORMALIZATION
# ============================================================

def remove_evidence_citations(
    text,
):

    return re.sub(
        r"\[(?:\s*E\d+\s*,?)+\]",
        "",
        str(text or ""),
    )


def clean_claim_text(
    text,
):

    text = str(
        text or ""
    )


    text = remove_evidence_citations(
        text
    )


    text = re.sub(
        r"\*\*",
        "",
        text,
    )


    text = re.sub(
        r"`",
        "",
        text,
    )


    text = text.replace(
        "$",
        "",
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    text = re.sub(
        r"\s+([.,;:!?])",
        r"\1",
        text,
    )


    return text.strip(
        " \t\r\n-*|"
    )


# ============================================================
# 4B.8 — ABBREVIATION-AWARE SENTENCE HANDLING
# ============================================================

CLAIM_ABBREVIATIONS = [

    "vs.",
    "e.g.",
    "i.e.",
    "etc.",
    "Fig.",
    "No.",
]


ABBREVIATION_DOT_TOKEN = (
    "<ABBR_DOT>"
)


def protect_abbreviations(
    text,
):

    protected = str(
        text or ""
    )


    for abbreviation in CLAIM_ABBREVIATIONS:

        protected = re.sub(

            re.escape(
                abbreviation
            ),

            lambda match:
                match.group(
                    0
                ).replace(
                    ".",
                    ABBREVIATION_DOT_TOKEN,
                ),

            protected,

            flags=re.IGNORECASE,
        )


    return protected


def restore_abbreviations(
    text,
):

    return str(
        text or ""
    ).replace(
        ABBREVIATION_DOT_TOKEN,
        ".",
    )


# ============================================================
# 4B.9 — STRUCTURAL STATEMENT FILTER
# ============================================================

STRUCTURAL_PATTERNS = [

    r"^the primary responsibilities.*(?:include|are|can be)",
    r"^its primary responsibilities.*(?:include|are|can be)",
    r"^the specific responsibilities.*(?:include|are|can be)",
    r"^the responsibilities.*(?:include|are|can be)",
    r"^the .* responsibilities .* categorized",

    r"^specific tasks include",
    r"^the following .* include",

    r"^in summary\b",
    r"^summary\b",

    r"^feature\s+[-—]\s+.*responsibility",

    r"^it is important to distinguish\b",
    r"^it is critical to distinguish\b",

    r"^to ensure .*clarity\b",
    r"^to avoid .*confusion\b",

    r"^the .* include:$",
    r"^the .* includes:$",

    r"^these include:$",

    r"^while .* plays .* role:$",
]


def is_structural_statement(
    text,
):

    text = clean_claim_text(
        text
    )


    if not text:

        return True


    lowered = text.lower()


    for pattern in STRUCTURAL_PATTERNS:

        if re.search(
            pattern,
            lowered,
        ):

            return True


    if text.endswith(
        ":"
    ):

        return True


    return False


# ============================================================
# 4B.10 — MARKDOWN TABLE DETECTION
# ============================================================

def is_markdown_table_separator(
    line,
):

    line = str(
        line
    ).strip()


    if "|" not in line:

        return False


    cells = [

        cell.strip()

        for cell
        in line.strip("|").split("|")
    ]


    if not cells:

        return False


    return all(

        re.fullmatch(
            r":?-{3,}:?",
            cell,
        )
        is not None

        for cell
        in cells
    )


def is_markdown_table_row(
    line,
):

    line = str(
        line
    ).strip()


    return (
        line.startswith("|")
        and
        line.endswith("|")
    )


# ============================================================
# 4B.11 — NON-PROPOSITIONAL LIST FRAGMENT FILTER
# ============================================================

PREDICATE_PATTERN = re.compile(
    r"\b("
    r"is|are|was|were|be|been|being|"
    r"has|have|had|"
    r"does|do|did|"
    r"can|could|may|might|must|shall|should|will|would|"
    r"acts|allows|applies|assists|"
    r"connects|contains|controls|coordinates|"
    r"enables|ensures|establishes|"
    r"facilitates|forwards|"
    r"handles|includes|interacts|"
    r"maintains|manages|maps|"
    r"offers|oversees|performs|processes|provides|"
    r"receives|represents|requires|responsible|routes|"
    r"selects|serves|supports|"
    r"terminates|tracks|triggers|uses|works"
    r")\b",
    flags=re.IGNORECASE,
)


def is_non_propositional_list_fragment(
    text,
):

    text = clean_claim_text(
        text
    )


    if not text:

        return True


    if ":" in text:

        return False


    if PREDICATE_PATTERN.search(
        text
    ):

        return False


    if re.search(
        r"[.!?]$",
        text,
    ):

        return False


    return True


# ============================================================
# 4B.12 — SENTENCE SPLITTER
# ============================================================

def split_into_sentence_units(
    text,
):

    text = clean_claim_text(
        text
    )


    if not text:

        return []


    protected = protect_abbreviations(
        text
    )


    parts = re.split(

        r"(?<=[.!?])\s+(?=[A-Z0-9])",

        protected,
    )


    cleaned_parts = []


    for part in parts:

        part = restore_abbreviations(
            part
        )


        part = clean_claim_text(
            part
        )


        if len(
            part
        ) < 20:

            continue


        if is_structural_statement(
            part
        ):

            continue


        cleaned_parts.append(
            part
        )


    return cleaned_parts


# ============================================================
# 4B.13 — REFINED FIXED CLAIM EXTRACTOR
# ============================================================

def extract_refined_claims(
    answer,
    prefix,
):

    answer = str(
        answer or ""
    )


    extracted = []


    skipped = {

        "headings":
            0,

        "tables":
            0,

        "structural":
            0,

        "fragments":
            0,

        "short":
            0,

        "duplicates":
            0,
    }


    for raw_line in answer.splitlines():

        line = raw_line.strip()


        if not line:

            continue


        if re.match(
            r"^#{1,6}\s+",
            line,
        ):

            skipped[
                "headings"
            ] += 1

            continue


        if (
            is_markdown_table_separator(
                line
            )
            or
            is_markdown_table_row(
                line
            )
        ):

            skipped[
                "tables"
            ] += 1

            continue


        bullet_match = re.match(

            r"^\s*(?:[-*+]|\d+[.)])\s+(.*)$",

            line,
        )


        if bullet_match:

            bullet_text = clean_claim_text(

                bullet_match.group(
                    1
                )
            )


            if is_structural_statement(
                bullet_text
            ):

                skipped[
                    "structural"
                ] += 1

                continue


            if is_non_propositional_list_fragment(
                bullet_text
            ):

                skipped[
                    "fragments"
                ] += 1

                continue


            sentence_units = (

                split_into_sentence_units(
                    bullet_text
                )
            )


            if not sentence_units:

                skipped[
                    "short"
                ] += 1


            extracted.extend(
                sentence_units
            )

            continue


        cleaned_line = clean_claim_text(
            line
        )


        if is_structural_statement(
            cleaned_line
        ):

            skipped[
                "structural"
            ] += 1

            continue


        sentence_units = (

            split_into_sentence_units(
                cleaned_line
            )
        )


        if not sentence_units:

            skipped[
                "short"
            ] += 1


        extracted.extend(
            sentence_units
        )


    # ========================================================
    # EXACT NORMALIZED DEDUPLICATION
    # ========================================================

    unique_claims = []

    seen = set()


    for claim in extracted:

        canonical = re.sub(

            r"[^a-z0-9]+",

            " ",

            claim.lower(),

        ).strip()


        if not canonical:

            continue


        if canonical in seen:

            skipped[
                "duplicates"
            ] += 1

            continue


        seen.add(
            canonical
        )


        unique_claims.append(
            claim
        )


    if not unique_claims:

        raise RuntimeError(
            f"No refined claims extracted for {prefix}."
        )


    numbered_claims = []


    for index, claim in enumerate(
        unique_claims,
        start=1,
    ):

        numbered_claims.append(

            {
                "claim_id":
                    f"{prefix}{index}",

                "claim":
                    claim,
            }
        )


    return {

        "claims":
            numbered_claims,

        "skipped":
            skipped,

        "final_claim_count":
            len(
                numbered_claims
            ),
    }


def format_claims_for_judge(
    claims,
):

    return "\n".join(

        f"{item['claim_id']}: {item['claim']}"

        for item
        in claims
    )


# ============================================================
# 4B.14 — GRANITE COMPARATIVE JUDGE PROMPT
# ============================================================
#
# ONLY CHANGE FROM THE LAST WORKING VERSION:
#
# Before final verdict, Granite checks whether another supplied
# evidence item supports the claim more directly.
#
# ============================================================

MODULE4_JUDGE_SYSTEM_PROMPT = """
You are an independent evaluator of technical answers.

Evaluate every supplied claim using ONLY the supplied evidence.

Classify each claim as exactly one of:

supported:
The evidence supports the claim.

partially_supported:
The evidence supports part of the claim, but does not fully establish it.

unsupported:
The evidence does not establish the claim.

contradicted:
The evidence conflicts with the claim.

Rules:
- Do not use external knowledge.
- Judge both answer groups using the same evidence and the same standard.
- Do not treat general topic similarity as sufficient evidence.
- A reasonable inference from the supplied evidence may be accepted.
- If the evidence supports only part of a claim, use partially_supported.
- If the evidence is missing, use unsupported rather than contradicted.
- Before assigning a verdict, check whether another supplied evidence item
  supports the claim more directly than the first evidence item considered.
- Evaluate every supplied claim exactly once.
- Copy every claim_id exactly and unchanged.
- Evidence must contain only relevant E# identifiers from the supplied evidence.
- Give a short reason for each judgment.
- Do not calculate totals, scores, percentages or risk.
- Return JSON only.

Return exactly this structure:

{
  "gemma4_only": {
    "claims": [
      {
        "claim_id": "A1",
        "status": "supported",
        "evidence": ["E1"],
        "reason": "Short evidence-based reason."
      }
    ]
  },
  "gemma4_rag_mcp": {
    "claims": [
      {
        "claim_id": "B1",
        "status": "supported",
        "evidence": ["E1"],
        "reason": "Short evidence-based reason."
      }
    ]
  }
}
""".strip()


# ============================================================
# 4B.15 — VALID EVIDENCE IDS
# ============================================================

def build_valid_evidence_ids(
    evidence_items,
):

    return {

        f"E{item['presentation_rank']}"

        for item
        in evidence_items
    }


# ============================================================
# 4B.16 — PARSE SINGLE COMPARATIVE GRANITE OUTPUT
# ============================================================

def parse_granite_comparative_output(
    raw_text,
):

    raw_text = str(
        raw_text or ""
    ).strip()


    if raw_text.startswith(
        "```"
    ):

        raw_text = re.sub(
            r"^```(?:json)?\s*",
            "",
            raw_text,
            flags=re.IGNORECASE,
        )


        raw_text = re.sub(
            r"\s*```$",
            "",
            raw_text,
        )


    payload = json.loads(
        raw_text
    )


    if not isinstance(
        payload,
        dict,
    ):

        raise RuntimeError(
            "Granite comparative output must be a JSON object."
        )


    gemma_only_group = payload.get(
        "gemma4_only"
    )


    rag_mcp_group = payload.get(
        "gemma4_rag_mcp"
    )


    if gemma_only_group is None:

        for key, value in payload.items():

            canonical = re.sub(
                r"[^a-z0-9]+",
                "",
                str(key).lower(),
            )


            if canonical in {
                "gemma4only",
                "standalone",
                "answera",
            }:

                gemma_only_group = value

                break


    if rag_mcp_group is None:

        for key, value in payload.items():

            canonical = re.sub(
                r"[^a-z0-9]+",
                "",
                str(key).lower(),
            )


            if canonical in {
                "gemma4ragmcp",
                "grounded",
                "answerb",
            }:

                rag_mcp_group = value

                break


    if gemma_only_group is None:

        raise RuntimeError(
            "Granite output missing Gemma 4 Only group."
        )


    if rag_mcp_group is None:

        raise RuntimeError(
            "Granite output missing Gemma 4 + RAG + MCP group."
        )


    def extract_claim_list(
        group,
        group_name,
    ):

        if isinstance(
            group,
            list,
        ):

            return group


        if isinstance(
            group,
            dict,
        ):

            if isinstance(
                group.get(
                    "claims"
                ),
                list,
            ):

                return group[
                    "claims"
                ]


            if isinstance(
                group.get(
                    "evaluations"
                ),
                list,
            ):

                return group[
                    "evaluations"
                ]


        raise RuntimeError(
            f"Granite group {group_name} does not contain a claim list."
        )


    return {

        "gemma4_only": {

            "claims":
                extract_claim_list(
                    gemma_only_group,
                    "gemma4_only",
                )
        },

        "gemma4_rag_mcp": {

            "claims":
                extract_claim_list(
                    rag_mcp_group,
                    "gemma4_rag_mcp",
                )
        },
    }


# ============================================================
# 4B.17 — STRUCTURAL VALIDATION + RISK CALCULATION
# ============================================================

def validate_and_summarize_judgment(
    original_claims,
    judge_group_payload,
    valid_evidence_ids,
):

    evaluations = judge_group_payload.get(
        "claims",
        [],
    )


    if not isinstance(
        evaluations,
        list,
    ):

        raise RuntimeError(
            "Granite claims must be a list."
        )


    expected_ids = [

        item[
            "claim_id"
        ]

        for item
        in original_claims
    ]


    expected_set = set(
        expected_ids
    )


    # --------------------------------------------------------
    # Hosted-model claim-ID resilience
    # --------------------------------------------------------
    #
    # Required original claim IDs remain strict:
    #   - every expected ID must be present;
    #   - expected IDs must not be duplicated;
    #   - each expected ID is validated exactly once.
    #
    # A hosted judge may occasionally append an invented extra
    # claim ID (for example B20 when the fixed claim list ends at
    # B19). Such extras are not part of the supplied evaluation
    # contract and are therefore discarded with an explicit
    # warning rather than failing an otherwise complete judgment.
    # --------------------------------------------------------

    unexpected_items = []

    filtered_evaluations = []

    for item in evaluations:

        if not isinstance(
            item,
            dict,
        ):
            continue

        claim_id = str(
            item.get(
                "claim_id",
                "",
            )
        ).strip()

        if claim_id in expected_set:
            filtered_evaluations.append(
                item
            )
        else:
            unexpected_items.append(
                item
            )


    unexpected_ids = sorted(
        {
            str(
                item.get(
                    "claim_id",
                    "",
                )
            ).strip()

            for item
            in unexpected_items

            if str(
                item.get(
                    "claim_id",
                    "",
                )
            ).strip()
        }
    )


    if unexpected_ids:

        print()
        print(
            "⚠ GRANITE JUDGE WARNING — "
            "discarding unexpected claim ID(s): "
            +
            ", ".join(
                unexpected_ids
            )
        )


    evaluations = filtered_evaluations


    returned_ids = [

        str(
            item.get(
                "claim_id",
                "",
            )
        ).strip()

        for item
        in evaluations
    ]


    returned_set = set(
        returned_ids
    )


    if len(
        returned_ids
    ) != len(
        returned_set
    ):

        raise RuntimeError(
            "Granite returned duplicate expected claim IDs."
        )


    missing_ids = sorted(

        expected_set
        -
        returned_set
    )


    if missing_ids:

        raise RuntimeError(

            "Granite omitted required claim IDs: "
            +
            ", ".join(
                missing_ids
            )
        )


    if len(
        evaluations
    ) != len(
        original_claims
    ):

        raise RuntimeError(
            "Granite output is not 1:1 with supplied required claims "
            "after removing unexpected claim IDs."
        )


    original_lookup = {

        item[
            "claim_id"
        ]:
            item[
                "claim"
            ]

        for item
        in original_claims
    }


    evaluation_lookup = {

        str(
            item[
                "claim_id"
            ]
        ).strip():
            item

        for item
        in evaluations
    }


    normalized = []


    counts = {

        "supported":
            0,

        "partially_supported":
            0,

        "unsupported":
            0,

        "contradicted":
            0,
    }


    invalid_evidence_references = (
        0
    )


    unsupported_with_evidence = (
        0
    )


    empty_reason_count = (
        0
    )


    for claim_id in expected_ids:

        evaluation = (

            evaluation_lookup[
                claim_id
            ]
        )


        status = str(

            evaluation.get(
                "status",
                "",
            )

        ).strip().lower()


        if status not in VALID_JUDGE_STATUSES:

            raise RuntimeError(

                f"Invalid Granite status for "
                f"{claim_id}: {status!r}"
            )


        evidence = evaluation.get(
            "evidence",
            [],
        )


        if evidence is None:

            evidence = []


        if isinstance(
            evidence,
            str,
        ):

            evidence = [
                evidence
            ]


        if not isinstance(
            evidence,
            list,
        ):

            evidence = []


        cleaned_evidence = []


        for evidence_id in evidence:

            evidence_id = str(
                evidence_id
            ).strip()


            if not evidence_id:

                continue


            if (
                evidence_id
                not in
                valid_evidence_ids
            ):

                invalid_evidence_references += 1

                continue


            if (
                evidence_id
                not in
                cleaned_evidence
            ):

                cleaned_evidence.append(
                    evidence_id
                )


        reason = str(

            evaluation.get(
                "reason",
                "",
            )

        ).strip()


        if not reason:

            empty_reason_count += 1


        if (
            status
            ==
            "unsupported"
            and
            cleaned_evidence
        ):

            unsupported_with_evidence += 1


        normalized.append(

            {
                "claim_id":
                    claim_id,

                "claim":
                    original_lookup[
                        claim_id
                    ],

                "status":
                    status,

                "evidence":
                    cleaned_evidence,

                "reason":
                    reason,
            }
        )


        counts[
            status
        ] += 1


    total_claims = len(
        normalized
    )


    weighted_risk = sum(

        counts[
            status
        ]
        *
        HALLUCINATION_RISK_WEIGHTS[
            status
        ]

        for status
        in HALLUCINATION_RISK_WEIGHTS
    )


    estimated_risk_pct = (

        weighted_risk
        /
        total_claims
        *
        100.0
    )


    return {

        "claims":
            normalized,

        "total_claims":
            total_claims,

        "supported":
            counts[
                "supported"
            ],

        "partially_supported":
            counts[
                "partially_supported"
            ],

        "unsupported":
            counts[
                "unsupported"
            ],

        "contradicted":
            counts[
                "contradicted"
            ],

        "invalid_evidence_references":
            invalid_evidence_references,

        "unsupported_with_evidence":
            unsupported_with_evidence,

        "empty_reason_count":
            empty_reason_count,

        "estimated_hallucination_risk_pct":
            float(
                min(
                    100.0,
                    max(
                        0.0,
                        estimated_risk_pct,
                    )
                )
            ),
    }


def hallucination_risk_label(
    risk_pct,
):

    if risk_pct <= 20:

        return "LOW"


    if risk_pct <= 40:

        return "MODERATE"


    if risk_pct <= 60:

        return "ELEVATED"


    if risk_pct <= 80:

        return "HIGH"


    return "VERY HIGH"

# 4B.19 — SINGLE GRANITE COMPARATIVE JUDGE HELPER
# ============================================================

def run_granite_comparative_judge(
    question,
    evidence_context,
    gemma_only_claims,
    rag_mcp_claims,
):

    gemma_only_ids = [
        item["claim_id"]
        for item in gemma_only_claims
    ]

    rag_mcp_ids = [
        item["claim_id"]
        for item in rag_mcp_claims
    ]

    user_prompt = (
        f"QUESTION\n"
        f"--------\n"
        f"{question}\n\n"

        f"TECHNICAL EVIDENCE\n"
        f"------------------\n"
        f"{evidence_context}\n\n"

        f"ANSWER A — {GEMMA_ONLY_LABEL}\n"
        f"FIXED CLAIMS\n"
        f"------------\n"
        f"{format_claims_for_judge(gemma_only_claims)}\n\n"

        f"ANSWER B — {GEMMA_RAG_MCP_LABEL}\n"
        f"FIXED CLAIMS\n"
        f"------------\n"
        f"{format_claims_for_judge(rag_mcp_claims)}\n\n"

        f"REQUIRED ANSWER A CLAIM IDS\n"
        f"---------------------------\n"
        f"{', '.join(gemma_only_ids)}\n\n"

        f"REQUIRED ANSWER B CLAIM IDS\n"
        f"---------------------------\n"
        f"{', '.join(rag_mcp_ids)}"
    )

    messages = [
        {
            "role": "system",
            "content": MODULE4_JUDGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    total_claims = (
        len(gemma_only_claims)
        +
        len(rag_mcp_claims)
    )

    judge_max_tokens = min(
        7000,
        max(
            4000,
            (130 * total_claims) + 800,
        )
    )

    # Hosted-provider resilience:
    # preserve the frozen judge prompt, claims, parser and token budget.
    # Retry once only when the returned structure cannot be parsed/validated.
    MAX_JUDGE_ATTEMPTS_4D = 2

    judge_start = time.perf_counter()

    last_exc = None
    last_raw_text = ""
    last_finish_reason = ""

    for attempt in range(
        1,
        MAX_JUDGE_ATTEMPTS_4D + 1,
    ):

        result = call_granite_judge(
            messages=messages,
            max_tokens=judge_max_tokens,
        )

        raw_text = str(
            result.get(
                "text",
                "",
            )
        ).strip()

        finish_reason = str(
            result.get(
                "finish_reason",
                "",
            )
        ).strip().lower()

        last_raw_text = raw_text
        last_finish_reason = finish_reason

        if (
            finish_reason
            and
            finish_reason
            !=
            "stop"
        ):
            print()
            print("GRANITE OUTPUT WARNING")
            print("-" * 116)
            print(
                f"Attempt             : "
                f"{attempt}/{MAX_JUDGE_ATTEMPTS_4D}"
            )
            print(
                f"Finish Reason       : "
                f"{finish_reason}"
            )
            print(
                f"Total Claims        : "
                f"{total_claims}"
            )
            print(
                f"Max Output Tokens   : "
                f"{judge_max_tokens}"
            )
            print(
                f"Returned Characters : "
                f"{len(raw_text):,}"
            )

        try:
            payload = parse_granite_comparative_output(
                raw_text
            )

            judge_elapsed_s = (
                time.perf_counter()
                -
                judge_start
            )

            if attempt > 1:
                print()
                print(
                    f"✓ Granite comparative judge recovered "
                    f"on attempt {attempt}/"
                    f"{MAX_JUDGE_ATTEMPTS_4D}."
                )

            return {
                "result": result,
                "payload": payload,
                "raw_text": raw_text,
                "elapsed_s": judge_elapsed_s,
                "max_tokens": judge_max_tokens,
                "total_claims": total_claims,
                "attempts": attempt,
                "retried": attempt > 1,
            }

        except Exception as exc:

            last_exc = exc

            print()
            print("RAW GRANITE OUTPUT")
            print("-" * 116)
            print(
                f"Attempt           : "
                f"{attempt}/{MAX_JUDGE_ATTEMPTS_4D}"
            )
            print(
                f"Finish Reason     : "
                f"{finish_reason or 'UNKNOWN'}"
            )
            print(
                f"Total Claims      : "
                f"{total_claims}"
            )
            print(
                f"Max Output Tokens : "
                f"{judge_max_tokens}"
            )
            print()
            print(
                raw_text
                if raw_text
                else "<EMPTY RESPONSE>"
            )

            if attempt < MAX_JUDGE_ATTEMPTS_4D:
                print()
                print(
                    "⚠ Granite comparative output was structurally "
                    "invalid. Retrying once with the same frozen "
                    "judge contract..."
                )
                continue

    raise RuntimeError(
        "Could not parse Granite comparative output after "
        f"{MAX_JUDGE_ATTEMPTS_4D} attempts. "
        f"Last finish reason: "
        f"{last_finish_reason or 'unknown'}. "
        f"Last returned characters: "
        f"{len(last_raw_text):,}."
    ) from last_exc


# ============================================================
# 4C.6Y.A — MODULE-4-ALIGNED LLM-SELECTED RETRIEVAL ORCHESTRATION
# ============================================================
#
# PURPOSE
# -------
# Restore the complete proven Module 4 natural-routing pattern:
#
#     Grounded Question
#          ↓
#     SAME LLM — Retrieval Planner
#          ├─ selects MINIMUM retrieval architecture
#          │      RAG_ONLY / MCP_ONLY / HYBRID
#          ├─ formulates initial retrieval query
#          └─ derives fixed answer requirements
#          ↓
#     Python executes ONLY the selected architecture
#          ↓
#     deterministic evidence-security scan
#          ↓
#     SAME LLM — Requirement-Bounded Sufficiency Assessor
#          ├─ sufficient → stop
#          └─ insufficient → formulate ONE refined next_query
#                                  ↓
#                    SAME retrieval architecture, next round
#          ↓
#     Final bounded evidence
#          ↓
#     EXISTING frozen grounded-answer generator
#          ↓
#     EXISTING frozen Granite comparative judge
#
# RESPONSIBILITY SPLIT
# --------------------
# LLM:
#   - selects RAG_ONLY / MCP_ONLY / HYBRID
#   - formulates retrieval query
#   - derives explicit answer requirements
#   - assesses evidence sufficiency
#   - formulates refined query when a requirement remains unsupported
#
# Python:
#   - validates the LLM-selected architecture
#   - executes only that architecture
#   - keeps source-family / shard / specification routing internal to MCP
#   - enforces maximum 3 retrieval rounds
#   - scans evidence for prompt injection
#   - accumulates / deduplicates evidence across rounds
#   - enforces maximum 5 final evidence items / context budget
#
# Native OpenAI-style tool_calls are NOT used.
# No retrieval route is hard-coded.
# ============================================================


MODULE4_ALLOWED_RETRIEVAL_MODES_4C6W = {
    "RAG_ONLY",
    "MCP_ONLY",
    "HYBRID",
}

MODULE4_MAX_MODEL_SEARCHES_4C6W = int(
    globals().get("MAX_MCP_SEARCHES", 3)
)

MODULE4_MAX_FINAL_EVIDENCE_4C6W = int(
    globals().get("MAX_RETRIEVED_SOURCES", 5)
)

MODULE4_EVIDENCE_EXCERPT_CHARS_4C6W = int(
    globals().get("MCP_EXCERPT_CHARS", 2500)
)

MODULE4_CONTEXT_MAX_CHARS_4C6W = 12500
MODULE4_MAX_ANSWER_REQUIREMENTS_4C6W = 5
MODULE4_PLANNER_MAX_TOKENS_4C6W = 420
MODULE4_SUFFICIENCY_MAX_TOKENS_4C6W = 320

if MODULE4_MAX_MODEL_SEARCHES_4C6W != 3:
    raise RuntimeError(
        "4C.6W preserves the frozen maximum of 3 adaptive retrieval rounds."
    )

if MODULE4_MAX_FINAL_EVIDENCE_4C6W != 5:
    raise RuntimeError(
        "4C.6W preserves the frozen maximum of 5 final evidence items."
    )


# ============================================================
# 4C.6Y.A1 — NATURAL RETRIEVAL-ARCHITECTURE PLANNER
# ============================================================

MODULE4_PLANNER_SYSTEM_PROMPT_4C6W = """
You are planning grounded retrieval for a technical question.

The question has already:
- passed deterministic security checks; and
- been classified as covered by the connected grounded corpus.

Retrieval is therefore required.

Select the MINIMUM retrieval architecture sufficient for the question:

RAG_ONLY
- Use for broad conceptual, architectural, explanatory or synthesis questions
  when semantic retrieval from the local technical corpus should be sufficient.
- Prefer this when the task is primarily about meaning, relationships, concepts,
  architecture or high-level technical explanation rather than locating a
  precise source-specific statement.

MCP_ONLY
- Use when the question primarily requires precise standards, protocol,
  specification, source-aware or authoritative-document evidence.
- Examples include questions explicitly tied to IETF/3GPP definitions,
  normative protocol mechanisms, specification clauses or source-specific facts.

HYBRID
- Use only when the question materially needs BOTH semantic corpus synthesis
  AND precise/source-aware technical evidence.
- Typical cases are cross-domain engineering questions where neither mechanism
  alone is likely to cover the requested scope adequately.

Do NOT default to HYBRID merely because both retrieval systems are available.
Choose the minimum sufficient architecture.

Your task is to:
1. select RAG_ONLY, MCP_ONLY or HYBRID;
2. formulate ONE focused standalone retrieval query; and
3. derive the minimum answer requirements explicitly requested by the question.

Important:
- You choose the retrieval architecture and query wording.
- If MCP is selected, the MCP service itself chooses its internal 3GPP / TCC /
  mixed source route. Do not choose collections, shards or specifications here.
- If HYBRID is selected, the SAME query is sent concurrently to RAG and MCP.

Rules for answer_requirements:
- Produce 1 to 5 short requirements.
- Derive them ONLY from what the question explicitly asks.
- Decompose the requested scope; do not expand it.
- Do not add signaling flows, protocol messages, parameters, interactions,
  implementation internals, root-cause steps or subtopics unless explicitly
  requested by the user.
- Each requirement must describe something the final answer needs to cover.
- Do not include generic requirements such as "be accurate" or "be detailed".

Return ONLY valid JSON:

{
  "mode": "RAG_ONLY | MCP_ONLY | HYBRID",
  "query": "standalone retrieval query",
  "reason": "brief routing and query rationale",
  "answer_requirements": [
    "first explicit answer requirement",
    "second explicit answer requirement"
  ]
}
""".strip()


# ============================================================
# 4C.6Y.A2 — REQUIREMENT-BOUNDED SUFFICIENCY ASSESSOR
# ============================================================

MODULE4_SUFFICIENCY_SYSTEM_PROMPT_4C6W = """
You are making a bounded retrieval stopping decision.

You will receive:
1. the technical question;
2. the fixed answer requirements derived from that question;
3. the retrieval architecture already selected for this question; and
4. the current accumulated retrieved evidence.

Your ONLY task is to determine which listed requirements are materially
supported by the evidence.

STRICT RULES:
- Judge ONLY the supplied answer requirements.
- Do NOT create, infer, add or demand any new requirement.
- Do NOT expand the scope of the question.
- Evidence does NOT need to be exhaustive.
- A requirement is supported when the evidence provides enough factual
  grounding to answer that requirement at the level requested by the question.
- Do NOT mark a requirement unsupported merely because deeper signaling,
  message-level, parameter-level, implementation, internal-logic or
  cross-function detail could also be provided.
- Additional retrieval is justified ONLY when at least one supplied requirement
  cannot be answered reliably from the current evidence.
- If every supplied requirement is materially supported, retrieval MUST stop.
- If retrieval is still needed, provide ONE targeted standalone retrieval query
  focused only on the missing listed requirement(s).
- The SAME previously selected retrieval architecture will be used for the next
  round. Do not change RAG_ONLY / MCP_ONLY / HYBRID here.

Return ONLY valid JSON:

{
  "supported_requirement_ids": ["R1", "R2"],
  "missing_requirement_ids": [],
  "next_query": "",
  "reason": "brief scope-bounded assessment"
}

Do not return any requirement ID that was not supplied.
""".strip()


# ============================================================
# 4C.6Y.A3 — OPENROUTER / JSON HELPERS
# ============================================================

def _4c6w_get_message(response):
    choices = getattr(response, "choices", []) or []
    if not choices:
        raise RuntimeError("OpenRouter returned no completion choices.")
    message = getattr(choices[0], "message", None)
    if message is None:
        raise RuntimeError("OpenRouter returned no assistant message.")
    return message


def _4c6w_extract_text(response):
    content = getattr(_4c6w_get_message(response), "content", "")
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                value = item.get("text", "")
            else:
                value = getattr(item, "text", "")
            if value:
                parts.append(str(value))
        return "\n".join(parts).strip()
    return str(content).strip()


def _4c6w_usage(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return {"input_tokens": 0, "output_tokens": 0}
    return {
        "input_tokens": int(getattr(usage, "prompt_tokens", 0) or 0),
        "output_tokens": int(getattr(usage, "completion_tokens", 0) or 0),
    }


def _4c6w_call_llm(messages, max_tokens):
    request_args = {
        "model": LLM_MODEL,
        "messages": messages,
        "max_tokens": int(max_tokens),
    }
    # Preserve the established provider-default temperature convention.
    return openrouter.chat.completions.create(**request_args)


def _4c6w_parse_json_object(raw_text):
    raw_text = str(raw_text or "").strip()

    if raw_text.startswith("```"):
        raw_text = re.sub(
            r"^```(?:json)?\s*",
            "",
            raw_text,
            flags=re.IGNORECASE,
        )
        raw_text = re.sub(r"\s*```$", "", raw_text).strip()

    try:
        parsed = json.loads(raw_text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start >= 0 and end > start:
        parsed = json.loads(raw_text[start:end + 1])
        if isinstance(parsed, dict):
            return parsed

    raise ValueError("Model response did not contain a valid JSON object.")


# ============================================================
# 4C.6Y.A4 — PLANNER
# ============================================================

def _4c6w_plan_retrieval(question):
    start = time.perf_counter()

    response = _4c6w_call_llm(
        messages=[
            {"role": "system", "content": MODULE4_PLANNER_SYSTEM_PROMPT_4C6W},
            {"role": "user", "content": question},
        ],
        max_tokens=MODULE4_PLANNER_MAX_TOKENS_4C6W,
    )

    elapsed = time.perf_counter() - start
    raw_text = _4c6w_extract_text(response)
    plan = _4c6w_parse_json_object(raw_text)

    mode = str(plan.get("mode", "") or "").strip().upper()
    if mode not in MODULE4_ALLOWED_RETRIEVAL_MODES_4C6W:
        raise ValueError(
            f"Invalid retrieval architecture selected by LLM: {mode!r}. "
            f"Allowed: {sorted(MODULE4_ALLOWED_RETRIEVAL_MODES_4C6W)}"
        )

    query = re.sub(r"\s+", " ", str(plan.get("query", "") or "")).strip()
    if not query:
        query = str(question).strip()

    raw_requirements = plan.get("answer_requirements", [])
    if not isinstance(raw_requirements, list):
        raw_requirements = []

    requirements = []
    for item in raw_requirements:
        item = re.sub(r"\s+", " ", str(item or "")).strip()
        if item and item not in requirements:
            requirements.append(item)

    requirements = requirements[:MODULE4_MAX_ANSWER_REQUIREMENTS_4C6W]

    if not requirements:
        requirements = [str(question).strip()]

    return {
        "mode": mode,
        "query": query,
        "reason": str(plan.get("reason", "") or "").strip(),
        "answer_requirements": requirements,
        "latency_s": elapsed,
        "usage": _4c6w_usage(response),
        "raw_text": raw_text,
    }


# ============================================================
# 4C.6Y.A5 — TRUE MODE-SPECIFIC RETRIEVAL EXECUTION
# ============================================================

async def _4c6w_execute_selected_retrieval_round(mode, query):
    """
    Execute ONLY the architecture selected by the LLM planner.

    RAG_ONLY:
        local semantic RAG only.

    MCP_ONLY:
        FastMCP knowledge retrieval only. Internal 3GPP/TCC routing remains
        owned by the MCP service.

    HYBRID:
        same query → RAG + MCP concurrently → existing Hybrid fusion.
    """

    if mode not in MODULE4_ALLOWED_RETRIEVAL_MODES_4C6W:
        raise ValueError(f"Unsupported retrieval mode: {mode}")

    round_start = time.perf_counter()

    if mode == "RAG_ONLY":
        rag = await asyncio.to_thread(
            retrieve_rag_candidates,
            query,
        )

        rag_s = float(
            rag.get("timing", {}).get("retrieval_time_s", 0.0) or 0.0
        )

        return {
            "mode": mode,
            "evidence": list(rag.get("evidence", []) or []),
            "rag_candidates": int(rag.get("candidate_count", 0) or 0),
            "mcp_candidates": 0,
            "rag_calls": 1,
            "mcp_calls": 0,
            "rag_retrieval_s": rag_s,
            "mcp_retrieval_s": 0.0,
            "selected_retrieval_s": float(time.perf_counter() - round_start),
            "rag": rag,
            "mcp": None,
            "hybrid": None,
        }

    if mode == "MCP_ONLY":
        mcp_result = await retrieve_mcp_candidates(query)

        mcp_s = float(
            mcp_result.get("timing", {}).get("retrieval_time_s", 0.0) or 0.0
        )

        return {
            "mode": mode,
            "evidence": list(mcp_result.get("evidence", []) or []),
            "rag_candidates": 0,
            "mcp_candidates": int(mcp_result.get("candidate_count", 0) or 0),
            "rag_calls": 0,
            "mcp_calls": 1,
            "rag_retrieval_s": 0.0,
            "mcp_retrieval_s": mcp_s,
            "selected_retrieval_s": float(time.perf_counter() - round_start),
            "rag": None,
            "mcp": mcp_result,
            "hybrid": None,
        }

    # HYBRID — preserve original Module 4 concurrency model.
    rag_task = asyncio.to_thread(
        retrieve_rag_candidates,
        query,
    )
    mcp_task = retrieve_mcp_candidates(query)

    rag, mcp_result = await asyncio.gather(
        rag_task,
        mcp_task,
    )

    parallel_wall_s = float(time.perf_counter() - round_start)

    hybrid = fuse_hybrid_candidates(
        rag_candidates=rag.get("evidence", []),
        mcp_candidates=mcp_result.get("evidence", []),
    )

    fusion_s = float(
        hybrid.get("timing", {}).get("fusion_only_s", 0.0) or 0.0
    )

    rag_s = float(
        rag.get("timing", {}).get("retrieval_time_s", 0.0) or 0.0
    )
    mcp_s = float(
        mcp_result.get("timing", {}).get("retrieval_time_s", 0.0) or 0.0
    )

    return {
        "mode": mode,
        "evidence": list(hybrid.get("evidence", []) or []),
        "rag_candidates": int(rag.get("candidate_count", 0) or 0),
        "mcp_candidates": int(mcp_result.get("candidate_count", 0) or 0),
        "rag_calls": 1,
        "mcp_calls": 1,
        "rag_retrieval_s": rag_s,
        "mcp_retrieval_s": mcp_s,
        "selected_retrieval_s": parallel_wall_s + fusion_s,
        "parallel_retrieval_wall_s": parallel_wall_s,
        "fusion_only_s": fusion_s,
        "rag": rag,
        "mcp": mcp_result,
        "hybrid": hybrid,
    }


# ============================================================
# 4C.6Y.A6 — EVIDENCE HELPERS
# ============================================================

def _4c6w_evidence_text(item):
    return str(item.get("text") or item.get("evidence") or "").strip()


def _4c6w_evidence_fingerprint(item):
    text = _4c6w_evidence_text(item)
    return "|".join([
        str(item.get("source_family", "") or "").strip().lower(),
        str(item.get("title", "") or "").strip().lower(),
        str(item.get("document_id", item.get("identifier", "")) or "").strip().lower(),
        re.sub(r"\s+", " ", text.lower())[:1200],
    ])


def _4c6w_normalize_systems(item):
    systems = (
        item.get("retrieval_systems")
        or item.get("retrieval_system")
        or item.get("retriever")
        or []
    )
    if isinstance(systems, str):
        systems = [systems]

    normalized = []
    for value in systems:
        value = str(value or "").strip().upper()
        if value and value not in normalized:
            normalized.append(value)
    return normalized


def _4c6w_merge_round_evidence(
    evidence_ledger,
    evidence_by_fingerprint,
    raw_evidence,
    round_number,
    query,
):
    """Merge one round into a stable cross-round evidence ledger."""

    round_items = []

    for fallback_rank, raw_item in enumerate(raw_evidence, start=1):
        item = dict(raw_item)

        source_rank = int(
            item.get("presentation_rank")
            or item.get("rank")
            or fallback_rank
        )

        item["text"] = _4c6w_evidence_text(item)[
            :MODULE4_EVIDENCE_EXCERPT_CHARS_4C6W
        ]

        systems = _4c6w_normalize_systems(item)
        fingerprint = _4c6w_evidence_fingerprint(item)
        rrf_increment = 1.0 / (60.0 + float(source_rank))

        if fingerprint in evidence_by_fingerprint:
            ledger_item = evidence_by_fingerprint[fingerprint]

            for system in systems:
                if system not in ledger_item["retrieval_systems"]:
                    ledger_item["retrieval_systems"].append(system)

            ledger_item["adaptive_rrf_score"] += rrf_increment
            ledger_item["occurrence_count"] += 1
            ledger_item["last_seen_round"] = int(round_number)
            ledger_item["round_numbers"].append(int(round_number))
            ledger_item["retrieval_queries"].append(str(query))

        else:
            ledger_item = dict(item)
            ledger_item["source_presentation_rank"] = source_rank
            ledger_item["presentation_rank"] = len(evidence_ledger) + 1
            ledger_item["evidence_id"] = f"E{ledger_item['presentation_rank']}"
            ledger_item["retrieval_systems"] = systems
            ledger_item["adaptive_rrf_score"] = rrf_increment
            ledger_item["occurrence_count"] = 1
            ledger_item["first_seen_round"] = int(round_number)
            ledger_item["last_seen_round"] = int(round_number)
            ledger_item["round_numbers"] = [int(round_number)]
            ledger_item["retrieval_queries"] = [str(query)]

            evidence_ledger.append(ledger_item)
            evidence_by_fingerprint[fingerprint] = ledger_item

        round_items.append(ledger_item)

    return round_items


def _4c6w_rank_evidence(evidence_ledger):
    return sorted(
        evidence_ledger,
        key=lambda item: (
            -float(item.get("adaptive_rrf_score", 0.0) or 0.0),
            -int(item.get("occurrence_count", 1) or 1),
            int(item.get("source_presentation_rank", 999999) or 999999),
            int(item.get("presentation_rank", 999999) or 999999),
        ),
    )


def _4c6w_select_evidence(evidence_ledger, mode):
    """
    RAG_ONLY / MCP_ONLY:
        top cumulative evidence by adaptive RRF.

    HYBRID:
        preserve the Module 4 representation principle by reserving up to two
        RAG-backed and two MCP-backed items, then fill by adaptive RRF.
    """

    ranked = _4c6w_rank_evidence(evidence_ledger)
    if not ranked:
        return []

    if mode in {"RAG_ONLY", "MCP_ONLY"}:
        return ranked[:MODULE4_MAX_FINAL_EVIDENCE_4C6W]

    available_rag = sum(
        "RAG" in item.get("retrieval_systems", [])
        for item in ranked
    )
    available_mcp = sum(
        "MCP" in item.get("retrieval_systems", [])
        for item in ranked
    )

    required_rag = min(2, available_rag)
    required_mcp = min(2, available_mcp)

    selected = []

    def selected_has(system):
        return sum(
            system in item.get("retrieval_systems", [])
            for item in selected
        )

    for item in ranked:
        if selected_has("RAG") >= required_rag:
            break
        if "RAG" in item.get("retrieval_systems", []) and item not in selected:
            selected.append(item)

    for item in ranked:
        if selected_has("MCP") >= required_mcp:
            break
        if "MCP" in item.get("retrieval_systems", []) and item not in selected:
            selected.append(item)

    for item in ranked:
        if len(selected) >= MODULE4_MAX_FINAL_EVIDENCE_4C6W:
            break
        if item not in selected:
            selected.append(item)

    return selected[:MODULE4_MAX_FINAL_EVIDENCE_4C6W]


def _4c6w_build_bounded_context(selected_evidence):
    bounded_items = []
    blocks = []
    chars_used = 0

    for item in selected_evidence:
        rank = int(item["presentation_rank"])
        systems = item.get("retrieval_systems", []) or []
        retrieval_label = "/".join(str(value) for value in systems if value)
        if not retrieval_label:
            retrieval_label = "GROUNDED"

        header = (
            f"[E{rank}] "
            f"Retrieval={retrieval_label} | "
            f"Source={item.get('source_family', '')} | "
            f"Title={item.get('title', '')} | "
            f"Document={item.get('document_id', item.get('identifier', ''))}"
        )

        separator_chars = 2 if blocks else 0
        remaining = (
            MODULE4_CONTEXT_MAX_CHARS_4C6W
            - chars_used
            - separator_chars
            - len(header)
            - 1
        )

        if remaining <= 0:
            break

        evidence_text = _4c6w_evidence_text(item)[:remaining]
        if not evidence_text:
            continue

        bounded_item = dict(item)
        bounded_item["text"] = evidence_text
        block = header + "\n" + evidence_text

        if blocks:
            chars_used += 2

        blocks.append(block)
        chars_used += len(block)
        bounded_items.append(bounded_item)

    return {
        "evidence": bounded_items,
        "context": "\n\n".join(blocks),
        "context_chars": chars_used,
    }


# ============================================================
# 4C.6Y.A7 — REQUIREMENT-BOUNDED SUFFICIENCY ASSESSOR
# ============================================================

def _4c6w_assess_sufficiency(
    question,
    requirements,
    mode,
    round_number,
    context,
):
    requirement_map = {
        f"R{index}": requirement
        for index, requirement in enumerate(requirements, start=1)
    }

    requirements_text = "\n".join(
        f"{requirement_id}: {requirement}"
        for requirement_id, requirement in requirement_map.items()
    )

    start = time.perf_counter()

    response = _4c6w_call_llm(
        messages=[
            {"role": "system", "content": MODULE4_SUFFICIENCY_SYSTEM_PROMPT_4C6W},
            {
                "role": "user",
                "content": (
                    f"QUESTION:\n{question}\n\n"
                    f"FIXED ANSWER REQUIREMENTS:\n{requirements_text}\n\n"
                    f"RETRIEVAL MODE:\n{mode}\n\n"
                    f"ROUND:\n{round_number}\n\n"
                    f"EVIDENCE:\n{context}"
                ),
            },
        ],
        max_tokens=MODULE4_SUFFICIENCY_MAX_TOKENS_4C6W,
    )

    elapsed = time.perf_counter() - start
    raw_text = _4c6w_extract_text(response)
    result = _4c6w_parse_json_object(raw_text)

    valid_ids = list(requirement_map.keys())

    supported = []
    for value in result.get("supported_requirement_ids", []) or []:
        value = str(value).strip().upper()
        if value in valid_ids and value not in supported:
            supported.append(value)

    missing = []
    for value in result.get("missing_requirement_ids", []) or []:
        value = str(value).strip().upper()
        if value in valid_ids and value not in missing:
            missing.append(value)

    # Structural reconciliation only; no semantic re-judging in Python.
    for requirement_id in valid_ids:
        if requirement_id not in supported and requirement_id not in missing:
            missing.append(requirement_id)

    supported = [
        requirement_id
        for requirement_id in supported
        if requirement_id not in missing
    ]

    sufficient = len(missing) == 0

    next_query = re.sub(
        r"\s+",
        " ",
        str(result.get("next_query", "") or ""),
    ).strip()

    missing_requirements = [
        requirement_map[requirement_id]
        for requirement_id in missing
    ]

    # Same safe Module 4 fallback: reuse only fixed requirements; invent no scope.
    if not sufficient and not next_query:
        focus = "; ".join(missing_requirements)
        next_query = (
            f"{question} Focus specifically on: {focus}"
        ).strip()

    if sufficient:
        next_query = ""

    return {
        "sufficient": sufficient,
        "supported_requirement_ids": supported,
        "missing_requirement_ids": missing,
        "missing_requirements": missing_requirements,
        "missing_evidence": "; ".join(missing_requirements),
        "next_query": next_query,
        "reason": str(result.get("reason", "") or "").strip(),
        "latency_s": elapsed,
        "usage": _4c6w_usage(response),
        "raw_text": raw_text,
    }


# ============================================================
# 4C.6Y.A8 — COMPLETE NATURAL-ROUTING ADAPTIVE LOOP
# ============================================================

async def run_module4_aligned_routed_grounded_4c6w(question):
    start_wall = time.perf_counter()

    total_input_tokens = 0
    total_output_tokens = 0
    total_llm_calls = 0
    total_llm_latency = 0.0

    rag_candidate_total = 0
    mcp_candidate_total = 0
    rag_call_total = 0
    mcp_call_total = 0

    rag_retrieval_total = 0.0
    mcp_retrieval_total = 0.0
    selected_retrieval_total = 0.0
    evidence_scan_total_ms = 0.0

    evidence_security_hits = []
    evidence_ledger = []
    evidence_by_fingerprint = {}

    search_queries = []
    round_traces = []

    # LLM selects architecture, query and fixed requirements.
    planner = await asyncio.to_thread(
        _4c6w_plan_retrieval,
        question,
    )

    total_llm_calls += 1
    total_llm_latency += float(planner["latency_s"])
    total_input_tokens += int(planner["usage"]["input_tokens"])
    total_output_tokens += int(planner["usage"]["output_tokens"])

    selected_mode = planner["mode"]
    current_query = planner["query"]
    requirements = list(planner["answer_requirements"])

    print()
    print(f"         Selected Mode       → {selected_mode}")
    print(f"         Planner Query       → {current_query}")
    print(f"         Planner Reason      → {planner['reason']}")
    print("         Fixed Requirements →")
    for index, requirement in enumerate(requirements, start=1):
        print(f"            R{index} → {requirement}")
    print(f"         Planner Latency     → {planner['latency_s']:.3f} s")

    final_selection = None
    final_assessment = None
    stop_reason = None

    for round_number in range(1, MODULE4_MAX_MODEL_SEARCHES_4C6W + 1):
        search_queries.append(current_query)

        print()
        print(
            f"         Retrieval Round    → {round_number}/"
            f"{MODULE4_MAX_MODEL_SEARCHES_4C6W} [{selected_mode}]"
        )
        print(f"         Query               → {current_query}")

        round_result = await _4c6w_execute_selected_retrieval_round(
            selected_mode,
            current_query,
        )

        rag_candidates = int(round_result["rag_candidates"])
        mcp_candidates = int(round_result["mcp_candidates"])

        rag_candidate_total += rag_candidates
        mcp_candidate_total += mcp_candidates
        rag_call_total += int(round_result["rag_calls"])
        mcp_call_total += int(round_result["mcp_calls"])

        rag_retrieval_s = float(round_result["rag_retrieval_s"])
        mcp_retrieval_s = float(round_result["mcp_retrieval_s"])
        selected_retrieval_s = float(round_result["selected_retrieval_s"])

        rag_retrieval_total += rag_retrieval_s
        mcp_retrieval_total += mcp_retrieval_s
        selected_retrieval_total += selected_retrieval_s

        raw_evidence = list(round_result.get("evidence", []) or [])

        # Deterministic evidence security boundary.
        scan_start = time.perf_counter()
        round_security_hits = []

        for raw_item in raw_evidence:
            evidence_text = _4c6w_evidence_text(raw_item)
            matches = get_injection_matches_4c3(evidence_text)

            if matches:
                hit = {
                    "round": round_number,
                    "query": current_query,
                    "title": raw_item.get("title", ""),
                    "matched_patterns": list(matches),
                }
                round_security_hits.append(hit)
                evidence_security_hits.append(hit)

        scan_ms = (time.perf_counter() - scan_start) * 1000.0
        evidence_scan_total_ms += scan_ms

        if round_security_hits:
            raise RuntimeError(
                "Retrieved evidence failed deterministic prompt-injection "
                "screening. Grounded generation has been blocked."
            )

        round_evidence = _4c6w_merge_round_evidence(
            evidence_ledger=evidence_ledger,
            evidence_by_fingerprint=evidence_by_fingerprint,
            raw_evidence=raw_evidence,
            round_number=round_number,
            query=current_query,
        )

        selected = _4c6w_select_evidence(
            evidence_ledger,
            selected_mode,
        )

        final_selection = _4c6w_build_bounded_context(selected)

        print(f"         RAG Candidates      → {rag_candidates}")
        print(f"         MCP Candidates      → {mcp_candidates}")
        print(f"         Round Evidence      → {len(round_evidence)}")
        print(f"         Cumulative Unique   → {len(evidence_ledger)}")
        print(f"         Presented Evidence  → {len(final_selection['evidence'])}")
        print(f"         Context             → {final_selection['context_chars']:,} chars")
        print(f"         Retrieval Wall      → {selected_retrieval_s:.3f} s")
        print(f"         Evidence Scan       → {scan_ms:.3f} ms")

        assessment = await asyncio.to_thread(
            _4c6w_assess_sufficiency,
            question,
            requirements,
            selected_mode,
            round_number,
            final_selection["context"],
        )

        total_llm_calls += 1
        total_llm_latency += float(assessment["latency_s"])
        total_input_tokens += int(assessment["usage"]["input_tokens"])
        total_output_tokens += int(assessment["usage"]["output_tokens"])
        final_assessment = assessment

        round_traces.append({
            "search_number": round_number,
            "mode": selected_mode,
            "query": current_query,
            "rag_candidates": rag_candidates,
            "mcp_candidates": mcp_candidates,
            "rag_calls": int(round_result["rag_calls"]),
            "mcp_calls": int(round_result["mcp_calls"]),
            "round_evidence_ids": [item["evidence_id"] for item in round_evidence],
            "presented_evidence_ids": [
                item["evidence_id"] for item in final_selection["evidence"]
            ],
            "cumulative_unique_evidence": len(evidence_ledger),
            "rag_retrieval_s": rag_retrieval_s,
            "mcp_retrieval_s": mcp_retrieval_s,
            "selected_retrieval_s": selected_retrieval_s,
            "evidence_scan_ms": scan_ms,
            "sufficiency": assessment,
        })

        print(f"         Sufficient          → {assessment['sufficient']}")
        print(f"         Supported Reqs      → {assessment['supported_requirement_ids']}")
        print(f"         Missing Reqs        → {assessment['missing_requirement_ids']}")
        print(f"         Sufficiency Latency → {assessment['latency_s']:.3f} s")

        if assessment["sufficient"]:
            stop_reason = "SUFFICIENT_EVIDENCE"
            break

        if round_number >= MODULE4_MAX_MODEL_SEARCHES_4C6W:
            stop_reason = "MAX_RETRIEVAL_ROUNDS"
            break

        current_query = assessment["next_query"]
        if not current_query:
            raise RuntimeError(
                "Evidence was insufficient but no refined retrieval query was available."
            )

        print(f"         Next Query          → {current_query}")

    if final_selection is None:
        raise RuntimeError(
            "Grounded adaptive retrieval completed without an evidence selection."
        )

    generation = await asyncio.to_thread(
        run_4c_telecom_grounded,
        question,
        final_selection["context"],
    )

    final_answer = str(generation.get("answer", "") or "").strip()
    final_generation_s = float(generation.get("elapsed_s", 0.0) or 0.0)

    total_llm_calls += 1
    total_llm_latency += final_generation_s

    if not final_answer:
        raise RuntimeError("Grounded generator returned an empty answer.")

    return {
        "answer": final_answer,
        "elapsed_s": final_generation_s,
        "tool_loop_wall_s": float(time.perf_counter() - start_wall),
        "llm_calls": total_llm_calls,
        "tool_requests": 0,
        "search_count": len(search_queries),
        "search_queries": list(search_queries),
        "round_traces": round_traces,
        "selected_mode": selected_mode,
        "planner": planner,
        "answer_requirements": requirements,
        "final_sufficiency_assessment": final_assessment,
        "stop_reason": stop_reason,
        "all_evidence": evidence_ledger,
        "final_evidence": list(final_selection["evidence"]),
        "final_context": final_selection["context"],
        "final_context_chars": final_selection["context_chars"],
        "rag_candidate_total": rag_candidate_total,
        "mcp_candidate_total": mcp_candidate_total,
        "rag_call_total": rag_call_total,
        "mcp_call_total": mcp_call_total,
        "rag_retrieval_s": rag_retrieval_total,
        "mcp_retrieval_s": mcp_retrieval_total,
        # Backwards-compatible field consumed by the surrounding runtime.
        # It now means total wall time of the LLM-selected retrieval architecture.
        "hybrid_retrieval_s": selected_retrieval_total,
        "selected_retrieval_s": selected_retrieval_total,
        "evidence_scan_ms": evidence_scan_total_ms,
        "evidence_security_hits": evidence_security_hits,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_llm_latency_s": total_llm_latency,
        "completion_mode": (
            "normal"
            if stop_reason == "SUFFICIENT_EVIDENCE"
            else "final_after_search_cap"
        ),
    }



# ============================================================
# 4C.6Y.L — THREE-WAY KNOWLEDGE ROUTER + LIVE EXTERNAL SEARCH
# ============================================================
#
# 6Y adds live-external and local-runtime knowledge capabilities without mutating the
# proven 6V connected-corpus retrieval architecture.
#
# ROUTES
# ------
#
#   TELECOM_GROUNDED
#       Legacy route name retained for compatibility.
#       Operational meaning in 6W:
#           connected technical corpus is the appropriate
#           authoritative knowledge source.
#
#   LIVE_EXTERNAL_GROUNDED
#       The answer materially depends on current / changing /
#       externally verifiable public information.
#
#   GENERAL_KNOWLEDGE_FALLBACK
#       Stable general knowledge that does not reasonably
#       require live verification.
#
# IMPORTANT
# ---------
# This is semantic routing, NOT keyword routing.
# A question does not need to contain "current", "today",
# "latest" or "now" to require live external grounding.
#
# The existing frozen 4B comparative judge remains applicable
# ONLY to the connected technical corpus path in this cell.
# Live-web answers are grounded and cited, but the frozen
# telecom-oriented judge is intentionally not repurposed.
# ============================================================

import datetime
import json
import logging

for _logger_name_4c6w in (
    "fastmcp",
    "fastmcp.client",
    "mcp",
):
    logging.getLogger(_logger_name_4c6w).setLevel(logging.WARNING)


try:
    from fastmcp import FastMCP, Client
except Exception as exc:
    raise RuntimeError(
        "FastMCP is required for Cell 4C.6Y. "
        f"Import failed: {exc}"
    )


LIVE_EXTERNAL_GROUNDED_MODE = "LIVE_EXTERNAL_GROUNDED"

VALID_RESPONSE_MODES_4C6W = {
    GENERAL_KNOWLEDGE_FALLBACK_MODE,
    TELECOM_GROUNDED_MODE,
    LIVE_EXTERNAL_GROUNDED_MODE,
}


# ============================================================
# 4C.6Y.L1 — THREE-WAY GRANITE KNOWLEDGE ROUTER
# ============================================================

KNOWLEDGE_ROUTER_SYSTEM_PROMPT_4C6W = """
You are the knowledge-scope router for an AI system.

Classify the user's sanitized question into exactly ONE response mode.

MODE 1 — TELECOM_GROUNDED
Use when the question is materially covered by the connected controlled
technical corpus and should be answered from that corpus.

The connected corpus materially covers:
- telecommunications and mobile networks;
- 3GPP, 5G, 5G SA, NR, RAN, Core, OSS/BSS;
- telecom standards, interfaces, procedures and architecture;
- IETF networking and Internet protocols;
- Kubernetes, cloud-native infrastructure and platform engineering;
- O-RAN, ETSI, GSMA, TM Forum, CAMARA and related technical material;
- technical research and industry material represented in the corpus.

The question does NOT need to explicitly contain telecom terminology.
If the connected technical corpus is the best authoritative source,
use TELECOM_GROUNDED.

MODE 2 — LIVE_EXTERNAL_GROUNDED
Use when a reliable answer materially depends on public information that
may have changed over time, may have changed after model training, or
should be verified against current external sources.

Examples include, but are not limited to:
- current office holders, leaders, executives or organizational roles;
- latest or recent news and events;
- current prices, market values or exchange rates;
- current schedules, availability, standings or results;
- current laws, regulations, policies or official guidance;
- latest software, product, standards or release status;
- recent company announcements, statistics or economic indicators;
- any real-world status whose answer can become stale.

IMPORTANT:
- This is NOT keyword matching.
- A question may require LIVE_EXTERNAL_GROUNDED even without words such as
  "current", "latest", "today", "now" or "recent".
- For example, "Who is the prime minister of the United Kingdom?" normally
  asks for the person holding that office now and should be live-verified.
- A telecom-related question may also require LIVE_EXTERNAL_GROUNDED when
  the requested claim is explicitly about the latest/current state and the
  controlled corpus cannot establish that current state.

MODE 3 — GENERAL_KNOWLEDGE_FALLBACK
Use when the question is outside the connected technical corpus AND its
answer is stable general knowledge that does not reasonably require live
external verification.

Examples:
- explain photosynthesis;
- explain Newton's second law;
- what causes volcanic eruptions;
- stable historical or conceptual general-knowledge questions.

Decision priority:
1. If current/fresh external verification is materially required,
   choose LIVE_EXTERNAL_GROUNDED.
2. Otherwise, if the connected controlled technical corpus is appropriate,
   choose TELECOM_GROUNDED.
3. Otherwise choose GENERAL_KNOWLEDGE_FALLBACK.

Return JSON only:

{
  "response_mode": "TELECOM_GROUNDED | LIVE_EXTERNAL_GROUNDED | GENERAL_KNOWLEDGE_FALLBACK",
  "reason": "brief semantic routing reason"
}
""".strip()


def classify_4c6w_knowledge_scope(question):
    start = time.perf_counter()

    response = openrouter.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "system",
                "content": KNOWLEDGE_ROUTER_SYSTEM_PROMPT_4C6W,
            },
            {
                "role": "user",
                "content": str(question).strip(),
            },
        ],
        max_tokens=160,
        extra_body={
            "reasoning": {
                "effort": "none"
            }
        },
    )

    elapsed = time.perf_counter() - start

    raw_text = _4c6w_extract_text(response)
    parsed = _4c6w_parse_json_object(raw_text)

    response_mode = str(
        parsed.get("response_mode", "")
        or ""
    ).strip().upper()

    reason = str(
        parsed.get("reason", "")
        or ""
    ).strip()

    if response_mode not in VALID_RESPONSE_MODES_4C6W:
        raise ValueError(
            f"Granite returned unsupported 6W response mode: "
            f"{response_mode!r}"
        )

    return {
        "response_mode": response_mode,
        "reason": reason,
        "elapsed_s": float(elapsed),
        "raw_text": raw_text,
    }


# ============================================================
# 4C.6Y.L2 — OPEN-WEBSEARCH CLI JSON BACKEND
# ============================================================
#
# This notebook uses the documented machine-readable CLI contract:
#
#   open-websearch search ... --json
#   open-websearch fetch-web ... --json
#
# Why CLI JSON:
#   - verified in this notebook environment
#   - no API key
#   - no FastMCP STDIO / fileno issue
#   - no long-lived daemon dependency
#   - structured title / URL / description output
#   - fetch-web returns actual page content when snippets are thin
#
# GOVERNANCE
# ----------
# HARD MAX = 3 external Open-WebSearch operations per user question.
#
# Typical sequence:
#   1. SEARCH
#   2. FETCH best source if needed
#   3. FINAL refined SEARCH if still needed
# ============================================================

import html
import subprocess
import shutil

OPEN_WEBSEARCH_VERSION_4C6W = "2.1.11"

OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W = 3
OPEN_WEBSEARCH_OPERATION_COUNT_4C6W = 0

OPEN_WEBSEARCH_SEARCH_TIMEOUT_S_4C6W = 25
OPEN_WEBSEARCH_FETCH_TIMEOUT_S_4C6W = 25

OPEN_WEBSEARCH_DEFAULT_ENGINE_4C6W = "duckduckgo"


def _4c6w_clean_search_text(value):
    value = html.unescape(
        str(value or "")
    )

    value = re.sub(
        r"<[^>]+>",
        "",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


def _4c6w_open_websearch_env():
    env = os.environ.copy()

    env.update({
        "DEFAULT_SEARCH_ENGINE": OPEN_WEBSEARCH_DEFAULT_ENGINE_4C6W,
        "ALLOWED_SEARCH_ENGINES": "duckduckgo,bing",
        "USE_PROXY": "false",
        "FETCH_WEB_INSECURE_TLS": "false",
    })

    return env


def _4c6w_open_websearch_budget_guard(operation_name):
    global OPEN_WEBSEARCH_OPERATION_COUNT_4C6W

    if (
        OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
        >=
        OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W
    ):
        raise RuntimeError(
            "Open-WebSearch hard operation budget reached: "
            f"{OPEN_WEBSEARCH_OPERATION_COUNT_4C6W}/"
            f"{OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W} "
            f"before {operation_name}."
        )

    OPEN_WEBSEARCH_OPERATION_COUNT_4C6W += 1


def _4c6w_run_open_websearch_cli(
    command_args,
    timeout_s,
    operation_name,
):
    """
    Execute ONE governed Open-WebSearch CLI operation.
    """

    _4c6w_open_websearch_budget_guard(
        operation_name
    )

    if shutil.which("npx") is None:
        raise RuntimeError(
            "Open-WebSearch requires npx in this runtime."
        )

    command = [
        "npx",
        "-y",
        f"open-websearch@{OPEN_WEBSEARCH_VERSION_4C6W}",
        *command_args,
    ]

    start = time.perf_counter()

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=float(timeout_s),
            env=_4c6w_open_websearch_env(),
        )

    except subprocess.TimeoutExpired:
        raise RuntimeError(
            f"Open-WebSearch {operation_name} timed out after "
            f"{timeout_s}s."
        )

    elapsed = time.perf_counter() - start

    stdout = str(
        result.stdout
        or ""
    ).strip()

    stderr = str(
        result.stderr
        or ""
    ).strip()

    if result.returncode != 0:
        raise RuntimeError(
            f"Open-WebSearch {operation_name} failed "
            f"(exit={result.returncode}).\n"
            f"stderr:\n{stderr[:2000]}"
        )

    if not stdout:
        raise RuntimeError(
            f"Open-WebSearch {operation_name} returned empty stdout."
        )

    try:
        payload = json.loads(
            stdout
        )

    except Exception as exc:
        raise RuntimeError(
            f"Open-WebSearch {operation_name} returned invalid JSON: "
            f"{exc}\nstdout preview:\n{stdout[:2000]}"
        )

    if payload.get("status") != "ok":
        raise RuntimeError(
            f"Open-WebSearch {operation_name} returned status="
            f"{payload.get('status')!r}: "
            f"{payload.get('error') or payload.get('hint')}"
        )

    return {
        "payload": payload,
        "elapsed_s": float(elapsed),
        "stderr": stderr,
        "command": command,
    }


def open_websearch_search_4c6w(
    query,
    top_k=5,
):
    """
    ONE external operation:
    Open-WebSearch CLI search --json
    """

    query = re.sub(
        r"\s+",
        " ",
        str(query or ""),
    ).strip()

    if not query:
        raise ValueError(
            "Open-WebSearch query cannot be empty."
        )

    top_k = max(
        1,
        min(
            int(top_k),
            5,
        ),
    )

    result = _4c6w_run_open_websearch_cli(
        [
            "search",
            query,
            "--json",
        ],
        timeout_s=OPEN_WEBSEARCH_SEARCH_TIMEOUT_S_4C6W,
        operation_name="search",
    )

    data = (
        result["payload"].get("data")
        or {}
    )

    raw_results = list(
        data.get("results")
        or []
    )

    normalized = []

    for rank, item in enumerate(
        raw_results[:top_k],
        start=1,
    ):
        title = _4c6w_clean_search_text(
            item.get("title")
        )
        url = str(
            item.get("url")
            or ""
        ).strip()
        description = _4c6w_clean_search_text(
            item.get("description")
        )
        source = str(
            item.get("source")
            or ""
        ).strip()
        engine = str(
            item.get("engine")
            or ""
        ).strip()

        if not url or not description:
            continue

        normalized.append({
            "evidence_id": "",
            "retrieval_system": "LIVE_WEB",
            "retrieval_systems": ["LIVE_WEB"],
            "source_family": "OPEN_WEBSEARCH",
            "source_families": ["OPEN_WEBSEARCH"],
            "document_id": url,
            "title": title or url,
            "chunk_id": "",
            "score": 0.0,
            "native_ranks": {
                "LIVE_WEB": rank,
            },
            "native_scores": {
                "LIVE_WEB": 0.0,
            },
            "fusion_score": None,
            "text": description,
            "text_chars": len(description),
            "published_date": None,
            "url": url,
            "retrieved_at_utc": datetime.datetime.now(
                datetime.timezone.utc
            ).isoformat(),
            "provenance": [{
                "retrieval_system": "LIVE_WEB",
                "source_family": "OPEN_WEBSEARCH",
                "provider": "Open-WebSearch",
                "operation": "search",
                "engine": engine,
                "source": source,
                "url": url,
                "native_rank": rank,
            }],
        })

    if not normalized:
        raise RuntimeError(
            "Open-WebSearch search succeeded but returned no "
            "normalizable evidence."
        )

    return {
        "operation": "search",
        "query": query,
        "candidate_count": len(normalized),
        "evidence": normalized,
        "elapsed_s": result["elapsed_s"],
        "trace": {
            "provider": "Open-WebSearch",
            "operation": "search",
            "engines": data.get("engines"),
            "total_results": data.get("totalResults"),
            "partial_failures": data.get("partialFailures"),
            "external_operation_number": OPEN_WEBSEARCH_OPERATION_COUNT_4C6W,
        },
    }


def _4c6w_best_fetch_url(evidence):
    """
    Prefer official / high-authority sources when available,
    otherwise preserve search rank.
    """

    if not evidence:
        return ""

    authority_domains = (
        ".gov.",
        ".gov/",
        "gov.uk",
        ".edu/",
        ".ac.",
        "who.int",
        "un.org",
        "europa.eu",
        "ietf.org",
        "3gpp.org",
        "gsma.com",
    )

    for item in evidence:
        url = str(
            item.get("url")
            or ""
        ).strip().lower()

        if any(
            token in url
            for token in authority_domains
        ):
            return str(
                item.get("url")
                or ""
            ).strip()

    return str(
        evidence[0].get("url")
        or ""
    ).strip()


def open_websearch_fetch_4c6w(
    url,
    max_chars=12000,
):
    """
    ONE external operation:
    Open-WebSearch CLI fetch-web <URL> --json

    Exact command form independently smoke-tested and passed.
    """

    url = str(
        url
        or ""
    ).strip()

    if not url:
        raise ValueError(
            "Open-WebSearch fetch URL cannot be empty."
        )

    result = _4c6w_run_open_websearch_cli(
        [
            "fetch-web",
            url,
            "--json",
        ],
        timeout_s=OPEN_WEBSEARCH_FETCH_TIMEOUT_S_4C6W,
        operation_name="fetch-web",
    )

    data = (
        result["payload"].get("data")
        or {}
    )

    content = _4c6w_clean_search_text(
        data.get("content")
    )

    final_url = str(
        data.get("finalUrl")
        or data.get("url")
        or url
    ).strip()

    title = _4c6w_clean_search_text(
        data.get("title")
    )

    if not content:
        raise RuntimeError(
            "Open-WebSearch fetch-web returned no content."
        )

    content = content[:int(max_chars)]

    evidence = [{
        "evidence_id": "",
        "retrieval_system": "LIVE_WEB",
        "retrieval_systems": ["LIVE_WEB"],
        "source_family": "OPEN_WEBSEARCH",
        "source_families": ["OPEN_WEBSEARCH"],
        "document_id": final_url,
        "title": title or final_url,
        "chunk_id": "",
        "score": 1.0,
        "native_ranks": {
            "LIVE_WEB": 1,
        },
        "native_scores": {
            "LIVE_WEB": 1.0,
        },
        "fusion_score": None,
        "text": content,
        "text_chars": len(content),
        "published_date": None,
        "url": final_url,
        "retrieved_at_utc": datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),
        "provenance": [{
            "retrieval_system": "LIVE_WEB",
            "source_family": "OPEN_WEBSEARCH",
            "provider": "Open-WebSearch",
            "operation": "fetch-web",
            "retrieval_method": data.get("retrievalMethod"),
            "content_type": data.get("contentType"),
            "url": final_url,
        }],
    }]

    return {
        "operation": "fetch",
        "url": final_url,
        "candidate_count": 1,
        "evidence": evidence,
        "elapsed_s": float(
            result["elapsed_s"]
        ),
        "trace": {
            "provider": "Open-WebSearch",
            "operation": "fetch-web",
            "retrieval_method": data.get("retrievalMethod"),
            "truncated": data.get("truncated"),
            "external_operation_number": (
                OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
            ),
        },
    }


# Backward-compatible adapter name used by the live loop.
async def retrieve_live_external_candidates_4c6w(
    query,
    top_k=5,
):
    result = await asyncio.to_thread(
        open_websearch_search_4c6w,
        query,
        top_k,
    )

    return {
        "query": result["query"],
        "retrieval_system": "LIVE_WEB",
        "candidate_count": result["candidate_count"],
        "evidence": result["evidence"],
        "timing": {
            "mcp_roundtrip_s": result["elapsed_s"],
            "retrieval_time_s": result["elapsed_s"],
        },
        "trace": result["trace"],
    }


# ============================================================
# 4C.6Y.L4 — DETERMINISTIC LOCAL DATE/TIME TOOL
# ============================================================
#
# Direct current date/time questions do not need web search.
#
# The runtime clock is:
# - deterministic;
# - local;
# - timezone-aware;
# - free;
# - effectively zero-latency;
# - more reliable than search snippets for clock/calendar facts.
#
# Granite still decides the high-level knowledge scope.
# Python then deterministically chooses the runtime clock for this
# narrow utility class. All other current/changing public facts retain
# the Open-WebSearch adaptive path.
# ============================================================

from datetime import datetime as DateTime
from zoneinfo import ZoneInfo


LOCAL_RUNTIME_DEFAULT_TZ_4C6Y = "Europe/London"


def classify_local_datetime_request_4c6y(question):
    """
    Narrow deterministic utility classifier.

    Intentionally handles only direct current date/time questions.
    It does NOT replace Granite knowledge-scope routing.
    """

    q = re.sub(
        r"\s+",
        " ",
        str(question or "").strip().lower(),
    )

    date_patterns = (
        r"\bwhat(?:'s| is) today'?s date\b",
        r"\bwhat(?:'s| is) todays date\b",
        r"\bwhat(?:'s| is) the date today\b",
        r"\bwhat(?:'s| is) the current date\b",
        r"\btoday'?s date\b",
        r"\btodays date\b",
        r"\bcurrent date\b",
        r"\bwhat day is it today\b",
    )

    time_patterns = (
        r"\bwhat time is it\b",
        r"\bwhat(?:'s| is) the current time\b",
        r"\bwhat time is it now\b",
        r"\bcurrent time\b",
        r"\btime now\b",
    )

    if any(
        re.search(pattern, q)
        for pattern in date_patterns
    ):
        return {
            "matched": True,
            "kind": "DATE",
            "timezone": LOCAL_RUNTIME_DEFAULT_TZ_4C6Y,
        }

    if any(
        re.search(pattern, q)
        for pattern in time_patterns
    ):
        return {
            "matched": True,
            "kind": "TIME",
            "timezone": LOCAL_RUNTIME_DEFAULT_TZ_4C6Y,
        }

    return {
        "matched": False,
        "kind": None,
        "timezone": None,
    }


def run_local_datetime_tool_4c6y(question):
    """
    Return a direct date/time answer from the Python runtime clock.
    """

    classification = classify_local_datetime_request_4c6y(
        question
    )

    if not classification["matched"]:
        return None

    tz_name = classification["timezone"]
    tz = ZoneInfo(tz_name)
    now = DateTime.now(tz)

    if classification["kind"] == "DATE":
        answer = (
            f"Today's date is "
            f"{now.strftime('%d %B %Y')}."
        )

    elif classification["kind"] == "TIME":
        answer = (
            f"The current time in London is "
            f"{now.strftime('%H:%M:%S %Z')} "
            f"on {now.strftime('%d %B %Y')}."
        )

    else:
        raise RuntimeError(
            "Unsupported local date/time request classification."
        )

    return {
        "answer": answer,
        "kind": classification["kind"],
        "timezone": tz_name,
        "timestamp_iso": now.isoformat(),
        "runtime_tool": "LOCAL_DATE_TIME",
    }


# ============================================================
# 4C.6Y.L5 — LIVE SEARCH PLANNER
# ============================================================

LIVE_SEARCH_PLANNER_SYSTEM_PROMPT_4C6W = """
You are planning live public-information retrieval.

The question has already been classified as requiring current external
verification.

Your task:
1. formulate ONE concise standalone web-search query;
2. derive the minimum fixed answer requirements explicitly requested by
   the user's question.

Rules:
- The query should retrieve current authoritative evidence.
- Prefer wording that is likely to surface primary or official sources
  when the requested fact concerns an official role, law, regulation,
  company announcement, release status or institutional fact.
- Do not answer the question yourself.
- Do not add scope the user did not request.
- Produce 1 to 5 fixed requirements.
- The requirements define the boundary for both sufficiency checking
  and final answer generation.

Return JSON only:

{
  "query": "standalone live-search query",
  "reason": "brief search rationale",
  "answer_requirements": [
    "first explicit requirement"
  ]
}
""".strip()


def plan_live_external_search_4c6w(question):
    current_date = datetime.date.today().isoformat()
    start = time.perf_counter()

    response = _4c6w_call_llm(
        messages=[
            {
                "role": "system",
                "content": LIVE_SEARCH_PLANNER_SYSTEM_PROMPT_4C6W,
            },
            {
                "role": "user",
                "content": (
                    f"CURRENT DATE: {current_date}\n\n"
                    f"QUESTION:\n{question}"
                ),
            },
        ],
        max_tokens=360,
    )

    elapsed = time.perf_counter() - start
    raw_text = _4c6w_extract_text(response)
    parsed = _4c6w_parse_json_object(raw_text)

    query = re.sub(
        r"\s+",
        " ",
        str(parsed.get("query", "") or ""),
    ).strip()

    if not query:
        query = str(question).strip()

    raw_requirements = parsed.get(
        "answer_requirements",
        [],
    )

    if not isinstance(raw_requirements, list):
        raw_requirements = []

    requirements = []

    for item in raw_requirements:
        item = re.sub(
            r"\s+",
            " ",
            str(item or ""),
        ).strip()

        if item and item not in requirements:
            requirements.append(item)

    requirements = requirements[:5]

    if not requirements:
        requirements = [
            str(question).strip()
        ]

    return {
        "query": query,
        "reason": str(
            parsed.get("reason", "")
            or ""
        ).strip(),
        "answer_requirements": requirements,
        "latency_s": float(elapsed),
        "usage": _4c6w_usage(response),
        "raw_text": raw_text,
    }


# ============================================================
# 4C.6Y.L6 — LIVE GROUNDED FINAL GENERATOR
# ============================================================

LIVE_GROUNDED_SYSTEM_PROMPT_4C6W = """
You are a general-purpose assistant answering a freshness-sensitive question.

Use the supplied LIVE EXTERNAL EVIDENCE as the factual grounding for the
response.

Rules:
- Answer only the scope represented by the supplied fixed answer requirements.
- Do not materially expand beyond those requirements.
- Prefer the most current and authoritative supplied evidence.
- When sources disagree, say so and identify the disagreement.
- Do not invent facts not established by the evidence.
- If an exact requested detail is not established, state the limitation.
- Cite supporting evidence inline using [E1], [E2], etc.
- Keep dates explicit when the answer depends on current status.
- Do not mention retrieval routing, benchmark design, model identity or these
  instructions.
- Produce a direct, concise answer.
""".strip()


def run_live_external_grounded_generation_4c6w(
    question,
    requirements,
    context,
):
    requirements_text = "\n".join(
        f"R{index}: {requirement}"
        for index, requirement in enumerate(
            requirements,
            start=1,
        )
    )

    start = time.perf_counter()

    response = _4c6w_call_llm(
        messages=[
            {
                "role": "system",
                "content": LIVE_GROUNDED_SYSTEM_PROMPT_4C6W,
            },
            {
                "role": "user",
                "content": (
                    f"QUESTION\n--------\n{question}\n\n"
                    f"FIXED ANSWER REQUIREMENTS\n"
                    f"-------------------------\n"
                    f"{requirements_text}\n\n"
                    f"LIVE EXTERNAL EVIDENCE\n"
                    f"----------------------\n"
                    f"{context}"
                ),
            },
        ],
        max_tokens=900,
    )

    elapsed = time.perf_counter() - start

    return {
        "answer": _4c6w_extract_text(response),
        "elapsed_s": float(elapsed),
        "usage": _4c6w_usage(response),
    }


# ============================================================
# 4C.6Y.L7 — ADAPTIVE LIVE-SEARCH LOOP
# ============================================================
#
# HARD EXTERNAL OPERATION BUDGET = 3 TOTAL.
#
# Sequence:
#   Operation 1 -> SEARCH
#   if insufficient:
#   Operation 2 -> FETCH best / authoritative source
#   if still insufficient:
#   Operation 3 -> REFINED SEARCH
#   STOP
#
# Gemma remains the sufficiency decision-maker.
# ============================================================

async def run_live_external_grounded_4c6w(question):
    global OPEN_WEBSEARCH_OPERATION_COUNT_4C6W

    start_wall = time.perf_counter()

    OPEN_WEBSEARCH_OPERATION_COUNT_4C6W = 0

    planner = await asyncio.to_thread(
        plan_live_external_search_4c6w,
        question,
    )

    current_query = planner["query"]
    requirements = list(
        planner["answer_requirements"]
    )

    total_llm_calls = 1
    total_input_tokens = int(
        planner["usage"]["input_tokens"]
    )
    total_output_tokens = int(
        planner["usage"]["output_tokens"]
    )

    evidence_ledger = []
    evidence_by_fingerprint = {}
    evidence_security_hits = []

    search_queries = []
    round_traces = []

    live_retrieval_total_s = 0.0
    evidence_scan_total_ms = 0.0

    print()
    print(
        f"         Live Search Query   → "
        f"{current_query}"
    )
    print("         Fixed Requirements →")

    for index, requirement in enumerate(
        requirements,
        start=1,
    ):
        print(
            f"            R{index} → "
            f"{requirement}"
        )

    print(
        f"         Planner Latency     → "
        f"{planner['latency_s']:.3f} s"
    )

    final_selection = None
    final_assessment = None
    stop_reason = None

    # --------------------------------------------------------
    # OPERATION 1 — SEARCH
    # --------------------------------------------------------
    search_queries.append(
        current_query
    )

    print()
    print(
        "         External Operation  → 1/3 SEARCH"
    )
    print(
        f"         Query               → "
        f"{current_query}"
    )

    search_result = await asyncio.to_thread(
        open_websearch_search_4c6w,
        current_query,
        5,
    )

    live_retrieval_total_s += float(
        search_result["elapsed_s"]
    )

    scan_start = time.perf_counter()

    for raw_item in search_result["evidence"]:
        evidence_text = _4c6w_evidence_text(
            raw_item
        )

        matches = get_injection_matches_4c3(
            evidence_text
        )

        if matches:
            evidence_security_hits.append({
                "operation": 1,
                "type": "search",
                "query": current_query,
                "title": raw_item.get(
                    "title",
                    "",
                ),
                "url": raw_item.get(
                    "url",
                    "",
                ),
                "matched_patterns": list(
                    matches
                ),
            })

    evidence_scan_total_ms += (
        time.perf_counter()
        -
        scan_start
    ) * 1000.0

    if evidence_security_hits:
        raise RuntimeError(
            "Live search evidence failed deterministic "
            "prompt-injection screening."
        )

    _4c6w_merge_round_evidence(
        evidence_ledger=evidence_ledger,
        evidence_by_fingerprint=evidence_by_fingerprint,
        raw_evidence=search_result["evidence"],
        round_number=1,
        query=current_query,
    )

    selected = _4c6w_rank_evidence(
        evidence_ledger
    )[
        :MODULE4_MAX_FINAL_EVIDENCE_4C6W
    ]

    final_selection = _4c6w_build_bounded_context(
        selected
    )

    assessment = await asyncio.to_thread(
        _4c6w_assess_sufficiency,
        question,
        requirements,
        "LIVE_EXTERNAL",
        1,
        final_selection["context"],
    )

    total_llm_calls += 1
    total_input_tokens += int(
        assessment["usage"]["input_tokens"]
    )
    total_output_tokens += int(
        assessment["usage"]["output_tokens"]
    )

    final_assessment = assessment

    round_traces.append({
        "external_operation": 1,
        "operation": "search",
        "query": current_query,
        "candidate_count": search_result[
            "candidate_count"
        ],
        "presented_evidence_ids": [
            item["evidence_id"]
            for item in final_selection[
                "evidence"
            ]
        ],
        "retrieval_s": search_result[
            "elapsed_s"
        ],
        "sufficiency": assessment,
    })

    print(
        f"         Candidates          → "
        f"{search_result['candidate_count']}"
    )
    print(
        f"         Presented Evidence  → "
        f"{len(final_selection['evidence'])}"
    )
    print(
        f"         Context             → "
        f"{final_selection['context_chars']:,} chars"
    )
    print(
        f"         Search Latency      → "
        f"{search_result['elapsed_s']:.3f} s"
    )
    print(
        f"         Sufficient          → "
        f"{assessment['sufficient']}"
    )

    if assessment["sufficient"]:
        stop_reason = (
            "SUFFICIENT_AFTER_SEARCH"
        )

    # --------------------------------------------------------
    # OPERATION 2 — FETCH BEST SOURCE
    # --------------------------------------------------------
    if (
        not final_assessment["sufficient"]
        and
        OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
        <
        OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W
    ):
        best_url = _4c6w_best_fetch_url(
            search_result["evidence"]
        )

        if best_url:
            print()
            print(
                "         External Operation  → 2/3 FETCH"
            )
            print(
                f"         Source              → "
                f"{best_url}"
            )

            fetch_result = None
            fetch_error = None

            try:
                fetch_result = await asyncio.to_thread(
                    open_websearch_fetch_4c6w,
                    best_url,
                    12000,
                )

            except Exception as exc:
                fetch_error = (
                    f"{type(exc).__name__}: {exc}"
                )

                print(
                    "         Fetch Status        → FAILED / continuing"
                )
                print(
                    f"         Fetch Error         → "
                    f"{fetch_error[:500]}"
                )

                round_traces.append({
                    "external_operation": 2,
                    "operation": "fetch",
                    "url": best_url,
                    "candidate_count": 0,
                    "presented_evidence_ids": [
                        item["evidence_id"]
                        for item in final_selection[
                            "evidence"
                        ]
                    ],
                    "retrieval_s": 0.0,
                    "fetch_error": fetch_error,
                    "sufficiency": final_assessment,
                })

            if fetch_result is not None:

                live_retrieval_total_s += float(
                    fetch_result["elapsed_s"]
                )

                scan_start = time.perf_counter()

                for raw_item in fetch_result[
                    "evidence"
                ]:
                    evidence_text = _4c6w_evidence_text(
                        raw_item
                    )

                    matches = get_injection_matches_4c3(
                        evidence_text
                    )

                    if matches:
                        evidence_security_hits.append({
                            "operation": 2,
                            "type": "fetch",
                            "url": best_url,
                            "matched_patterns": list(
                                matches
                            ),
                        })

                evidence_scan_total_ms += (
                    time.perf_counter()
                    -
                    scan_start
                ) * 1000.0

                if evidence_security_hits:
                    raise RuntimeError(
                        "Fetched live evidence failed deterministic "
                        "prompt-injection screening."
                    )

                _4c6w_merge_round_evidence(
                    evidence_ledger=evidence_ledger,
                    evidence_by_fingerprint=evidence_by_fingerprint,
                    raw_evidence=fetch_result[
                        "evidence"
                    ],
                    round_number=2,
                    query=f"FETCH::{best_url}",
                )

                selected = _4c6w_rank_evidence(
                    evidence_ledger
                )[
                    :MODULE4_MAX_FINAL_EVIDENCE_4C6W
                ]

                final_selection = _4c6w_build_bounded_context(
                    selected
                )

                assessment = await asyncio.to_thread(
                    _4c6w_assess_sufficiency,
                    question,
                    requirements,
                    "LIVE_EXTERNAL",
                    2,
                    final_selection["context"],
                )

                total_llm_calls += 1
                total_input_tokens += int(
                    assessment[
                        "usage"
                    ][
                        "input_tokens"
                    ]
                )
                total_output_tokens += int(
                    assessment[
                        "usage"
                    ][
                        "output_tokens"
                    ]
                )

                final_assessment = assessment

                round_traces.append({
                    "external_operation": 2,
                    "operation": "fetch",
                    "url": best_url,
                    "candidate_count": 1,
                    "presented_evidence_ids": [
                        item["evidence_id"]
                        for item in final_selection[
                            "evidence"
                        ]
                    ],
                    "retrieval_s": fetch_result[
                        "elapsed_s"
                    ],
                    "sufficiency": assessment,
                })

                print(
                    f"         Presented Evidence  → "
                    f"{len(final_selection['evidence'])}"
                )
                print(
                    f"         Context             → "
                    f"{final_selection['context_chars']:,} chars"
                )
                print(
                    f"         Fetch Latency       → "
                    f"{fetch_result['elapsed_s']:.3f} s"
                )
                print(
                    f"         Sufficient          → "
                    f"{assessment['sufficient']}"
                )

                if assessment["sufficient"]:
                    stop_reason = (
                        "SUFFICIENT_AFTER_FETCH"
                    )

    # --------------------------------------------------------
    # OPERATION 3 — FINAL REFINED SEARCH
    # --------------------------------------------------------
    if (
        not final_assessment["sufficient"]
        and
        OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
        <
        OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W
    ):
        refined_query = str(
            final_assessment.get(
                "next_query",
                "",
            )
            or current_query
        ).strip()

        if refined_query:
            search_queries.append(
                refined_query
            )

            print()
            print(
                "         External Operation  → 3/3 SEARCH"
            )
            print(
                f"         Query               → "
                f"{refined_query}"
            )

            final_search = await asyncio.to_thread(
                open_websearch_search_4c6w,
                refined_query,
                5,
            )

            live_retrieval_total_s += float(
                final_search["elapsed_s"]
            )

            scan_start = time.perf_counter()

            for raw_item in final_search[
                "evidence"
            ]:
                evidence_text = _4c6w_evidence_text(
                    raw_item
                )

                matches = get_injection_matches_4c3(
                    evidence_text
                )

                if matches:
                    evidence_security_hits.append({
                        "operation": 3,
                        "type": "search",
                        "query": refined_query,
                        "title": raw_item.get(
                            "title",
                            "",
                        ),
                        "url": raw_item.get(
                            "url",
                            "",
                        ),
                        "matched_patterns": list(
                            matches
                        ),
                    })

            evidence_scan_total_ms += (
                time.perf_counter()
                -
                scan_start
            ) * 1000.0

            if evidence_security_hits:
                raise RuntimeError(
                    "Final live search evidence failed deterministic "
                    "prompt-injection screening."
                )

            _4c6w_merge_round_evidence(
                evidence_ledger=evidence_ledger,
                evidence_by_fingerprint=evidence_by_fingerprint,
                raw_evidence=final_search[
                    "evidence"
                ],
                round_number=3,
                query=refined_query,
            )

            selected = _4c6w_rank_evidence(
                evidence_ledger
            )[
                :MODULE4_MAX_FINAL_EVIDENCE_4C6W
            ]

            final_selection = _4c6w_build_bounded_context(
                selected
            )

            assessment = await asyncio.to_thread(
                _4c6w_assess_sufficiency,
                question,
                requirements,
                "LIVE_EXTERNAL",
                3,
                final_selection["context"],
            )

            total_llm_calls += 1
            total_input_tokens += int(
                assessment[
                    "usage"
                ][
                    "input_tokens"
                ]
            )
            total_output_tokens += int(
                assessment[
                    "usage"
                ][
                    "output_tokens"
                ]
            )

            final_assessment = assessment

            round_traces.append({
                "external_operation": 3,
                "operation": "search",
                "query": refined_query,
                "candidate_count": final_search[
                    "candidate_count"
                ],
                "presented_evidence_ids": [
                    item["evidence_id"]
                    for item in final_selection[
                        "evidence"
                    ]
                ],
                "retrieval_s": final_search[
                    "elapsed_s"
                ],
                "sufficiency": assessment,
            })

            print(
                f"         Candidates          → "
                f"{final_search['candidate_count']}"
            )
            print(
                f"         Presented Evidence  → "
                f"{len(final_selection['evidence'])}"
            )
            print(
                f"         Context             → "
                f"{final_selection['context_chars']:,} chars"
            )
            print(
                f"         Search Latency      → "
                f"{final_search['elapsed_s']:.3f} s"
            )
            print(
                f"         Sufficient          → "
                f"{assessment['sufficient']}"
            )

            stop_reason = (
                "SUFFICIENT_AFTER_FINAL_SEARCH"
                if assessment["sufficient"]
                else
                "OPERATION_BUDGET_EXHAUSTED_SAFE_INSUFFICIENCY"
            )

    if final_selection is None:
        raise RuntimeError(
            "Live external route produced no evidence selection."
        )

    generation = await asyncio.to_thread(
        run_live_external_grounded_generation_4c6w,
        question,
        requirements,
        final_selection["context"],
    )

    total_llm_calls += 1
    total_input_tokens += int(
        generation["usage"]["input_tokens"]
    )
    total_output_tokens += int(
        generation["usage"]["output_tokens"]
    )

    answer = str(
        generation.get(
            "answer",
            "",
        )
        or ""
    ).strip()

    if not answer:
        raise RuntimeError(
            "Live external grounded generator returned an empty answer."
        )

    return {
        "answer": answer,
        "elapsed_s": float(
            generation["elapsed_s"]
        ),
        "tool_loop_wall_s": float(
            time.perf_counter()
            -
            start_wall
        ),
        "search_count": len(
            search_queries
        ),
        "external_operation_count": int(
            OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
        ),
        "search_queries": list(
            search_queries
        ),
        "round_traces": round_traces,
        "planner": planner,
        "answer_requirements": requirements,
        "final_sufficiency_assessment": final_assessment,
        "stop_reason": stop_reason,
        "final_evidence": list(
            final_selection["evidence"]
        ),
        "final_context": final_selection[
            "context"
        ],
        "final_context_chars": final_selection[
            "context_chars"
        ],
        "live_retrieval_s": float(
            live_retrieval_total_s
        ),
        "evidence_scan_ms": float(
            evidence_scan_total_ms
        ),
        "evidence_security_hits": evidence_security_hits,
        "llm_calls": total_llm_calls,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
    }


# ============================================================
# 4C.6Y.0 — VERIFY CORPUS-AWARE ROUTER FROM CELL 4C.6R
# ============================================================
#
# Cell 4C.6R remains a prerequisite because 6W preserves its
# validated runtime constants and earlier routing baseline.
# 6W itself performs the new three-way knowledge-scope routing.
# ============================================================

if not globals().get(
    "MODULE4_4C6R_PASS",
    False,
):

    raise RuntimeError(
        "Run Cell 4C.6R successfully before running Cell 4C.6Y."
    )


# ============================================================
# 4C.6Y.1 — VERIFY QUESTION
# ============================================================

if (
    "MODULE4_4C6_RAW_PROMPT"
    not in globals()
    or
    not isinstance(
        MODULE4_4C6_RAW_PROMPT,
        str,
    )
    or
    not MODULE4_4C6_RAW_PROMPT.strip()
):

    raise RuntimeError(
        "Run Cell 4C.6A with a non-empty question "
        "before running Cell 4C.6Y."
    )


# ============================================================
# 4C.6Y.2 — VERIFY REQUIRED RUNTIME FUNCTIONS
# ============================================================

required_functions_4c6 = [

    "preprocess_user_prompt_4c3",

    "scan_for_injection_4c3",

    "scan_for_evidence_manipulation_4c3",

    "get_injection_matches_4c3",

    "run_4c_gemma_only",

    "run_4c_general_fallback",

    "run_4c_telecom_grounded",

    "retrieve_rag_candidates",

    "retrieve_mcp_candidates",

    "fuse_hybrid_candidates",

    "call_granite_judge",
]


for function_name in required_functions_4c6:

    if (
        function_name not in globals()
        or
        not callable(
            globals()[function_name]
        )
    ):

        raise RuntimeError(
            f"Required runtime function unavailable: "
            f"{function_name}"
        )


# ============================================================
# 4C.6Y.3 — VERIFY REQUIRED RUNTIME CONSTANTS
# ============================================================

required_constants_4c6 = [

    "GENERAL_KNOWLEDGE_FALLBACK_MODE",

    "TELECOM_GROUNDED_MODE",

    "VALID_RESPONSE_MODES",

    "JUDGE_MODEL",

    "LLM_MODEL",
]


for constant_name in required_constants_4c6:

    if constant_name not in globals():

        raise RuntimeError(
            f"Required runtime constant unavailable: "
            f"{constant_name}"
        )


if "openrouter" not in globals():
    raise RuntimeError(
        "Required OpenRouter client 'openrouter' is unavailable."
    )


# ============================================================
# 4C.6Y.4 — MEMORY BASELINE
# ============================================================

gc.collect()


PROCESS_4C6 = psutil.Process(
    os.getpid()
)


RAM_BEFORE_4C6_GIB = (

    PROCESS_4C6.memory_info().rss
    /
    (1024 ** 3)
)


AVAILABLE_RAM_BEFORE_4C6_GIB = (

    psutil.virtual_memory().available
    /
    (1024 ** 3)
)


# ============================================================
# 4C.6Y.5 — START WORKFLOW
# ============================================================

WORKFLOW_START_4C6 = (
    time.perf_counter()
)


print("=" * 116)
print("MODULE 4C.6Y — THREE-WAY KNOWLEDGE ROUTER + OPEN-WEBSEARCH CLI")
print("=" * 116)


print(
    f"Raw User Question           : "
    f"{MODULE4_4C6_RAW_PROMPT}"
)


print()


# ============================================================
# 4C.6Y.6 — GATE 1: DETERMINISTIC SECURITY
# ============================================================

GUARDRAIL_4C6 = (
    preprocess_user_prompt_4c3(
        MODULE4_4C6_RAW_PROMPT
    )
)


INJECTION_DETECTED_4C6 = (
    GUARDRAIL_4C6[
        "injection_detected"
    ]
)


MATCHED_PATTERNS_4C6 = (
    GUARDRAIL_4C6[
        "matched_patterns"
    ]
)


EVIDENCE_MANIPULATION_DETECTED_4C6 = (
    GUARDRAIL_4C6[
        "evidence_manipulation_detected"
    ]
)


SANITIZED_QUESTION_4C6 = (
    GUARDRAIL_4C6[
        "sanitized_prompt"
    ]
)


GUARDRAIL_LATENCY_4C6_S = (
    GUARDRAIL_4C6[
        "elapsed_s"
    ]
)


GUARDRAIL_LATENCY_4C6_MS = (

    GUARDRAIL_LATENCY_4C6_S
    *
    1000
)


print(
    "GATE 1 — DETERMINISTIC SECURITY"
)


print(
    f"         Injection Detected → "
    f"{INJECTION_DETECTED_4C6}"
)


print(
    f"         Matched Patterns   → "
    f"{MATCHED_PATTERNS_4C6}"
)


print(
    f"         Evidence Attack    → "
    f"{EVIDENCE_MANIPULATION_DETECTED_4C6}"
)


print(
    f"         Sanitized Intent   → "
    f"{SANITIZED_QUESTION_4C6}"
)


print(
    f"         Guardrail Latency  → "
    f"{GUARDRAIL_LATENCY_4C6_MS:.3f} ms"
)


# ============================================================
# 4C.6Y.7 — GENERIC SECURITY VALIDATION
# ============================================================
#
# No assumption is made about whether an attack SHOULD exist.
#
# We validate only:
#
#   - useful intent remains
#   - sanitized text contains no currently detectable attack
#   - if an attack WAS detected, raw text was not forwarded
#
# ============================================================

SANITIZED_INTENT_EXISTS_PASS_4C6 = (

    isinstance(
        SANITIZED_QUESTION_4C6,
        str,
    )

    and

    len(
        SANITIZED_QUESTION_4C6.strip()
    )
    >=
    3
)


SANITIZED_INJECTION_FREE_PASS_4C6 = (

    not scan_for_injection_4c3(
        SANITIZED_QUESTION_4C6
    )
)


SANITIZED_EVIDENCE_ATTACK_FREE_PASS_4C6 = (

    not scan_for_evidence_manipulation_4c3(
        SANITIZED_QUESTION_4C6
    )
)


if (
    INJECTION_DETECTED_4C6
    or
    EVIDENCE_MANIPULATION_DETECTED_4C6
):

    RAW_ATTACK_BLOCKED_PASS_4C6 = (

        SANITIZED_QUESTION_4C6
        !=
        MODULE4_4C6_RAW_PROMPT
    )

else:

    # Clean questions require no transformation.
    RAW_ATTACK_BLOCKED_PASS_4C6 = True


PRE_LLM_SECURITY_PASS_4C6 = all(

    [
        SANITIZED_INTENT_EXISTS_PASS_4C6,

        SANITIZED_INJECTION_FREE_PASS_4C6,

        SANITIZED_EVIDENCE_ATTACK_FREE_PASS_4C6,

        RAW_ATTACK_BLOCKED_PASS_4C6,
    ]
)


if not PRE_LLM_SECURITY_PASS_4C6:

    raise RuntimeError(
        "4C.6W deterministic security boundary failed. "
        "No LLM or retrieval system will be called."
    )


print()

print(
    "✓ GATE 1 PASSED — trusted sanitized intent created."
)


# ============================================================
# 4C.6Y.8 — START BASELINE GEMMA + GRANITE ROUTER
# ============================================================
#
# These are allowed to run concurrently:
#
#   Gemma 4 baseline
#   Granite semantic router
#
# No RAG or MCP has been called yet.
#
# The second Gemma call will NOT begin until the baseline
# Gemma call has completed.
#
# ============================================================

print()

print(
    "GATE 2 — BASELINE GENERATION + KNOWLEDGE-SCOPE ROUTING"
)


BASELINE_ROUTER_START_4C6 = (
    time.perf_counter()
)


GENERAL_LLM_TASK_4C6 = asyncio.create_task(

    asyncio.to_thread(

        run_4c_gemma_only,

        SANITIZED_QUESTION_4C6,
    )
)


ROUTER_TASK_4C6 = asyncio.create_task(

    asyncio.to_thread(

        classify_4c6w_knowledge_scope,

        SANITIZED_QUESTION_4C6,
    )
)


(
    GENERAL_LLM_4C6,
    ROUTER_4C6,
) = await asyncio.gather(

    GENERAL_LLM_TASK_4C6,

    ROUTER_TASK_4C6,
)


BASELINE_ROUTER_WALL_4C6_S = (

    time.perf_counter()
    -
    BASELINE_ROUTER_START_4C6
)


# ============================================================
# 4C.6Y.9 — BASELINE RESULT
# ============================================================

GENERAL_LLM_ANSWER_4C6 = (
    GENERAL_LLM_4C6[
        "answer"
    ]
)


GENERAL_LLM_LATENCY_4C6_S = (
    GENERAL_LLM_4C6[
        "elapsed_s"
    ]
)


GENERAL_LLM_RAG_EXECUTED_4C6 = False

GENERAL_LLM_MCP_EXECUTED_4C6 = False


# ============================================================
# 4C.6Y.10 — GRANITE ROUTER RESULT
# ============================================================

RESPONSE_MODE_4C6 = (
    ROUTER_4C6[
        "response_mode"
    ]
)


ROUTER_REASON_4C6 = (
    ROUTER_4C6[
        "reason"
    ]
)


ROUTER_LATENCY_4C6_S = (
    ROUTER_4C6[
        "elapsed_s"
    ]
)


ROUTER_OUTPUT_PASS_4C6 = (

    RESPONSE_MODE_4C6
    in
    VALID_RESPONSE_MODES_4C6W
)


if not ROUTER_OUTPUT_PASS_4C6:

    raise RuntimeError(
        f"Granite returned an unsupported response mode: "
        f"{RESPONSE_MODE_4C6}"
    )


print(
    f"         Baseline Model     → "
    f"{LLM_MODEL}"
)


print(
    f"         Baseline Latency   → "
    f"{GENERAL_LLM_LATENCY_4C6_S:.3f} s"
)


print()


print(
    f"         Knowledge Router   → "
    f"{JUDGE_MODEL}"
)


print(
    f"         Response Mode      → "
    f"{RESPONSE_MODE_4C6}"
)


print(
    f"         Router Latency     → "
    f"{ROUTER_LATENCY_4C6_S:.3f} s"
)


print(
    f"         Parallel Wall      → "
    f"{BASELINE_ROUTER_WALL_4C6_S:.3f} s"
)


# ============================================================
# 4C.6Y.11 — INITIALIZE CONDITIONAL RUNTIME STATE
# ============================================================

RAG_EXECUTED_4C6 = False

MCP_EXECUTED_4C6 = False

HYBRID_EXECUTED_4C6 = False

LIVE_EXTERNAL_EXECUTED_4C6 = False
LIVE_EXTERNAL_RETRIEVAL_4C6_S = 0.0

LOCAL_RUNTIME_USED_4C6Y = False
LOCAL_RUNTIME_KIND_4C6Y = None
LOCAL_RUNTIME_TIMEZONE_4C6Y = None
LOCAL_RUNTIME_TIMESTAMP_4C6Y = None

CLAIM_JUDGE_EXECUTED_4C6 = False

GRANITE_JUDGE_4C6_S = 0.0
GRANITE_EXECUTION_4C6 = None

GEMMA_ONLY_EXTRACTION_4C6 = None
GEMMA_RAG_MCP_EXTRACTION_4C6 = None

MODULE4_GEMMA_ONLY_CLAIMS_4C6 = []
MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6 = []

GEMMA_ONLY_ASSESSMENT_4C6 = None
GEMMA_RAG_MCP_ASSESSMENT_4C6 = None

GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6 = None
GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6 = None

GEMMA_ONLY_RISK_LABEL_4C6 = None
GEMMA_RAG_MCP_RISK_LABEL_4C6 = None
ESTIMATED_RISK_DIFFERENCE_PCT_4C6 = None

GRANITE_FINISH_PASS_4C6 = True
JUDGE_ONE_TO_ONE_PASS_4C6 = True
JUDGE_REASON_PASS_4C6 = True
ESTIMATED_HALLUCINATION_RISK_PASS_4C6 = True
JUDGE_EXECUTION_PASS_4C6 = True


RAG_CANDIDATES_4C6 = 0

MCP_CANDIDATES_4C6 = 0


RAG_RETRIEVAL_4C6_S = 0.0

MCP_RETRIEVAL_4C6_S = 0.0

HYBRID_RETRIEVAL_4C6_S = 0.0


EVIDENCE_4C6 = []

CONTEXT_4C6 = ""

CONTEXT_CHARS_4C6 = 0


EVIDENCE_SCAN_LATENCY_4C6_MS = 0.0

EVIDENCE_INJECTION_HITS_4C6 = []


TELECOM_EVIDENCE_USED_4C6 = False


SELECTED_RETRIEVAL_MODE_4C6 = None

MODEL_DRIVEN_SEARCH_COUNT_4C6 = 0
MODEL_DRIVEN_TOOL_REQUESTS_4C6 = 0
MODEL_DRIVEN_LLM_CALLS_4C6 = 0
MODEL_DRIVEN_SEARCH_QUERIES_4C6 = []
MODEL_DRIVEN_ROUND_TRACES_4C6 = []
MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S = 0.0
MODEL_DRIVEN_INPUT_TOKENS_4C6 = 0
MODEL_DRIVEN_OUTPUT_TOKENS_4C6 = 0


# ============================================================
# 4C.6Y.12 — GATE 3: CONDITIONAL KNOWLEDGE ACCESS
# ============================================================

print()
print(
    "GATE 3 — CONDITIONAL KNOWLEDGE ACCESS"
)


# ============================================================
# PATH A — STABLE GENERAL KNOWLEDGE
# ============================================================

if (
    RESPONSE_MODE_4C6
    ==
    GENERAL_KNOWLEDGE_FALLBACK_MODE
):

    print(
        "         Knowledge Scope    → STABLE GENERAL KNOWLEDGE"
    )
    print(
        "         Connected Corpus   → SKIPPED"
    )
    print(
        "         Open-WebSearch CLI    → SKIPPED"
    )
    print(
        "         Comparative Judge  → NOT APPLICABLE"
    )
    print()

    FLEXIBLE_4C6 = await asyncio.to_thread(
        run_4c_general_fallback,
        SANITIZED_QUESTION_4C6,
    )

    FLEXIBLE_ANSWER_4C6 = (
        FLEXIBLE_4C6[
            "answer"
        ]
    )

    FLEXIBLE_GENERATION_4C6_S = (
        FLEXIBLE_4C6[
            "elapsed_s"
        ]
    )

    TELECOM_EVIDENCE_USED_4C6 = False
    RISK_APPLICABLE_4C6 = False

    RISK_DISPLAY_4C6 = (
        "N/A — no connected grounded evidence "
        "was used in General-Knowledge Fallback mode."
    )


# ============================================================
# PATH B — LIVE EXTERNAL GROUNDED
# ============================================================
#
# 6Y FINAL LIVE-SCOPE SUBROUTING
#
#   direct current date/time
#       -> deterministic local runtime tool
#
#   other changing/current public facts
#       -> Open-WebSearch adaptive grounding
# ============================================================

elif (
    RESPONSE_MODE_4C6
    ==
    LIVE_EXTERNAL_GROUNDED_MODE
):

    OPEN_WEBSEARCH_OPERATION_COUNT_4C6W = 0

    LOCAL_RUNTIME_RESULT_4C6Y = (
        run_local_datetime_tool_4c6y(
            SANITIZED_QUESTION_4C6
        )
    )

    print(
        "         Knowledge Scope    → LIVE EXTERNAL"
    )
    print(
        "         Connected Corpus   → SKIPPED"
    )

    # --------------------------------------------------------
    # PATH B1 — DETERMINISTIC LOCAL DATE/TIME
    # --------------------------------------------------------
    if LOCAL_RUNTIME_RESULT_4C6Y is not None:

        LOCAL_RUNTIME_USED_4C6Y = True
        LOCAL_RUNTIME_KIND_4C6Y = (
            LOCAL_RUNTIME_RESULT_4C6Y[
                "kind"
            ]
        )
        LOCAL_RUNTIME_TIMEZONE_4C6Y = (
            LOCAL_RUNTIME_RESULT_4C6Y[
                "timezone"
            ]
        )
        LOCAL_RUNTIME_TIMESTAMP_4C6Y = (
            LOCAL_RUNTIME_RESULT_4C6Y[
                "timestamp_iso"
            ]
        )

        print(
            "         Live Tool          → LOCAL DATE/TIME RUNTIME"
        )
        print(
            "         Open-WebSearch     → SKIPPED"
        )
        print(
            "         External Ops       → 0"
        )
        print(
            "         Gemma Planner      → SKIPPED"
        )
        print(
            "         Sufficiency Loop   → SKIPPED"
        )
        print(
            "         Frozen 4B Judge    → SKIPPED (telecom contract preserved)"
        )

        print()
        print(
            "STEP 4 — Local Runtime Date/Time Response..."
        )

        FLEXIBLE_ANSWER_4C6 = (
            LOCAL_RUNTIME_RESULT_4C6Y[
                "answer"
            ]
        )

        FLEXIBLE_4C6 = {
            "answer": FLEXIBLE_ANSWER_4C6,
            "elapsed_s": 0.0,
            "tool_loop_wall_s": 0.0,
            "search_count": 0,
            "external_operation_count": 0,
            "search_queries": [],
            "round_traces": [],
            "llm_calls": 0,
            "input_tokens": 0,
            "output_tokens": 0,
            "final_evidence": [],
            "final_context": "",
            "live_retrieval_s": 0.0,
            "evidence_scan_ms": 0.0,
            "evidence_security_hits": [],
            "runtime_tool": "LOCAL_DATE_TIME",
            "runtime_kind": LOCAL_RUNTIME_KIND_4C6Y,
            "runtime_timezone": LOCAL_RUNTIME_TIMEZONE_4C6Y,
            "runtime_timestamp_iso": LOCAL_RUNTIME_TIMESTAMP_4C6Y,
            "stop_reason": "LOCAL_RUNTIME_DATE_TIME",
        }

        FLEXIBLE_GENERATION_4C6_S = 0.0

        LIVE_EXTERNAL_EXECUTED_4C6 = True
        LIVE_EXTERNAL_RETRIEVAL_4C6_S = 0.0

        MODEL_DRIVEN_SEARCH_COUNT_4C6 = 0
        MODEL_DRIVEN_SEARCH_QUERIES_4C6 = []
        MODEL_DRIVEN_ROUND_TRACES_4C6 = []
        MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S = 0.0
        MODEL_DRIVEN_LLM_CALLS_4C6 = 0
        MODEL_DRIVEN_INPUT_TOKENS_4C6 = 0
        MODEL_DRIVEN_OUTPUT_TOKENS_4C6 = 0

        EVIDENCE_4C6 = []
        CONTEXT_4C6 = ""
        CONTEXT_CHARS_4C6 = 0

        EVIDENCE_SCAN_LATENCY_4C6_MS = 0.0
        EVIDENCE_INJECTION_HITS_4C6 = []

        TELECOM_EVIDENCE_USED_4C6 = False

        RISK_APPLICABLE_4C6 = False
        RISK_DISPLAY_4C6 = (
            "N/A — deterministic local runtime utility; "
            "connected-corpus comparative judge not applicable."
        )

        print(
            f"         Runtime Tool       → LOCAL_DATE_TIME"
        )
        print(
            f"         Timezone           → "
            f"{LOCAL_RUNTIME_TIMEZONE_4C6Y}"
        )
        print(
            f"         Runtime Timestamp  → "
            f"{LOCAL_RUNTIME_TIMESTAMP_4C6Y}"
        )
        print(
            "STEP 4 — Local Runtime Response Complete"
        )

    # --------------------------------------------------------
    # PATH B2 — OPEN-WEBSEARCH
    # --------------------------------------------------------
    else:

        print(
            "         Live Tool          → OPEN-WEBSEARCH"
        )
        print(
            "         Open-WebSearch     → ACTIVATED"
        )
        print(
            "         Search Policy      → 1 required / 2 if needed / 3 exceptional"
        )
        print(
            "         External Ops       → HARD MAX 3"
        )
        print(
            "         Query Formulation  → GEMMA PLANNER"
        )
        print(
            "         Sufficiency        → SAME LLM / REQUIREMENT-BOUNDED"
        )
        print(
            "         Frozen 4B Judge    → SKIPPED (telecom contract preserved)"
        )

        print()
        print(
            "STEP 4 — Starting Live External Adaptive Search + Generation..."
        )

        FLEXIBLE_4C6 = await run_live_external_grounded_4c6w(
            SANITIZED_QUESTION_4C6
        )

        FLEXIBLE_ANSWER_4C6 = (
            FLEXIBLE_4C6[
                "answer"
            ]
        )

        FLEXIBLE_GENERATION_4C6_S = (
            FLEXIBLE_4C6[
                "elapsed_s"
            ]
        )

        LIVE_EXTERNAL_EXECUTED_4C6 = True

        MODEL_DRIVEN_SEARCH_COUNT_4C6 = (
            FLEXIBLE_4C6[
                "search_count"
            ]
        )
        OPEN_WEBSEARCH_OPERATION_COUNT_4C6W = int(
            FLEXIBLE_4C6.get(
                "external_operation_count",
                OPEN_WEBSEARCH_OPERATION_COUNT_4C6W,
            )
        )
        MODEL_DRIVEN_SEARCH_QUERIES_4C6 = list(
            FLEXIBLE_4C6[
                "search_queries"
            ]
        )
        MODEL_DRIVEN_ROUND_TRACES_4C6 = list(
            FLEXIBLE_4C6[
                "round_traces"
            ]
        )
        MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S = (
            FLEXIBLE_4C6[
                "tool_loop_wall_s"
            ]
        )
        MODEL_DRIVEN_LLM_CALLS_4C6 = (
            FLEXIBLE_4C6[
                "llm_calls"
            ]
        )
        MODEL_DRIVEN_INPUT_TOKENS_4C6 = (
            FLEXIBLE_4C6[
                "input_tokens"
            ]
        )
        MODEL_DRIVEN_OUTPUT_TOKENS_4C6 = (
            FLEXIBLE_4C6[
                "output_tokens"
            ]
        )

        EVIDENCE_4C6 = list(
            FLEXIBLE_4C6[
                "final_evidence"
            ]
        )
        CONTEXT_4C6 = (
            FLEXIBLE_4C6[
                "final_context"
            ]
        )
        CONTEXT_CHARS_4C6 = len(
            CONTEXT_4C6
        )

        LIVE_EXTERNAL_RETRIEVAL_4C6_S = float(
            FLEXIBLE_4C6[
                "live_retrieval_s"
            ]
        )

        EVIDENCE_SCAN_LATENCY_4C6_MS = float(
            FLEXIBLE_4C6[
                "evidence_scan_ms"
            ]
        )
        EVIDENCE_INJECTION_HITS_4C6 = list(
            FLEXIBLE_4C6[
                "evidence_security_hits"
            ]
        )

        TELECOM_EVIDENCE_USED_4C6 = False

        RISK_APPLICABLE_4C6 = False
        RISK_DISPLAY_4C6 = (
            "N/A — frozen 4B comparative judge is reserved "
            "for connected technical-corpus evidence."
        )

        print()
        print(
            f"STEP 4 — Live External Grounded Response Complete → "
            f"{MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S:.3f} s wall"
        )
        print(
            f"         Live Searches       → "
            f"{MODEL_DRIVEN_SEARCH_COUNT_4C6}"
        )
        print(
            f"         Open-WebSearch Calls     → "
            f"{OPEN_WEBSEARCH_OPERATION_COUNT_4C6W}/"
            f"{OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W}"
        )
        print(
            f"         Final Evidence      → "
            f"{len(EVIDENCE_4C6)}"
        )


# ============================================================
# PATH C — CONNECTED TECHNICAL CORPUS
# ============================================================

elif (
    RESPONSE_MODE_4C6
    ==
    TELECOM_GROUNDED_MODE
):

    print(
        "         Knowledge Scope     → CONNECTED TECHNICAL CORPUS"
    )

    print(
        "         Retrieval Control   → LLM-SELECTED ARCHITECTURE / PYTHON-EXECUTED"
    )

    print(
        "         Available Modes     → RAG_ONLY / MCP_ONLY / HYBRID"
    )

    print(
        "         Search Policy       → 1 required / 2 if needed / 3 exceptional"
    )

    print(
        "         Mode + Query        → LLM PLANNER"
    )

    print(
        "         Sufficiency         → SAME LLM / REQUIREMENT-BOUNDED"
    )

    print(
        "         Follow-up Query     → LLM WHEN REQUIREMENT IS MISSING"
    )

    print()
    print(
        "STEP 4 — Starting Natural-Routing Adaptive Retrieval + Generation..."
    )

    FLEXIBLE_4C6 = await run_module4_aligned_routed_grounded_4c6w(
        SANITIZED_QUESTION_4C6
    )

    FLEXIBLE_ANSWER_4C6 = FLEXIBLE_4C6["answer"]
    FLEXIBLE_GENERATION_4C6_S = FLEXIBLE_4C6["elapsed_s"]

    MODEL_DRIVEN_SEARCH_COUNT_4C6 = FLEXIBLE_4C6["search_count"]
    MODEL_DRIVEN_TOOL_REQUESTS_4C6 = FLEXIBLE_4C6["tool_requests"]
    MODEL_DRIVEN_LLM_CALLS_4C6 = FLEXIBLE_4C6["llm_calls"]
    MODEL_DRIVEN_SEARCH_QUERIES_4C6 = list(
        FLEXIBLE_4C6["search_queries"]
    )
    MODEL_DRIVEN_ROUND_TRACES_4C6 = list(
        FLEXIBLE_4C6["round_traces"]
    )
    MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S = FLEXIBLE_4C6[
        "tool_loop_wall_s"
    ]
    MODEL_DRIVEN_INPUT_TOKENS_4C6 = FLEXIBLE_4C6["input_tokens"]
    MODEL_DRIVEN_OUTPUT_TOKENS_4C6 = FLEXIBLE_4C6["output_tokens"]

    SELECTED_RETRIEVAL_MODE_4C6 = FLEXIBLE_4C6["selected_mode"]

    RAG_EXECUTED_4C6 = (
        MODEL_DRIVEN_SEARCH_COUNT_4C6 > 0
        and SELECTED_RETRIEVAL_MODE_4C6 in {"RAG_ONLY", "HYBRID"}
    )
    MCP_EXECUTED_4C6 = (
        MODEL_DRIVEN_SEARCH_COUNT_4C6 > 0
        and SELECTED_RETRIEVAL_MODE_4C6 in {"MCP_ONLY", "HYBRID"}
    )
    HYBRID_EXECUTED_4C6 = (
        MODEL_DRIVEN_SEARCH_COUNT_4C6 > 0
        and SELECTED_RETRIEVAL_MODE_4C6 == "HYBRID"
    )

    RAG_CANDIDATES_4C6 = FLEXIBLE_4C6["rag_candidate_total"]
    MCP_CANDIDATES_4C6 = FLEXIBLE_4C6["mcp_candidate_total"]

    RAG_RETRIEVAL_4C6_S = FLEXIBLE_4C6["rag_retrieval_s"]
    MCP_RETRIEVAL_4C6_S = FLEXIBLE_4C6["mcp_retrieval_s"]
    HYBRID_RETRIEVAL_4C6_S = FLEXIBLE_4C6["hybrid_retrieval_s"]

    EVIDENCE_4C6 = list(FLEXIBLE_4C6["final_evidence"])
    CONTEXT_4C6 = FLEXIBLE_4C6["final_context"]
    CONTEXT_CHARS_4C6 = len(CONTEXT_4C6)

    EVIDENCE_SCAN_LATENCY_4C6_MS = FLEXIBLE_4C6[
        "evidence_scan_ms"
    ]
    EVIDENCE_INJECTION_HITS_4C6 = list(
        FLEXIBLE_4C6["evidence_security_hits"]
    )

    TELECOM_EVIDENCE_USED_4C6 = bool(EVIDENCE_4C6)
    RISK_APPLICABLE_4C6 = True

    RISK_DISPLAY_4C6 = (
        "PENDING — immediate comparative Granite evaluation "
        "will run in this 4C.6W cell."
    )

    print()
    print(
        f"STEP 4 — Adaptive Grounded Response Complete → "
        f"{MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S:.3f} s wall"
    )
    print(
        f"         LLM Calls           → {MODEL_DRIVEN_LLM_CALLS_4C6}"
    )
    print(
        f"         Retrieval Searches  → {MODEL_DRIVEN_SEARCH_COUNT_4C6}"
    )
    print(
        f"         Native Tool Calls    → {MODEL_DRIVEN_TOOL_REQUESTS_4C6} (not used)"
    )
    print(
        f"         Final Evidence      → {len(EVIDENCE_4C6)}"
    )
    print(
        f"         Judge Context       → {CONTEXT_CHARS_4C6:,} chars"
    )

else:

    raise RuntimeError(
        f"Unhandled response mode: "
        f"{RESPONSE_MODE_4C6}"
    )


print(
    f"STEP 4 — Flexible Response Complete → "
    f"{FLEXIBLE_GENERATION_4C6_S:.3f} s LLM time"
)


# ============================================================
# 4C.6Y.13 — CITATION DIAGNOSTIC
# ============================================================

CITATION_PATTERN_4C6 = re.compile(

    r"\[\s*E\d+"
    r"(?:\s*,\s*E\d+)*"
    r"\s*\]",

    flags=re.IGNORECASE,
)


GENERAL_LLM_CITATION_PRESENT_4C6 = (

    CITATION_PATTERN_4C6.search(
        GENERAL_LLM_ANSWER_4C6
    )
    is not None
)


FLEXIBLE_CITATION_PRESENT_4C6 = (

    CITATION_PATTERN_4C6.search(
        FLEXIBLE_ANSWER_4C6
    )
    is not None
)


# ============================================================
# 4C.6Y.14 — GENERIC ROUTE-AWARE VALIDATION
# ============================================================
#
# These checks do NOT assume which route Granite should choose.
#
# They validate that execution behaved consistently with
# Granite's actual semantic routing decision.
#
# ============================================================

BASELINE_ANSWER_PASS_4C6 = (

    isinstance(
        GENERAL_LLM_ANSWER_4C6,
        str,
    )

    and

    len(
        GENERAL_LLM_ANSWER_4C6.strip()
    )
    >=
    100
)


# Route-aware answer validation.
#
# Connected technical-corpus answers are generally substantive engineering
# responses, so preserve the historical >=100 character check there.
#
# Stable-general and live-current questions may correctly produce a concise
# factual answer (for example, a person's name). Requiring 100 characters
# would falsely fail a valid deployment response.
if (
    RESPONSE_MODE_4C6
    ==
    TELECOM_GROUNDED_MODE
):
    FLEXIBLE_MIN_ANSWER_CHARS_4C6 = 100
else:
    FLEXIBLE_MIN_ANSWER_CHARS_4C6 = 20


FLEXIBLE_ANSWER_PASS_4C6 = (

    isinstance(
        FLEXIBLE_ANSWER_4C6,
        str,
    )

    and

    len(
        FLEXIBLE_ANSWER_4C6.strip()
    )
    >=
    FLEXIBLE_MIN_ANSWER_CHARS_4C6
)


BASELINE_ISOLATION_PASS_4C6 = (

    GENERAL_LLM_RAG_EXECUTED_4C6
    is False

    and

    GENERAL_LLM_MCP_EXECUTED_4C6
    is False
)


# ============================================================
# GENERAL-KNOWLEDGE ROUTE VALIDATION
# ============================================================

if (
    RESPONSE_MODE_4C6
    ==
    GENERAL_KNOWLEDGE_FALLBACK_MODE
):

    CAVEAT_PRESENT_4C6 = any(
        caveat_text
        in
        FLEXIBLE_ANSWER_4C6

        for caveat_text in (
            "Outside Telecom Knowledge Domain",
            "Outside Connected Knowledge Scope",
        )
    )


    ROUTE_EXECUTION_PASS_4C6 = all(

        [
            SELECTED_RETRIEVAL_MODE_4C6
            is None,

            RAG_EXECUTED_4C6
            is False,

            MCP_EXECUTED_4C6
            is False,

            HYBRID_EXECUTED_4C6
            is False,

            len(
                EVIDENCE_4C6
            )
            ==
            0,

            CONTEXT_CHARS_4C6
            ==
            0,

            TELECOM_EVIDENCE_USED_4C6
            is False,

            CLAIM_JUDGE_EXECUTED_4C6
            is False,

            RISK_APPLICABLE_4C6
            is False,

            CAVEAT_PRESENT_4C6,

            FLEXIBLE_CITATION_PRESENT_4C6
            is False,
        ]
    )



# ============================================================
# LIVE-EXTERNAL ROUTE VALIDATION
# ============================================================

elif (
    RESPONSE_MODE_4C6
    ==
    LIVE_EXTERNAL_GROUNDED_MODE
):

    CAVEAT_PRESENT_4C6 = False

    if LOCAL_RUNTIME_USED_4C6Y:

        ROUTE_EXECUTION_PASS_4C6 = all(
            [
                LIVE_EXTERNAL_EXECUTED_4C6
                is True,

                SELECTED_RETRIEVAL_MODE_4C6
                is None,

                RAG_EXECUTED_4C6
                is False,

                MCP_EXECUTED_4C6
                is False,

                HYBRID_EXECUTED_4C6
                is False,

                len(EVIDENCE_4C6)
                ==
                0,

                CONTEXT_CHARS_4C6
                ==
                0,

                len(EVIDENCE_INJECTION_HITS_4C6)
                ==
                0,

                CLAIM_JUDGE_EXECUTED_4C6
                is False,

                RISK_APPLICABLE_4C6
                is False,

                FLEXIBLE_CITATION_PRESENT_4C6
                is False,

                MODEL_DRIVEN_SEARCH_COUNT_4C6
                ==
                0,

                OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
                ==
                0,

                LOCAL_RUNTIME_TIMESTAMP_4C6Y
                is not None,
            ]
        )

    else:

        ROUTE_EXECUTION_PASS_4C6 = all(
            [
                LIVE_EXTERNAL_EXECUTED_4C6
                is True,

                SELECTED_RETRIEVAL_MODE_4C6
                is None,

                RAG_EXECUTED_4C6
                is False,

                MCP_EXECUTED_4C6
                is False,

                HYBRID_EXECUTED_4C6
                is False,

                len(EVIDENCE_4C6)
                >
                0,

                CONTEXT_CHARS_4C6
                >
                0,

                len(EVIDENCE_INJECTION_HITS_4C6)
                ==
                0,

                CLAIM_JUDGE_EXECUTED_4C6
                is False,

                RISK_APPLICABLE_4C6
                is False,

                FLEXIBLE_CITATION_PRESENT_4C6
                is True,

                MODEL_DRIVEN_SEARCH_COUNT_4C6
                >=
                1,

                MODEL_DRIVEN_SEARCH_COUNT_4C6
                <=
                MODULE4_MAX_MODEL_SEARCHES_4C6W,

                OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
                >=
                1,

                OPEN_WEBSEARCH_OPERATION_COUNT_4C6W
                <=
                OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W,
            ]
        )


# ============================================================
# TELECOM-GROUNDED ROUTE VALIDATION
# ============================================================

elif (
    RESPONSE_MODE_4C6
    ==
    TELECOM_GROUNDED_MODE
):

    CAVEAT_PRESENT_4C6 = (

        "Outside Connected Knowledge Scope"
        in
        FLEXIBLE_ANSWER_4C6
    )


    MODE_EXECUTION_MATCH_PASS_4C6 = (
        (
            SELECTED_RETRIEVAL_MODE_4C6 == "RAG_ONLY"
            and RAG_EXECUTED_4C6 is True
            and MCP_EXECUTED_4C6 is False
            and HYBRID_EXECUTED_4C6 is False
        )
        or
        (
            SELECTED_RETRIEVAL_MODE_4C6 == "MCP_ONLY"
            and RAG_EXECUTED_4C6 is False
            and MCP_EXECUTED_4C6 is True
            and HYBRID_EXECUTED_4C6 is False
        )
        or
        (
            SELECTED_RETRIEVAL_MODE_4C6 == "HYBRID"
            and RAG_EXECUTED_4C6 is True
            and MCP_EXECUTED_4C6 is True
            and HYBRID_EXECUTED_4C6 is True
        )
    )

    ROUTE_EXECUTION_PASS_4C6 = all(

        [
            SELECTED_RETRIEVAL_MODE_4C6
            in
            MODULE4_ALLOWED_RETRIEVAL_MODES_4C6W,

            MODE_EXECUTION_MATCH_PASS_4C6,

            len(
                EVIDENCE_4C6
            )
            >
            0,

            CONTEXT_CHARS_4C6
            >
            0,

            len(
                EVIDENCE_INJECTION_HITS_4C6
            )
            ==
            0,

            TELECOM_EVIDENCE_USED_4C6
            is True,

            RISK_APPLICABLE_4C6
            is True,

            CAVEAT_PRESENT_4C6
            is False,

            FLEXIBLE_CITATION_PRESENT_4C6
            is True,

            MODEL_DRIVEN_SEARCH_COUNT_4C6
            >=
            1,

            MODEL_DRIVEN_SEARCH_COUNT_4C6
            <=
            MODULE4_MAX_MODEL_SEARCHES_4C6W,

            len(MODEL_DRIVEN_SEARCH_QUERIES_4C6)
            ==
            MODEL_DRIVEN_SEARCH_COUNT_4C6,
        ]
    )


else:

    ROUTE_EXECUTION_PASS_4C6 = False



# ============================================================
# 4C.6Y.15 — IMMEDIATE ROUTE-AWARE COMPARATIVE JUDGE
# ============================================================
#
# Grounded route:
#   - extract fixed claims from the two EXISTING answers
#   - reuse the EXISTING Hybrid evidence/context
#   - execute ONE Granite comparative request
#   - validate exact 1:1 claim IDs
#   - calculate evidence-relative risk deterministically
#
# General fallback route:
#   - skip the judge
#   - risk remains N/A
#
# No Gemma generation or retrieval is repeated here.
# ============================================================

if (
    RESPONSE_MODE_4C6
    ==
    TELECOM_GROUNDED_MODE
):

    print()

    print(
        "STEP 5 — Preparing Fixed Claims for Comparative Granite Evaluation..."
    )


    GEMMA_ONLY_EXTRACTION_4C6 = (
        extract_refined_claims(
            GENERAL_LLM_ANSWER_4C6,
            "A",
        )
    )


    GEMMA_RAG_MCP_EXTRACTION_4C6 = (
        extract_refined_claims(
            FLEXIBLE_ANSWER_4C6,
            "B",
        )
    )


    MODULE4_GEMMA_ONLY_CLAIMS_4C6 = (
        GEMMA_ONLY_EXTRACTION_4C6[
            "claims"
        ]
    )


    MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6 = (
        GEMMA_RAG_MCP_EXTRACTION_4C6[
            "claims"
        ]
    )


    VALID_EVIDENCE_IDS_4C6 = (
        build_valid_evidence_ids(
            EVIDENCE_4C6
        )
    )


    print(
        f"         Gemma 4 Only Claims        → "
        f"{len(MODULE4_GEMMA_ONLY_CLAIMS_4C6)}"
    )


    print(
        f"         Gemma 4 + RAG + MCP Claims → "
        f"{len(MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6)}"
    )


    print()


    print(
        "STEP 6 — Starting ONE Granite Fixed Claim-by-Claim "
        "Comparative Assessment..."
    )


    GRANITE_EXECUTION_4C6 = (
        await asyncio.to_thread(
            run_granite_comparative_judge,
            SANITIZED_QUESTION_4C6,
            CONTEXT_4C6,
            MODULE4_GEMMA_ONLY_CLAIMS_4C6,
            MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6,
        )
    )


    GRANITE_JUDGE_4C6_S = (
        GRANITE_EXECUTION_4C6[
            "elapsed_s"
        ]
    )


    print(
        f"STEP 6 — Granite Comparative Assessment Complete → "
        f"{GRANITE_JUDGE_4C6_S:.3f} s"
    )


    print(
        f"         Claims evaluated    → "
        f"{GRANITE_EXECUTION_4C6['total_claims']}"
    )


    print(
        f"         Max output tokens   → "
        f"{GRANITE_EXECUTION_4C6['max_tokens']}"
    )


    GEMMA_ONLY_ASSESSMENT_4C6 = (
        validate_and_summarize_judgment(
            original_claims=
                MODULE4_GEMMA_ONLY_CLAIMS_4C6,
            judge_group_payload=
                GRANITE_EXECUTION_4C6[
                    "payload"
                ][
                    "gemma4_only"
                ],
            valid_evidence_ids=
                VALID_EVIDENCE_IDS_4C6,
        )
    )


    GEMMA_RAG_MCP_ASSESSMENT_4C6 = (
        validate_and_summarize_judgment(
            original_claims=
                MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6,
            judge_group_payload=
                GRANITE_EXECUTION_4C6[
                    "payload"
                ][
                    "gemma4_rag_mcp"
                ],
            valid_evidence_ids=
                VALID_EVIDENCE_IDS_4C6,
        )
    )


    GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6 = (
        GEMMA_ONLY_ASSESSMENT_4C6[
            "estimated_hallucination_risk_pct"
        ]
    )


    GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6 = (
        GEMMA_RAG_MCP_ASSESSMENT_4C6[
            "estimated_hallucination_risk_pct"
        ]
    )


    GEMMA_ONLY_RISK_LABEL_4C6 = (
        hallucination_risk_label(
            GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
        )
    )


    GEMMA_RAG_MCP_RISK_LABEL_4C6 = (
        hallucination_risk_label(
            GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
        )
    )


    ESTIMATED_RISK_DIFFERENCE_PCT_4C6 = (
        GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
        -
        GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
    )


    CLAIM_JUDGE_EXECUTED_4C6 = True


    RISK_DISPLAY_4C6 = (
        f"{GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6:.1f}% "
        f"({GEMMA_RAG_MCP_RISK_LABEL_4C6})"
    )


    GRANITE_FINISH_PASS_4C6 = (
        str(
            GRANITE_EXECUTION_4C6[
                "result"
            ].get(
                "finish_reason",
                "",
            )
        ).strip().lower()
        ==
        "stop"
    )


    JUDGE_ONE_TO_ONE_PASS_4C6 = (
        GEMMA_ONLY_ASSESSMENT_4C6[
            "total_claims"
        ]
        ==
        len(
            MODULE4_GEMMA_ONLY_CLAIMS_4C6
        )

        and

        GEMMA_RAG_MCP_ASSESSMENT_4C6[
            "total_claims"
        ]
        ==
        len(
            MODULE4_GEMMA_RAG_MCP_CLAIMS_4C6
        )
    )


    JUDGE_REASON_PASS_4C6 = (
        GEMMA_ONLY_ASSESSMENT_4C6[
            "empty_reason_count"
        ]
        ==
        0

        and

        GEMMA_RAG_MCP_ASSESSMENT_4C6[
            "empty_reason_count"
        ]
        ==
        0
    )


    ESTIMATED_HALLUCINATION_RISK_PASS_4C6 = (
        0.0
        <=
        GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
        <=
        100.0

        and

        0.0
        <=
        GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6
        <=
        100.0
    )


    JUDGE_EXECUTION_PASS_4C6 = all(
        [
            CLAIM_JUDGE_EXECUTED_4C6,
            GRANITE_FINISH_PASS_4C6,
            JUDGE_ONE_TO_ONE_PASS_4C6,
            JUDGE_REASON_PASS_4C6,
            ESTIMATED_HALLUCINATION_RISK_PASS_4C6,
        ]
    )


else:

    # General-knowledge fallback intentionally has no connected
    # evidence and therefore no evidence-relative judge.
    CLAIM_JUDGE_EXECUTED_4C6 = False

    GRANITE_JUDGE_4C6_S = 0.0

    JUDGE_EXECUTION_PASS_4C6 = (
        RISK_APPLICABLE_4C6
        is False
    )


# ============================================================
# 4C.6Y.16 — COMPLETE RUNTIME VALIDATION
# ============================================================

MODULE4_CELL4C6B_PASS = all(

    [
        PRE_LLM_SECURITY_PASS_4C6,

        ROUTER_OUTPUT_PASS_4C6,

        BASELINE_ANSWER_PASS_4C6,

        FLEXIBLE_ANSWER_PASS_4C6,

        BASELINE_ISOLATION_PASS_4C6,

        ROUTE_EXECUTION_PASS_4C6,

        JUDGE_EXECUTION_PASS_4C6,
    ]
)


# ============================================================
# 4C.6Y.17 — END-TO-END TIMING
# ============================================================

TOTAL_WORKFLOW_4C6_S = (

    time.perf_counter()
    -
    WORKFLOW_START_4C6
)


# ============================================================
# DEPLOYED PATH COMPONENT SUM
# ============================================================
#
# Excludes Gemma 4 Only because that branch is experimental.
#
# GENERAL:
#
#   Guardrail + Router + Fallback Gemma
#
# TELECOM:
#
#   Guardrail + Router + Hybrid Retrieval + Grounded Gemma
#
# ============================================================

DEPLOYED_PATH_COMPONENT_SUM_4C6_S = (

    GUARDRAIL_LATENCY_4C6_S
    +
    ROUTER_LATENCY_4C6_S
    +
    HYBRID_RETRIEVAL_4C6_S
    +
    LIVE_EXTERNAL_RETRIEVAL_4C6_S
    +
    FLEXIBLE_GENERATION_4C6_S
)


# ============================================================
# 4C.6Y.18 — MEMORY
# ============================================================

gc.collect()


RAM_AFTER_4C6_GIB = (

    PROCESS_4C6.memory_info().rss
    /
    (1024 ** 3)
)


RAM_DELTA_4C6_GIB = (

    RAM_AFTER_4C6_GIB
    -
    RAM_BEFORE_4C6_GIB
)


AVAILABLE_RAM_AFTER_4C6_GIB = (

    psutil.virtual_memory().available
    /
    (1024 ** 3)
)


# ============================================================
# 4C.6Y.19 — ANSWER A
# ============================================================

print()

print("=" * 116)
print("ANSWER A — GEMMA 4 ONLY")
print("=" * 116)


print(
    f"Input                       : "
    f"{SANITIZED_QUESTION_4C6}"
)


print(
    f"Model                       : "
    f"{LLM_MODEL}"
)


print(
    "Knowledge Access            : NONE"
)


print(
    f"RAG Executed                : "
    f"{GENERAL_LLM_RAG_EXECUTED_4C6}"
)


print(
    f"MCP Executed                : "
    f"{GENERAL_LLM_MCP_EXECUTED_4C6}"
)


print()

print(
    GENERAL_LLM_ANSWER_4C6
)


# ============================================================
# 4C.6Y.20 — ANSWER B
# ============================================================

print()

print("=" * 116)
print("ANSWER B — FLEXIBLE TELECOM AI")
print("=" * 116)


print(
    f"Input                       : "
    f"{SANITIZED_QUESTION_4C6}"
)


print(
    f"Generator                   : "
    f"{LLM_MODEL}"
)


print(
    f"Response Mode               : "
    f"{RESPONSE_MODE_4C6}"
)


print(
    f"Selected Retrieval Mode     : "
    f"{SELECTED_RETRIEVAL_MODE_4C6}"
)


print(
    f"RAG Executed                : "
    f"{RAG_EXECUTED_4C6}"
)


print(
    f"MCP Executed                : "
    f"{MCP_EXECUTED_4C6}"
)


print(
    f"Hybrid Retrieval Executed   : "
    f"{HYBRID_EXECUTED_4C6}"
)


print(
    f"Connected Corpus Evidence       : "
    f"{TELECOM_EVIDENCE_USED_4C6}"
)


print(
    f"Evidence Items              : "
    f"{len(EVIDENCE_4C6)}"
)


print(
    f"Grounded Citation Present   : "
    f"{FLEXIBLE_CITATION_PRESENT_4C6}"
)


print(
    f"Estimated Hallucination Risk: "
    f"{RISK_DISPLAY_4C6}"
)


print()

# ------------------------------------------------------------
# USER-FACING KNOWLEDGE-SCOPE / GROUNDING NOTICE
# ------------------------------------------------------------
#
# Preserve an explicit warning whenever the question is outside the
# connected telecom technical corpus. This makes it clear which knowledge
# source is being used before the user reads the answer.
#
if (
    RESPONSE_MODE_4C6
    ==
    LIVE_EXTERNAL_GROUNDED_MODE
):
    if LOCAL_RUNTIME_USED_4C6Y:
        print(
            "⚠ GROUNDING NOTICE — This is not a connected telecom-domain "
            "knowledge question. The response below is supplied by a "
            "deterministic local runtime date/time tool rather than the "
            "connected telecom corpus."
        )
    else:
        print(
            "⚠ GROUNDING NOTICE — This is not a connected telecom-domain "
            "knowledge question. The response below is grounded using live "
            "external web evidence rather than the connected telecom corpus."
        )
    print()

elif (
    RESPONSE_MODE_4C6
    ==
    GENERAL_KNOWLEDGE_FALLBACK_MODE
):
    print(
        "⚠ GROUNDING NOTICE — This is not a connected telecom-domain "
        "knowledge question. The response below uses stable general "
        "model knowledge and is not grounded in the connected telecom corpus."
    )
    print()


print(
    FLEXIBLE_ANSWER_4C6
)


# ============================================================
# 4C.6Y.21 — RETRIEVED EVIDENCE PREVIEW
# ============================================================

if EVIDENCE_4C6:

    print()

    print("=" * 116)

    if (
        RESPONSE_MODE_4C6
        ==
        LIVE_EXTERNAL_GROUNDED_MODE
    ):
        print("LIVE EXTERNAL EVIDENCE — PREVIEW")
    else:
        print("TELECOM EVIDENCE — PREVIEW")

    print("=" * 116)


    for index, item in enumerate(
        EVIDENCE_4C6,
        start=1,
    ):

        evidence_id = (
            item.get(
                "evidence_id"
            )
            or
            item.get(
                "presentation_id"
            )
            or
            f"E{index}"
        )


        retriever = (
            item.get(
                "retriever"
            )
            or
            item.get(
                "retrieval_system"
            )
            or
            item.get(
                "source_system"
            )
            or
            "UNKNOWN"
        )


        title = (
            item.get(
                "title"
            )
            or
            item.get(
                "document_title"
            )
            or
            ""
        )


        text = str(
            item.get(
                "text",
                "",
            )
        )


        preview = (
            text[:300]
            .replace(
                "\n",
                " ",
            )
            .strip()
        )


        print()

        print(
            f"[{evidence_id}] "
            f"{retriever} | {title}"
        )


        print(
            f"Preview: "
            f"{preview}"
        )



# ============================================================
# 4C.6Y.22 — IMMEDIATE GRANITE JUDGE RESULT
# ============================================================

if CLAIM_JUDGE_EXECUTED_4C6:

    print()

    print("=" * 116)
    print("ESTIMATED HALLUCINATION RISK")
    print("=" * 116)


    print()

    print("GEMMA 4 ONLY")
    print("-" * 116)


    print(
        f"Claims Evaluated            : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['total_claims']}"
    )


    print(
        f"Supported                   : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['supported']}"
    )


    print(
        f"Partially Supported         : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['partially_supported']}"
    )


    print(
        f"Unsupported                 : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['unsupported']}"
    )


    print(
        f"Contradicted                : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['contradicted']}"
    )


    print(
        f"{RISK_DISPLAY_LABEL:<28}: "
        f"{GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6:.1f}% "
        f"({GEMMA_ONLY_RISK_LABEL_4C6})"
    )


    print()

    print("GEMMA 4 + RAG + MCP")
    print("-" * 116)


    print(
        f"Claims Evaluated            : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['total_claims']}"
    )


    print(
        f"Supported                   : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['supported']}"
    )


    print(
        f"Partially Supported         : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['partially_supported']}"
    )


    print(
        f"Unsupported                 : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['unsupported']}"
    )


    print(
        f"Contradicted                : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['contradicted']}"
    )


    print(
        f"{RISK_DISPLAY_LABEL:<28}: "
        f"{GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6:.1f}% "
        f"({GEMMA_RAG_MCP_RISK_LABEL_4C6})"
    )


    print()


    print(
        f"Estimated Risk Difference   : "
        f"{ESTIMATED_RISK_DIFFERENCE_PCT_4C6:+.1f} percentage points"
    )


    print()


    print(
        "Interpretation              : "
        "Evidence-based risk estimate; unsupported does not "
        "necessarily mean factually false."
    )


    print()

    print("=" * 116)
    print("GRANITE 4.2 — CLAIM-BY-CLAIM ASSESSMENT")
    print("=" * 116)


    def display_claim_assessment_4c6(
        title,
        assessment,
    ):

        print()

        print(title)

        print("-" * 116)


        for item in assessment[
            "claims"
        ]:

            evidence_text = (
                ", ".join(
                    item[
                        "evidence"
                    ]
                )
                if item[
                    "evidence"
                ]
                else
                "—"
            )


            print(
                f"{item['claim_id']} | "
                f"{item['status'].upper()} | "
                f"Evidence: {evidence_text}"
            )


            print(
                f"Claim : "
                f"{item['claim']}"
            )


            print(
                f"Reason: "
                f"{item['reason'] or '—'}"
            )


            print()


    display_claim_assessment_4c6(
        GEMMA_ONLY_LABEL,
        GEMMA_ONLY_ASSESSMENT_4C6,
    )


    display_claim_assessment_4c6(
        GEMMA_RAG_MCP_LABEL,
        GEMMA_RAG_MCP_ASSESSMENT_4C6,
    )


    print()

    print("JUDGE OUTPUT AUDIT")
    print("-" * 116)


    print(
        f"Gemma 4 Only Invalid E# Refs         : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['invalid_evidence_references']}"
    )


    print(
        f"Gemma 4 + RAG + MCP Invalid E# Refs : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['invalid_evidence_references']}"
    )


    print(
        f"Gemma 4 Only Unsupported + E#       : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['unsupported_with_evidence']}"
    )


    print(
        f"Gemma 4 + RAG + MCP Unsupported + E#: "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['unsupported_with_evidence']}"
    )


    print(
        f"Gemma 4 Only Empty Reasons           : "
        f"{GEMMA_ONLY_ASSESSMENT_4C6['empty_reason_count']}"
    )


    print(
        f"Gemma 4 + RAG + MCP Empty Reasons   : "
        f"{GEMMA_RAG_MCP_ASSESSMENT_4C6['empty_reason_count']}"
    )


else:

    print()

    print("=" * 116)
    print("ESTIMATED HALLUCINATION RISK")
    print("=" * 116)

    print(
        "Granite Comparative Judge   : SKIPPED"
    )

    print(
        "Estimated Hallucination Risk: N/A — no connected grounded evidence."
    )


# ============================================================
# 4C.6Y.23 — GENERALIZED EXECUTION SUMMARY
# ============================================================

print()

print("=" * 116)
print("MODULE 4C.6Y — NATURAL-ROUTING EXECUTION SUMMARY")
print("=" * 116)


print(
    f"Raw Question                : "
    f"{MODULE4_4C6_RAW_PROMPT}"
)


print(
    f"Sanitized Question          : "
    f"{SANITIZED_QUESTION_4C6}"
)


print(
    f"Injection Detected          : "
    f"{INJECTION_DETECTED_4C6}"
)


print(
    f"Evidence Manipulation       : "
    f"{EVIDENCE_MANIPULATION_DETECTED_4C6}"
)


print()


print(
    f"Generator Model             : "
    f"{LLM_MODEL}"
)


print(
    f"Granite Knowledge Router     : "
    f"{JUDGE_MODEL}"
)


print(
    f"Granite Knowledge Route      : "
    f"{RESPONSE_MODE_4C6}"
)


print()


print(
    f"RAG Executed                : "
    f"{RAG_EXECUTED_4C6}"
)


print(
    f"MCP Executed                : "
    f"{MCP_EXECUTED_4C6}"
)


print(
    f"Hybrid Executed             : "
    f"{HYBRID_EXECUTED_4C6}"
)

print(
    f"Live External Scope         : "
    f"{LIVE_EXTERNAL_EXECUTED_4C6}"
)

print(
    f"Live External Search        : "
    f"{bool(LIVE_EXTERNAL_EXECUTED_4C6 and not LOCAL_RUNTIME_USED_4C6Y)}"
)

print(
    f"Local Runtime Date/Time     : "
    f"{LOCAL_RUNTIME_USED_4C6Y}"
)

if LIVE_EXTERNAL_EXECUTED_4C6 and not LOCAL_RUNTIME_USED_4C6Y:
    print(
        f"Open-WebSearch Calls        : "
        f"{OPEN_WEBSEARCH_OPERATION_COUNT_4C6W}/"
        f"{OPEN_WEBSEARCH_OPERATION_LIMIT_4C6W}"
    )


print(
    f"Evidence Retrieved          : "
    f"{len(EVIDENCE_4C6)}"
)


print(
    f"Connected Corpus Evidence       : "
    f"{TELECOM_EVIDENCE_USED_4C6}"
)


print(
    f"Comparative Risk Applicable     : "
    f"{RISK_APPLICABLE_4C6}"
)


print(
    f"Granite Judge Executed      : "
    f"{CLAIM_JUDGE_EXECUTED_4C6}"
)


if RESPONSE_MODE_4C6 in {TELECOM_GROUNDED_MODE, LIVE_EXTERNAL_GROUNDED_MODE}:
    print(
        f"Model-Driven Searches       : "
        f"{MODEL_DRIVEN_SEARCH_COUNT_4C6}"
    )
    print(
        f"Grounded LLM Calls          : "
        f"{MODEL_DRIVEN_LLM_CALLS_4C6}"
    )
    print(
        f"Search Queries              : "
        f"{MODEL_DRIVEN_SEARCH_QUERIES_4C6}"
    )


# ============================================================
# 4C.6Y.24 — LATENCY
# ============================================================

print()

print("LATENCY")
print("-" * 116)


print(
    f"Deterministic Guardrail     : "
    f"{GUARDRAIL_LATENCY_4C6_MS:.3f} ms"
)


print(
    f"Granite Knowledge Router       : "
    f"{ROUTER_LATENCY_4C6_S:.3f} s"
)


print(
    f"Gemma 4 Only Generation     : "
    f"{GENERAL_LLM_LATENCY_4C6_S:.3f} s"
)


print(
    f"Baseline + Router Wall      : "
    f"{BASELINE_ROUTER_WALL_4C6_S:.3f} s"
)


print(
    f"RAG Retrieval               : "
    f"{RAG_RETRIEVAL_4C6_S:.3f} s"
)


print(
    f"MCP Retrieval               : "
    f"{MCP_RETRIEVAL_4C6_S:.3f} s"
)

print(
    f"Open-WebSearch Retrieval     : "
    f"{LIVE_EXTERNAL_RETRIEVAL_4C6_S:.3f} s"
)


print(
    f"Selected Retrieval Wall    : "
    f"{HYBRID_RETRIEVAL_4C6_S:.3f} s"
)


print(
    f"Flexible Generation         : "
    f"{FLEXIBLE_GENERATION_4C6_S:.3f} s"
)


if RESPONSE_MODE_4C6 in {TELECOM_GROUNDED_MODE, LIVE_EXTERNAL_GROUNDED_MODE}:
    print(
        f"Grounded Tool-Loop Wall     : "
        f"{MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S:.3f} s"
    )


print(
    f"Granite Comparative Judge   : "
    f"{GRANITE_JUDGE_4C6_S:.3f} s"
)


print(
    f"Deployed Path Component Sum : "
    f"{DEPLOYED_PATH_COMPONENT_SUM_4C6_S:.3f} s"
)


print(
    f"Full Comparison Cell Wall   : "
    f"{TOTAL_WORKFLOW_4C6_S:.3f} s"
)


# ============================================================
# 4C.6Y.25 — MEMORY
# ============================================================

print()

print("MEMORY")
print("-" * 116)


print(
    f"Process RAM Before           : "
    f"{RAM_BEFORE_4C6_GIB:.3f} GiB"
)


print(
    f"Process RAM After            : "
    f"{RAM_AFTER_4C6_GIB:.3f} GiB"
)


print(
    f"RAM Delta                   : "
    f"{RAM_DELTA_4C6_GIB:+.3f} GiB"
)


print(
    f"System RAM Before           : "
    f"{AVAILABLE_RAM_BEFORE_4C6_GIB:.3f} GiB"
)


print(
    f"System RAM Available        : "
    f"{AVAILABLE_RAM_AFTER_4C6_GIB:.3f} GiB"
)


# ============================================================
# 4C.6Y.26 — FORMAL JUDGE / AUDIT INPUT CONTRACT
# ============================================================
#
# Cell 4C.6Y executes the comparative Granite judge above when
# grounded evidence is available.
#
# This contract preserves the exact artifacts used by that judge
# for auditability and downstream dashboard use.
#
# No generation or retrieval is repeated here.
# ============================================================

JUDGE_INPUT_4C6W = {
    "contract_version": "4C.6W.model_driven_retrieval.integrated_judge.v1",
    "applicable": bool(RISK_APPLICABLE_4C6),
    "response_mode": RESPONSE_MODE_4C6,
    "question": SANITIZED_QUESTION_4C6,
    "gemma4_only_answer": GENERAL_LLM_ANSWER_4C6,
    "grounded_answer": (
        FLEXIBLE_ANSWER_4C6
        if RISK_APPLICABLE_4C6
        else None
    ),
    "evidence": (
        list(EVIDENCE_4C6)
        if RISK_APPLICABLE_4C6
        else []
    ),
    "evidence_context": (
        CONTEXT_4C6
        if RISK_APPLICABLE_4C6
        else ""
    ),
    "evidence_count": (
        len(EVIDENCE_4C6)
        if RISK_APPLICABLE_4C6
        else 0
    ),
    # Use the actual packaged context string length here.
    # The Hybrid runtime's context_chars field is retrieval metadata
    # and is not required to be byte-for-byte identical to len(context).
    "context_chars": (
        len(CONTEXT_4C6)
        if RISK_APPLICABLE_4C6
        else 0
    ),
}

if RISK_APPLICABLE_4C6:
    JUDGE_INPUT_CONTRACT_PASS_4C6W = all([
        JUDGE_INPUT_4C6W["applicable"] is True,
        bool(str(JUDGE_INPUT_4C6W["question"]).strip()),
        bool(str(JUDGE_INPUT_4C6W["gemma4_only_answer"]).strip()),
        bool(str(JUDGE_INPUT_4C6W["grounded_answer"]).strip()),
        JUDGE_INPUT_4C6W["evidence_count"] > 0,
        len(JUDGE_INPUT_4C6W["evidence"]) == JUDGE_INPUT_4C6W["evidence_count"],
        bool(str(JUDGE_INPUT_4C6W["evidence_context"]).strip()),
        JUDGE_INPUT_4C6W["context_chars"] == len(JUDGE_INPUT_4C6W["evidence_context"]),
    ])
else:
    JUDGE_INPUT_CONTRACT_PASS_4C6W = all([
        JUDGE_INPUT_4C6W["applicable"] is False,
        JUDGE_INPUT_4C6W["grounded_answer"] is None,
        JUDGE_INPUT_4C6W["evidence"] == [],
        JUDGE_INPUT_4C6W["evidence_count"] == 0,
        JUDGE_INPUT_4C6W["evidence_context"] == "",
        JUDGE_INPUT_4C6W["context_chars"] == 0,
    ])

if not JUDGE_INPUT_CONTRACT_PASS_4C6W:
    raise RuntimeError(
        "Cell 4C.6Y judge-input contract validation failed."
    )

# ============================================================
# 4C.6Y.27 — ROUTE-AWARE VALIDATION
# ============================================================

print()

print("VALIDATION")
print("-" * 116)


print(
    f"Pre-LLM Security PASS       : "
    f"{PRE_LLM_SECURITY_PASS_4C6}"
)


print(
    f"Router Output PASS          : "
    f"{ROUTER_OUTPUT_PASS_4C6}"
)


print(
    f"Baseline Answer PASS        : "
    f"{BASELINE_ANSWER_PASS_4C6}"
)


print(
    f"Flexible Answer PASS        : "
    f"{FLEXIBLE_ANSWER_PASS_4C6}"
)

print(
    f"Flexible Answer Min Chars   : "
    f"{FLEXIBLE_MIN_ANSWER_CHARS_4C6}"
)


print(
    f"Baseline Isolation PASS     : "
    f"{BASELINE_ISOLATION_PASS_4C6}"
)


print(
    f"Route Execution PASS        : "
    f"{ROUTE_EXECUTION_PASS_4C6}"
)


# Route-aware adaptive retrieval validation.
#
# Stable general fallback:
#   no retrieval expected.
#
# 6Y deterministic local date/time runtime:
#   no search, no planner, no sufficiency LLM calls expected.
#
# Connected-corpus / live-web paths:
#   preserve the existing model-driven retrieval requirements.
if (
    RESPONSE_MODE_4C6
    ==
    GENERAL_KNOWLEDGE_FALLBACK_MODE
):
    MODEL_DRIVEN_RETRIEVAL_PASS_4C6 = True

elif (
    RESPONSE_MODE_4C6
    ==
    LIVE_EXTERNAL_GROUNDED_MODE
    and
    LOCAL_RUNTIME_USED_4C6Y
):
    MODEL_DRIVEN_RETRIEVAL_PASS_4C6 = all([
        MODEL_DRIVEN_SEARCH_COUNT_4C6 == 0,
        len(MODEL_DRIVEN_SEARCH_QUERIES_4C6) == 0,
        MODEL_DRIVEN_LLM_CALLS_4C6 == 0,
        OPEN_WEBSEARCH_OPERATION_COUNT_4C6W == 0,
    ])

else:
    MODEL_DRIVEN_RETRIEVAL_PASS_4C6 = all([
        MODEL_DRIVEN_SEARCH_COUNT_4C6 >= 1,
        MODEL_DRIVEN_SEARCH_COUNT_4C6 <= MODULE4_MAX_MODEL_SEARCHES_4C6W,
        len(MODEL_DRIVEN_SEARCH_QUERIES_4C6) == MODEL_DRIVEN_SEARCH_COUNT_4C6,
        MODEL_DRIVEN_LLM_CALLS_4C6 >= 2,
    ])

print(
    f"Adaptive Retrieval PASS     : "
    f"{MODEL_DRIVEN_RETRIEVAL_PASS_4C6}"
)


print(
    f"Judge Input Contract PASS   : "
    f"{JUDGE_INPUT_CONTRACT_PASS_4C6W}"
)


print(
    f"Judge Execution PASS        : "
    f"{JUDGE_EXECUTION_PASS_4C6}"
)


COMPLETE_4C6Y_PASS = bool(
    MODULE4_CELL4C6B_PASS
    and
    MODEL_DRIVEN_RETRIEVAL_PASS_4C6
    and
    JUDGE_INPUT_CONTRACT_PASS_4C6W
)

print(
    f"Complete 4C.6Y PASS         : "
    f"{COMPLETE_4C6Y_PASS}"
)


# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# A PASS means:
#
#   "The runtime behaved consistently with Granite's route."
#
# It does NOT independently prove:
#
#   "Granite's semantic classification was objectively correct."
#
# Granite remains the probabilistic semantic decision-maker.
#
# ============================================================


if not MODULE4_CELL4C6B_PASS:

    raise RuntimeError(
        "Module 4C.6Y generalized runtime validation failed."
    )


# ============================================================
# 4C.6Y.28 — STORE GENERALIZED RESULT PACKAGE
# ============================================================

MODULE4_4C6Y_RESULT = {

    "raw_question":
        MODULE4_4C6_RAW_PROMPT,

    "sanitized_question":
        SANITIZED_QUESTION_4C6,

    "security": {

        "injection_detected":
            INJECTION_DETECTED_4C6,

        "matched_patterns":
            MATCHED_PATTERNS_4C6,

        "evidence_manipulation_detected":
            EVIDENCE_MANIPULATION_DETECTED_4C6,

        "guardrail_latency_ms":
            GUARDRAIL_LATENCY_4C6_MS,
    },

    "routing": {

        "model":
            JUDGE_MODEL,

        "response_mode":
            RESPONSE_MODE_4C6,

        "reason":
            ROUTER_REASON_4C6,

        "latency_s":
            ROUTER_LATENCY_4C6_S,
    },

    "gemma4_only": {

        "model":
            LLM_MODEL,

        "answer":
            GENERAL_LLM_ANSWER_4C6,

        "generation_s":
            GENERAL_LLM_LATENCY_4C6_S,

        "rag_executed":
            False,

        "mcp_executed":
            False,
    },

    "flexible_telecom_ai": {

        "model":
            LLM_MODEL,

        "answer":
            FLEXIBLE_ANSWER_4C6,

        "response_mode":
            RESPONSE_MODE_4C6,

        "rag_executed":
            RAG_EXECUTED_4C6,

        "mcp_executed":
            MCP_EXECUTED_4C6,

        "hybrid_executed":
            HYBRID_EXECUTED_4C6,

        "evidence_count":
            len(
                EVIDENCE_4C6
            ),

        "context_chars":
            CONTEXT_CHARS_4C6,

        "telecom_evidence_used":
            TELECOM_EVIDENCE_USED_4C6,

        "citation_present":
            FLEXIBLE_CITATION_PRESENT_4C6,

        "generation_s":
            FLEXIBLE_GENERATION_4C6_S,

        "retrieval_control":
            (
                "llm_planned_python_executed"
                if RESPONSE_MODE_4C6 == TELECOM_GROUNDED_MODE
                else "not_applicable"
            ),

        "model_driven_search_count":
            MODEL_DRIVEN_SEARCH_COUNT_4C6,

        "model_driven_tool_requests":
            MODEL_DRIVEN_TOOL_REQUESTS_4C6,

        "grounded_llm_calls":
            MODEL_DRIVEN_LLM_CALLS_4C6,

        "search_queries":
            list(MODEL_DRIVEN_SEARCH_QUERIES_4C6),

        "search_rounds":
            list(MODEL_DRIVEN_ROUND_TRACES_4C6),

        "tool_loop_wall_s":
            MODEL_DRIVEN_TOOL_LOOP_WALL_4C6_S,
    },

    "judge_input":
        JUDGE_INPUT_4C6W,

    "evaluation": {

        "telecom_risk_applicable":
            RISK_APPLICABLE_4C6,

        "claim_judge_applicable":
            RISK_APPLICABLE_4C6,

        "claim_judge_executed":
            CLAIM_JUDGE_EXECUTED_4C6,

        "judge_model":
            (
                JUDGE_MODEL
                if CLAIM_JUDGE_EXECUTED_4C6
                else None
            ),

        "judge_latency_s":
            GRANITE_JUDGE_4C6_S,

        "gemma4_only_assessment":
            GEMMA_ONLY_ASSESSMENT_4C6,

        "grounded_assessment":
            GEMMA_RAG_MCP_ASSESSMENT_4C6,

        "gemma4_only_risk_pct":
            GEMMA_ONLY_ESTIMATED_HALLUCINATION_RISK_PCT_4C6,

        "grounded_risk_pct":
            GEMMA_RAG_MCP_ESTIMATED_HALLUCINATION_RISK_PCT_4C6,

        "risk_difference_pct":
            ESTIMATED_RISK_DIFFERENCE_PCT_4C6,

        "risk_display":
            RISK_DISPLAY_4C6,

        "note":
            (
                "ONE comparative Granite judge request executed after model-driven grounded retrieval in 4C.6W."
                if CLAIM_JUDGE_EXECUTED_4C6
                else
                "Judge skipped because the selected route has no connected grounded evidence."
            ),
    },

    "timing": {

        "guardrail_ms":
            GUARDRAIL_LATENCY_4C6_MS,

        "router_s":
            ROUTER_LATENCY_4C6_S,

        "gemma4_only_s":
            GENERAL_LLM_LATENCY_4C6_S,

        "baseline_router_wall_s":
            BASELINE_ROUTER_WALL_4C6_S,

        "rag_s":
            RAG_RETRIEVAL_4C6_S,

        "mcp_s":
            MCP_RETRIEVAL_4C6_S,

        "hybrid_s":
            HYBRID_RETRIEVAL_4C6_S,

        "flexible_generation_s":
            FLEXIBLE_GENERATION_4C6_S,

        "granite_comparative_judge_s":
            GRANITE_JUDGE_4C6_S,

        "deployed_path_component_sum_s":
            DEPLOYED_PATH_COMPONENT_SUM_4C6_S,

        "full_comparison_wall_s":
            TOTAL_WORKFLOW_4C6_S,
    },

    "validation": {

        "security_pass":
            PRE_LLM_SECURITY_PASS_4C6,

        "router_pass":
            ROUTER_OUTPUT_PASS_4C6,

        "baseline_pass":
            BASELINE_ANSWER_PASS_4C6,

        "flexible_answer_pass":
            FLEXIBLE_ANSWER_PASS_4C6,

        "baseline_isolation_pass":
            BASELINE_ISOLATION_PASS_4C6,

        "route_execution_pass":
            ROUTE_EXECUTION_PASS_4C6,

        "model_driven_retrieval_pass":
            MODEL_DRIVEN_RETRIEVAL_PASS_4C6,

        "local_runtime_used":
            LOCAL_RUNTIME_USED_4C6Y,

        "judge_input_contract_pass":
            JUDGE_INPUT_CONTRACT_PASS_4C6W,

        "judge_execution_pass":
            JUDGE_EXECUTION_PASS_4C6,

        "complete_pass":
            (
                MODULE4_CELL4C6B_PASS
                and
                MODEL_DRIVEN_RETRIEVAL_PASS_4C6
                and
                JUDGE_INPUT_CONTRACT_PASS_4C6W
            ),
    },
}

MODULE4_4C6Y_PASS = bool(
    MODULE4_4C6Y_RESULT[
        "validation"
    ][
        "complete_pass"
    ]
)

if not MODULE4_4C6Y_PASS:
    raise RuntimeError(
        "Cell 4C.6Y final result-package validation failed."
    )

# Backward-compatible aliases for existing downstream notebook code.
MODULE4_4C6W_RESULT = MODULE4_4C6Y_RESULT
MODULE4_4C6W_PASS = MODULE4_4C6Y_PASS
MODULE4_4C6_RESULT = MODULE4_4C6Y_RESULT


# ============================================================
# 4C.6Y.29 — FINAL STATUS
# ============================================================

print()

print("✓ CELL 4C.6Y PASSED")

print(
    "✓ No expected domain was hard-coded."
)

print(
    "✓ Corpus-aware Granite router was supplied by Cell 4C.6R."
)

print(
    "✓ Grounded retrieval architecture, query and refinements are selected by the routed LLM."
)

print(
    "✓ Python executes only the LLM-selected retrieval architecture, scans evidence, and enforces the 3-round cap."
)

print(
    "✓ Existing RAG, MCP, Hybrid fusion and frozen 4B judge remain unchanged."
)

print(
    "✓ Formal judge-input/audit contract packaged."
)

print(
    "✓ Comparative Granite judge executed immediately when grounded evidence was available."
)

print(
    "✓ Deterministic security executed before all LLM/tool activity."
)

print(
    "✓ Only sanitized intent reached downstream components."
)

print(
    "✓ Gemma 4 Only remained retrieval-independent."
)

print(
    "✓ Granite made the corpus-aware knowledge-scope routing decision."
)

print(
    "✓ Python deterministically enforced the selected execution boundary."
)


if (
    RESPONSE_MODE_4C6
    ==
    TELECOM_GROUNDED_MODE
):

    print(
        "✓ Grounded LLM selected RAG_ONLY / MCP_ONLY / HYBRID and formulated the retrieval query."
    )

    print(
        "✓ Retrieved evidence passed deterministic security screening."
    )

    print(
        "✓ Grounded Gemma generation used Telecom evidence."
    )

    print(
        "✓ ONE Granite comparative request judged both answer groups."
    )

    print(
        "✓ Connected-corpus evidence-relative risk was calculated and displayed immediately."
    )

else:

    print(
        "✓ Non-telecom route skipped RAG + MCP completely."
    )

    print(
        "✓ General-knowledge fallback retained the domain caveat."
    )

    print(
        "✓ Comparative judge was skipped because no connected grounded evidence existed."
    )

    print(
        "✓ Connected-corpus evidence-relative risk remained N/A."
    )


print("=" * 116)


MODULE 4C.6Y — THREE-WAY KNOWLEDGE ROUTER + OPEN-WEBSEARCH CLI
Raw User Question           : What is 5G?

GATE 1 — DETERMINISTIC SECURITY
         Injection Detected → False
         Matched Patterns   → []
         Evidence Attack    → False
         Sanitized Intent   → What is 5G?
         Guardrail Latency  → 0.090 ms

✓ GATE 1 PASSED — trusted sanitized intent created.

GATE 2 — BASELINE GENERATION + KNOWLEDGE-SCOPE ROUTING
         Baseline Model     → google/gemma-4-26b-a4b-it
         Baseline Latency   → 10.564 s

         Knowledge Router   → ibm-granite/granite-4.2-8b
         Response Mode      → TELECOM_GROUNDED
         Router Latency     → 2.219 s
         Parallel Wall      → 10.568 s

GATE 3 — CONDITIONAL KNOWLEDGE ACCESS
         Knowledge Scope     → CONNECTED TECHNICAL CORPUS
         Retrieval Control   → LLM-SELECTED ARCHITECTURE / PYTHON-EXECUTED
         Available Modes     → RAG_ONLY / MCP_ONLY / HYBRID
         Search Policy       → 1 required / 2 if needed / 

##### Cell 4D — Observation

The validated 6Y runtime completed successfully for `What is 5G?`: Granite selected `TELECOM_GROUNDED`, Gemma selected `RAG_ONLY`, one retrieval round returned **5 evidence items**, and requirement-bounded sufficiency passed with no refinement required.

The comparison evaluated **21 baseline claims** versus **15 grounded claims**. The baseline produced an estimated evidence-relative hallucination risk of **26.7% (MODERATE)**, while the grounded response achieved **0.0% (LOW)** with **15/15 claims supported**. All final validation checks passed, including security, routing, adaptive retrieval, judge-input contract and judge execution.

**Cell 4D / 6Y v1.3 is therefore frozen as the canonical runtime.**


# SECTION 5 — Final Validation + Deployment Readiness

#### Fresh Sequential Validation — Canonical 5G Run

The cleaned notebook was executed through the final orchestration runtime with `What is 5G?` as the reproducibility question.

| Validation element | Observed result |
|---|---|
| Knowledge route | `TELECOM_GROUNDED` |
| Retrieval architecture | `RAG_ONLY` |
| Retrieval rounds | 1 / 3 |
| Evidence items | 5 |
| Context | 12,500 characters |
| Requirement-bounded sufficiency | `True` |
| Baseline claims | 21 |
| Grounded claims | 15 |
| Baseline evidence-relative risk | 26.7% — MODERATE |
| Grounded evidence-relative risk | 0.0% — LOW |
| Grounded supported claims | 15 / 15 |
| Complete 4C.6Y validation | `True` |

The fresh run demonstrates that the cleaned notebook can execute the canonical connected-corpus path without relying on the earlier experimental notebook state.

#### Previously Validated Knowledge-Scope Branches

| Test | Observed path | Key result |
|---|---|---|
| `What is 5G?` | Connected technical corpus → `RAG_ONLY` | Fresh sequential run passed; 5 evidence items; 15/15 grounded claims supported |
| `What is malaria?` | Stable general knowledge | RAG/MCP/web skipped; scope caveat retained; validation passed |
| `Who is Clifford Imaguezegie?` | Live external | Open-WebSearch activated; 1/3 external operations; 5 cited evidence items; validation passed |
| `What is todays date?` | Live external scope → local runtime | `Europe/London` runtime path; 0 web calls; planner/sufficiency loop skipped |
| Current UK Prime Minister query | Live external | Date-aware Open-WebSearch path returned a grounded current answer |

These branches collectively validate the three knowledge scopes plus the deterministic local runtime utility path.


#### Final Architectural Decisions

1. **Telecom AI Knowledge Orchestration Runtime** is the final public notebook topic.
2. **Cell 4D / 6Y v1.3 is frozen as the canonical runtime.**
3. **Legacy `MODULE 4C.6Y` / `4C6*` identifiers remain internally** to avoid unnecessary regression after successful validation.
4. **6Z is not adopted.** Correctly phrased person/current-information queries already route directly to live external grounding.
5. **Open-WebSearch uses the CLI JSON interface**, not MCP transport, because the notebook environment exposed a Jupyter/STDIO `fileno` incompatibility while the CLI path proved reliable.
6. **Date/time uses the local runtime clock**, not web search.
7. **The frozen Granite comparative judge remains connected-corpus-only.** Live external and stable-general answers report comparative risk as not applicable.
8. **The baseline comparison is an evaluation feature, not the preferred production critical path.** A user-facing deployment should return the operational answer first and make comparative evaluation optional or deferred.
9. **Hosted-judge resilience is retained:** one retry is allowed for structurally invalid Granite output, and unexpected extra claim IDs are discarded only when all required fixed claim IDs remain present exactly once.


#### Git / Portfolio Readiness

This validated notebook is suitable as the canonical Module 4 portfolio artifact because it:

- uses the public title **Telecom AI Knowledge Orchestration Runtime**;
- preserves the successful fresh execution outputs as reproducibility evidence;
- keeps concise description and observation Markdown around every executable cell;
- retains only one canonical final 6Y runtime;
- documents the relationship between public naming and legacy internal identifiers;
- keeps secrets out of source control;
- documents external data, model and CLI dependencies;
- distinguishes operational response logic from evaluation-only baseline/judge work;
- records connected-corpus, live-external, local-runtime and general-fallback strategies;
- preserves validated security, adaptive retrieval and comparative-evaluation behavior.

For GitHub, this notebook should now be paired with the repository README, architecture, strategy, features, requirements, deployment, security and validation documents.
